# High-scoring source-built solution: Verifiable Quantum Circuit Compilation

This notebook builds all three submission components from readable source, validates them with the competition's official full-shot verifiers, and creates `/kaggle/working/submission.zip`.

The recorded reference run for this source passed every verifier and scored **6.434** by the published geometric-mean formula:

| Task | Qubits | Toffoli | Reference / cost |
|---|---:|---:|---:|
| `invsqrt` | 84 | 1,538 | 13.33× |
| `dqi` | 1,789 | 445,410 mean | 13.95× |
| `dfthcsos` whole block encoding | 681 | 72,404 | 1.43× |

The DQI cost varies slightly with the verifier seed. Kaggle derives leaderboard tests from the submitted circuit plus a secret salt, so the recorded score is evidence rather than a guaranteed leaderboard value.

## Required Kaggle inputs

1. The **Google - Verifiable Quantum Circuit Compilation** competition data (`kickmix/` and `verifiers/`).
2. Version 2 of **VQCC FeMoCo public factorization** by Dmitrii Gluzdov, containing `femoco_public_words.npz` (Kaggle dataset version ID `21143008`). The preflight cell checks its SHA-256 exactly.

Internet access is not needed. Use **Restart & Run All**. A normal CPU run takes about 18 minutes; the final cell prints the archive path, member list, size, and SHA-256.

The construction combines signed-digit inverse-square-root arithmetic, a compact reversible Berlekamp–Massey decoder, and compressed banked FeMoCo angle lookups. Full derivations, provenance, and Apache-2.0 attributions appear below.


In [ ]:
import hashlib
import json
import math
import os
import pathlib
import shutil
import sys

os.environ.update(
    OMP_NUM_THREADS="1",
    OPENBLAS_NUM_THREADS="1",
    MKL_NUM_THREADS="1",
)

import numpy as np
import pandas as pd
from IPython.display import display

INPUT = pathlib.Path(os.environ.get("QCC_INPUT", "/kaggle/input"))
WORK = pathlib.Path(os.environ.get("QCC_WORK", "/kaggle/working"))
SUB = pathlib.Path(os.environ.get("QCC_SUB", "/tmp/qcc_submission"))
SLUG = "google-verifiable-quantum-circuit-compilation"
CHEM_SHA256 = "9afd0ca5e90dd1740fe713e4ac1c26736184ea9bc4d4b57a6eeefafd608ee193"
REFERENCE = {
    "invsqrt": 2076774.233715282,
    "dqi": 24439474177519.598,
    "dfthcsos": 3355599588.3155365,
}


def sha256_file(path: pathlib.Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def competition_candidates(root: pathlib.Path) -> list[pathlib.Path]:
    preferred = [root, root / "competitions" / SLUG, root / SLUG]
    discovered = []
    if root.exists():
        discovered = [path.parent for path in root.rglob("kickmix") if path.is_dir()]
    unique = []
    for path in preferred + sorted(discovered):
        path = path.resolve()
        if path not in unique and (path / "kickmix").is_dir() and (path / "verifiers").is_dir():
            unique.append(path)
    return unique


if not INPUT.exists():
    raise FileNotFoundError(f"Kaggle input directory does not exist: {INPUT}")

competition_dirs = competition_candidates(INPUT)
if len(competition_dirs) != 1:
    raise RuntimeError(
        "Expected exactly one competition input containing kickmix/ and verifiers/. "
        f"Found {len(competition_dirs)}: {[str(path) for path in competition_dirs]}"
    )
COMP = competition_dirs[0]

chem_candidates = sorted(INPUT.rglob("femoco_public_words.npz"))
chem_hashes = {path: sha256_file(path) for path in chem_candidates}
valid_chem = [path for path, digest in chem_hashes.items() if digest == CHEM_SHA256]
if len(valid_chem) != 1:
    observed = {str(path): digest for path, digest in chem_hashes.items()}
    raise RuntimeError(
        "Attach version 2 of the VQCC FeMoCo public factorization dataset. "
        f"Expected one femoco_public_words.npz with SHA-256 {CHEM_SHA256}; observed {observed or 'none'}."
    )
CHEM_DATA = valid_chem[0]

with np.load(CHEM_DATA) as chemistry:
    required_keys = {"words", "W", "H2_I"}
    missing_keys = required_keys.difference(chemistry.files)
    if missing_keys:
        raise ValueError(f"FeMoCo input is missing arrays: {sorted(missing_keys)}")
    expected_shapes = {"words": (3976, 75), "W": (65, 60, 30), "H2_I": (65, 30)}
    actual_shapes = {key: chemistry[key].shape for key in expected_shapes}
    if actual_shapes != expected_shapes:
        raise ValueError(f"Unexpected FeMoCo shapes: {actual_shapes}; expected {expected_shapes}")

WORK.mkdir(parents=True, exist_ok=True)
SUB.mkdir(parents=True, exist_ok=True)
dfthcsos_out = SUB / "dfthcsos"
dfthcsos_out.mkdir(parents=True, exist_ok=True)

# Make Restart & Run All deterministic without deleting unrelated user files.
for generated in [SUB / "invsqrt.kmb", SUB / "dqi.kmb", dfthcsos_out / "submission.npz"]:
    generated.unlink(missing_ok=True)
for generated in dfthcsos_out.glob("angle_k_*.kmb"):
    generated.unlink()
unexpected = list(dfthcsos_out.iterdir())
if unexpected:
    raise RuntimeError(f"Refusing to mix unknown files into the submission: {unexpected}")

source_dirs = [
    "src",
    "src/gen_dqi_v6",
    "src/gen_dqi_v6/ref",
    "src/gen_invsqrt_r2",
    "src/gen_invsqrt_r3",
    "src/gen_invsqrt_r3/vendor",
    "src/ports",
    "src/ports/evidence_inv",
    "src/ports/public_sources/anazemcev_quantum-circuits-source-exact-compilation/embedded/src",
    "src/ports/public_sources/lightning_v2/embedded",
]
for relative in source_dirs:
    (WORK / relative).mkdir(parents=True, exist_ok=True)

available_cpus = os.cpu_count() or 4
requested_workers = int(os.environ.get("QCC_WORKERS", available_cpus))
WORKERS = max(1, min(requested_workers, available_cpus))
free_gib = shutil.disk_usage(WORK).free / 2**30
if free_gib < 0.5:
    raise OSError(f"At least 0.5 GiB of free working space is required; found {free_gib:.2f} GiB")

os.chdir(WORK)
print(f"Competition data : {COMP}")
print(f"FeMoCo data      : {CHEM_DATA}")
print(f"FeMoCo SHA-256   : {CHEM_SHA256}")
print(f"Scratch output   : {SUB}")
print(f"Workers / CPUs   : {WORKERS} / {available_cpus}")
print(f"Free work space  : {free_gib:.2f} GiB")


In [ ]:
%%writefile src/qcc_pipeline.py
# SPDX-License-Identifier: Apache-2.0
"""Build the three benchmark circuits from source and verify them with the official verifiers."""
from __future__ import annotations

import json
import math
import os
import pathlib
import sys
import time
import zipfile

import numpy as np

SRC = pathlib.Path(__file__).resolve().parent
COMP: pathlib.Path | None = None


def setup(competition_dir) -> None:
    global COMP
    COMP = pathlib.Path(competition_dir)
    for p in (COMP / 'kickmix', COMP / 'verifiers', SRC / 'gen_dqi_v6', SRC / 'gen_invsqrt_r2'):
        if str(p) not in sys.path:
            sys.path.append(str(p))


def build_invsqrt(out: pathlib.Path) -> pathlib.Path:
    import sumit_reference as H
    import round2
    data = json.loads((SRC / 'gen_invsqrt_r2' / 'tables.json').read_text())
    tabs = {(e, i): tuple(data['rows'][128 * e + i]) for e in range(2) for i in range(128)}
    circuit, _ = round2.build(H.Cfg(*data['params']), tabs, deferred=True, anf=True, parity_input=True)
    circuit.to_file(str(out / 'invsqrt.kmb'), format='kmb')
    return out / 'invsqrt.kmb'


def build_dqi(out: pathlib.Path, work: pathlib.Path) -> pathlib.Path:
    import gu_formula_derivation as F
    tables, _ = F.run_b12()
    assert F.certify_kt(tables['KT12'], F.FIELD)
    constants = json.loads((SRC / 'gen_dqi_v6' / 'ref' / 'gu_constants.json').read_text())
    constants.update(tables)
    work.mkdir(parents=True, exist_ok=True)
    (work / 'dqi_constants.json').write_text(json.dumps(constants))
    os.environ['DQI_CONSTANTS'] = str(work / 'dqi_constants.json')
    os.environ['DQI_BORROW'] = 'affine'
    import gu_port as G
    import run_gu_port as RG
    import round2_gadgets as R2
    G.borrow_template = RG.borrow_template
    assert R2.certify_norm_identity(G)
    R2.install(G, top_first=True, pebble_last=True, norm_pebble=True, compact_from=70)
    G.emit_sweep = R2.stream_sweep
    b, _ = G.build()
    b.k.free(b.measurement)
    b.k.finish_circuit().to_file(str(out / 'dqi.kmb'), format='kmb')
    return out / 'dqi.kmb'


def log_depth_tree_edges(n: int) -> np.ndarray:
    """Breadth-first edges of the organizers' log-depth Givens tree."""
    edges, level, depth = [], [0], 0
    while len(edges) < n - 1:
        nxt = []
        for label in level:
            nxt.append(label)
            right = label + (1 << depth)
            if right < n:
                edges.append((label, right))
                nxt.append(right)
        level, depth = nxt, depth + 1
    return np.asarray(edges, dtype=np.int64)


def banked_lookup(delta: np.ndarray, brot: int):
    """XOR oracle |a>|z> -> |a>|z ^ delta[a]> as two banks of a unary-iteration lookup."""
    import kickmix as km
    A, J = delta.shape
    s = (A - 1).bit_length()
    W = J * brot
    table = [sum(int(v) << (j * brot) for j, v in enumerate(row)) for row in delta]
    b = km.CircuitBuilder()
    sel = b.create_quantum_register(s, 'selection')
    tgt = [q for j in range(J) for q in b.create_quantum_register(brot, f'word_{j}')]
    H = (A + 1) // 2
    even, odd = table[::2], table[1::2] + ([table[-1]] if A % 2 else [])
    diff = [x ^ y for x, y in zip(even, odd)]
    bank = b.alloc_qubits(W)
    values = [x | (y << W) for x, y in zip(even, diff)]
    outputs = tgt + list(bank)
    bits = list(sel[1:])[::-1]
    anc = b.alloc_qubits(s - 2)

    def node(ctrl, level, lo, hi):
        if lo >= H:
            return
        if hi - lo == 1 or len(set(values[lo:min(H, hi)])) == 1:
            ts = [q for j, q in enumerate(outputs) if (values[lo] >> j) & 1]
            if ts:
                b.cx(ctrl, km.array(ts))
            return
        mid = (lo + hi) // 2
        if H <= mid:
            node(ctrl, level + 1, lo, mid)
            return
        bit, a = bits[level], anc[level - 1]
        b.x(bit); b.init_and(ctrl, bit, a); b.x(bit)
        node(a, level + 1, lo, mid)
        b.cx(ctrl, a)
        node(a, level + 1, mid, hi)
        b.del_and(ctrl, bit, a)

    half = 1 << (s - 2)
    b.x(bits[0]); node(bits[0], 1, 0, half); b.x(bits[0])
    node(bits[0], 1, half, 2 * half)
    b.free(anc)
    for j in range(W):
        b.ccx(sel[0], bank[j], tgt[j])
    b.del_lookup(table=diff + [0] * ((1 << (s - 1)) - H), address=sel[1:], target=bank)
    b.free(bank)
    return b.finish_circuit()


def build_dfthcsos(out: pathlib.Path, data_file: pathlib.Path, brot: int = 10, J: int = 5) -> pathlib.Path:
    """ANGLE circuits for the packed angle words, plus the factorization file the verifier reads."""
    from dfthcsos.scoreinfra import dfthc_math as M
    data = np.load(data_file)
    words, W, H2 = data['words'].astype(np.int64), data['W'], data['H2_I']
    R, B, _ = W.shape
    edges = log_depth_tree_edges(76)
    U = M.reconstruct_u_vectors(words[76:] / 2 ** (brot - 1), edges).reshape(R, B, 76)
    shifts = np.load(COMP / 'verifiers' / 'dfthcsos' / 'FeMoCo76o113e.fixed_shifts.npz')
    folder = out / 'dfthcsos'
    folder.mkdir(parents=True, exist_ok=True)
    np.savez(folder / 'submission.npz', U=U, W=W, H2_I=H2, H1_I=shifts['H1_I'], Hsym_lower=shifts['Hsym_lower'],
             tree_edges=edges)
    K = math.ceil(75 / J)
    padded = np.zeros((len(words), K * J), dtype=np.int64)
    padded[:, :75] = words
    batches = padded.reshape(len(words), K, J).transpose(1, 0, 2)
    deltas = batches.copy()
    deltas[1:] ^= batches[:-1]
    for k, d in enumerate(deltas):
        banked_lookup(d, brot).to_file(str(folder / f'angle_k_{k}.kmb'), format='kmb')
    return folder


def verify_all(sub: pathlib.Path, salt: str, workers: int | None = None) -> dict:
    """The three official verifiers at their full shot counts."""
    import kickmix as km
    from invsqrt import verify_submission_invsqrt as VI
    from dqi import verify_submission_dqi as VD
    from dfthcsos import verify as VC
    load = lambda p: km.Circuit.from_file(str(p), format='kmb')  # noqa: E731
    res = {'invsqrt': VI.verify(load(sub / 'invsqrt.kmb'), VI.ProblemSpecInvSqrt.for_n_p(10), salt=salt,
                                max_workers=workers),
           'dqi': VD.verify(load(sub / 'dqi.kmb'), VD.ProblemSpecDQI.benchmark_instance(), salt=salt,
                            max_workers=min(workers or 4, 8))}
    folder = sub / 'dfthcsos'
    files = sorted(folder.glob('angle_k_*.kmb'), key=lambda p: int(p.stem.rsplit('_', 1)[1]))
    with np.load(folder / 'submission.npz') as z:
        npz = {k: np.array(z[k]) for k in z.files}
    tree = npz.pop('tree_edges')
    res['dfthcsos'] = VC.verify([load(p) for p in files], npz, tree, salt=salt)
    return res


def write_zip(sub: pathlib.Path, path: pathlib.Path) -> pathlib.Path:
    members = [sub / 'invsqrt.kmb', sub / 'dqi.kmb'] + sorted((sub / 'dfthcsos').iterdir())
    with zipfile.ZipFile(path, 'w', zipfile.ZIP_DEFLATED, compresslevel=6) as z:
        for m in members:
            z.write(m, m.relative_to(sub).as_posix())
    return path

In [ ]:
%%writefile src/gen_invsqrt_r2/sumit_reference.py
"""Builds an optimized kickmix circuit for the invsqrt benchmark (Benchmark I).

|x>|0> -> exp(2 pi i / sqrt(x)) |x>|0>,   x in [1, 3133452], 22-bit register 'x'.

Design (all arithmetic exactly mirrored by the numpy model below):
  1. Normalize x by single-bit binary stages (16, 8, 4, 2, 1) into M = x_norm / 2^21 in [1, 2);
     flags f0..f4 hold the shift count s' in binary.  e = f0 (parity), s = s' >> 1.
     1/sqrt(x) = 2^(s-10) * y,  y = 2^((e-1)/2) / sqrt(M)  in (0.5, 1].
  2. One 256-entry QROM indexed by (7 mantissa bits, e) XORs quadratic coefficients directly into
     the working registers (no separate coefficient copies).
  3. Centered signed-digit Horner: delta_c = sum_i (2 d_i - 1) 2^(i-22) over the 14 low bits.
     R = K1 - C2*delta_c,  y = K0 - R*delta_c.  Each row is an UNcontrolled Gidney addition
     wrapped in CNOT flips (Y - A = ~(~Y + A)), with a rounding carry-in.
  4. The y register is split: rows only touch a 21-bit low register L; the constant high part
     sits in a separate register and the overflow bit is merged in before phasing.
  5. Binary de-normalization (shift by s), one Z_POW per bit, then exact mirror uncompute.
     Lookup uncompute is measurement based (del_lookup); normalization flags are cleared by HMR
     with classically conditioned phase fixups.
Usage:  python build_invsqrt.py OUT.kmb   (needs kickmix importable and numpy)
"""
import sys, math
from fractions import Fraction
import numpy as np
import kickmix as km

W = 22; MB = 21; DB = 14; NSEG = 128
F2, F1, FYR, GY, DROP2 = 10, 19, 26, 2, (5,)     # precision configuration (verified)

# ----------------------------------------------------------------- numpy model / tables
def fitc(lo, hi, mid, scale, deg=2, iters=80):
    f = lambda M: scale / np.sqrt(M)
    d = np.linspace(lo - mid, hi - mid, 3001)
    A = np.vander(d, deg + 1, increasing=True)
    w = np.sqrt(mid + d) / scale
    wt = np.ones_like(d)
    for _ in range(iters):                       # Lawson iterations -> weighted minimax
        sw = w * np.sqrt(wt)
        coef = np.linalg.lstsq(A * sw[:, None], f(mid + d) * sw, rcond=None)[0]
        err = np.abs((A @ coef - f(mid + d)) * w)
        wt = wt * err; wt /= wt.sum()
    return coef

class Cfg:
    def __init__(self, f2, f1, fyr, gy, drop2=()):
        self.f2, self.f1, self.fyr, self.gy = f2, f1, fyr, gy
        self.fy = fyr + gy
        self.w2 = f2 - 1
        self.sh2 = [f2 + 22 - i - f1 for i in range(DB)]
        self.sh1 = [f1 + 22 - i - self.fy for i in range(DB)]
        self.drop2 = set(drop2)
    def rrow_list(self):
        return [i for i in reversed(range(DB)) if i not in self.drop2 and self.sh2[i] < self.w2]

def rd(v, s):
    if s < 0: return v << (-s)
    r = v >> s
    if s >= 1: r = r + ((v >> (s - 1)) & 1)
    return r

def pipeline(cfg, K2, K1, d):
    sig = [2 * ((d >> i) & 1) - 1 for i in range(DB)]
    R = np.full_like(d, K1)
    for i in cfg.rrow_list():
        R = R - sig[i] * rd(K2, cfg.sh2[i])
    Q = np.zeros_like(d)
    for i in reversed(range(DB)):
        Q = Q + sig[i] * rd(R, cfg.sh1[i])
    return R, Q

def build_tables(cfg, search=2):
    tabs = {}; worst = 0
    dmid = (2**DB - 1) / 2**(MB + 1)
    d = np.arange(2**DB, dtype=np.int64)
    g = cfg.gy
    for e in (0, 1):
        scale = 2**((e - 1) / 2)
        for a in range(NSEG):
            lo = 1 + a * 2.0**(DB - MB); mid = lo + dmid
            c = fitc(lo, lo + 2.0**(DB - MB), mid, scale)
            C1, C2 = -c[1], c[2]
            M = lo + d * 2.0**-MB
            fM = scale / np.sqrt(M); tgt = fM * 2.0**cfg.fy; w = 1 / fM
            K2c = int(round(C2 * 2**cfg.f2)); K1c = int(round(C1 * 2**cfg.f1))
            best = None
            for K2 in range(K2c - 1, K2c + 2):
                for K1 in range(K1c - search, K1c + search + 1):
                    R, Q = pipeline(cfg, K2, K1, d)
                    def mx(K0):
                        Y = ((K0 - Q) >> g) << g
                        return float(np.max(np.abs(Y - tgt) * w))
                    t = tgt + Q
                    a_, b_ = math.floor(t.min()), math.ceil(t.max())
                    while b_ - a_ > 2:
                        m1 = a_ + (b_ - a_) // 3; m2 = b_ - (b_ - a_) // 3
                        if mx(m1) < mx(m2): b_ = m2
                        else: a_ = m1
                    for K0 in range(a_, b_ + 1):
                        v = mx(K0) / 2.0**cfg.fy
                        if best is None or v < best[0]: best = (v, K2, K1, K0)
            tabs[(e, a)] = best[1:]
            worst = max(worst, best[0])
    return tabs, worst

def layout(cfg, tabs, LB=21, S=20):
    d = np.arange(2**DB, dtype=np.int64)
    assert max(t[0] for t in tabs.values()) < (1 << cfg.w2)
    Rmax = 0; per = {}; Ymax = 0
    for key, (K2, K1, K0) in tabs.items():
        R, Q = pipeline(cfg, K2, K1, d)
        assert R.min() >= 0
        Rmax = max(Rmax, int(R.max()), K1)
        per[key] = (int(Q.min()), int(Q.max()))
        Ymax = max(Ymax, K0 - per[key][0])
    w1 = Rmax.bit_length()
    lo_tab, hi_tab = {}, {}
    for key, (K2, K1, K0) in tabs.items():
        qmin, qmax = per[key]
        base = K0 % (1 << S); j = 0
        while base + j * (1 << S) - qmax < 0: j += 1
        k0lo = base + j * (1 << S)
        assert k0lo - qmin < (1 << LB), 'layout infeasible'
        lo_tab[key] = k0lo; hi_tab[key] = (K0 - k0lo) >> S
        assert hi_tab[key] >= 0
    whi = max(max(hi_tab.values()).bit_length(), 1)
    return dict(w1=w1, LB=LB, S=S, whi=whi, lo=lo_tab, hi=hi_tab)

# ----------------------------------------------------------------- circuit pieces
def and_all_into(b, controls, target):
    k = len(controls)
    if k == 1: b.cx(controls[0], target); return
    if k == 2: b.ccx(controls[0], controls[1], target); return
    anc = b.alloc_qubits(k - 2)
    b.init_and(controls[0], controls[1], anc[0])
    for j in range(1, k - 2): b.init_and(anc[j - 1], controls[j + 1], anc[j])
    b.ccx(anc[k - 3], controls[k - 1], target)
    for j in reversed(range(1, k - 2)): b.del_and(anc[j - 1], controls[j + 1], anc[j])
    b.del_and(controls[0], controls[1], anc[0])
    b.free(anc)

def phase_all(b, controls):
    k = len(controls)
    if k == 1: b.z(controls[0]); return
    if k == 2: b.cz(controls[0], controls[1]); return
    anc = b.alloc_qubits(k - 2)
    b.init_and(controls[0], controls[1], anc[0])
    for j in range(1, k - 2): b.init_and(anc[j - 1], controls[j + 1], anc[j])
    b.cz(anc[k - 3], controls[k - 1])
    for j in reversed(range(1, k - 2)): b.del_and(anc[j - 1], controls[j + 1], anc[j])
    b.del_and(controls[0], controls[1], anc[0])
    b.free(anc)

NSTAGES = [(4, 16), (3, 8), (2, 4), (1, 2), (0, 1)]

def normalize(b, x, f):
    for (k, d) in NSTAGES:
        top = x[W - d:W]
        if d == 1:
            b.cx(x[W - 1], f[0]); b.x(f[0])
        else:
            b.x(top); and_all_into(b, list(top), f[k]); b.x(top)
        for i in reversed(range(0, W - d)): b.cswap(f[k], x[i], x[i + d])

def unnormalize(b, x, f):
    for (k, d) in reversed(NSTAGES):
        for i in range(0, W - d): b.cswap(f[k], x[i], x[i + d])
        top = x[W - d:W]
        if d == 1:
            b.x(f[0]); b.cx(x[W - 1], f[0])
        else:
            m = b.alloc_bits(1)[0]
            b.hmr(f[k], m)
            b.push_condition(m)                     # fixup runs on half the shots on average
            b.x(top); phase_all(b, list(top)); b.x(top)
            b.pop_condition()
            b.free(m)

def gadd(b, a, t, cin=None, sub=False):
    """t += a + cin (or t -= a + cin), Gidney adder: len(t)-1 Toffolis, uncompute by HMR."""
    a = list(a); t = list(t); o = len(a); n = len(t)
    assert o <= n
    if sub: b.x(km.array(t))
    c = b.alloc_qubits(n - 1) if n > 1 else km.array([])
    def C(i): return cin if i == 0 else c[i - 1]
    for i in range(0, n - 1):
        if i == 0 and cin is None:
            if o: b.init_and(a[0], t[0], C(1))
        elif i < o:
            b.cx(C(i), a[i]); b.cx(C(i), t[i]); b.init_and(a[i], t[i], C(i + 1)); b.cx(C(i), C(i + 1))
        else:
            b.init_and(C(i), t[i], C(i + 1))
    if n - 1 < o: b.cx(a[n - 1], t[n - 1])
    if n > 1 or cin is not None: b.cx(C(n - 1), t[n - 1])
    for i in reversed(range(0, n - 1)):
        if i == 0 and cin is None:
            if o: b.del_and(a[0], t[0], C(1)); b.cx(a[0], t[0])
        elif i < o:
            b.cx(C(i), C(i + 1)); b.del_and(a[i], t[i], C(i + 1)); b.cx(C(i), a[i]); b.cx(a[i], t[i])
        else:
            b.del_and(C(i), t[i], C(i + 1)); b.cx(C(i), t[i])
    if n > 1: b.free(c)
    if sub: b.x(km.array(t))

def shift_stages(b, f, reg, L, inverse):
    stages = [(1, 1), (2, 2), (3, 4), (4, 8)]
    occ = []; o = L
    for (k, d) in stages: occ.append(o); o = min(o + d, len(reg))
    seq = list(enumerate(stages))
    if not inverse:
        for idx, (k, d) in seq:
            for i in reversed(range(0, occ[idx])):
                if i + d < len(reg): b.cswap(f[k], reg[i], reg[i + d])
    else:
        for idx, (k, d) in reversed(seq):
            for i in range(0, occ[idx]):
                if i + d < len(reg): b.cswap(f[k], reg[i], reg[i + d])

def build_circuit(cfg, tabs, lay):
    w2, w1, LB, S, whi = cfg.w2, lay['w1'], lay['LB'], lay['S'], lay['whi']
    gy, fy = cfg.gy, cfg.fy
    b = km.CircuitBuilder()
    x = b.create_quantum_register(W, name='x')
    f = b.alloc_qubits(5)
    c2 = b.alloc_qubits(w2); R = b.alloc_qubits(w1); Lr = b.alloc_qubits(LB); Yh = b.alloc_qubits(whi)
    tgt = km.array(list(Lr) + list(R) + list(c2) + list(Yh))
    table = [0] * 256
    for (e, a), (K2, K1, K0) in tabs.items():
        table[a + 128 * e] = lay['lo'][(e, a)] | (K1 << LB) | (K2 << (LB + w1)) | (lay['hi'][(e, a)] << (LB + w1 + w2))
    addr = km.array(list(x[14:21]) + [f[0]])
    delta = x[0:DB]
    def rrows(inverse):
        seq = cfg.rrow_list()
        if inverse: seq = list(reversed(seq))
        for i in seq:
            sh = cfg.sh2[i]
            if sh >= 0: off = c2[sh:w2]; t = R[0:w1]; cin = c2[sh - 1] if sh >= 1 else None
            else: off = c2[0:w2]; t = R[-sh:w1]; cin = None
            b.cx(delta[i], t); gadd(b, off, t, cin, sub=inverse); b.cx(delta[i], t)
    def yrows(inverse):
        seq = list(reversed(range(DB)))
        if inverse: seq.reverse()
        for i in seq:
            sh = cfg.sh1[i]; assert 0 <= sh < w1
            t = Lr[0:LB]; cin = R[sh - 1] if sh >= 1 else None
            b.cx(delta[i], t); gadd(b, R[sh:w1], t, cin, sub=inverse); b.cx(delta[i], t)
    normalize(b, x, f)
    b.init_lookup(table=table, address=addr, target=tgt)
    rrows(False); yrows(False)
    ext = b.alloc_qubits(10)
    if LB > S: gadd(b, Lr[S:LB], Yh)
    Cx = km.array(list(Lr[gy:S]) + list(Yh) + list(ext))
    shift_stages(b, f, Cx, S - gy + whi, False)
    for j in range(len(Cx)):
        e = gy + j - fy - 10                       # bit weight 2^e turns
        if e < 0: b.z_pow(Cx[j], Fraction(2)**(e + 1))
    shift_stages(b, f, Cx, S - gy + whi, True)
    if LB > S: gadd(b, Lr[S:LB], Yh, sub=True)
    b.free(ext)
    yrows(True); rrows(True)
    b.del_lookup(table=table, address=addr, target=tgt)
    b.free(tgt)
    unnormalize(b, x, f)
    b.free(f)
    return b.finish_circuit()

def main(out):
    cfg = Cfg(F2, F1, FYR, GY, DROP2)
    tabs, worst = build_tables(cfg)
    print(f'model worst relative error / 2^-24 = {worst * 2**24:.3f}')
    lay = layout(cfg, tabs)
    c = build_circuit(cfg, tabs, lay)
    s = c.operation_statistics()
    print('qubits', c.num_qubits, 'toffoli(static)', s.num_toffoli_like,
          'rotations', s.num_z_pow(min_clifford_hierarchy_level=4))
    c.to_file(out)
    print('wrote', out)

if __name__ == '__main__':
    main(sys.argv[1] if len(sys.argv) > 1 else 'invsqrt.kmb')

In [ ]:
%%writefile src/gen_invsqrt_r2/signed_digit_lab.py
# Attribution: Sumit Chongder, Apache-2.0 as published.
# https://www.kaggle.com/code/sumitchongder/notebook2f196e646b
# Read-only source downloaded from latest version 2. Kirill's v9 cites v1.
# This isolated lab regenerates the circuit from readable source; no binaries or
# remote notebook outputs are inputs. Our added exact integer certificate and
# reporting helpers and memory-schedule adaptations follow the original generator.
"""Builds an optimized kickmix circuit for the invsqrt benchmark (Benchmark I).

|x>|0> -> exp(2 pi i / sqrt(x)) |x>|0>,   x in [1, 3133452], 22-bit register 'x'.

Design (all arithmetic exactly mirrored by the numpy model below):
  1. Normalize x by single-bit binary stages (16, 8, 4, 2, 1) into M = x_norm / 2^21 in [1, 2);
     flags f0..f4 hold the shift count s' in binary.  e = f0 (parity), s = s' >> 1.
     1/sqrt(x) = 2^(s-10) * y,  y = 2^((e-1)/2) / sqrt(M)  in (0.5, 1].
  2. One 256-entry QROM indexed by (7 mantissa bits, e) XORs quadratic coefficients directly into
     the working registers (no separate coefficient copies).
  3. Centered signed-digit Horner: delta_c = sum_i (2 d_i - 1) 2^(i-22) over the 14 low bits.
     R = K1 - C2*delta_c,  y = K0 - R*delta_c.  Each row is an UNcontrolled Gidney addition
     wrapped in CNOT flips (Y - A = ~(~Y + A)), with a rounding carry-in.
  4. The y register is split: rows only touch a 21-bit low register L; the constant high part
     sits in a separate register and the overflow bit is merged in before phasing.
  5. Binary de-normalization (shift by s), one Z_POW per bit, then exact mirror uncompute.
     Lookup uncompute is measurement based (del_lookup); normalization flags are cleared by HMR
     with classically conditioned phase fixups.
Usage: python signed_digit_lab.py OUT.kmb [--baseline] (requires kickmix and NumPy)
"""
import sys, math
from fractions import Fraction
import numpy as np
import kickmix as km

W = 22; MB = 21; DB = 14; NSEG = 128
F2, F1, FYR, GY, DROP2 = 10, 19, 26, 2, (5,)     # precision configuration (verified)

# ----------------------------------------------------------------- numpy model / tables
def fitc(lo, hi, mid, scale, deg=2, iters=80):
    f = lambda M: scale / np.sqrt(M)
    d = np.linspace(lo - mid, hi - mid, 3001)
    A = np.vander(d, deg + 1, increasing=True)
    w = np.sqrt(mid + d) / scale
    wt = np.ones_like(d)
    for _ in range(iters):                       # Lawson iterations -> weighted minimax
        sw = w * np.sqrt(wt)
        coef = np.linalg.lstsq(A * sw[:, None], f(mid + d) * sw, rcond=None)[0]
        err = np.abs((A @ coef - f(mid + d)) * w)
        wt = wt * err; wt /= wt.sum()
    return coef

class Cfg:
    def __init__(self, f2, f1, fyr, gy, drop2=()):
        self.f2, self.f1, self.fyr, self.gy = f2, f1, fyr, gy
        self.fy = fyr + gy
        self.w2 = f2 - 1
        self.sh2 = [f2 + 22 - i - f1 for i in range(DB)]
        self.sh1 = [f1 + 22 - i - self.fy for i in range(DB)]
        self.drop2 = set(drop2)
    def rrow_list(self):
        return [i for i in reversed(range(DB)) if i not in self.drop2 and self.sh2[i] < self.w2]

def rd(v, s):
    if s < 0: return v << (-s)
    r = v >> s
    if s >= 1: r = r + ((v >> (s - 1)) & 1)
    return r

def pipeline(cfg, K2, K1, d):
    sig = [2 * ((d >> i) & 1) - 1 for i in range(DB)]
    R = np.full_like(d, K1)
    for i in cfg.rrow_list():
        R = R - sig[i] * rd(K2, cfg.sh2[i])
    Q = np.zeros_like(d)
    for i in reversed(range(DB)):
        Q = Q + sig[i] * rd(R, cfg.sh1[i])
    return R, Q

def build_tables(cfg, search=2):
    tabs = {}; worst = 0
    dmid = (2**DB - 1) / 2**(MB + 1)
    d = np.arange(2**DB, dtype=np.int64)
    g = cfg.gy
    for e in (0, 1):
        scale = 2**((e - 1) / 2)
        for a in range(NSEG):
            lo = 1 + a * 2.0**(DB - MB); mid = lo + dmid
            c = fitc(lo, lo + 2.0**(DB - MB), mid, scale)
            C1, C2 = -c[1], c[2]
            M = lo + d * 2.0**-MB
            fM = scale / np.sqrt(M); tgt = fM * 2.0**cfg.fy; w = 1 / fM
            K2c = int(round(C2 * 2**cfg.f2)); K1c = int(round(C1 * 2**cfg.f1))
            best = None
            for K2 in range(K2c - 1, K2c + 2):
                for K1 in range(K1c - search, K1c + search + 1):
                    R, Q = pipeline(cfg, K2, K1, d)
                    def mx(K0):
                        Y = ((K0 - Q) >> g) << g
                        return float(np.max(np.abs(Y - tgt) * w))
                    t = tgt + Q
                    a_, b_ = math.floor(t.min()), math.ceil(t.max())
                    while b_ - a_ > 2:
                        m1 = a_ + (b_ - a_) // 3; m2 = b_ - (b_ - a_) // 3
                        if mx(m1) < mx(m2): b_ = m2
                        else: a_ = m1
                    for K0 in range(a_, b_ + 1):
                        v = mx(K0) / 2.0**cfg.fy
                        if best is None or v < best[0]: best = (v, K2, K1, K0)
            tabs[(e, a)] = best[1:]
            worst = max(worst, best[0])
    return tabs, worst

def layout(cfg, tabs, LB=21, S=20):
    d = np.arange(2**DB, dtype=np.int64)
    assert max(t[0] for t in tabs.values()) < (1 << cfg.w2)
    Rmax = 0; per = {}; Ymax = 0
    for key, (K2, K1, K0) in tabs.items():
        R, Q = pipeline(cfg, K2, K1, d)
        assert R.min() >= 0
        Rmax = max(Rmax, int(R.max()), K1)
        per[key] = (int(Q.min()), int(Q.max()))
        Ymax = max(Ymax, K0 - per[key][0])
    w1 = Rmax.bit_length()
    lo_tab, hi_tab = {}, {}
    for key, (K2, K1, K0) in tabs.items():
        qmin, qmax = per[key]
        base = K0 % (1 << S); j = 0
        while base + j * (1 << S) - qmax < 0: j += 1
        k0lo = base + j * (1 << S)
        assert k0lo - qmin < (1 << LB), 'layout infeasible'
        lo_tab[key] = k0lo; hi_tab[key] = (K0 - k0lo) >> S
        assert hi_tab[key] >= 0
    whi = max(max(hi_tab.values()).bit_length(), 1)
    return dict(w1=w1, LB=LB, S=S, whi=whi, lo=lo_tab, hi=hi_tab)

# ----------------------------------------------------------------- circuit pieces
def and_all_into(b, controls, target):
    k = len(controls)
    if k == 1: b.cx(controls[0], target); return
    if k == 2: b.ccx(controls[0], controls[1], target); return
    anc = b.alloc_qubits(k - 2)
    b.init_and(controls[0], controls[1], anc[0])
    for j in range(1, k - 2): b.init_and(anc[j - 1], controls[j + 1], anc[j])
    b.ccx(anc[k - 3], controls[k - 1], target)
    for j in reversed(range(1, k - 2)): b.del_and(anc[j - 1], controls[j + 1], anc[j])
    b.del_and(controls[0], controls[1], anc[0])
    b.free(anc)

def phase_all(b, controls):
    k = len(controls)
    if k == 1: b.z(controls[0]); return
    if k == 2: b.cz(controls[0], controls[1]); return
    anc = b.alloc_qubits(k - 2)
    b.init_and(controls[0], controls[1], anc[0])
    for j in range(1, k - 2): b.init_and(anc[j - 1], controls[j + 1], anc[j])
    b.cz(anc[k - 3], controls[k - 1])
    for j in reversed(range(1, k - 2)): b.del_and(anc[j - 1], controls[j + 1], anc[j])
    b.del_and(controls[0], controls[1], anc[0])
    b.free(anc)

NSTAGES = [(4, 16), (3, 8), (2, 4), (1, 2), (0, 1)]

def normalize(b, x, f):
    for (k, d) in NSTAGES:
        top = x[W - d:W]
        if d == 1:
            b.cx(x[W - 1], f[0]); b.x(f[0])
        else:
            b.x(top); and_all_into(b, list(top), f[k]); b.x(top)
        for i in reversed(range(0, W - d)): b.cswap(f[k], x[i], x[i + d])

def unnormalize(b, x, f):
    for (k, d) in reversed(NSTAGES):
        for i in range(0, W - d): b.cswap(f[k], x[i], x[i + d])
        top = x[W - d:W]
        if d == 1:
            b.x(f[0]); b.cx(x[W - 1], f[0])
        else:
            m = b.alloc_bits(1)[0]
            b.hmr(f[k], m)
            b.push_condition(m)                     # fixup runs on half the shots on average
            b.x(top); phase_all(b, list(top)); b.x(top)
            b.pop_condition()
            b.free(m)

def gadd(b, a, t, cin=None, sub=False):
    """t += a + cin (or t -= a + cin), Gidney adder: len(t)-1 Toffolis, uncompute by HMR."""
    a = list(a); t = list(t); o = len(a); n = len(t)
    assert o <= n
    if sub: b.x(km.array(t))
    c = b.alloc_qubits(n - 1) if n > 1 else km.array([])
    def C(i): return cin if i == 0 else c[i - 1]
    for i in range(0, n - 1):
        if i == 0 and cin is None:
            if o: b.init_and(a[0], t[0], C(1))
        elif i < o:
            b.cx(C(i), a[i]); b.cx(C(i), t[i]); b.init_and(a[i], t[i], C(i + 1)); b.cx(C(i), C(i + 1))
        else:
            b.init_and(C(i), t[i], C(i + 1))
    if n - 1 < o: b.cx(a[n - 1], t[n - 1])
    if n > 1 or cin is not None: b.cx(C(n - 1), t[n - 1])
    for i in reversed(range(0, n - 1)):
        if i == 0 and cin is None:
            if o: b.del_and(a[0], t[0], C(1)); b.cx(a[0], t[0])
        elif i < o:
            b.cx(C(i), C(i + 1)); b.del_and(a[i], t[i], C(i + 1)); b.cx(C(i), a[i]); b.cx(a[i], t[i])
        else:
            b.del_and(C(i), t[i], C(i + 1)); b.cx(C(i), t[i])
    if n > 1: b.free(c)
    if sub: b.x(km.array(t))

def shift_stages(b, f, reg, L, inverse):
    stages = [(1, 1), (2, 2), (3, 4), (4, 8)]
    occ = []; o = L
    for (k, d) in stages: occ.append(o); o = min(o + d, len(reg))
    seq = list(enumerate(stages))
    if not inverse:
        for idx, (k, d) in seq:
            for i in reversed(range(0, occ[idx])):
                if i + d < len(reg): b.cswap(f[k], reg[i], reg[i + d])
    else:
        for idx, (k, d) in reversed(seq):
            for i in range(0, occ[idx]):
                if i + d < len(reg): b.cswap(f[k], reg[i], reg[i + d])

def build_circuit(cfg, tabs, lay, adder=None):
    # The default preserves the public generator exactly. An injected adder
    # supports isolated, explicitly tested memory experiments.
    if adder is None: adder=gadd
    w2, w1, LB, S, whi = cfg.w2, lay['w1'], lay['LB'], lay['S'], lay['whi']
    gy, fy = cfg.gy, cfg.fy
    b = km.CircuitBuilder()
    x = b.create_quantum_register(W, name='x')
    f = b.alloc_qubits(5)
    c2 = b.alloc_qubits(w2); R = b.alloc_qubits(w1); Lr = b.alloc_qubits(LB); Yh = b.alloc_qubits(whi)
    tgt = km.array(list(Lr) + list(R) + list(c2) + list(Yh))
    table = [0] * 256
    for (e, a), (K2, K1, K0) in tabs.items():
        table[a + 128 * e] = lay['lo'][(e, a)] | (K1 << LB) | (K2 << (LB + w1)) | (lay['hi'][(e, a)] << (LB + w1 + w2))
    addr = km.array(list(x[14:21]) + [f[0]])
    delta = x[0:DB]
    def rrows(inverse):
        seq = cfg.rrow_list()
        if inverse: seq = list(reversed(seq))
        for i in seq:
            sh = cfg.sh2[i]
            if sh >= 0: off = c2[sh:w2]; t = R[0:w1]; cin = c2[sh - 1] if sh >= 1 else None
            else: off = c2[0:w2]; t = R[-sh:w1]; cin = None
            b.cx(delta[i], t); adder(b, off, t, cin, sub=inverse); b.cx(delta[i], t)
    def yrows(inverse):
        seq = list(reversed(range(DB)))
        if inverse: seq.reverse()
        for i in seq:
            sh = cfg.sh1[i]; assert 0 <= sh < w1
            t = Lr[0:LB]; cin = R[sh - 1] if sh >= 1 else None
            b.cx(delta[i], t); adder(b, R[sh:w1], t, cin, sub=inverse); b.cx(delta[i], t)
    normalize(b, x, f)
    b.init_lookup(table=table, address=addr, target=tgt)
    rrows(False); yrows(False)
    ext = b.alloc_qubits(10)
    if LB > S: adder(b, Lr[S:LB], Yh)
    Cx = km.array(list(Lr[gy:S]) + list(Yh) + list(ext))
    shift_stages(b, f, Cx, S - gy + whi, False)
    for j in range(len(Cx)):
        e = gy + j - fy - 10                       # bit weight 2^e turns
        if e < 0: b.z_pow(Cx[j], Fraction(2)**(e + 1))
    shift_stages(b, f, Cx, S - gy + whi, True)
    if LB > S: adder(b, Lr[S:LB], Yh, sub=True)
    b.free(ext)
    yrows(True); rrows(True)
    b.del_lookup(table=table, address=addr, target=tgt)
    b.free(tgt)
    unnormalize(b, x, f)
    b.free(f)
    return b.finish_circuit()

def main(out):
    cfg = Cfg(F2, F1, FYR, GY, DROP2)
    tabs, worst = build_tables(cfg)
    print(f'model worst relative error / 2^-24 = {worst * 2**24:.3f}')
    lay = layout(cfg, tabs)
    c = build_circuit(cfg, tabs, lay)
    s = c.operation_statistics()
    print('qubits', c.num_qubits, 'toffoli(static)', s.num_toffoli_like,
          'rotations', s.num_z_pow(min_clifford_hierarchy_level=4))
    c.to_file(out, format='kmb')
    print('wrote', out)


def selected_cfg():
    return Cfg(F2,F1,FYR,GY,DROP2)


def config_record(cfg):
    return dict(f2=cfg.f2,f1=cfg.f1,fyr=cfg.fyr,gy=cfg.gy,drop2=sorted(cfg.drop2))


def load_or_build_tables(path=None):
    """Regenerate open-source coefficients, optionally retaining a local cache."""
    import json
    from pathlib import Path
    cfg=selected_cfg()
    path=Path(path) if path else None
    if path and path.exists():
        saved=json.loads(path.read_text())
        assert saved['config']==config_record(cfg)
        rows=saved['rows']
        assert len(rows)==256 and all(len(row)==3 for row in rows)
        tabs={(i//128,i%128):tuple(map(int,row)) for i,row in enumerate(rows)}
        return cfg,tabs,saved['normalized_worst_relative_error']
    tabs,worst=build_tables(cfg)
    if path:
        path.write_text(json.dumps({'source':'Sumit Chongder open generator, latest v2',
            'config':config_record(cfg),'normalized_worst_relative_error':worst,
            'rows':[tabs[(e,a)] for e in range(2) for a in range(128)]},indent=2))
    return cfg,tabs,worst


def integer_model(xs,cfg,tabs):
    """Exact emitted arithmetic in units 2^-(cfg.fy+10) turns, before mod 1."""
    xs=np.asarray(xs,dtype=np.int64)
    if not ((xs>=1)&(xs<=3133452)).all():
        raise ValueError('The supported competition domain is x=1..3133452')
    # All inputs fit exactly in float64; frexp gives the exact binary exponent.
    shifts=21-(np.frexp(xs)[1].astype(np.int64)-1)
    norm=xs << shifts
    assert ((norm >> 21)==1).all(), 'Normalized top-bit borrowing requires x[21]=1'
    parity=shifts&1
    index=((norm >> 14)&127)+128*parity
    table=np.asarray([tabs[(e,a)] for e in range(2) for a in range(128)],dtype=np.int64)
    K2,K1,K0=(table[index,j] for j in range(3))
    d=norm&((1 << DB)-1)
    signs=[2*((d >> i)&1)-1 for i in range(DB)]
    R=K1.copy()
    for i in cfg.rrow_list():
        R-=signs[i]*rd(K2,cfg.sh2[i])
    Q=np.zeros_like(xs)
    for i in reversed(range(DB)):
        Q+=signs[i]*rd(R,cfg.sh1[i])
    Y=((K0-Q) >> cfg.gy) << cfg.gy
    if (R<0).any() or (Y<0).any():
        raise OverflowError('An unsigned intermediate is negative')
    return Y << (shifts >> 1)


def exhaustive_certificate(cfg,tabs,chunk_size=131072):
    """Every promised basis input, exact integer target and tolerance.

    This is an arithmetic certificate. Ancillas, restored inputs and phase
    feed-forward require fresh circuit simulation and the official verifier.
    """
    output_fraction=cfg.fy+10
    worst_ratio,worst_x,min_margin=0.,0,1 << 64
    for start in range(1,3133453,chunk_size):
        xs=np.arange(start,min(start+chunk_size,3133453),dtype=np.int64)
        values=integer_model(xs,cfg,tabs).astype(np.uint64)
        phase=values << np.uint64(64-output_fraction)
        roots=[math.isqrt((1 << 130)//int(x)) for x in xs]
        target=np.array([((r+1) >> 1)&((1 << 64)-1) for r in roots],dtype=np.uint64)
        tolerance=np.array([min(r >> 25,1 << 63) for r in roots],dtype=np.uint64)
        error=np.abs((phase-target).view(np.int64)).view(np.uint64)
        ratio=error/tolerance
        i=int(np.argmax(ratio))
        if float(ratio[i])>worst_ratio:
            worst_ratio,worst_x=float(ratio[i]),int(xs[i])
        if (error>tolerance).any():
            i=int(np.flatnonzero(error>tolerance)[0])
            return {'pass':False,'first_bad_x':int(xs[i]),'ratio':float(ratio[i])}
        min_margin=min(min_margin,int((tolerance-error).min()))
    return {'pass':True,'tested_inputs':3133452,'worst_ratio':worst_ratio,
            'worst_x':worst_x,'min_margin_units':min_margin,
            'phase_fraction_bits':output_fraction}


def estimated_resources(circuit,certificate):
    """Model-based cost; full unchanged official verification remains required."""
    assert certificate['pass'] and certificate['min_margin_units']>0
    stats=circuit.operation_statistics()
    rotations=int(stats.num_z_pow(min_clifford_hierarchy_level=4))
    epsilon=math.pi*(certificate['min_margin_units']/2**64)/rotations
    per_rotation=0.56*math.log2(1/epsilon)+5.3
    static_toffoli=int(stats.num_toffoli_like)
    # Four X-basis flag measurements independently activate phase_all on half
    # the shots. Its 16,8,4,2-control paths have 14+6+2+0 Toffolis in total.
    normalization_discount=sum(max(0,d-2) for _,d in NSTAGES if d>1)/2
    expected_toffoli=static_toffoli-normalization_discount
    t_like=int(stats.num_t_like)
    extra=(t_like+rotations*per_rotation)/4
    return {'qubits':circuit.num_qubits,'static_toffoli':static_toffoli,
            'expected_mean_toffoli':expected_toffoli,'t_like':t_like,
            'rotations':rotations,'rotation_epsilon':epsilon,
            'estimated_expected_cost':circuit.num_qubits*(expected_toffoli+extra),
            'static_cost_upper_estimate':circuit.num_qubits*(static_toffoli+extra),
            'full_official_verification_required':True}




# ---- Our isolated phase scheduling, folded carry and borrowed workspace ----
def folded_gadd(b,a,t,cin=None,sub=False,borrowed=()):
    a,t=list(a),list(t)
    o,n=len(a),len(t)
    assert o<=n and n>=1
    if sub:
        for q in t:b.x(q)
    if n==1:
        if o:b.cx(a[0],t[0])
        if cin is not None:b.cx(cin,t[0])
        if sub:
            for q in t:b.x(q)
        return
    borrowed=list(borrowed)[:max(0,n-2)]
    count=max(0,n-2-len(borrowed))
    allocated=b.alloc_qubits(count) if count else []
    c=borrowed+list(allocated)
    def C(i):return cin if i==0 else c[i-1]
    for i in range(n-2):
        if i==0 and cin is None:
            if o:b.init_and(a[0],t[0],C(1))
        elif i<o:
            b.cx(C(i),a[i]);b.cx(C(i),t[i])
            b.init_and(a[i],t[i],C(i+1));b.cx(C(i),C(i+1))
        else:
            b.init_and(C(i),t[i],C(i+1))
    i=n-2
    carry=C(i)
    if carry is None:
        if i<o:
            b.ccx(a[i],t[i],t[-1]);b.cx(a[i],t[i])
    elif i<o:
        b.cx(carry,a[i]);b.cx(carry,t[i])
        b.ccx(a[i],t[i],t[-1]);b.cx(carry,t[-1])
        b.cx(carry,a[i]);b.cx(a[i],t[i])
    else:
        b.ccx(carry,t[i],t[-1]);b.cx(carry,t[i])
    if n-1<o:b.cx(a[n-1],t[n-1])
    for i in reversed(range(n-2)):
        if i==0 and cin is None:
            if o:b.del_and(a[0],t[0],C(1));b.cx(a[0],t[0])
        elif i<o:
            b.cx(C(i),C(i+1));b.del_and(a[i],t[i],C(i+1))
            b.cx(C(i),a[i]);b.cx(a[i],t[i])
        else:
            b.del_and(C(i),t[i],C(i+1));b.cx(C(i),t[i])
    if count:b.free(allocated)
    if sub:
        for q in t:b.x(q)


def truth_table_check(max_width=6,borrowed_count=0):
    class Bits:
        def __init__(self):self.v=[]
        def alloc_qubits(self,n):
            ids=list(range(len(self.v),len(self.v)+n));self.v.extend([0]*n);return ids
        def x(self,q):self.v[q]^=1
        def cx(self,a,t):self.v[t]^=self.v[a]
        def ccx(self,a,b,t):self.v[t]^=self.v[a]&self.v[b]
        def init_and(self,a,b,t):
            assert self.v[t]==0
            self.v[t]=self.v[a]&self.v[b]
        def del_and(self,a,b,t):
            assert self.v[t]==(self.v[a]&self.v[b])
            self.v[t]=0
        def free(self,reg):assert not any(self.v[q] for q in reg)
        def put(self,reg,value):
            for i,q in enumerate(reg):self.v[q]=(value >> i)&1
        def read(self,reg):return sum(self.v[q] << i for i,q in enumerate(reg))
    count=0
    for n in range(1,max_width+1):
        for m in range(n+1):
            for a in range(1 << m):
                for value in range(1 << n):
                    for cin_value in (None,0,1):
                        for sub in (False,True):
                            b=Bits();areg=b.alloc_qubits(m);target=b.alloc_qubits(n)
                            cin=b.alloc_qubits(1)[0] if cin_value is not None else None
                            if cin is not None:b.v[cin]=cin_value
                            b.put(areg,a);b.put(target,value)
                            borrowed=b.alloc_qubits(borrowed_count)
                            folded_gadd(b,areg,target,cin,sub,borrowed=borrowed)
                            delta=a+(cin_value or 0)
                            expected=(value-delta if sub else value+delta) % (1 << n)
                            assert b.read(target)==expected,(n,m,a,value,cin_value,sub)
                            assert b.read(areg)==a
                            assert cin is None or b.v[cin]==cin_value
                            assert not any(b.v[q] for q in borrowed)
                            count+=1
    return count


def build_split_circuit(cfg,tabs,lay,adder=None,borrow_top=False):
    if adder is None:adder=gadd
    w2,w1,LB,S,whi=cfg.w2,lay['w1'],lay['LB'],lay['S'],lay['whi']
    b=km.CircuitBuilder()
    x=b.create_quantum_register(W,name='x')
    flags=b.alloc_qubits(5)
    normalize(b,x,flags)
    # Every promised positive input normalizes to x[21]=1. The arithmetic
    # address uses x[14:21] and delta x[:14], so this wire is idle in the core.
    if borrow_top:
        b.x(x[-1])
        raw_adder=adder
        def adder(builder,a,t,cin=None,sub=False):
            return raw_adder(builder,a,t,cin,sub,borrowed=(x[-1],))
    c2=b.alloc_qubits(w2)
    R=b.alloc_qubits(w1)
    Lr=b.alloc_qubits(LB)
    Yh=b.alloc_qubits(whi)
    remaining=km.array(list(Lr)+list(R)+list(c2))
    target=km.array(list(remaining)+list(Yh))
    table=[0]*256
    rest_table=[0]*256
    high_table=[0]*256
    for (e,a),(K2,K1,K0) in tabs.items():
        idx=a+128*e
        rest_table[idx]=lay['lo'][(e,a)] | (K1 << LB) | (K2 << (LB+w1))
        high_table[idx]=lay['hi'][(e,a)]
        table[idx]=rest_table[idx] | (high_table[idx] << (LB+w1+w2))
    address=km.array(list(x[14:21])+[flags[0]])
    delta=x[:DB]
    def rrows(inverse):
        seq=cfg.rrow_list()
        if inverse:seq=list(reversed(seq))
        for i in seq:
            sh=cfg.sh2[i]
            if sh>=0:off=c2[sh:w2];t=R[:w1];cin=c2[sh-1] if sh>=1 else None
            else:off=c2[:w2];t=R[-sh:w1];cin=None
            b.cx(delta[i],t);adder(b,off,t,cin,sub=inverse);b.cx(delta[i],t)
    def yrows(inverse):
        seq=list(reversed(range(DB)))
        if inverse:seq.reverse()
        for i in seq:
            sh=cfg.sh1[i]
            assert 0<=sh<w1
            cin=R[sh-1] if sh>=1 else None
            b.cx(delta[i],Lr);adder(b,R[sh:w1],Lr,cin,sub=inverse);b.cx(delta[i],Lr)
    def phase(reg,base_bit):
        ext=b.alloc_qubits(10)
        word=km.array(list(reg)+list(ext))
        shift_stages(b,flags,word,len(reg),False)
        for j in range(len(word)):
            exponent=base_bit+j-cfg.fy-10
            if exponent<0:b.z_pow(word[j],Fraction(2)**(exponent+1))
        shift_stages(b,flags,word,len(reg),True)
        b.free(ext)
    b.init_lookup(table=table,address=address,target=target)
    phase(Yh,S)
    b.del_lookup(table=high_table,address=address,target=Yh)
    b.free(Yh)
    rrows(False)
    yrows(False)
    phase(Lr[cfg.gy:LB],cfg.gy)
    yrows(True)
    rrows(True)
    b.del_lookup(table=rest_table,address=address,target=remaining)
    b.free(remaining)
    if borrow_top:b.x(x[-1])
    unnormalize(b,x,flags)
    b.free(flags)
    return b.finish_circuit()


def build_selected(cfg,tabs,lay=None):
    """Selected unchanged-Sumit model with split high phase and 93-qubit schedule.

    The normalized input's top bit is known to be one throughout the core;
    it is temporarily zeroed for a carry wire and restored before unnormalizing.
    Every call regenerates a circuit from source and ordinary numeric tables.
    """
    if lay is None:lay=layout(cfg,tabs)
    return build_split_circuit(cfg,tabs,lay,adder=folded_gadd,borrow_top=True)


def selected_descriptor():
    return {
        'variant':'sumit_split_folded_borrow',
        'source_author':'Sumit Chongder',
        'source_url':'https://www.kaggle.com/code/sumitchongder/notebook2f196e646b?scriptVersionId=361357149',
        'source_download':'Latest v2 readable generator; regenerated baseline canonical bytes match cited v1 report',
        'license_as_published':'Apache-2.0',
        'coefficient_changes':'None; original Sumit Cfg(10,19,26,2,(5,)) and generated coefficient table',
        'own_changes':[
            'Allocate coefficient registers after normalization',
            'Phase constant high word and measurement-unlookup it before Horner arithmetic',
            'Phase the low accumulator including its overflow bit separately; remove high/low merge adders',
            'Fold final ripple carry directly into highest target bit, including the rounding carry-in',
            'Borrow normalized x[21] as a clean carry wire; restore its value before unnormalization'],
        'baseline_canonical_sha256':'f19a8207e282859f663abe7f655b06eb3224d32ac99fbf899e1422b3795ca0be',
        'selected_canonical_sha256':'85611d22c524a5e982f2beafd11e32e6d39c86e174d3438734ff5a088679090a',
        'measured_counts':{'qubits':93,'static_toffoli':1582,'rotations':44},
        'expected_mean_toffoli':1571.0,
        'estimated_cost':175203.2709288877,
        'official_full_verification':'Required separately; no arithmetic certificate substitutes for it'}


def comparison_report(cfg,tabs,certificate,lay=None):
    """Measured circuit counts and model-based estimates, without executing verification."""
    if lay is None:lay=layout(cfg,tabs)
    baseline=build_circuit(cfg,tabs,lay)
    selected=build_selected(cfg,tabs,lay)
    return {'baseline':estimated_resources(baseline,certificate),
            'selected':estimated_resources(selected,certificate),
            'selected_descriptor':selected_descriptor()}


def cli():
    import argparse
    import json
    from pathlib import Path
    parser = argparse.ArgumentParser(description='Generate the credited signed-digit circuit and memory-schedule adaptation.')
    parser.add_argument('output', nargs='?', default='invsqrt.kmb')
    parser.add_argument('--baseline', action='store_true')
    args = parser.parse_args()
    cfg, tabs, _ = load_or_build_tables()
    certificate = exhaustive_certificate(cfg, tabs)
    if not certificate['pass']:
        raise ArithmeticError(certificate)
    lay = layout(cfg, tabs)
    circuit = build_circuit(cfg, tabs, lay) if args.baseline else build_selected(cfg, tabs, lay)
    circuit.to_file(args.output, format='kmb')
    report = estimated_resources(circuit, certificate)
    report['arithmetic_certificate'] = certificate
    report['variant'] = 'sumit_baseline' if args.baseline else selected_descriptor()['variant']
    Path(args.output + '.resources.json').write_text(json.dumps(report, indent=2), encoding='utf-8')
    print(json.dumps(report, indent=2))


if __name__ == '__main__':
    cli()

In [ ]:
%%writefile src/gen_invsqrt_r2/deferred_lookup.py
"""Deferred lookup demolition and shared one-hot/unary phase repair.

The scheduling idea follows Gu, Kemeny, Yelin and Lukin's Gaussian circuit
(andigu/uncompute, commit 64b5a13aa249f1360846657392e6c66817b26d0f).
This is a new implementation against the kickmix API, Apache-2.0.
"""
import kickmix as km

def demolish(b, register):
 bits=b.alloc_bits(len(register))
 for q,m in zip(register,bits):b.hmr(q,m)
 return list(bits)

def repair_anf(b,address,table,measurements,low_bits=4):
 """Möbius-transform the truth table and share monomials of each half."""
 address=list(address);bits=list(measurements);coef=list(table)
 for i in range(len(address)):
  for mask in range(len(coef)):
   if mask&(1<<i):coef[mask]^=coef[mask^(1<<i)]
 def monomials(register):
  nodes={1<<i:q for i,q in enumerate(register)};undo=[]
  for mask in range(1,1<<len(register)):
   if mask in nodes:continue
   bit=mask&-mask;left=nodes[mask^bit];right=nodes[bit]
   target=b.alloc_qubits(1)[0];b.init_and(left,right,target)
   nodes[mask]=target;undo.append((left,right,target))
  return nodes,undo
 lo,lundo=monomials(address[:low_bits]);hi,hundo=monomials(address[low_bits:])
 one=b.alloc_qubits(1)[0];b.x(one)
 for mask,word in enumerate(coef):
  l=mask&((1<<low_bits)-1);h=mask>>low_bits
  while word:
   pos=(word&-word).bit_length()-1;b.push_condition(bits[pos])
   if l and h:b.cz(lo[l],hi[h])
   else:b.z(lo[l] if l else hi[h] if h else one)
   b.pop_condition();word&=word-1
 b.x(one);b.free(one)
 for left,right,target in reversed(hundo+lundo):
  b.del_and(left,right,target);b.free(target)
 for bit in bits:b.free(bit)

def repair(b,address,table,measurements,low_bits=4,anf=False):
 """Repair (-1)^(table[address] dot measurements), without output wires."""
 if anf:return repair_anf(b,address,table,measurements,low_bits)
 address=list(address);bits=list(measurements)
 low=address[:low_bits];high=address[low_bits:]
 rails=b.alloc_qubits(2);b.cx(low[-1],rails[0]);b.x(rails[0]);b.cx(low[-1],rails[1])
 current=list(rails);levels=[]
 for bit in reversed(low[:-1]):
  following=[];level=[]
  for parent in current:
   pair=b.alloc_qubits(2);lo,hi=pair
   b.init_and(parent,bit,hi);b.cx(parent,lo);b.cx(hi,lo)
   following.extend([lo,hi]);level.append((parent,bit,pair))
  levels.append(level);current=following
 def leaf(prefix,control):
  for j,rail in enumerate(current):
   word=table[(prefix<<low_bits)+j]
   while word:
    pos=(word&-word).bit_length()-1
    b.push_condition(bits[pos]);b.cz(control,rail);b.pop_condition()
    word&=word-1
 def walk(remaining,prefix,parent):
  if not remaining:leaf(prefix,parent);return
  bit=remaining[-1];rest=remaining[:-1]
  if parent is None:
   b.x(bit);walk(rest,prefix*2,bit);b.x(bit);walk(rest,prefix*2+1,bit)
  else:
   tmp=b.alloc_qubits(1)[0]
   b.x(bit);b.init_and(parent,bit,tmp);b.x(bit)
   walk(rest,prefix*2,tmp)
   b.cx(parent,tmp);walk(rest,prefix*2+1,tmp);b.cx(parent,tmp)
   b.x(bit);b.del_and(parent,bit,tmp);b.x(bit);b.free(tmp)
 walk(high,0,None)
 for level in reversed(levels):
  for parent,bit,pair in reversed(level):
   lo,hi=pair;b.cx(hi,lo);b.cx(parent,lo);b.del_and(parent,bit,hi);b.free(pair)
 b.cx(low[-1],rails[1]);b.x(rails[0]);b.cx(low[-1],rails[0]);b.free(rails)
 for bit in bits:b.free(bit)

In [ ]:
%%writefile src/gen_invsqrt_r2/hybrid_add.py
"""Cuccaro MAJ/UMA prefix followed by a measured-carry Gidney suffix.

Trades clean carry workspace for additional Toffolis. New implementation,
Apache-2.0; circuit identities credited to Cuccaro et al. and Gidney.
"""
import signed_digit_lab as A

def add(b,a,t,cin=None,sub=False,borrowed=(),prefix=0):
 a,t=list(a),list(t);k=min(prefix,len(a),len(t)-1)
 if not k:return A.folded_gadd(b,a,t,cin,sub,borrowed=borrowed)
 if sub:
  for q in t:b.x(q)
 initial=None;available=list(borrowed)
 if cin is None:
  if available:cin=available.pop(0)
  else:initial=b.alloc_qubits(1)[0];cin=initial
 carry=cin
 for i in range(k):
  b.cx(a[i],t[i]);b.cx(a[i],carry);b.ccx(t[i],carry,a[i]);carry=a[i]
 A.folded_gadd(b,a[k:],t[k:],carry,borrowed=available)
 for i in reversed(range(k)):
  carry=cin if i==0 else a[i-1]
  b.ccx(t[i],carry,a[i]);b.cx(a[i],carry);b.cx(carry,t[i])
 if initial is not None:b.free(initial)
 if sub:
  for q in t:b.x(q)

In [ ]:
%%writefile src/gen_invsqrt_r2/round2.py
"""Signed-digit oracle schedules, derived from Sumit Chongder and Rudra R4/R5.

Changes made here: configurable precisions and split widths; borrow the
normalized leading one for a clean carry; prune the final phase-shift stage
using the promised s<=10 bound. No verifier logic is changed.
"""
from fractions import Fraction
import numpy as np
import kickmix as km
import sumit_reference as H
import signed_digit_lab as A
import deferred_lookup as D
import hybrid_add as B

def prepare(cfg,tabs):
 d=np.arange(1<<H.DB,dtype=np.int64);Rmax=0;spread=0;minimum={};maxima={}
 for key,(k2,k1,k0) in tabs.items():
  r,_=H.pipeline(cfg,k2,k1,d);minimum[key]=int(r.min());maxima[key]=int(r.max());Rmax=max(Rmax,int(r.max()),k1)
  spread=max(spread,int(r.max()-r.min()))
 # Pick the narrowest unsigned residual which also contains the initial K1.
 for block in range(1,Rmax.bit_length()):
  bases={k:v>>block for k,v in minimum.items()}
  if all(0<=tabs[k][1]-(v<<block)<(1<<(block+1)) and maxima[k]-(v<<block)<(1<<(block+1)) for k,v in bases.items()):break
 return {"block":block,"rw":block+1,"cw":Rmax.bit_length()-block,
         "coarse":bases,"residual":{k:tabs[k][1]-(v<<block) for k,v in bases.items()}}

def shift(b,flags,word,L,inverse,prune=True):
 stages=[(1,1),(2,2),(3,4),(4,8)];occ=[];n=L
 for k,d in stages:
  occ.append(min(n,L+2) if prune and k==4 else n);n=min(n+d,len(word))
 seq=list(enumerate(stages));seq=seq[::-1] if inverse else seq
 for j,(k,d) in seq:
  indices=range(occ[j]) if inverse else reversed(range(occ[j]))
  for i in indices:
   if i+d<len(word):
    zero_boundary = i+d >= L+sum(amount for _,amount in stages[:j])
    if zero_boundary:
     if inverse:b.cx(word[i+d],word[i]);b.del_and(flags[k],word[i],word[i+d])
     else:b.init_and(flags[k],word[i],word[i+d]);b.cx(word[i+d],word[i])
    else:b.cswap(flags[k],word[i],word[i+d])

def build(cfg,tabs,*,borrow=True,prune=True,merge=False,parity_input=False,carry_block=None,phase_cut=0,deferred=False,anf=False):
 lay=H.layout(cfg,tabs,LB=cfg.fy-7,S=cfg.fy-8);split=prepare(cfg,tabs)
 b=km.CircuitBuilder();x=b.create_quantum_register(22,name="x");flags=b.alloc_qubits(4 if parity_input else 5)
 f=([x[21]]+list(flags)) if parity_input else flags
 if parity_input:
  for k,amount in H.NSTAGES[:-1]:
   top=x[22-amount:22];b.x(top);H.and_all_into(b,list(top),f[k]);b.x(top)
   for i in reversed(range(22-amount)):b.cswap(f[k],x[i],x[i+amount])
  b.x(x[21]);b.left_rotate(x[:21],1,control=x[21]);b.cx(x[21],x[0])
 else:H.normalize(b,x,f)
 borrowed=[]
 if borrow and not parity_input:b.x(x[21]);borrowed=[x[21]]
 c2,res,coarse,low,high=[b.alloc_qubits(n) for n in [cfg.w2,split["rw"],split["cw"],lay["LB"],lay["whi"]]]
 checkpoints=[]
 def mark(label):
  z=b # stats collected from final tagged operation counts externally when supported
 def add(a,t,cin=None,sub=False):
  if carry_block:return B.add(b,a,t,cin,sub,borrowed=borrowed,prefix=carry_block)
  return A.folded_gadd(b,a,t,cin,sub,borrowed=borrowed)
 full=km.array(list(low)+list(res)+list(c2)+list(high)+list(coarse))
 rest=km.array(list(low)+list(res)+list(c2)+list(coarse));fulltable=[];resttable=[];hightable=[]
 for e in range(2):
  for i in range(128):
   k=(e,i);common=lay["lo"][k]|(split["residual"][k]<<lay["LB"])|(tabs[k][0]<<(lay["LB"]+split["rw"]))
   resttable.append(common|(split["coarse"][k]<<(lay["LB"]+split["rw"]+cfg.w2)))
   fulltable.append(common|(lay["hi"][k]<<(lay["LB"]+split["rw"]+cfg.w2))|(split["coarse"][k]<<(lay["LB"]+split["rw"]+cfg.w2+lay["whi"])))
   hightable.append(lay["hi"][k])
 address=km.array(list(x[14:21])+[f[0]]);delta=x[:14]
 def phase(reg,lowbit):
  ext=b.alloc_qubits(10-len(borrowed));word=km.array(list(reg)+borrowed+list(ext))
  shift(b,f,word,len(reg),False,prune)
  for i,q in enumerate(word):
   exponent=lowbit+i-cfg.fy-10
   if exponent<0 and not(lowbit==cfg.gy and i<phase_cut):b.z_pow(q,Fraction(2)**(exponent+1))
  shift(b,f,word,len(reg),True,prune);b.free(ext)
 def inner(inverse):
  rows=cfg.rrow_list();rows=rows[::-1] if inverse else rows
  for i in rows:
   sh=cfg.sh2[i]
   if sh>=0:a=c2[sh:];t=res;cin=c2[sh-1] if sh else None
   else:a=c2;t=res[-sh:];cin=None
   b.cx(delta[i],t);add(a,t,cin,sub=inverse);b.cx(delta[i],t)
 def outer(derivative,inverse):
  for i in range(14) if inverse else reversed(range(14)):
   sh=cfg.sh1[i]
   if sh>=0:a=derivative[sh:];t=low;cin=derivative[sh-1] if sh else None
   else:a=derivative;t=low[-sh:];cin=None
   b.cx(delta[i],t);add(a,t,cin,sub=inverse);b.cx(delta[i],t)
 b.init_lookup(table=fulltable,address=address,target=full)
 deferred_bits=[]
 if not merge:
  phase(high,lay["S"])
  if deferred:deferred_bits=D.demolish(b,high)
  else:b.del_lookup(table=hightable,address=address,target=high)
  b.free(high)
 inner(False);add(res[split["block"]:],coarse)
 derivative=res[:split["block"]]+coarse
 outer(derivative,False)
 if merge:
  add(low[lay["S"]:],high)
  phase(km.array(list(low[cfg.gy:lay["S"]])+list(high)),cfg.gy)
  add(low[lay["S"]:],high,sub=True)
 else:phase(low[cfg.gy:],cfg.gy)
 outer(derivative,True);add(res[split["block"]:],coarse,sub=True);inner(True)
 remaining=full if merge else rest
 if deferred:
  deferred_bits+=D.demolish(b,remaining);b.free(remaining)
  table=fulltable if merge else [hi|(rr<<lay['whi']) for hi,rr in zip(hightable,resttable)]
  D.repair(b,address,table,deferred_bits,anf=anf)
 else:
  b.del_lookup(table=fulltable if merge else resttable,address=address,target=remaining);b.free(remaining)
 if borrowed:b.x(x[21])
 if parity_input:
  b.cx(x[21],x[0]);b.right_rotate(x[:21],1,control=x[21]);b.x(x[21])
  for k,amount in reversed(H.NSTAGES[:-1]):
   for i in range(22-amount):b.cswap(f[k],x[i],x[i+amount])
   top=x[22-amount:22];m=b.alloc_bits(1)[0];b.hmr(f[k],m);b.push_condition(m)
   b.x(top);H.phase_all(b,list(top));b.x(top);b.pop_condition();b.free(m)
 else:H.unnormalize(b,x,f)
 b.free(flags)
 return b.finish_circuit(),{k:v for k,v in split.items() if k not in ["coarse","residual"]}

In [ ]:
%%writefile src/gen_invsqrt_r2/tables.json
{"params": [9, 19, 26, 2, [5]], "rows": [[134, 184284, 189442912], [132, 182154, 188710052], [129, 180064, 187985637], [127, 178014, 187269495], [124, 176003, 186561483], [122, 174029, 185861435], [120, 172091, 185169211], [118, 170190, 184484662], [116, 168323, 183807650], [113, 166490, 183138044], [111, 164690, 182475698], [109, 162922, 181820487], [107, 161186, 181172283], [105, 159481, 180530965], [104, 157804, 179896403], [102, 156158, 179268491], [100, 154540, 178647110], [98, 152949, 178032146], [97, 151386, 177423482], [95, 149849, 176821026], [93, 148338, 176224664], [92, 146852, 175634293], [90, 145391, 175049821], [89, 143953, 174471139], [88, 142540, 173898159], [86, 141149, 173330793], [85, 139781, 172768939], [83, 138435, 172212518], [82, 137110, 171661436], [81, 135807, 171115610], [79, 134523, 170574962], [78, 133260, 170039402], [77, 132017, 169508856], [76, 130793, 168983245], [75, 129587, 168462493], [73, 128400, 167946531], [72, 127231, 167435277], [71, 126080, 166928663], [70, 124945, 166426621], [69, 123828, 165929081], [68, 122727, 165435977], [67, 121643, 164947242], [66, 120574, 164462814], [65, 119521, 163982630], [64, 118483, 163506628], [63, 117460, 163034746], [63, 116452, 162566922], [61, 115458, 162103112], [60, 114478, 161643246], [60, 113512, 161187265], [59, 112560, 160735127], [58, 111621, 160286773], [57, 110695, 159842149], [57, 109781, 159401199], [56, 108880, 158963885], [55, 107991, 158530151], [54, 107114, 158099947], [54, 106249, 157673222], [53, 105395, 157249938], [52, 104553, 156830046], [51, 103722, 156413500], [51, 102903, 156000250], [50, 102094, 155590264], [49, 101295, 155183492], [49, 100507, 154779890], [48, 99729, 154379426], [48, 98960, 153982050], [47, 98201, 153587730], [46, 97453, 153196425], [46, 96714, 152808091], [45, 95984, 152422699], [45, 95263, 152040205], [44, 94552, 151660581], [44, 93848, 151283781], [43, 93154, 150909781], [42, 92468, 150538541], [42, 91791, 150170023], [41, 91122, 149804203], [41, 90461, 149441037], [40, 89808, 149080506], [40, 89163, 148722566], [40, 88525, 148367192], [39, 87895, 148014360], [38, 87272, 147664032], [38, 86657, 147316174], [38, 86048, 146970766], [37, 85448, 146627778], [37, 84853, 146287178], [36, 84266, 145948944], [36, 83686, 145613040], [36, 83112, 145279446], [35, 82544, 144948138], [35, 81983, 144619082], [34, 81429, 144292261], [34, 80881, 143967641], [33, 80339, 143645207], [33, 79802, 143324927], [33, 79272, 143006779], [33, 78748, 142690739], [32, 78229, 142376791], [32, 77716, 142064901], [31, 77208, 141755057], [31, 76706, 141447225], [31, 76210, 141141391], [30, 75719, 140837539], [30, 75233, 140535635], [30, 74752, 140235663], [29, 74277, 139937609], [29, 73806, 139641443], [29, 73341, 139347148], [29, 72879, 139054708], [28, 72424, 138764106], [28, 71973, 138475312], [28, 71526, 138188316], [27, 71084, 137903100], [27, 70647, 137619640], [27, 70214, 137337918], [26, 69785, 137057926], [26, 69361, 136779636], [26, 68941, 136503032], [26, 68525, 136228098], [25, 68114, 135954826], [25, 67706, 135683186], [25, 67303, 135413170], [25, 66903, 135144756], [24, 66508, 134877940], [24, 66116, 134612692], [24, 65728, 134349002], [190, 260618, 267912733], [186, 257604, 266876317], [183, 254649, 265851833], [179, 251750, 264839062], [176, 248905, 263837776], [173, 246114, 262847760], [170, 243374, 261868805], [166, 240685, 260900717], [163, 238045, 259943280], [160, 235452, 258996306], [157, 232907, 258059608], [155, 230407, 257132994], [152, 227952, 256216297], [149, 225539, 255309335], [147, 223170, 254411933], [144, 220841, 253523934], [141, 218553, 252645166], [139, 216303, 251775472], [137, 214092, 250914696], [134, 211918, 250062693], [133, 209781, 249219303], [130, 207680, 248384401], [128, 205614, 247557827], [126, 203582, 246739450], [124, 201582, 245929136], [122, 199616, 245126758], [120, 197681, 244332178], [118, 195777, 243545277], [116, 193903, 242765929], [114, 192059, 241994019], [112, 190245, 241229423], [110, 188458, 240472031], [109, 186700, 239721722], [107, 184968, 238978398], [105, 183264, 238241948], [104, 181585, 237512258], [102, 179931, 236789238], [101, 178303, 236072775], [100, 176699, 235362777], [98, 175119, 234659153], [97, 173563, 233961797], [95, 172029, 233270625], [94, 170518, 232585538], [92, 169029, 231906460], [91, 167561, 231233288], [90, 166114, 230565944], [88, 164689, 229904352], [87, 163283, 229248415], [86, 161897, 228598063], [84, 160531, 227953219], [84, 159184, 227313793], [82, 157855, 226679727], [81, 156545, 226050933], [80, 155254, 225427342], [79, 153979, 224808882], [78, 152723, 224195488], [77, 151482, 223587086], [76, 150259, 222983610], [74, 149052, 222385000], [74, 147861, 221791176], [73, 146686, 221202087], [72, 145527, 220617669], [71, 144382, 220037857], [70, 143252, 219462597], [69, 142138, 218891821], [68, 141037, 218325477], [67, 139951, 217763505], [67, 138879, 217205848], [66, 137820, 216652456], [65, 136774, 216103274], [64, 135742, 215558248], [63, 134723, 215017324], [62, 133716, 214480452], [62, 132722, 213947576], [61, 131740, 213418658], [60, 130770, 212893643], [59, 129812, 212372483], [58, 128866, 211855133], [58, 127931, 211341541], [57, 127007, 210831671], [57, 126095, 210325469], [56, 125193, 209822899], [55, 124302, 209323915], [54, 123422, 208828475], [54, 122551, 208336532], [53, 121691, 207848052], [53, 120841, 207362990], [52, 120001, 206881312], [51, 119171, 206402974], [51, 118350, 205927936], [50, 117538, 205456166], [50, 116736, 204987620], [49, 115943, 204522268], [49, 115157, 204060068], [48, 114383, 203600993], [48, 113616, 203144995], [47, 112858, 202692053], [47, 112108, 202242123], [46, 111366, 201795181], [46, 110632, 201351185], [45, 109907, 200910111], [44, 109189, 200471923], [44, 108480, 200036585], [44, 107777, 199604071], [43, 107083, 199174353], [43, 106396, 198747396], [42, 105716, 198323178], [41, 105043, 197901664], [41, 104378, 197482822], [41, 103719, 197066628], [41, 103068, 196653054], [40, 102422, 196242078], [40, 101784, 195833664], [39, 101153, 195427794], [38, 100528, 195024436], [38, 99909, 194623562], [38, 99297, 194225148], [38, 98691, 193829172], [37, 98091, 193435613], [37, 97497, 193044437], [36, 96909, 192655629], [36, 96327, 192269157], [36, 95751, 191884999], [35, 95181, 191503141], [35, 94616, 191123549], [35, 94056, 190746205], [34, 93502, 190371093], [34, 92954, 189998181]], "phase_cut": 0}

In [ ]:
%%writefile src/gen_dqi_v6/port_runtime.py
# SPDX-License-Identifier: Apache-2.0
"""Small explicit adapter for Gu et al.'s published source to official kickmix.

Integer wire handles and LIFO ownership match the published builder. Physical
kickmix qubits stay allocated while this adapter accounts for their lifetimes;
every recycle emits the same reset as the authors' Builder. Named replay holds
are checked, so the journal cannot silently alias live scratch.
"""
from contextlib import contextmanager
import json, os, pathlib, time
import kickmix as km
from ref.gu_classical import GF2n, f2_matinv, f2_transpose, parity
from ref.gu_linear import linear_in_place


def load_contract(_):
    contract=json.loads((pathlib.Path(__file__).parent/'ref/gu_contract.json').read_text())
    # The paper uses 0x10eb. The competition uses 0x1053. All arithmetic
    # formulas are regenerated from this field; no field labels are conflated.
    contract['gf_poly']=km.GF2Field(12).modulus
    return contract


class Journal:
    """Gu et al., uncompute.bennett.Journal; pure conjugate choreography."""
    def __init__(self): self._steps=[];self.recording=True
    def mark(self):return len(self._steps)
    def step(self, fwd, rev=None):
        fwd()
        if self.recording:self._steps.append((fwd,rev if rev is not None else fwd))
    def defer(self, fwd, rev=None):self._steps.append((fwd,rev if rev is not None else fwd))
    def forward(self,since=0,until=None):
        for fwd,_ in self._steps[since:until]:fwd()
    def reverse(self,since=0,until=None):
        for _,rev in reversed(self._steps[since:until]):rev()


class Builder:
    def __init__(self):
        self.k=km.CircuitBuilder()
        self.physical=[]
        self.free_qubits=[];self.free_set=set();self.active=set()
        self.errors=[self.k.create_classical_register(12,name=f'E{j}') for j in range(4095)]
        # Published emitter uses each HMR outcome only for the immediately
        # following phase repayments, before the next HMR. Reusing one live
        # classical bit preserves that ordering and avoids dead classical ids.
        self.measurement=self.k.alloc_bits(1)[0]
        self.phase_stats=[];self.gate_counts={}
        self.phase_name='init';self.started=time.monotonic()
        self.owners={};self.live_profile={};self.current_phase=None
        self.progress=pathlib.Path(os.environ['QCC_OUT'])/'build_progress.json' if 'QCC_OUT' in os.environ else None

    def qubit(self):
        if self.free_qubits:
            q=self.free_qubits.pop();self.free_set.remove(q)
        else:
            q=len(self.physical);self.physical.append(self.k.alloc_qubits(1)[0])
        assert q not in self.active
        self.active.add(q)
        self._profile()
        return q
    def qubits(self,n):return [self.qubit() for _ in range(n)]
    def reclaim(self,qs):
        qs=list(qs);want=set(qs)
        assert len(want)==len(qs) and want<=self.free_set, ('reclaim live ids',want-self.free_set)
        self.free_qubits[:]=[q for q in self.free_qubits if q not in want]
        self.free_set-=want;self.active.update(want)
        self._profile()
    def recycle(self,q):
        assert q in self.active,('recycle not active',q)
        self.k.reset(self.physical[q])
        self.active.remove(q);self.free_qubits.append(q);self.free_set.add(q)
        self.owners.pop(q,None)
    def recycle_all(self,qs):
        for q in qs:self.recycle(q)
    def _profile(self,force=False):
        if self.current_phase is None:return
        p=self.live_profile.setdefault(self.current_phase,{'peak_active':0,'peak_physical':0})
        p['peak_physical']=max(p['peak_physical'],len(self.physical))
        if len(self.active)>p['peak_active'] or (force and len(self.active)==p['peak_active']):
            p['peak_active']=len(self.active)
            counts={}
            for q in self.active:
                name=self.owners.get(q,'unclaimed')
                counts[name]=counts.get(name,0)+1
            p['owners_at_peak']=counts
    def claim(self,name,qs):
        for q in qs:self.owners[q]=name
        self._profile(force=True)
    def release(self,name,qs):
        for q in qs:self.owners.pop(q,None)
    def release_recycle(self,name,qs):self.recycle_all(reversed(qs))
    def _gate(self,name,*qs,cond=None):
        if cond is not None:self.k.push_condition(cond)
        self.gate_counts[name]=self.gate_counts.get(name,0)+1
        getattr(self.k,name)(*[self.physical[q] for q in qs])
        if cond is not None:self.k.pop_condition()
    def x(self,q,cond=None):self._gate('x',q,cond=cond)
    def z(self,q,cond=None):self._gate('z',q,cond=cond)
    def cx(self,c,t,cond=None):self._gate('cx',c,t,cond=cond)
    def cz(self,a,b,cond=None):self._gate('cz',a,b,cond=cond)
    def ccx(self,a,b,t,cond=None):self._gate('ccx',a,b,t,cond=cond)
    def ccz(self,a,b,c,cond=None):self._gate('ccz',a,b,c,cond=cond)
    def swap(self,a,b,cond=None):self._gate('swap',a,b,cond=cond)
    def cswap(self,c,a,b,cond=None):self._gate('cswap',c,a,b,cond=cond)
    def neg(self,cond=None):
        assert cond is not None
        self.k.neg_if(cond)
    def hmr(self,q):
        self.k.hmr(self.physical[q],self.measurement)
        return self.measurement
    @contextmanager
    def phase(self,name):
        self.k.info_tag('@enter:'+name)
        self.phase_name=name
        prev=self.current_phase;self.current_phase=name
        self._profile()
        before=dict(self.gate_counts)
        t=time.monotonic()
        p={'phase':name,'elapsed':t-self.started,'physical_qubits':len(self.physical),'active_qubits':len(self.active)}
        print(json.dumps(p),flush=True)
        if self.progress:self.progress.write_text(json.dumps(p,indent=2))
        yield
        self.k.info_tag('@exit:'+name)
        self.phase_stats.append({'phase':name,'seconds':time.monotonic()-t,
                                 'gates':{k:v-before.get(k,0) for k,v in self.gate_counts.items()}})
        self.live_profile[name]['exit_active']=len(self.active)
        self.current_phase=prev


def declare_registers(b,decls):
    regs={}
    for index,(_,width) in enumerate(decls):
        assert not b.free_qubits
        start=len(b.physical)
        reg=b.k.create_quantum_register(width,name=f'S{index}')
        b.physical.extend(reg)
        qs=list(range(start,start+width));b.active.update(qs)
        regs[f'S{index}']=qs
    return regs


def assert_metrics(b,_):
    return b.gate_counts.get('ccx',0)+b.gate_counts.get('cswap',0)+b.gate_counts.get('ccz',0)


def emit_and_report(b):
    raise RuntimeError('Use run_gu_port.py; no implicit publication or emission destination.')

In [ ]:
%%writefile src/gen_dqi_v6/gu_formula_derivation.py
# SPDX-License-Identifier: Apache-2.0
# Gu et al., deterministic field formula derivation, adapted imports.
#!/usr/bin/env python3
"""Regenerate constants.json for the dqi-reed-solomon discovered circuit.

main() reruns every search and writes constants.json.  The searches, in
file order:

  Formulas     the multiplication formulas and tower maps (KT12, KT6,
               KT3, CUBE3Q, PAIR3, LIN).  Deterministic; seconds.
  BM windows   the per-round coefficient windows (IMAXV, KMSV, KMMV).
               search.c samples the full Berlekamp-Massey recursion over
               403.2M weight-law draws (56 fixed seeds, 7.2M draws each)
               and counts, per (round, slot), how often the slot changes
               state.  Each window is the smallest one whose clipped
               slots all have mass under 4e-5 per draw.  About 2.7
               CPU-hours; the seeds run on a thread pool (3.5 min on 56
               cores).
  Psi windows  the cofactor windows of the Psi route: per CR op, the
               minimal cyclic interval covering the union support where
               the op acts, over 400k law draws (16 fixed seeds) plus
               the w = 0 anchor and every w = 1 position.  BM runs under
               the freshly derived BM windows.

The models section holds the bit-exact classical models of the circuit
that the Psi window search runs: the fused-Q BM recursion and the Psi
route (stages 5-7).  The script derives the instance from contract.json
and does not import circuit.py, so it runs without constants.json.

Formula derivation.  The field is GF(2^12), poly 4331, x = 2
primitive.  The tower is GF(2^12) = GF(64)(t) = GF(8)(s)(t), read top
down: A = x + x^64, t = x / A, nu = x^65 / A^2, u = x^65,
A' = u + u^8, s = u / A', nu' = u^9 / A'^2, v = u^9.  Tower
coordinates are E12[i + 3j + 6k] = v^i s^j t^k.  The norm descends
both quadratic levels (N = x xbar in GF(64), then M = N Nbar in GF(8)),
so the only nonlinear inverse left is M -> M^3 in GF(8), a 3-product
quadratic form; M^-1 = (M^3)^2 is a free squaring.  A full inversion
costs N (15) + M (6) + M^3 (3) + Nbar M^-1 (12) + xbar N^-1 (30) = 66
Toffoli, and every stage maps 0 to 0.  Rejected towers: GF(64)^2 alone
needs N^31 (five 1-bits, so a generic GF(64) product on top of two
quadratic forms); GF(16)^3 pays a cubic norm; plain Itoh-Tsujii needs
at least four 42-Toffoli products.

  KT12    GF(2^12) = GF(4)[x]: a 6-term product over GF(4) by CRT at
          the four GF(4) points, infinity, and the irreducible
          quadratics x^2+x+w, x^2+x+w^2, x^2+wx+1.  14 GF(4) products
          x 3 = 42; the greedy prune removes nothing.  GF(8)-CRT and
          Chudnovsky-style curves over GF(4) tie at 42; Karatsuba
          variants give 45-51.
  KT6     GF(64) = GF(4)[u], 3 terms, CRT at four points + infinity:
          15 products, the best known bilinear complexity of GF(2^6).
  KT3     the 3-term formula M3TERMS reduced modulo v's minimal
          polynomial: 6 products.
  CUBE3Q  exhaustive search over product subsets: 3 is the minimum.
  PAIR3   elimination rank decomposition of Tr3(r s^2): rank 3.
  LIN     the tower's linear maps (pure linear algebra on the
          elements above).

RUN: uv run python gen_constants.py   (writes constants.json)
"""

import collections
import ctypes
import json
import os
import pathlib
import random
import sys
import time
from multiprocessing import Pool
from multiprocessing.pool import ThreadPool

import numpy as np

from port_runtime import GF2n, f2_matinv, f2_transpose, parity
from port_runtime import load_contract

HERE = pathlib.Path(__file__).resolve().parent
PATH = HERE / "constants.json"

# The instance (contract.json), derived as circuit.py derives it.
CONTRACT = load_contract(__file__)
M = CONTRACT["m"]  # code length = nonzero field points
N = CONTRACT["n_synd"]  # syndrome symbols = BM rounds
B = CONTRACT["b"]  # bits per symbol
W = CONTRACT["w_max"]  # unique-decoding radius n/2 = max locator degree
LAW_CUM = CONTRACT["weight_cum_2p16"]  # the grader's 2^16 cumulative weight law
NH = W // 2 + 1  # words per Euclid half E', O' (and per cofactor)
FIELD = GF2n(B, CONTRACT["gf_poly"])
EXP, LOG = FIELD.exp, FIELD.log
gmul, ginv, gsqrt, trace = FIELD.mul, FIELD.inv, FIELD.sqrt, FIELD.trace
DWB = (N - 2).bit_length() + 1  # the signed d counter dW: two's complement
DMAX = 1 << (DWB - 1)  # two's-complement range of dW


def bitlist(m, n):
    return [i for i in range(n) if (m >> i) & 1]


def bitmask(n, pred):
    return sum(1 << i for i in range(n) if pred(i))


DWMASK = (1 << DWB) - 1  # the dW register (two's complement, DWB bits)
MMASK = 31  # m = (dW + 2) >> 2 lives in 5 bits (dW[2:7])
# The cores this process may use (its CPU affinity set, not the whole node).
NCPU = len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else os.cpu_count()

# #########################################################################
# Formulas
# #########################################################################
# Every table is deterministic; the module docstring gives the derivation.
# search_kt keeps the CRT formula only if certify_kt accepts it.

# ---------------------------------------------------------------------------
# small F2 helpers
# ---------------------------------------------------------------------------


def rows_apply(rows, x):
    """bit r of the result = parity(rows[r] & x)."""
    v = 0
    for r in range(len(rows)):
        if parity(rows[r] & x):
            v |= 1 << r
    return v


def compose_rows(A, B, nb):
    """Rows of A o B (apply B, then A); B has nb rows."""
    out = []
    for r in range(len(A)):
        m = 0
        for i in bitlist(A[r], nb):
            m ^= B[i]
        out.append(m)
    return out


# ---------------------------------------------------------------------------
# GF(2^b) arithmetic
# ---------------------------------------------------------------------------


def reduce_raw(raw):
    """Reduce a raw F2[x] product (degree <= 2B - 2) mod the field
    polynomial: x^k = EXP[k], since gamma = x.
    """
    v = 0
    for k in bitlist(raw, 2 * B - 1):
        v ^= EXP[k]
    return v


# ---------------------------------------------------------------------------
# polynomial-multiplication formulas (Karatsuba fallback, GF(8) terms)
# ---------------------------------------------------------------------------

M3TERMS = [
    (1, 1, 0b00111),
    (2, 2, 0b01110),
    (4, 4, 0b11100),
    (3, 3, 0b00010),
    (6, 6, 0b01000),
    (5, 5, 0b00100),
]


def kara(n):
    """Recursive symmetric Karatsuba formula for n-term F2 polynomial
    multiplication, [(Amask, Bmask, raw-product mask)].
    """
    if n == 1:
        return [(1, 1, 1)]
    if n == 3:
        return list(M3TERMS)
    n0 = n // 2
    n1 = n - n0
    m = max(n0, n1)
    out = []
    for A, B, C in kara(n0):
        out.append((A, B, C ^ (C << n0)))
    for A, B, C in kara(n1):
        out.append((A << n0, B << n0, (C << n0) ^ (C << (2 * n0))))
    for A, B, C in kara(m):
        Am = 0
        Bm = 0
        for i in range(m):
            if (A >> i) & 1:
                if i < n0:
                    Am ^= 1 << i
                if i < n1:
                    Am ^= 1 << (n0 + i)
            if (B >> i) & 1:
                if i < n0:
                    Bm ^= 1 << i
                if i < n1:
                    Bm ^= 1 << (n0 + i)
        out.append((Am, Bm, C << n0))
    return out


# ---------------------------------------------------------------------------
# GF(4)-tower CRT bilinear formulas (deterministic; KT / KT12 / KT6)
# ---------------------------------------------------------------------------


class F4:
    """GF(4) = {0, 1, w, w+1} inside fld, index-encoded (bit 0 = 1, bit 1 =
    w), so index addition is XOR.
    """

    def __init__(self, fld):
        gmul = fld.mul
        w3 = fld.exp[fld.order // 3]
        assert gmul(w3, w3) == (w3 ^ 1) and gmul(w3, w3 ^ 1) == 1
        self.w3 = w3
        self.E = [0, 1, w3, w3 ^ 1]
        self.I = {self.E[i]: i for i in range(4)}
        self.gmul = gmul

    def mul(self, a, bq):
        return self.I[self.gmul(self.E[a], self.E[bq])]


def f4_irreducible_quadratics(f4):
    """Monic irreducible x^2 + a x + b over GF(4), (a, b) in index order."""
    out = []
    for a in range(4):
        for bq in range(4):
            if all((f4.mul(r, r) ^ f4.mul(a, r) ^ bq) != 0 for r in range(4)):
                out.append((a, bq))
    assert len(out) == 6
    return out


def f4_lams(f4, k, nquad):
    """Return the CRT evaluation functionals for a k-term product over GF(4):
    evaluation at 0, 1, w, w^2, at infinity, then for each of the first
    nquad irreducible quadratics q the two reduction coordinates
    (x^i mod q = c0_i + c1_i x) and their sum (Karatsuba over GF(4)).
    Modulus degree = 4 + 1 + 2 nquad; the product has 2k - 1 coefficients.
    """
    assert 4 + 1 + 2 * nquad == 2 * k - 1, (k, nquad)
    lams = []
    for pt in range(4):
        lam, p = [], 1
        for _ in range(k):
            lam.append(p)
            p = f4.mul(p, pt)
        lams.append(lam)
    lams.append([0] * (k - 1) + [1])
    for a, bq in f4_irreducible_quadratics(f4)[:nquad]:
        L0, L1 = [], []
        c0, c1 = 1, 0
        for _ in range(k):
            L0.append(c0)
            L1.append(c1)
            c0, c1 = f4.mul(c1, bq), c0 ^ f4.mul(c1, a)
        lams.extend([L0, L1, [p ^ q for p, q in zip(L0, L1)]])
    return lams


def f4_digits(f4, fld, coords, gens, k):
    """GF(4)-digit extraction for an n = 2k dimensional coordinate system:
    gens[i] (i < k) are field elements whose GF(4)-span {gens[i], gens[i]
    w} is the space; coords(elt) -> n-bit coordinate vector.  Returns
    DIG[bq] = the k GF(4)-digit indices of coordinate basis vector bq.
    """
    n = 2 * k
    colsE = []
    for i in range(k):
        for j in range(2):
            e = gens[i]
            if j:
                e = fld.mul(e, f4.w3)
            colsE.append(coords(e))
    EXTR = f2_matinv(f2_transpose(colsE, n), n)
    DIG = []
    for bq in range(n):
        c = rows_apply(EXTR, 1 << bq)
        DIG.append([(c >> (2 * i)) & 3 for i in range(k)])
    return DIG


def solve_sym(n, prodlist, mulcoord):
    """Reconstruction solve for symmetric operand masks: target masks T[t]
    per product such that a*b = XOR_t parity(a&m_t) parity(b&m_t) T[t] on all
    n x n basis pairs (verified), or None.
    """
    piv = {}
    for i in range(n):
        for j in range(n):
            a = 0
            for t, mk in enumerate(prodlist):
                if ((mk >> i) & 1) and ((mk >> j) & 1):
                    a |= 1 << t
            r = mulcoord(i, j)
            while a:
                cc = a.bit_length() - 1
                if cc in piv:
                    pa, pr = piv[cc]
                    a ^= pa
                    r ^= pr
                else:
                    piv[cc] = (a, r)
                    a = 0
                    r = 0
            if r:
                return None
    T = [0] * len(prodlist)
    for cc in sorted(piv):
        a, r = piv[cc]
        acc = r
        aa = a & ~(1 << cc)
        while aa:
            c2 = aa.bit_length() - 1
            aa ^= 1 << c2
            acc ^= T[c2]
        T[cc] = acc
    for i in range(n):
        for j in range(n):
            acc = 0
            for t, mk in enumerate(prodlist):
                if ((mk >> i) & 1) and ((mk >> j) & 1):
                    acc ^= T[t]
            if acc != mulcoord(i, j):
                return None
    return T


def crt_f4_formula(n, DIG, k, lams, f4mul, mulcoord):
    """GF(4)-CRT bilinear formula on n = 2k coordinate bits: products =
    3 F2 parities per lambda, reconstruction solved by
    Gaussian elimination over all n x n basis pairs, deterministic greedy
    prune in index order.  Returns [(prodmask, targetmask)] (symmetric
    operand masks) or None.
    """
    prods = []
    for lam in lams:
        m0 = 0
        m1 = 0
        for bq in range(n):
            U = 0
            for i in range(k):
                U ^= f4mul(lam[i], DIG[bq][i])
            if U & 1:
                m0 |= 1 << bq
            if U & 2:
                m1 |= 1 << bq
        prods.extend([m0, m1, m0 ^ m1])
    assert len(prods) == 3 * len(lams)

    T = solve_sym(n, prods, mulcoord)
    if T is None:
        return None
    keep = list(range(len(prods)))
    for t in range(len(prods)):
        if t not in keep:
            continue
        trial = [prods[u] for u in keep if u != t]
        if solve_sym(n, trial, mulcoord) is not None:
            keep = [u for u in keep if u != t]
    plist = [prods[u] for u in keep]
    T = solve_sym(n, plist, mulcoord)
    if T is None:
        return None
    out = []
    for t, mk in enumerate(plist):
        if T[t] and mk:
            out.append((mk, T[t]))
    return out


def build_f4_crt_formula(fld):
    """GF(2^b) multiplication over the poly basis via GF(2^b) = GF(4)[x]:
    b/2 terms over GF(4), moduli = 4 points + infinity + (b/2 - 3)
    irreducible quadratics (b = 12: 42 products).
    """
    b = fld.n
    k = b // 2
    try:
        f4 = F4(fld)
        theta = 2
        gens = [fld.pow(theta, i) for i in range(k)]
        DIG = f4_digits(f4, fld, lambda e: e, gens, k)
        for v in range(1 << b):  # exhaustive digit round-trip
            dg = [0] * k
            for bq in bitlist(v, b):
                dg = [x ^ y for x, y in zip(dg, DIG[bq])]
            w = 0
            for i in range(k):
                w ^= fld.mul(gens[i], f4.E[dg[i]])
            assert w == v
        lams = f4_lams(f4, k, k - 3)
        return crt_f4_formula(b, DIG, k, lams, f4.mul, lambda i, j: fld.mul(1 << i, 1 << j))
    except Exception as ex:  # noqa: BLE001  (best-effort alternative: any failure falls back to kara(b))
        print("f4-crt formula construction failed:", repr(ex))
        return None


def search_kt(fld):
    """Return the GF(2^b) multiplication formula, constants.json shape
    [[Ab, Bb, Tb], ...].  Deterministic: the Karatsuba formula kara(b)
    (b = 12: 54 products), replaced by the GF(4)-tower CRT formula when
    that is smaller.
    """
    b = fld.n
    RAW = kara(b)
    # basis-exhaustive check of RAW (complete for a bilinear form)
    for i in range(b):
        for j in range(b):
            pr = 0
            for Am, Bm, Cm in RAW:
                if parity((1 << i) & Am) and parity((1 << j) & Bm):
                    pr ^= Cm
            assert reduce_raw(pr) == fld.mul(1 << i, 1 << j)
    KT = []
    for Am, Bm, Cm in RAW:
        tm = reduce_raw(Cm)
        if tm:
            KT.append([bitlist(Am, b), bitlist(Bm, b), bitlist(tm, b)])
    F4KT = build_f4_crt_formula(fld)
    if F4KT is not None:
        KT_NEW = [[bitlist(mk, b), bitlist(mk, b), bitlist(tm, b)] for (mk, tm) in F4KT]
        if certify_kt(KT_NEW, fld) and len(KT_NEW) < len(KT):
            KT = KT_NEW
    return KT


# ---------------------------------------------------------------------------
# The towers
# ---------------------------------------------------------------------------


class Tower:
    """The (2, 2, 3) tower of the module docstring.  GF(2^12) = GF(64)(t),
    GF(64) = GF(8)(s), GF(8) = F2[v]; E_list = [v^i s^j t^k] indexed
    i + 3j + 6k; the GF(64) coordinates are E6 = [v^i s^j] (i + 3j), the
    GF(8) coordinates E3 = [v^i].
    """

    def __init__(self, fld):
        self.fld = fld
        b = fld.n
        assert b == 12
        h = b // 2
        EXP, gmul, ginv, gpow = fld.exp, fld.mul, fld.inv, fld.pow
        Ael = EXP[1 << h] ^ 2  # x^(2^h) + x = Tr(x)
        t_el = gmul(2, ginv(Ael))  # t = x / A
        u_el = EXP[(1 << h) + 1]  # u = x^(2^h + 1) = N(x)
        nu_el = gmul(u_el, ginv(gmul(Ael, Ael)))  # nu = u / A^2
        assert gpow(u_el, 1 << h) == u_el and u_el != 0
        assert gpow(Ael, 1 << h) == Ael and Ael != 0
        assert gpow(nu_el, 1 << h) == nu_el
        assert gmul(t_el, t_el) ^ t_el == nu_el
        self.Ael, self.t_el, self.u_el, self.nu_el = Ael, t_el, u_el, nu_el
        self.h = h
        # sub-level GF(64) = GF(8)(s)
        Ap = gpow(u_el, 8) ^ u_el  # Tr_{6/3}(u)
        s_el = gmul(u_el, ginv(Ap))
        v_el = gpow(u_el, 9)  # N_{6/3}(u)
        nup_el = gmul(v_el, ginv(gmul(Ap, Ap)))
        assert gpow(Ap, 8) == Ap and Ap != 0
        assert gpow(v_el, 8) == v_el and gpow(v_el, 7) == 1 and v_el != 1
        assert gpow(nup_el, 8) == nup_el
        assert gmul(s_el, s_el) ^ s_el == nup_el
        self.Ap, self.s_el, self.v_el, self.nup_el = (Ap, s_el, v_el, nup_el)
        E3 = [gpow(v_el, i) for i in range(3)]
        E6 = E3 + [gmul(e, s_el) for e in E3]
        E_list = E6 + [gmul(e, t_el) for e in E6]
        self.E_sub = E6
        self.E3 = E3
        self.E_list = E_list
        self.MINVR = f2_matinv(f2_transpose(E_list, b), b)
        for v in range(1 << b):
            c = self.coords_full(v)
            w = 0
            for u in bitlist(c, b):
                w ^= E_list[u]
            assert w == v
        # projections poly -> (a0, a1) and the conjugate's constant part
        self.A0ROWS = self.MINVR[0:h]
        self.A1ROWS = self.MINVR[h:b]
        self.A01ROWS = [self.A0ROWS[r] ^ self.A1ROWS[r] for r in range(h)]
        # squaring and multiply-by-nu on subfield coords
        self.SQR = f2_transpose([self.coords_sub(gmul(e, e)) for e in self.E_sub], h)
        self.NUR = f2_transpose([self.coords_sub(gmul(nu_el, e)) for e in self.E_sub], h)
        NSR = compose_rows(self.NUR, self.SQR, h)
        # linear part of the norm N = a0 a1 + a0^2 + nu a1^2 on poly words
        self.LNROWS = []
        for r in range(h):
            m = 0
            for j in bitlist(self.SQR[r], h):
                m ^= self.A0ROWS[j]
            for j in bitlist(NSR[r], h):
                m ^= self.A1ROWS[j]
            self.LNROWS.append(m)
        # GF(8) level: coords3 = low 3 bits of the GF(64) coords of a
        # GF(8) element; RED3 = v^k for the KT3 reduction
        self.RED3 = [self.coords3(gpow(v_el, k)) for k in range(5)]
        self.SQ3R = f2_transpose([self.coords3(gmul(e, e)) for e in self.E3], 3)
        self.NU3R = f2_transpose([self.coords3(gmul(nup_el, e)) for e in self.E3], 3)
        NSR3 = compose_rows(self.NU3R, self.SQ3R, 3)
        # linear part of M = n0 n1 + n0^2 + nu' n1^2 on GF(64) coords
        self.LMROWS = [self.SQ3R[r] ^ (NSR3[r] << 3) for r in range(3)]
        self.FCUBE3 = [self.coords3(gpow(self.elt3(c), 3)) for c in range(8)]

    # -- coordinates --
    def coords_full(self, v):
        return rows_apply(self.MINVR, v)

    def coords_sub(self, v):
        c = self.coords_full(v)
        assert c < (1 << self.h), "not a subfield element"
        return c

    def coords3(self, v):
        c = self.coords_full(v)
        assert c < 8, "not a GF(8) element"
        return c

    def elt3(self, c):
        w = 0
        for i in bitlist(c, 3):
            w ^= self.E3[i]
        return w

    def lin_tables(self):
        """Return the LIN block of constants.json: the linear maps circuit.py
        reads.
        """
        return {
            "E12": list(self.E_list),
            "A0ROWS": list(self.A0ROWS),
            "A1ROWS": list(self.A1ROWS),
            "LNROWS": list(self.LNROWS),
            "SQ3R": list(self.SQ3R),
            "LMROWS": list(self.LMROWS),
        }


# ---------------------------------------------------------------------------
# subfield multiplication formulas
# ---------------------------------------------------------------------------


def search_kt6(fld, tower):
    """Return the GF(64) multiplication formula in tower coords (E6), 15
    products: GF(64) = GF(4)[u], 3 terms over GF(4), CRT at the four points
    + infinity, built directly on the 6-bit coordinate vectors.
    """
    f4 = F4(fld)
    u = tower.u_el
    gens = [fld.pow(u, i) for i in range(3)]
    DIG = f4_digits(f4, fld, tower.coords_sub, gens, 3)
    lams = f4_lams(f4, 3, 0)
    assert len(lams) == 5
    E6 = tower.E_sub
    F = crt_f4_formula(
        6, DIG, 3, lams, f4.mul, lambda i, j: tower.coords_sub(fld.mul(E6[i], E6[j]))
    )
    assert F is not None, "GF(64) CRT formula failed"
    return [[bitlist(mk, 6), bitlist(mk, 6), bitlist(tm, 6)] for (mk, tm) in F]


def search_kt3(tower):
    """Return the GF(8) multiplication formula in the v-basis, 6 products:
    M3TERMS reduced modulo the minimal polynomial of v.
    """
    KT3 = []
    for Am, Bm, Cm in M3TERMS:
        tm = 0
        for kk in range(5):
            if (Cm >> kk) & 1:
                tm ^= tower.RED3[kk]
        if tm:
            KT3.append([bitlist(Am, 3), bitlist(Bm, 3), bitlist(tm, 3)])
    return KT3


# ---------------------------------------------------------------------------
# quadratic-form formulas: exact solve and exhaustive search (CUBE3Q)
# ---------------------------------------------------------------------------


def qp_pairs(n):
    return [(i, j) for i in range(n) for j in range(i + 1, n)]


def _offvec(n, u, w):
    v = 0
    for idx, (i, j) in enumerate(qp_pairs(n)):
        if (((u >> i) & (w >> j)) ^ ((u >> j) & (w >> i))) & 1:
            v |= 1 << idx
    return v


def quad_solve(n, prods, fvals):
    """Given product pairs (u, w),
    solve for output masks + linear correction so the quadratic formula
    reproduces fvals on all 2^n elements; returns (tris, corr) or None.
    """
    pairs = qp_pairs(n)
    bas = {}
    for t_, (u, w) in enumerate(prods):
        x = _offvec(n, u, w)
        sel = 1 << t_
        while x:
            mb = x.bit_length() - 1
            if mb in bas:
                x ^= bas[mb][0]
                sel ^= bas[mb][1]
            else:
                bas[mb] = (x, sel)
                break
    tg = [0] * n
    for idx, (i, j) in enumerate(pairs):
        cdiff = fvals[(1 << i) | (1 << j)] ^ fvals[1 << i] ^ fvals[1 << j]
        for m in range(n):
            if (cdiff >> m) & 1:
                tg[m] |= 1 << idx
    outs = []
    for m in range(n):
        x = tg[m]
        sel = 0
        while x:
            mb = x.bit_length() - 1
            if mb not in bas:
                return None
            x ^= bas[mb][0]
            sel ^= bas[mb][1]
        outs.append(sel)
    tris = []
    for t_, (u, w) in enumerate(prods):
        om = 0
        for m in range(n):
            if (outs[m] >> t_) & 1:
                om |= 1 << m
        if om:
            tris.append((u, w, om))
    corr = [0] * n
    for i in range(n):
        cv = fvals[1 << i]
        for u, w, om in tris:
            if (u >> i) & 1 and (w >> i) & 1:
                cv ^= om
        for m in range(n):
            if (cv >> m) & 1:
                corr[m] |= 1 << i
    for x in range(1 << n):
        if quad_apply((tris, corr), x, n) != fvals[x]:
            return None
    return (tris, corr)


def quad_exhaustive(n, fvals):
    """EXHAUSTIVE minimum-product quadratic formula for fvals on n bits:
    candidate products are the unordered pairs u < w of nonzero masks (a
    pair (u, u) is linear and never needed); subsets are tried in
    increasing size, lexicographic order, and the first solvable one wins.
    Feasible for n <= 4; used for GF(8).
    """
    from itertools import combinations

    cands = [(u, w) for u in range(1, 1 << n) for w in range(u + 1, 1 << n)]
    for r in range(len(qp_pairs(n)) + 1):
        for sub in combinations(cands, r):
            res = quad_solve(n, list(sub), fvals)
            if res is not None:
                tris, corr = res
                return [[list(t) for t in tris], list(corr)]
    raise AssertionError("no quadratic formula (impossible)")


def search_cube3q(tower):
    """Exhaustive, deterministic: x -> x^3 in GF(8), v-basis."""
    return quad_exhaustive(3, tower.FCUBE3)


# ---------------------------------------------------------------------------
# pairing-form rank decomposition (elimination)
# ---------------------------------------------------------------------------


def pair_decomp(Brows, n):
    """B[u] = mask over v of the form on basis pair (u, v).  Returns
    [(p, qm), ...] with B(a, c) = XOR_k parity(a & p_k) & parity(c & qm_k).
    """
    Rw = list(Brows)
    out = []
    while True:
        u0 = None
        for u in range(n):
            if Rw[u]:
                u0 = u
                break
        if u0 is None:
            break
        qm = Rw[u0]
        v0 = (qm & -qm).bit_length() - 1
        p = bitmask(n, lambda u: (Rw[u] >> v0) & 1)
        for u in bitlist(p, n):
            Rw[u] ^= qm
        out.append((p, qm))
    chk = [0] * n
    for p, qm in out:
        for u in bitlist(p, n):
            chk[u] ^= qm
    assert chk == list(Brows)
    return out


def search_pair3(fld, tower):
    """Rank decomposition of B(r, s) = Tr3(r s^2) on GF(8) v-coords."""
    gmul = fld.mul
    E3 = tower.E3
    B33 = [bitmask(3, lambda j: trace(gmul(E3[i], gmul(E3[j], E3[j])), 3)) for i in range(3)]
    return [[p, qm] for (p, qm) in pair_decomp(B33, 3)]


# ---------------------------------------------------------------------------
# formula evaluation (search_kt's certify_kt, quad_solve's quad_apply)
# ---------------------------------------------------------------------------


def kt_mul_fn(KT):
    ktm = [
        (sum(1 << u for u in Ab), sum(1 << u for u in Bb), sum(1 << u for u in Tb))
        for (Ab, Bb, Tb) in KT
    ]

    def mul(a, bb):
        acc = 0
        for Am, Bm, Tm in ktm:
            if parity(a & Am) and parity(bb & Bm):
                acc ^= Tm
        return acc

    return mul


def certify_kt(KT, fld):
    """Return True iff KT multiplies exactly in GF(2^b) over the poly basis:
    checked on the b x b basis pairs, complete for a bilinear form.
    """
    kt_mul = kt_mul_fn(KT)
    return all(
        kt_mul(1 << i, 1 << j) == fld.mul(1 << i, 1 << j)
        for i in range(fld.n)
        for j in range(fld.n)
    )


def quad_apply(Q, x, n):
    tris, corr = Q
    acc = 0
    for u, w, om in tris:
        if parity(u & x) and parity(w & x):
            acc ^= om
    for m in range(n):
        if parity(corr[m] & x):
            acc ^= 1 << m
    return acc


def run_b12():
    """All formula searches (every one deterministic); returns (tables,
    timings) with tables in constants.json shape.
    """
    fld = FIELD
    tower = Tower(fld)
    tables, tm = {}, {}
    for name, search in (
        ("KT12", lambda: search_kt(fld)),
        ("KT6", lambda: search_kt6(fld, tower)),
        ("KT3", lambda: search_kt3(tower)),
        ("CUBE3Q", lambda: search_cube3q(tower)),
        ("PAIR3", lambda: search_pair3(fld, tower)),
    ):
        t0 = time.time()
        tables[name] = search()
        tm[name + "_s"] = time.time() - t0
    tables["LIN"] = tower.lin_tables()
    return tables, tm


# #########################################################################
# Models
# #########################################################################
# Bit-exact classical models of the circuit (Builder-free): the one Python
# model of its fused-Q BM recursion, and the Psi route (stages 5-7) op for
# op.  The Psi window search runs them.

# =========================================================================
# BM recursion (q-form conditional-swap, fused-Q B bank)
# =========================================================================
# The B bank rotates at bank-1.  A per-round hook instruments the loop.

In [ ]:
%%writefile src/gen_dqi_v6/gu_port.py
# SPDX-License-Identifier: Apache-2.0
# Adapted from andigu/uncompute by Gu et al.; see REPORT.md and ref/gu_README.md.
#!/usr/bin/env python3
"""dqi-reed-solomon discovered circuit: fused-Q truncated Berlekamp-Massey
decode at (m, n, b) = (4095, 70, 12), then a half-degree Bezout pass that
turns the per-point Forney division into a precomputed Psi = 1/sigma'
mod sigma, so the Chien/Forney sweep costs 23 Toffoli per point (root
test 11 + a rank-12 gated pairing 12) instead of 80.  Built natively on
uncompute.Builder.

The BM body runs the (2,2,3)-tower field arithmetic under the measured
window schedule and stores no accepted discrepancy: the scale identity
sigma_0 * DEL == 1 replaces it (README).
The Psi route (stages 5-7) holds the Bezout pair in the syndrome TAIL,
zeroed by extending the in-place omega transform to all 70 rows.

Layout (flat):
  circuit.py         the tables (field, formulas, schedule; loaded from
                    constants.json), the circuit layer (term lists,
                    emitter, Psi-route gadgets, sweep body), and build(),
                    the stage sequence with its phase marks and cost pins.
  constants.json    every searched value: the formula tables, the BM
                    windows (bm_windows), the Psi windows (psi_windows).
  gen_constants.py  reruns every search and writes constants.json; it
                    holds the bit-exact classical models the searches run.
  search.c          the C sampler behind the BM window census.

RUN: uv run python circuit.py -- builds, asserts its cost, and
streams to $UNCOMPUTE_OPS_PATH (else ./ops.bin).  Importing is cheap and
builds nothing; build() feeds the occupancy/liveness profilers.
Evaluate: python tools/eval.py dqi-reed-solomon.

README.md in this folder describes the construction.
"""

import functools
import json
import pathlib
from contextlib import contextmanager

import numpy as np

from port_runtime import (Builder, Journal, GF2n, f2_matinv, f2_transpose, parity,
    declare_registers, linear_in_place, assert_metrics, emit_and_report, load_contract)
import os

HERE = pathlib.Path(__file__).resolve().parent


# =========================================================================
# Field and instance
# =========================================================================

CONTRACT = load_contract(__file__)
POLY = CONTRACT["gf_poly"]
M = CONTRACT["m"]  # code length = nonzero field points
N = CONTRACT["n_synd"]  # syndrome symbols = BM rounds
B = CONTRACT["b"]  # bits per symbol
W = CONTRACT["w_max"]  # unique-decoding radius n/2 = max locator degree
LAW_CUM = CONTRACT["weight_cum_2p16"]  # the grader's 2^16 cumulative weight law
assert M == (1 << B) - 1 and N == 2 * W
NH = W // 2 + 1  # words per Euclid half E', O' (and per cofactor)
SQRT_EXP = (M + 1) // 2  # sqrt(x) = x^SQRT_EXP (2048)

# EXP/LOG tables (gamma = 2): EXP[k] = gamma^k for k < 2M, LOG its inverse.
# ginv is totalized (zero maps to zero), as the tower inversion chain.
FIELD = GF2n(B, POLY)
assert FIELD.gen == 2, "gamma = 2 not primitive?!"
EXP, LOG = FIELD.exp, FIELD.log
gmul, ginv, gpow, gsqrt = FIELD.mul, FIELD.inv, FIELD.pow, FIELD.sqrt
# trace(x) is the absolute trace; trace(x, 3) the GF(8) trace of a GF(8) element.
trace = FIELD.trace


def bitlist(m, n):
    return [i for i in range(n) if (m >> i) & 1]


def bitmask(n, pred):
    return sum(1 << i for i in range(n) if pred(i))


def gf_matinv(Mx):
    """Inverse of a square matrix over GF(2^B) (Gauss-Jordan)."""
    n = len(Mx)
    A = [row[:] + [1 if i == j else 0 for j in range(n)] for i, row in enumerate(Mx)]
    for col in range(n):
        piv = next(r for r in range(col, n) if A[r][col])
        A[col], A[piv] = A[piv], A[col]
        iv = ginv(A[col][col])
        A[col] = [gmul(iv, v) for v in A[col]]
        for r in range(n):
            if r != col and A[r][col]:
                f = A[r][col]
                A[r] = [A[r][c] ^ gmul(f, A[col][c]) for c in range(2 * n)]
    return [row[n:] for row in A]


# =========================================================================
# A. Fixup vector c (contract.json; task README section 3.2:
# counter-mode SHA-256 XOF over c_tag, grader-asserted)
# =========================================================================
_buf = bytes.fromhex(CONTRACT["c_stream_hex"])
CBITS = [(_buf[t >> 3] >> (t & 7)) & 1 for t in range(M * B)]
CMASKJ = [bitmask(B, lambda bp: CBITS[B * j + bp]) for j in range(M)]

# =========================================================================
# B. Frozen multiplication formulas + tower row masks (constants.json
# beside circuit.py, written by gen_constants.py).
# =========================================================================
CONSTS = json.loads(pathlib.Path(os.environ["DQI_CONSTANTS"]).read_bytes())
KT = [tuple(map(tuple, t)) for t in CONSTS["KT12"]]  # GF(2^12) mult: 42 products
KT6 = [tuple(map(tuple, t)) for t in CONSTS["KT6"]]  # GF(64) mult: 15 products
KT3 = [tuple(map(tuple, t)) for t in CONSTS["KT3"]]  # GF(8) mult: 6 products
# x^3 in GF(8): 3 products
CUBE3Q = tuple(map(tuple, CONSTS["CUBE3Q"][0])), tuple(CONSTS["CUBE3Q"][1])
PAIR3 = [tuple(pq) for pq in CONSTS["PAIR3"]]  # Tr3(r s^2) pairing: rank 3
LIN = CONSTS["LIN"]  # the tower's linear maps

# =========================================================================
# C. Omega schedule: in-place triangular product from middle-product
# blocks, each at its field-rank a+b-1 by evaluation-interpolation.
# Constant GF(2^B) maps are free CX networks, so evaluating both
# operands at a+b-1 fixed points and interpolating the results back
# costs only the pointwise quantum-quantum products; the recursion
# below (balanced binary) is the DP optimum of the alias-free schedule
# family: 147 products against the dense triangle's W(W+1)/2 = 630.
# Blocks read syn columns strictly below the rows they write, so the
# hi -> block -> lo emission order stays in-place.
# =========================================================================
_OMC = {1: 0}
_OMS = {1: None}
for _n in range(2, W + 1):
    _best = None
    _bh = None
    for _h in range(1, _n // 2 + 1):
        _v = _OMC[_h] + _OMC[_n - _h]
        if _best is None or _v < _best:
            _best, _bh = _v, _h
    assert _best is not None  # range(1, _n // 2 + 1) is non-empty
    _OMC[_n] = (_n - 1) + _best
    _OMS[_n] = _bh

MACS = []  # (r0, a, c0, b): reads syn cols [c0, c0+b), writes [r0, r0+a)


def _build_tri(lo, n):
    if n == 1:
        return
    h = _OMS[n]
    _build_tri(lo + h, n - h)
    MACS.append((lo + h, n - h, lo, h))
    _build_tri(lo, h)


# omega70: the same in-place transform extended to all N = 2W rows.  Rows
# W..N-1 of sigma_m*S mod z^N vanish on every correct decode (the key
# equation: deg omega < deg sigma <= W), which frees the 35-word syndrome
# TAIL as workspace for the Bezout pair (stage 5).  Emission
# order hi -> lo keeps it in place: the tail triangle (rows W..N-1 from
# rows W..N-2, diagonals 1..W-1), then one off-diagonal middle-product
# block (rows W..N-1 from rows 0..W-1; its Toeplitz diagonals W+1..N-1
# are zero -- sigma has W+1 coefficients -- and are clipped from the
# sigma-side evaluation), then the omega triangle.  The map is unit
# triangular on all N rows, so its literal reverse restores S on EVERY
# path, including approximate-BM paths where the tail is not zero.
_build_tri(W, W)
MACS.append((W, W, 0, W))
_build_tri(0, W)
assert len(MACS) == 2 * (W - 1) + 1
for _r0, _a, _c0, _b in MACS:
    # alias-free in place: every block reads rows strictly below the rows
    # it writes; the triangles span diagonals 1..W-1, the off-diagonal
    # block 1..N-1 (clipped to 1..W at emission)
    assert _c0 + _b == _r0, (_r0, _a, _c0, _b)
    assert _r0 + _a - 1 - _c0 <= (N - 1 if _b == W else W - 1), (_r0, _a, _c0, _b)

_VINV = {}


def vinv(npts):
    """(points, inverse Vandermonde) for evaluation at gamma^0..gamma^(npts-1),
    cached per size.
    """
    if npts not in _VINV:
        pts = [EXP[t] for t in range(npts)]
        _VINV[npts] = (
            pts,
            gf_matinv([[gpow(pts[t], k) for k in range(npts)] for t in range(npts)]),
        )
    return _VINV[npts]


@functools.cache
def cmrows(c):
    """Row masks of the constant multiplication x -> c*x (a free CX network)."""
    return f2_transpose([gmul(c, 1 << j) for j in range(B)], B)


# =========================================================================
# D. (2,2,3) tower: GF(2^12) = GF(64)(t) = GF(8)(s)(t), read top-down
# (gen_constants.py docstring).  A 12-bit tower word is a0 (low 6) + a1 (high 6) t with
# t^2 + t = nu, conjugate xbar = (a0 + a1) + a1 t; a GF(64) word is
# n0 (low 3) + n1 (high 3) s, conjugate Nbar = (n0 + n1) + n1 s.  The
# norm descends both quadratic levels (N = x xbar in GF(64) = a0 a1 +
# linear, M = N Nbar in GF(8) = n0 n1 + linear); the only nonlinear
# inverse left is M -> M^3 (a 3-product quadratic form), M^-1 = (M^3)^2
# a free squaring.  Coordinates: E12[i + 3j + 6k] = v^i s^j t^k; the
# GF(8) projection of a GF(64) word is a bit slice (no row parities).
# Every stage maps 0 to 0, so the zflag armor (monicize) carries over.
# local: gadgets.gf_tower.GF asserts q0 in {4, 8}; this tower is
# GF(8)(s)(t) and its in-circuit chain runs on A0/A1 row projections of
# 12-bit poly-basis words, never on tower-packed words.
# =========================================================================
E12 = LIN["E12"]  # the 12 tower basis elements (poly ints)
A0ROWS = LIN["A0ROWS"]  # poly(12) -> a0 (GF(64) coords)
A1ROWS = LIN["A1ROWS"]  # poly(12) -> a1
A01ROWS = [A0ROWS[r] ^ A1ROWS[r] for r in range(6)]  # a0 + a1 (xbar's constant part)
LNROWS = LIN["LNROWS"]  # poly(12) -> the linear part of N: a0^2 + nu a1^2
LMROWS = LIN["LMROWS"]  # GF(64) coords -> the linear part of M: n0^2 + nu' n1^2
SQ3R = LIN["SQ3R"]  # squaring on GF(8) coords
assert (len(E12), len(A0ROWS), len(A1ROWS), len(LNROWS), len(LMROWS), len(SQ3R)) == (
    12,
    6,
    6,
    6,
    3,
    3,
)
# the tower elements re-derived from the field (the recipe in the
# gen_constants.py docstring); E12 must be their products
_A_el = 2 ^ gpow(2, 64)
_t_el = gmul(2, ginv(_A_el))
_u_el = gpow(2, 65)
_Ap_el = _u_el ^ gpow(_u_el, 8)
_s_el = gmul(_u_el, ginv(_Ap_el))
_v_el = gpow(_u_el, 9)
assert E12 == [
    gmul(gmul(gpow(_v_el, i), gpow(_s_el, j)), gpow(_t_el, k))
    for k in range(2)
    for j in range(2)
    for i in range(3)
], "E12 != v^i s^j t^k"
MINVR = f2_matinv(f2_transpose(E12, B), B)
assert MINVR[0:6] == A0ROWS and MINVR[6:12] == A1ROWS, "A0/A1 rows != inverse basis matrix"

# =========================================================================
# E. Trace dual, free-Z interpolation split, GF(8) fixed pairing form
# =========================================================================
TRT = [trace(x) for x in range(1 << B)]  # absolute trace table
TINV = f2_matinv([bitmask(B, lambda c: TRT[gmul(1 << a, 1 << c)]) for a in range(B)], B)
MU = f2_transpose(TINV, B)
for _a in range(B):
    for _bb in range(B):
        assert TRT[gmul(1 << _a, MU[_bb])] == (1 if _a == _bb else 0)

LAMJ = [0] * M
for _j in range(M):
    _v = 0
    for _bpos in bitlist(CMASKJ[_j], B):
        _v ^= MU[_bpos]
    LAMJ[_j] = _v

# ---- move N sweep points into free Z gates (Vandermonde interpolation) --
KILLJ = [j for j in range(M) if LAMJ[j] == 0]
KILLJ += [j for j in range(M) if LAMJ[j] != 0][: N - len(KILLJ)]
assert len(KILLJ) == N and len(set(KILLJ)) == N
_tgt = [gmul(LAMJ[j], ginv(EXP[j])) for j in KILLJ]
_Ai = gf_matinv([[gpow(EXP[j], i) for i in range(N)] for j in KILLJ])
UCO = [0] * (N + 1)
for _r in range(N):
    _v = 0
    for _c in range(N):
        _v ^= gmul(_Ai[_r][_c], _tgt[_c])
    UCO[_r + 1] = _v


def U_eval(j):
    acc = 0
    for i in range(1, N + 1):
        if UCO[i]:
            acc ^= EXP[(LOG[UCO[i]] + i * j) % M]
    return acc


LAMJ2 = [LAMJ[j] ^ U_eval(j) for j in range(M)]
for _j in KILLJ:
    assert LAMJ2[_j] == 0, ("interpolation failed", _j)
NLIVE = sum(1 for j in range(M) if LAMJ2[j])
ZMASKI = [bitmask(B, lambda bb: TRT[gmul(UCO[i + 1], 1 << bb)]) for i in range(N)]


def _elt3(c):
    w = 0
    for i in bitlist(c, 3):
        w ^= gpow(_v_el, i)
    return w


def _tr3(x):
    t = trace(x, 3)
    assert t in (0, 1)
    return t


# the GF(8) pairing form B(r, s) = Tr3(r * s^2) as PAIR3 spells it: rank 3
for _r in range(8):
    for _s3 in range(8):
        _acc = 0
        for _p, _qm in PAIR3:
            _acc ^= parity(_r & _p) & parity(_s3 & _qm)
        assert _acc == _tr3(gmul(_elt3(_r), gmul(_elt3(_s3), _elt3(_s3)))), (
            "pairing form drifted",
            _r,
            _s3,
        )


def pair_rows(lam):
    """r_a(lam), a < B: parity(r_a & U) = Tr(e_a sqrt(lam U)), lam != 0."""
    assert lam
    sq = [gsqrt(gmul(lam, 1 << c)) for c in range(B)]
    return [bitmask(B, lambda c: TRT[gmul(1 << a_, sq[c])]) for a_ in range(B)]


# =========================================================================
# Constant linear maps of monicize and the Euclid pass
# =========================================================================
# MONB[k] = the (a, b) basis pairs whose product has bit k set
MONB = [[] for _ in range(B)]
for _a in range(B):
    for _b2 in range(B):
        _vv = gmul(1 << _a, 1 << _b2)
        for _k2 in range(B):
            if (_vv >> _k2) & 1:
                MONB[_k2].append((_a, _b2))
SQRT_ROWS = f2_transpose([gsqrt(1 << j) for j in range(B)], B)
SQUARE_ROWS = f2_transpose([gmul(1 << j, 1 << j) for j in range(B)], B)


# =========================================================================
# Truncation schedule
# =========================================================================
# Reversal-stack capacity (tagged slots; overflow shots form error
# class E2, the stack overflow that README.md describes).  Overflow rate
# by cap under the DQI weight law, measured under the derived windows
# (403.2M draws): cap 1 -> 1.02e-4 [1.01e-4, 1.03e-4], cap 2 -> 4.1e-7,
# cap 3 -> 0 seen; max pushes 3.
# Cap 1 saves STKW = 19 wires at the global peak (both slots materialize
# by round 4 and stay live through the sweep) and 19 Toffoli per push
# cascade forward and again in the reverse pass: 2,470 Toffoli and 19
# qubits against cap 2, a 1.5% lower TQ-cost.  The aggregate sign-error
# rate is ~1.4e-4 (E1 3.8e-5 + E2 1.02e-4), 7x under the 1e-3 allowance
# (cap 2: 3.8e-5, 26x).
STK_CAP = 1

# Per-round coefficient windows (MEASURED; constants.json bm_windows,
# derived by gen_constants.py from 403.2M draws under the DQI weight
# law).  IMAXV bounds the discrepancy convolution, KMSV the
# flag-controlled slot swaps, KMMV the update macs.  The rule: the
# smallest per-round windows such that every clipped slot has marginal
# mass < 4e-5; r=0 and r=N-1 are pinned by the circuit.  Events outside
# a window form error class E1, the cutoff crossings of README.md.
_WIN = CONSTS["bm_windows"]
IMAXV = list(_WIN["IMAXV"])
KMSV = list(_WIN["KMSV"])
KMMV = list(_WIN["KMMV"])
assert len(IMAXV) == len(KMSV) == len(KMMV) == N
for _r in range(N):
    # never wider than the reachable envelope, never narrower than
    # round 0's seed path needs
    assert IMAXV[_r] <= min(_r, W, (_r + 1) // 2 + 1)
    assert KMMV[_r] <= KMSV[_r] <= min(_r + 1, W, _r // 2 + 2)
assert IMAXV[N - 1] == W and KMSV[N - 1] == KMMV[N - 1] == W
assert KMSV[0] == 1 and KMMV[0] == 0

# Signed d counter: two's complement, wide enough for the all-zero
# syndrome's monotone climb (d = N-2 after the last increment) -- DWB-1
# magnitude bits + the sign bit dW[DWB-1].
DWB = (N - 2).bit_length() + 1
assert (1 << (DWB - 1)) - 1 >= N - 2
SGN = DWB - 1  # the sign bit index (pqc / ctl anchor)
DMAX = 1 << (DWB - 1)  # two's-complement range of dW
# Stack round tag: TAGB bits hold every pushing round 0..N-1.
TAGB = (N - 1).bit_length()
TAGMASK = (1 << TAGB) - 1
STKW = B + TAGB  # stack slot: B value lanes + the round tag
assert DWB == 8  # MBITS = dW[2:7] and search.c hard-code the 8-bit dW


def caps_schedule():
    """Exact reachable-(d, depth) DP over the d-automaton: d>=0 branches to
    d+1 (ctl=0, needs D==0) or -d (ctl=1); d<0 forces ctl=0 and a PUSH
    (pqc = dW[SGN]), d -> d+1.  r=N-1 pushes unconditionally.  CAPS[r] =
    max reachable stack depth AFTER a push at round r = the emitted
    cascade extent; CAPS[r]==0 => a push is unreachable => the cascade is
    elided.
    """
    states = {(0, 0)}
    caps = [0] * N
    for r in range(N):
        ns = set()
        mx = 0
        for d_, k_ in states:
            if r == N - 1:
                ns.add((d_, k_ + 1))
                mx = max(mx, k_ + 1)
            elif d_ < 0:
                ns.add((d_ + 1, k_ + 1))
                mx = max(mx, k_ + 1)
            else:
                ns.add((d_ + 1, k_))
                ns.add((-d_, k_))
        caps[r] = mx
        states = ns
    return caps


CAPS_FULL = caps_schedule()
assert max(CAPS_FULL) <= W, ("stack cap bound broken", CAPS_FULL)
assert CAPS_FULL[0] == 0 and CAPS_FULL[1] == 0
# reachable depth is W-1 before the unconditional r=N-1 push, W after
assert max(CAPS_FULL[: N - 1]) == W - 1 and CAPS_FULL[N - 1] == W
# CAP truncation: clamp every emitted cascade extent to STK_CAP.
# min(c, STK_CAP) == 0 iff c == 0, so the set of rounds carrying a cascade
# is unchanged; only extents shrink.  Overflow shots (depth would exceed
# STK_CAP) are the failure class E2 (README.md).  With
# STK_CAP = 1 every cascade is the single Dw <-> STK[0] Fredkin level.
CAPS = [min(c, STK_CAP) for c in CAPS_FULL]
assert max(CAPS) == STK_CAP

# Psi-route cofactor windows (cyclic intervals (lo, n) of the 18-word
# registers, array-index frame) per CR op of R: the measured union
# support, recorded only where the op acts (swap: flag = 1; rotation:
# c = 1; product: (q_lo, q_hi) != 0), over 400k weight-law draws plus the
# w = 0 / w = 1 anchors (constants.json psi_windows, searched by
# gen_constants.py).  A support outside a window only mis-sets the
# Psi value (a sign error on that shot): R^-1 mirrors R op for op, so the
# registers are restored exactly either way.
PSI_WIN_IV = {k: {int(j): tuple(v) for j, v in d.items()} for k, d in CONSTS["psi_windows"].items()}
assert sorted(PSI_WIN_IV) == ["mul", "rz", "swap"] and all(
    sorted(d) == list(range(NH)) for d in PSI_WIN_IV.values()
)
PSI_WIN = {(k, jj): v for k, d in PSI_WIN_IV.items() for jj, v in d.items()}


# =========================================================================
# Terms (Builder-free term-list construction)
# =========================================================================
# A term list is the build's unit of arithmetic: a list of normalized
# terms over hard wire ids,
#
#   ('L', ws, c, tws)             tws ^= parity(ws) ^ c      (free CX/X)
#   ('P', wa, ca, wb, cb, tws)    tws ^= (parity(wa) ^ ca) AND
#                                        (parity(wb) ^ cb)   (one CCX)
#
# built by folding the frozen formula tables (section B) through operand
# specs (one wire-set per operand bit).  Em emits a term list and
# erases it.  Everything here is pure construction: no gate is emitted.


def spec_plain(ws):
    return [frozenset((w,)) for w in ws]


def spec_rows(rows, ws):
    return [
        frozenset(ws[j] for j in range(len(ws)) if (rows[r] >> j) & 1) for r in range(len(rows))
    ]


def fold(spec, bits_):
    ws = frozenset()
    for i in bits_:
        ws = ws ^ spec[i]
    return tuple(sorted(ws))


def tmask(Tb):
    """Return a formula product's target index set as a B-bit mask."""
    u = 0
    for jj in Tb:
        u ^= 1 << jj
    return u


def xfold(pairs):
    """Operand spec bit from (word, row mask) pairs: the XOR (not union)
    of the selected wires, so repeated wires cancel.
    """
    s = set()
    for word, row in pairs:
        for j in bitlist(row, B):
            s ^= {word[j]}
    return frozenset(s)


def norm_term(wa, ca, wb, cb, tws):
    if (not wa) and ca == 0:
        return None
    if (not wb) and cb == 0:
        return None
    if not wa:
        return ("L", wb, cb, tws)
    if not wb:
        return ("L", wa, ca, tws)
    if wa == wb:
        if ca == cb:
            return ("L", wa, ca, tws)
        return None
    return ("P", wa, ca, wb, cb, tws)


def bil_terms(rt, table, specA, specB, tgt=None, tws_of=None):
    """Bilinear terms of one frozen formula table over operand specs:
    fold each product's operand bit-sets through the specs, normalize,
    append.  Targets come from tgt (indexed by the table's target set)
    or a custom tws_of(Tb).
    """
    for Ab, Bb, Tb in table:
        tws = tws_of(Tb) if tws_of else [tgt[jj] for jj in Tb]  # ty: ignore[not-subscriptable]  # tgt is given when tws_of is not
        tm = norm_term(fold(specA, Ab), 0, fold(specB, Bb), 0, tws)
        if tm:
            rt.append(tm)


def kt6_terms_poly(rt, specA, specB, tgt12, off):
    """KT6 terms whose GF(64)-coordinate targets land in the 12-bit poly
    word tgt12 through the tower basis: coord j -> E12[j + off].
    """

    def tws_of(Tb):
        pm = 0
        for u in Tb:
            pm ^= E12[u + off]
        return [tgt12[jj] for jj in bitlist(pm, B)]

    bil_terms(rt, KT6, specA, specB, tws_of=tws_of)


def quad_terms(rt, Qf, Xs, tgt, n):
    tris, corr = Qf
    sp = spec_plain(Xs)
    for u, w, om in tris:
        tm = norm_term(
            fold(sp, bitlist(u, n)), 0, fold(sp, bitlist(w, n)), 0, [tgt[m] for m in bitlist(om, n)]
        )
        if tm:
            rt.append(tm)
    for m in range(n):
        if corr[m]:
            rt.append(("L", tuple(sorted(Xs[i] for i in bitlist(corr[m], n))), 0, (tgt[m],)))


def lin_rows_terms(rt, rows, src, tgt):
    """Free linear rows: tgt[r] ^= parity(rows[r] & src)."""
    for rr in range(len(rows)):
        if rows[rr]:
            rt.append(
                ("L", tuple(sorted(src[jj] for jj in bitlist(rows[rr], len(src)))), 0, (tgt[rr],))
            )


def prod_rt(tgt, wa, wb):
    """tgt ^= wa * wb (KT12, 42 products) as a term list."""
    rt = []
    bil_terms(rt, KT, spec_plain(wa), spec_plain(wb), tgt)
    return rt


def one_rt(wa, ca, wb, cb, t):
    """t ^= (wa ^ ca) AND (wb ^ cb) over wire tuples as a one-term list."""
    return [norm_term(tuple(sorted(wa)), ca, tuple(sorted(wb)), cb, [t])]


# ---- AND ladders ----------------------------------------------------------
def ladder_rt(rungs, tree):
    """AND-ladder term lists over rungs [(wire tuple, polarity), ...]:
    tree[i-1] <- AND of the first i+1 rungs, each parity tested against
    its polarity bit (len(rungs) - 1 products).
    """
    (w0, c0), (w1, c1) = rungs[0], rungs[1]
    rts = [one_rt(w0, c0, w1, c1, tree[0])]
    for i in range(2, len(rungs)):
        ws, cc = rungs[i]
        rts.append(one_rt((tree[i - 2],), 0, ws, cc, tree[i - 1]))
    return rts


def eq_rungs(bits, val):
    """Rungs whose ladder top is [bits == val] (val = 0: the zero test)."""
    return [((w,), 1 - ((val >> i) & 1)) for i, w in enumerate(bits)]


# ---- the tower inversion chain -------------------------------------------
CHAIN_WINS = [("nqd", 6), ("m3d", 3), ("s3d", 3), ("yd", 6)]


def norm_chain_terms(src, a0, a1, nq, m3, s3):
    """Term lists for the norm chain on src: N (KT6 bilinear + linear
    rows into nq), M (KT3 on the halves of nq + linear rows into m3),
    M^3 (CUBE3Q into s3).
    """
    rtN = []
    bil_terms(rtN, KT6, a0, a1, nq)
    lin_rows_terms(rtN, LNROWS, src, nq)
    rtM = []
    bil_terms(rtM, KT3, spec_plain(nq[0:3]), spec_plain(nq[3:6]), m3)
    lin_rows_terms(rtM, LMROWS, nq, m3)
    rtS = []
    quad_terms(rtS, CUBE3Q, m3, s3, 3)
    return rtN, rtM, rtS


def chain_rts(src, wins, out):
    """Forward tower inversion-chain term lists for out = inv(src): the
    norm chain, N^-1 = Nbar M^-1 (two KT3 into yd, M^-1 = (M^3)^2 folded
    into the operand spec as SQ3R rows), then x^-1 = xbar N^-1 (two KT6)
    recombined into the 12-bit poly-basis out.  wins are the CHAIN_WINS
    scratch words; rts[:-1] write wins in order, rts[-1] writes out.
    """
    nq, m3, s3, yd = wins
    a0 = spec_rows(A0ROWS, src)
    a1 = spec_rows(A1ROWS, src)
    a01 = spec_rows(A01ROWS, src)
    rtN, rtM, rtS = norm_chain_terms(src, a0, a1, nq, m3, s3)
    m6 = spec_rows(SQ3R, s3)
    n0 = spec_plain(nq[0:3])
    n1 = spec_plain(nq[3:6])
    n01 = [n0[i] ^ n1[i] for i in range(3)]
    rtY = []
    bil_terms(rtY, KT3, n01, m6, yd[0:3])
    bil_terms(rtY, KT3, n1, m6, yd[3:6])
    y = spec_plain(yd)
    rtI = []
    kt6_terms_poly(rtI, a01, y, out, 0)
    kt6_terms_poly(rtI, a1, y, out, 6)
    return [rtN, rtM, rtS, rtY, rtI]


# ---- evaluation/interpolation products -----------------------------------
def polyprod_rt(Qw, Fw, tgt):
    """Evaluation/interpolation product: for the polynomials Q(y) =
    sum_i Qw[i] y^i and F(y) = sum_k Fw[k] y^k (lists of words), XOR
    coefficient n of Q*F into every word in tgt[n] (a list; empty = drop).
    Costs len(Qw) + len(Fw) - 1 products (one scalar product when both
    have one word); operand evaluations and the interpolation rows are
    free CX networks folded into the operand specs and target sets.
    """
    npts = len(Qw) + len(Fw) - 1
    assert len(tgt) == npts
    if npts == 1:
        rt = []
        for T in tgt[0]:
            bil_terms(rt, KT, spec_plain(Qw[0]), spec_plain(Fw[0]), T)
        return rt
    pts, Iv = vinv(npts)
    rt = []
    for t in range(npts):
        cq = [cmrows(gpow(pts[t], i)) for i in range(len(Qw))]
        cf = [cmrows(gpow(pts[t], k)) for k in range(len(Fw))]
        sA = [xfold([(Qw[i], cq[i][bb]) for i in range(len(Qw))]) for bb in range(B)]
        sB = [xfold([(Fw[k], cf[k][bb]) for k in range(len(Fw))]) for bb in range(B)]

        def tws_of(Tb, t=t):
            u = tmask(Tb)
            acc = {}
            for n in range(npts):
                if not tgt[n]:
                    continue
                v = gmul(Iv[n][t], u)
                for T in tgt[n]:
                    for mm in bitlist(v, B):
                        acc[T[mm]] = acc.get(T[mm], 0) ^ 1
            return [w for w in sorted(acc) if acc[w]]

        bil_terms(rt, KT, sA, sB, tws_of=tws_of)
    return [tm for tm in rt if tm[-1]]


def omega_rts(sig, syn):
    """Return the in-place omega70 block term lists (MACS order) for the
    monic sigma bank sig (words sig[1..W]) and the syndrome words syn.
    Each block folds its evaluation maps into the operand parity specs
    (sigma side over the bank, S side over the rows it reads) and its
    interpolation rows into the KT target sets, so the a+b-1 pointwise
    products are the only Toffoli gates.
    """
    out = []
    for r0, a_, c0, b_ in MACS:
        npts = a_ + b_ - 1
        pts, Iv = vinv(npts)
        rt = []
        for t in range(npts):
            cmA = [cmrows(Iv[k][t]) for k in range(npts)]
            cmB = [cmrows(gpow(pts[t], j)) for j in range(b_)]
            # diagonals beyond W are zero (clipped): sigma has W+1 coefficients
            sA = [
                frozenset(
                    sig[1 + k][j] for k in range(min(npts, W)) for j in bitlist(cmA[k][bb], B)
                )
                for bb in range(B)
            ]
            sB = [
                frozenset(
                    syn[c0 + b_ - 1 - j][jb] for j in range(b_) for jb in bitlist(cmB[j][bb], B)
                )
                for bb in range(B)
            ]
            wcoef = [gpow(pts[t], i2) for i2 in range(a_)]

            def om_tws(Tb, wcoef=wcoef, r0=r0, a_=a_):
                u = tmask(Tb)
                tws = []
                for i2 in range(a_):
                    tws.extend(syn[r0 + i2][m] for m in bitlist(gmul(wcoef[i2], u), B))
                return tws

            bil_terms(rt, KT, sA, sB, tws_of=om_tws)
        out.append(rt)
    return out


# =========================================================================
# Emission (term-list emitter, journaled vocabulary, allocators)
# =========================================================================
# Em emits the term lists of the Terms section through a live Builder: emit_mac
# writes a term list forward (one CCX per product; a shared multi-target
# product fans out through the sT scratch and erases it for free), and
# emit_free erases a written term list by demolition measurement + phase
# fixups, 0 Toffoli.  The journaled vocabulary (mac / xor / free / x /
# cx / cswap / swap / swaps, and the ladder, chain and increment helpers
# built on them) runs each step now and records its conjugate in a
# uncompute.bennett.Journal, so a later Journal.reverse() replays the
# conjugates over the same hard wire ids.
#
# Scratch comes from Em's current allocator, an Alloc: it round-trips
# windows through the Builder allocator and logs the window ids a journal
# touches, so a replay can hold exactly those ids (Alloc.replay).


def word_pairs(xs, ys):
    """Bitwise wire pairs of two word lists: word by word, bit 0 first."""
    return [(x[t], y[t]) for x, y in zip(xs, ys) for t in range(B)]


# NOJ never records: unjournaled emission through the journaled vocabulary.
NOJ = Journal()
NOJ.recording = False


class Em:
    """Term-list emitter over the held pivots sA, sB and the fan-out
    scratch sT, with the journaled vocabulary.  The vocabulary records
    into the current journal J (NOJ outside any using() block), and the
    scratch helpers draw from the current allocator A.
    """

    def __init__(self, b, sA, sB, sT, A):
        self.b = b
        self.sA = sA
        self.sB = sB
        self.sT = sT
        self.J = NOJ
        self.A = A

    @contextmanager
    def using(self, J=None, A=None):
        """Make J and/or A the current journal / allocator inside the block."""
        prev = self.J, self.A
        if J is not None:
            self.J = J
        if A is not None:
            self.A = A
        yield
        self.J, self.A = prev

    # ---- the unjournaled core --------------------------------------------
    def _build(self, ws, cc, s):
        if len(ws) == 1 and cc == 0:
            return ws[0], False
        for w in ws:
            self.b.cx(w, s)
        if cc:
            self.b.x(s)
        return s, True

    def _unbuild(self, ws, cc, s, built):
        if not built:
            return
        if cc:
            self.b.x(s)
        for w in reversed(ws):
            self.b.cx(w, s)

    def pivot_pair(self, wa, ca, wb, cb, mid):
        """Emit the shared pivot-operand sandwich: build both pivot operands
        into sA/sB, run the caller's middle 2-qubit op on the built
        pivots (ua, ub), then unbuild in reverse.  One core for the
        single-target CCX in emit_mac, the conditioned-CZ repayment in
        emit_free, and fan_pair's full sT window.
        """
        ua, ba = self._build(wa, ca, self.sA)
        ub, bb2 = self._build(wb, cb, self.sB)
        mid(ua, ub)
        self._unbuild(wb, cb, self.sB, bb2)
        self._unbuild(wa, ca, self.sA, ba)

    def fan_pair(self, wa, ca, wb, cb, mid):
        """Emit the product-pair skeleton over pivot_pair: CCX the pivots'
        AND into the sT fan-out scratch, run the caller's mid(sT) on it,
        then measure sT away with its phase repayment.
        """
        b = self.b

        def sandwich(ua, ub):
            b.ccx(ua, ub, self.sT)
            mid(self.sT)
            m = b.hmr(self.sT)
            b.cz(ua, ub, cond=m)

        self.pivot_pair(wa, ca, wb, cb, sandwich)

    def emit_mac(self, rt):
        b = self.b
        for t in rt:
            k = t[0]
            if k == "L":
                ws, cc, tws = t[1], t[2], t[3]
                for tw in tws:
                    for w in ws:
                        b.cx(w, tw)
                    if cc:
                        b.x(tw)
            else:
                wa, ca, wb, cb, tws = t[1], t[2], t[3], t[4], t[5]
                if len(tws) == 1:
                    self.pivot_pair(wa, ca, wb, cb, lambda ua, ub, tw=tws[0]: b.ccx(ua, ub, tw))
                else:

                    def mid(fan, tws=tws):
                        for tw in tws:
                            b.cx(fan, tw)

                    self.fan_pair(wa, ca, wb, cb, mid)

    def emit_free(self, rt):
        b = self.b
        per = {}
        order = []
        for t in rt:
            for tw in t[-1]:
                if tw not in per:
                    per[tw] = []
                    order.append(tw)
                per[tw].append(t)
        for tw in reversed(order):
            m = b.hmr(tw)
            for t in per[tw]:
                k = t[0]
                if k == "L":
                    for w in t[1]:
                        b.z(w, cond=m)
                    if t[2]:
                        b.neg(cond=m)
                else:
                    self.pivot_pair(t[1], t[2], t[3], t[4], lambda ua, ub: b.cz(ua, ub, cond=m))

    def swap_fn(self, c, pairs):
        """One closure applying cswap(c, x, y) over pairs in order (c None:
        plain swaps).
        """
        b = self.b
        if c is None:

            def fn():
                for x, y in pairs:
                    b.swap(x, y)
        else:

            def fn():
                for x, y in pairs:
                    b.cswap(c, x, y)

        return fn

    # ---- the journaled vocabulary ----------------------------------------
    def mac(self, rt):
        """Write a term list into ZERO targets; the conjugate erases it
        for free.
        """
        self.J.step(lambda: self.emit_mac(rt), lambda: self.emit_free(rt))
        return rt

    def xor(self, rt):
        """XOR-accumulate a term list into live targets: self-inverse."""
        self.J.step(lambda: self.emit_mac(rt))
        return rt

    def free(self, rt):
        """Measurement-erase targets that equal the term list (a claim);
        the conjugate recomputes them.
        """
        self.J.step(lambda: self.emit_free(rt), lambda: self.emit_mac(rt))

    def macs(self, rts):
        for rt in rts:
            self.mac(rt)

    def frees(self, rts):
        """Erase term lists in reverse order."""
        for rt in reversed(rts):
            self.free(rt)

    def x(self, q):
        self.J.step(lambda: self.b.x(q))

    def cx(self, c, t):
        self.J.step(lambda: self.b.cx(c, t))

    def cswap(self, c, x, y):
        self.J.step(lambda: self.b.cswap(c, x, y))

    def swap(self, x, y):
        self.J.step(lambda: self.b.swap(x, y))

    def swaps(self, c, pairs):
        """cswap(c, .) (c None: swap) over pairs as ONE journal step."""
        self.J.step(self.swap_fn(c, pairs))

    def ladder(self, rungs, tree):
        """Emit an AND ladder (ladder_rt) and return its term
        lists; the caller consumes the top and erases with frees().
        """
        rts = ladder_rt(rungs, tree)
        self.macs(rts)
        return rts

    def inc1(self, x, tree):
        """x += 1 (mod 2^len(x)) as a constant ripple: len(x) - 2 CCX
        ladder carries into tree (len(x) - 2 zero wires), each erased by
        measurement right after its use.
        """
        crts = self.ladder([((x[i],), 0) for i in range(len(x) - 1)], tree)
        for j in range(len(x) - 1, 1, -1):
            self.cx(tree[j - 2], x[j])
            self.free(crts[j - 2])
        self.cx(x[0], x[1])
        self.x(x[0])


def chain_open(E, src, out):
    """Open the inversion-chain scratch on E's allocator and build the
    chain term lists for out = inv(src) (chain_rts).
    """
    wins = [E.A.open(name, n) for (name, n) in CHAIN_WINS]
    return wins, chain_rts(src, wins, out)


def chain_close(E, wins):
    for (name, _n), win in zip(reversed(CHAIN_WINS), reversed(wins)):
        E.A.close(name, win)


def chain_fwd(E, src, out):
    wins, rts = chain_open(E, src, out)
    E.macs(rts)
    return wins, rts


def chain_undo(E, wins, rts):
    E.frees(rts)
    chain_close(E, wins)


@contextmanager
def chain_live(E, src, out):
    """out <- inv(src) with the chain intermediates live inside the block;
    exit erases out and the intermediates.
    """
    wins, rts = chain_fwd(E, src, out)
    yield
    chain_undo(E, wins, rts)


@contextmanager
def opened(E, name, n):
    """Open an n-wire scratch window on E's allocator; exit closes it."""
    qs = E.A.open(name, n)
    yield qs
    E.A.close(name, qs)


@contextmanager
def eq_test(E, bits, val):
    """Yield the top wire of a [bits == val] ladder; exit
    erases the ladder.
    """
    with opened(E, "tree", len(bits) - 1) as tr:
        rts = E.ladder(eq_rungs(bits, val), tr)
        yield tr[len(bits) - 2]
        E.frees(rts)


class Alloc:
    """Scratch windows through the Builder allocator: open allocs and
    claims a window, close releases it and frees it in reverse alloc
    order.  Journals replay closures over hard wire ids, so open records
    each window id into the current log set (logging()), and replay()
    re-materializes exactly a logged set via reclaim before a replay.
    """

    def __init__(self, b):
        self.b = b
        self.log = None  # the set open records window ids into, if any
        self.round = None  # BM round tag for bank bookkeeping
        self.bank_log = {}  # round -> bank registers first touched there

    def open(self, name, n):
        qs = self.b.qubits(n)
        if self.log is not None:
            self.log.update(qs)
        self.b.claim(name, qs)
        return qs

    def reopen(self, name, qs):
        """Re-acquire these exact (free) ids as a window."""
        self.b.reclaim(qs)
        self.b.claim(name, qs)

    def close(self, name, qs):
        self.b.release_recycle(name, qs)

    @contextmanager
    def logging(self, ids):
        """Record every window id opened inside the block into the set ids."""
        prev, self.log = self.log, ids
        yield
        self.log = prev

    @contextmanager
    def replay(self, ids, name="jscratch"):
        """Hold exactly the (free) ids inside the block, for a journal replay."""
        pool = sorted(ids)
        if pool:
            self.reopen(name, pool)
        yield
        if pool:
            self.close(name, pool)


class BankReg:
    """One lazy bank register: B wires materialized at the first operand
    touch (allocated, claimed into the bank's set, and logged with the
    touch round).  The fused-Q rotation relabels handles for free, so a
    physical slot's first touch is discovered mechanically instead of
    derived from the clipped-window schedule.  The mirror replay of the
    touch round returns the slot to |0> (post-replay state equals the
    forward state before that round), which is where the reverse pass
    frees it.
    """

    __slots__ = ("A", "name", "round", "width", "wires")

    def __init__(self, A, name, width=B):
        self.A = A
        self.name = name
        self.width = width
        self.wires = None
        self.round = None

    def mat(self):
        if self.wires is None:
            A = self.A
            self.wires = A.b.qubits(self.width)
            A.b.claim(self.name, self.wires)
            self.round = A.round
            A.bank_log.setdefault(A.round, []).append(self)
        return self.wires

    def __getitem__(self, k):
        return self.mat()[k]

    def __iter__(self):
        return iter(self.mat())

    def __len__(self):
        return self.width


# =========================================================================
# Psi route (half-degree Euclid F, R, cofactor recurrence)
# =========================================================================
# README.md, "Division-free Forney recovery".  The half-degree Euclid
# pass F compresses the padded locator halves (E', O') into quotients;
# R replays F backwards (restoring E', O') while the cofactor recurrence
# CR builds the Bezout pair (a, b), A E' + B O' = z^m, so
# Psi = B^2 + z A^2 = 1/sigma_m' at every root.  Every step runs
# through a uncompute.bennett.Journal over hard wire ids: F records into its
# journal JF; R is a deferred journal that replays JF's block segments
# backwards (interleaved with the CR steps), R^-1 replays them forwards,
# and F^-1 replays all of JF backwards.  The allocator logs every scratch
# id these journals touch (PSI_IDS), and R and R^-1..F^-1 hold exactly that
# set, so every replay reuses exactly the forward wire ids.  gen_constants.py holds the bit-exact
# classical model of these gadgets, op for op.


def flag_nz(E, word, q):
    """q ^= [word != 0]: a zero-test ladder, then NOT top into q (free:
    CX + X, one journal step).
    """
    b = E.b
    with eq_test(E, word, 0) as top:

        def fn():
            b.cx(top, q)
            b.x(q)

        E.J.step(fn)


@contextmanager
def inv_word(E, src):
    """Yield out = inv(src) (66 Toffoli), chain intermediates erased for
    free; exit erases out (36 Toffoli: the intermediates are recomputed).
    """
    with opened(E, "inv", B) as out:
        wins, rts = chain_fwd(E, src, out)
        chain_undo(E, wins, rts[:-1])
        yield out
        # the recompute runs rts built on the chain's exact ids
        for (name, _n), w in zip(CHAIN_WINS, wins):
            E.A.reopen(name, w)
        E.macs(rts[:-1])
        E.free(rts[-1])
        E.frees(rts[:-1])
        chain_close(E, wins)


def pad_barrel(E, mbits, ew, ow):
    """Shift both halves up by m = the 5-bit register mbits (non-cyclic:
    the barrel of cswap chains, one per set bit of m).
    """
    for sft in (16, 8, 4, 2, 1):
        ctl = mbits[sft.bit_length() - 1]
        for X in (ew, ow):
            for k in range(NH - 1 - sft, -1, -1):
                E.swaps(ctl, word_pairs([X[k]], [X[k + sft]]))


def psi_pre(E, ew, ow, pre):
    """pre <- [E'_17 == 0]; cswap the halves on pre (odd L with
    sigma_{L-1} = 0).  pre stays live until the literal reverse.
    """
    with eq_test(E, ew[17], 0) as top:
        E.cx(top, pre)
    E.swaps(pre, word_pairs(ew, ow))


def psi_forward(E, f, g, eps, idx):
    """F on the Euclid bank (f, g: lists of 18 words, top-aligned, f[17]
    the pivot): round 1, the block-0 zero-pivot repair, then 17 blocks
    s0..s9.  Mirrors gen_constants.py forward() op for op.  Every free
    erase claim holds whenever f17 != 0 at the elimination, i.e. on every
    correct decode (the pre-swap and padding guarantee deg E' = 17) and on
    every garbage locator the model has seen.  Returns the final (f, g)
    word arrays and the per-block records that build_R consumes: the
    flag / c / q wires and the journal marks where R interleaves the
    cofactor ops (start; swap, after the flag cswap; q, quotients ready;
    c, c set; end).
    """
    f, g = list(f), list(g)
    recs = {}
    # ---- round 1: q1 = g17/f17; g -= q1 f; store q1 in g17's word ----
    mk = {"start": E.J.mark(), "swap": E.J.mark()}  # block 0 swaps on pre, before F
    with inv_word(E, f[17]) as fi:
        with opened(E, "q1", B) as q1:
            E.mac(prod_rt(q1, g[17], fi))
            E.free(prod_rt(g[17], q1, f[17]))  # claim g17 == q1 f17
            E.swaps(None, word_pairs([q1], [g[17]]))  # g17's word <- q1
        mk["q"] = E.J.mark()  # q1 lives in g[17]'s word
        q1w = g[17]
        E.xor([op for k in range(17) for op in prod_rt(g[k], q1w, f[k])])
    g = g[17:] + g[:17]  # rot_up (relabel)
    # ---- block 0: zero-pivot test + repair (degree defect after round 1)
    with eq_test(E, g[17], 0) as top:
        E.cx(top, eps)  # eps <- [g17 == 0]
    mk["c"] = E.J.mark()
    for k in range(17, 1, -1):  # rot_up(g, lo=1) on eps
        E.swaps(eps, word_pairs([g[k]], [g[k - 1]]))
    mk["end"] = E.J.mark()
    recs[0] = {"marks": mk, "c": eps, "q": [q1w]}
    for j in range(1, 18):
        mk = {"start": E.J.mark()}
        f[j - 1], g[j - 1] = g[j - 1], f[j - 1]  # boundary swap (relabel)
        with opened(E, "flag", 1) as (flag,):
            # s0: set flag to [g17 != 0]
            flag_nz(E, g[17], flag)
            # s1: cswap window [j, 17] on flag
            E.swaps(flag, word_pairs([f[k] for k in range(j, 18)], [g[k] for k in range(j, 18)]))
            mk["swap"] = E.J.mark()
            # s2: erase flag against [g17 != 0] (post-swap)
            flag_nz(E, g[17], flag)
        # s3: invert f17 into fi; s9 (the with exit) erases it
        with inv_word(E, f[17]) as fi:
            # s4: q_hi, t, q_lo
            with opened(E, "qh", B) as qh, opened(E, "ql", B) as ql:
                E.mac(prod_rt(qh, g[17], fi))
                if j <= 16:
                    with opened(E, "t", B) as t:
                        trt = E.mac(prod_rt(t, qh, f[16]))
                        rtl = []
                        bil_terms(
                            rtl,
                            KT,
                            [frozenset((g[16][i], t[i])) for i in range(B)],
                            spec_plain(fi),
                            ql,
                        )
                        E.mac(rtl)
                        E.free(trt)
                else:
                    E.mac(prod_rt(ql, g[16], fi))
                mk["q"] = E.J.mark()
                # s5: block elimination: target word g[j-1+n] ^= coef_n of
                # (ql + qh y) * (f[j] + f[j+1] y + ... + f[16] y^(16-j)), n = 0..17-j
                if j <= 16:
                    E.xor(
                        polyprod_rt(
                            [ql, qh],
                            [f[k] for k in range(j, 17)],
                            [[g[j - 1 + n]] for n in range(18 - j)],
                        ),
                    )
                E.free(prod_rt(g[17], qh, f[17]))  # claim g17 == qh f17
                E.free(prod_rt(g[16], ql, f[17]))  # claim g16' == ql f17
                E.swaps(None, word_pairs([qh, ql], [g[17], g[16]]))
            g = g[16:] + g[:16]  # rot_up twice (relabel)
            with opened(E, "c", 1) as (c,):
                # s6: c = [g17 == 0] AND NOT eps
                with eq_test(E, g[17], 0) as top:
                    E.mac(one_rt((top,), 0, (eps,), 1, c))
                mk["c"] = E.J.mark()
                # s7: controlled rot_up(g, lo=j+1) on c; eps ^= c; idx ^= c*j
                for k in range(17, j + 1, -1):
                    E.swaps(c, word_pairs([g[k]], [g[k - 1]]))
                E.cx(c, eps)
                for bp in bitlist(j, 5):
                    E.cx(c, idx[bp])
                # s8: erase c == eps AND [idx == j]
                with eq_test(E, idx, j) as top:
                    E.free(one_rt((top,), 0, (eps,), 0, c))
        mk["end"] = E.J.mark()
        recs[j] = {"marks": mk, "flag": flag, "c": c, "q": [ql, qh]}
    return f, g, recs


def build_R(E, JF, recs, A, Bw, pre, win, eb):
    """R as a deferred Journal: JF's block segments replayed backwards,
    interleaved with the cofactor recurrence CR on the 18-word cyclic
    registers A (a) and Bw (b).  win maps ('mul'|'rz'|'swap', j) to the
    measured union support (lo, n) of the register(s) the op touches
    (array index frame).  R.forward() runs R; R.reverse() runs R^-1.
    Returns (R, a, b), the cofactor word lists in their final labels.
    """
    a, bb = list(A), list(Bw)
    R = Journal()

    def seg_rev(x, y):
        R.defer(lambda: JF.reverse(x, y), lambda: JF.forward(x, y))

    def cr_rz_ctl(c, j):
        # controlled cyclic rotation up by one (new ws[k] = old ws[k-1])
        # of the window words: len-1 word cswaps
        lo, n = win[("rz", j)]
        if n == 0:
            return
        ws = [bb[(lo + i) % NH] for i in range(min(n + 1, NH))]
        hi = [ws[k] for k in range(len(ws) - 1, 0, -1)]
        low = [ws[k - 1] for k in range(len(ws) - 1, 0, -1)]
        R.defer(E.swap_fn(c, word_pairs(hi, low)), E.swap_fn(c, word_pairs(hi[::-1], low[::-1])))

    def cr_mul(qws, j):
        # b <- z^len(qws) b (relabel); a ^= (sum_i q_i z^(i+1)) b_old,
        # with qws listed low -> high (ql, qh for a block; [q1] for round 1)
        b0 = list(bb)
        lo, n = win[("mul", j)]
        if n:
            Fw = [b0[(lo + i) % NH] for i in range(n)]
            tgt = [[a[(lo + k + 1) % NH]] for k in range(len(qws) + n - 1)]
            rt = polyprod_rt(list(qws), Fw, tgt)
            R.defer(lambda: E.emit_mac(rt))
        return b0[-len(qws) :] + b0[: -len(qws)]

    def cr_swap(c, j):
        lo, n = win[("swap", j)]
        ks = [(lo + i) % NH for i in range(n)]
        R.defer(E.swap_fn(c, word_pairs([a[k] for k in ks], [bb[k] for k in ks])))

    for j in range(17, -1, -1):
        r = recs[j]
        mk = r["marks"]
        seg_rev(mk["c"], mk["end"])
        cr_rz_ctl(r["c"], j)
        seg_rev(mk["q"], mk["c"])
        bb = cr_mul(r["q"], j)
        seg_rev(mk["swap"], mk["q"])
        cr_swap(pre if j == 0 else r["flag"], j)
        seg_rev(mk["start"], mk["swap"])
    # undo the pre-swap of the halves too: the sweep's root test needs E'
    # in the E words (psi_pre's swap itself stays live until the literal
    # reverse, which redoes this swap first)
    ew, ow = eb
    R.defer(E.swap_fn(pre, word_pairs(ew, ow)))
    return R, a, bb


def psi_seed(E, h, mbits, A):
    """A[(m+1) % 18] ^= inv(h), m = the 5-bit register mbits (0..17):
    one [m == v] ladder (4 Toffoli) and one controlled 12-bit copy per v
    (12 Toffoli, paid again by the literal reverse).
    """
    with inv_word(E, h) as ih:
        for v in range(NH):
            with eq_test(E, mbits, v) as top:
                # XOR-accumulate, recomputed on the reverse: the TAIL word is zero
                # only on a correct decode, so a measurement erase here would
                # break syndrome restoration on approximate-BM paths
                E.xor([norm_term((top,), 0, (ih[t],), 0, [A[(v + 1) % NH][t]]) for t in range(B)])


# =========================================================================
# Sweep (Chien/Forney root test + Psi pairing, one gated repeat body)
# =========================================================================
# Per point x = gamma^t (error position j = M - t): with E'_k, b_k read as
# if rotated by gamma^(kt) and O'_k, a_k by gamma^((k+SQRT_EXP)t) (parity sets
# over the unrotated words; no rotation is emitted),
#   XOR of the 36 EB words = E'(x) + sqrt(x) O'(x) = x^m sqrt(sigma_m(x)),
#   XOR of the 36 (a, b) words = V = B(x) + sqrt(x) A(x),  Psi(x) = V^2,
# so flag = [XOR of EB == 0] (11 Toffoli) and the Forney bit
# Tr(lam omega(x) / sigma_m'(x)) = Tr(lam omega(x) Psi(x)) = Tr(sqrt(lam U) V)
# = sum_a parity(r_a(lam) . U) V_a (U = omega(x), loaded into som): 12
# flag-gated rank-1 terms, one Toffoli each.  23 Toffoli per point.
#
# Loop form: the words never move, so every CX of every point acts on
# fixed wires; only WHICH stored bits a fan-in reads changes with t.  At
# point t, word k would hold g_k^t times its stored value, so bit bb of the
# rotated word is the parity of the stored bits selected by row bb of
# cmrows(g_k^t).  The body emits the union of each fan-in's possible bits
# in the per-point emission order, each CX data-gated on its membership
# bit at t, so a point's executed ops are exactly its per-point spelling
# (the Em core's pivot_pair / fan_pair / emit_free order).  Every other op
# is gated on the point being live (lam != 0).  A pairing row r_a(lam)
# with a single bit uses that som wire directly (no sA build), so the
# pairing CCX/CZ appear once per case (built sA, or som[c]), each gated on
# its case.

LOG_EB = list(range(NH)) + [(k + SQRT_EXP) % M for k in range(NH)]
LOG_AB = [(k + SQRT_EXP) % M for k in range(NH)] + list(range(NH))
CM_ARR = np.array(
    [[0] * B] + [cmrows(EXP[e]) for e in range(M)], dtype=np.int64
)  # CM_ARR[1 + e, bb]: cmrows(gamma^e)


def rot_member(logs):
    """mem[bb, k, j, t]: bit j of word k is in operand group bb at t."""
    t = np.arange(M, dtype=np.int64)
    mem = np.zeros((B, len(logs), B, M), dtype=bool)
    for k, lg in enumerate(logs):
        rows = CM_ARR[1 + (lg * t) % M]  # [t, bb]
        for j in range(B):
            mem[:, k, j, :] = ((rows >> j) & 1).T.astype(bool)
    return mem


LIVE = np.array([bool(LAMJ2[(M - t) % M]) for t in range(M)])
assert int(LIVE.sum()) == NLIVE


def sweep_plan(b, A, omw, ebw, abw):
    """Return the body as [(gate vector over t, emitter(s_))] in emission order,
    s_ the body's wire handles; a None vector emits ungated (the tree
    window open on allocator A).  Mirrors the per-point form: som fan-ins, the root-test
    ladder (pivot_pair builds, sorted operands), the rank-12 pairing
    (fan_pair), the ladder erase (emit_free, reversed), the som unwind.
    """
    m_om = rot_member(list(range(W))) & LIVE  # som fan-ins
    m_eb = rot_member(LOG_EB) & LIVE  # root-test groups
    m_v = rot_member(LOG_AB) & LIVE  # pairing V groups
    prow = np.zeros((B, M), dtype=np.int64)  # r_a(lam) masks
    for t in range(M):
        if LIVE[t]:
            prow[:, t] = pair_rows(LAMJ2[(M - t) % M])
    single = np.stack([prow == (1 << c) for c in range(B)], axis=1)  # [a, c, t]
    built = ~single.any(axis=1) & LIVE  # [a, t]
    m_sa = np.stack([((prow >> c) & 1).astype(bool) & built for c in range(B)], axis=1)  # [a, c, t]
    # the per-point spelling the body mirrors: every root-test group is
    # non-empty (both pivots build), every V group has >= 2 wires (it builds)
    assert (m_eb.any(axis=(1, 2)) | ~LIVE).all(), "empty root-test group"
    assert (m_v.sum(axis=(1, 2)) >= 2)[:, LIVE].all(), "degenerate V group"
    eb_sorted = sorted(
        (w, k, j) for k, word in enumerate(ebw) for j, w in enumerate(word)
    )  # terms sort operands
    assert len({w for w, _, _ in eb_sorted}) == len(eb_sorted)

    plan = []

    def op(vec, fn):
        plan.append((vec, fn))

    def eb_build(g, s, inverse):
        cands = [(w, m_eb[g, k, j]) for w, k, j in eb_sorted]
        if inverse:
            op(LIVE, lambda s_: b.x(s_[s]))
            cands.reverse()
        for w, v in cands:
            op(v, lambda s_, w=w: b.cx(w, s_[s]))
        if not inverse:
            op(LIVE, lambda s_: b.x(s_[s]))

    def ua(i, s_):
        return s_["sA"] if i == 1 else s_["tr"][i - 2]

    def rung(i, mid):
        """ladder rung i >= 1: wa = group 0 (i == 1, built in sA) or
        tree[i-2] (direct), wb = group i (built in sB).
        """
        if i == 1:
            eb_build(0, "sA", False)
        eb_build(i, "sB", False)
        op(LIVE, mid)
        eb_build(i, "sB", True)
        if i == 1:
            eb_build(0, "sA", True)

    om = [(bb, k, j) for bb in range(B) for k in range(W) for j in range(B)]
    for bb, k, j in om:
        op(m_om[bb, k, j], lambda s_, bb=bb, k=k, j=j: b.cx(omw[k][j], s_["som"][bb]))
    op(None, lambda s_: s_.__setitem__("tr", A.open("tree", B - 1)))
    for i in range(1, B):
        rung(i, lambda s_, i=i: b.ccx(ua(i, s_), s_["sB"], s_["tr"][i - 1]))
    vw = [(k, j) for k in range(len(abw)) for j in range(B)]
    for a_ in range(B):
        cases = [(built[a_], lambda s_: s_["sA"])] + [
            (single[a_, c] & LIVE, lambda s_, c=c: s_["som"][c]) for c in range(B)
        ]
        for c in range(B):
            op(m_sa[a_, c], lambda s_, c=c: b.cx(s_["som"][c], s_["sA"]))
        for k, j in vw:
            op(m_v[a_, k, j], lambda s_, k=k, j=j: b.cx(abw[k][j], s_["sB"]))
        for v, u in cases:
            op(v, lambda s_, u=u: b.ccx(u(s_), s_["sB"], s_["sT"]))
        op(LIVE, lambda s_: b.cz(s_["tr"][B - 2], s_["sT"]))
        op(LIVE, lambda s_: s_.__setitem__("m", b.hmr(s_["sT"])))
        for v, u in cases:
            op(v, lambda s_, u=u: b.cz(u(s_), s_["sB"], cond=s_["m"]))
        for k, j in reversed(vw):
            op(m_v[a_, k, j], lambda s_, k=k, j=j: b.cx(abw[k][j], s_["sB"]))
        for c in reversed(range(B)):
            op(m_sa[a_, c], lambda s_, c=c: b.cx(s_["som"][c], s_["sA"]))
    for i in range(B - 1, 0, -1):
        op(LIVE, lambda s_, i=i: s_.__setitem__("m", b.hmr(s_["tr"][i - 1])))
        rung(i, lambda s_, i=i: b.cz(ua(i, s_), s_["sB"], cond=s_["m"]))
    op(LIVE, lambda s_: A.close("tree", s_["tr"]))
    for bb, k, j in reversed(om):
        op(m_om[bb, k, j], lambda s_, bb=bb, k=k, j=j: b.cx(omw[k][j], s_["som"][bb]))
    return plan


def emit_sweep(b, E, A, omw, ebw, abw):
    """Emit the sweep: one gate-table row of M bits per distinct gate
    vector (the forward and unwind passes share rows; row r gates
    iteration t on bit r * M + t), then the repeat(M) body over the held
    pivots and the som / tree windows of allocator A.
    """
    plan = sweep_plan(b, A, omw, ebw, abw)
    rows = {}
    for vec, _ in plan:
        if vec is not None:
            rows.setdefault(vec.tobytes(), vec)
    rows.setdefault(LIVE.tobytes(), LIVE)
    row_of = {key: r for r, key in enumerate(rows)}
    table = b.declare_table(np.concatenate(list(rows.values())).astype(np.uint8).tolist())
    with b.repeat(M) as it:

        def gate(vec):
            return b.gated(table, it + row_of[vec.tobytes()] * M)

        s_ = {"sA": E.sA, "sB": E.sB, "sT": E.sT, "som": A.open("som", B)}
        for vec, fn in plan:
            if vec is None:
                fn(s_)
            else:
                with gate(vec):
                    fn(s_)
        with gate(LIVE):
            A.close("som", s_["som"])


# =========================================================================
# BM (stage 2 forward, stage 10 journaled reverse)
# =========================================================================
def flag_test(E, src, gate, ctl):
    """ctl <- [src != 0] AND parity(gate) (gate=() folds to the bare
    nonzero test): the tree window, the (B-1)-rung zero-test ladder, the
    flag AND, and the rung teardown.
    local: every zero-test tree here has gadgets.mcx's n-1 Toffoli cost,
    but must hold its flag live across later gates and journal each step
    for the reverse pass; mcx computes and releases inside one call and
    records nothing.
    """
    with eq_test(E, src, 0) as top:
        E.mac(one_rt((top,), 1, gate, 1, ctl))


class BM:
    """The q-form conditional-swap BM forward (stage 2) and its journaled
    reverse (stage 10).  Round 0 and round N-1 are peeled; the general
    round is a sequence of step methods over explicit round state.  run()
    makes the BM journal J current, so every step records
    into it; JROUND holds the
    journal index at the seed start and at every round start.
    """

    def __init__(self, E, A, syn, Dw, dW):
        self.b, self.E, self.A, self.J = E.b, E, A, Journal()
        self.J.recording = False
        self.pools = {}  # round -> window ids its journal touches
        self.syn, self.Dw, self.dW = syn, Dw, dW
        self.SIG = [BankReg(A, "sig") for _ in range(W + 1)]  # sigma_0..sigma_W
        self.BB = [BankReg(A, "bb") for _ in range(W + 2)]  # B bank (fused-Q: wrap at W+1)
        # STK_CAP-slot LIFO push stack, STKW wires per slot: B value lanes + a
        # TAGB-bit round tag.  The tag records the pushing round and gives
        # every round's control flag an erase anchor at the round tail --
        # after the d complement, ctl == dW[SGN] AND NOT [top tag == r] -- so
        # no flag outlives its round.
        self.STK = [BankReg(A, "stk", STKW) for _ in range(STK_CAP)]
        assert max(CAPS) <= len(self.STK), ("push cascade would overrun STK", max(CAPS))
        self.JROUND = [0]

    def round_head(self, r, bb_l):
        """Open round r (inside its bm/rNN phase block): journal mark, the
        round's cascade stack slots, and the fused-Q bank rotation (a
        relabel).  Returns the rotated bank.
        """
        self.A.round = r
        self.A.log = self.pools.setdefault(r, set())
        self.JROUND.append(self.J.mark())
        # Lazy-materialization invariant: a register touched mid-round may
        # only take free-list ids that strictly earlier rounds' windows freed
        # (those return before the descending replay reaches them).  At the
        # bank/ctl touch sites this holds because the round's own freed ids
        # are always reconsumed by the windows still open there; the cascade
        # runs after every window closed, so its stack depths materialize at
        # the round head instead.  reclaim fails closed on any breach.
        for s in self.STK[: CAPS[r]]:
            s.mat()
        return [bb_l[W + 1]] + bb_l[: W + 1]

    def discrepancy(self, r):
        """Dw <- sum_{i <= IMAXV[r]} sigma_i S_{r-i} (the truncated window)."""
        rtD = []
        for i in range(IMAXV[r] + 1):
            bil_terms(rtD, KT, spec_plain(self.SIG[i]), spec_plain(self.syn[r - i]), self.Dw)
        self.E.mac(rtD)

    @contextmanager
    def quotient(self, ctl, bb_l):
        """Yield the round quotient m without a stored accepted discrepancy.
        Invariant (README): sigma_0 * DEL == 1 on every
        correct-path round, DEL the accepted discrepancy of the q-form
        recursion, so the recursion's m = D'/DEL' is
          ctl=0: m = Delta * sigma_0 = p,
          ctl=1: m = DEL_old / Delta = 1/(Delta * sigma_0) = inv(p).
        Exit erases m (a function of p, inv(p), ctl), the chain and inv(p),
        then p.  Dw = Delta on every branch there; p = Delta * sigma_0_old,
        and sigma_0_old now sits in bb_l[0] (ctl=1) or SIG[0] (ctl=0, where
        Delta != 0 forces bb_l[0] == 0); on ctl=1 the update left SIG[0] =
        1/Delta, so Delta*(bb_l[0] + SIG[0]) = p + ctl:
          p == Dw * (bb_l[0] ^ SIG[0]) ^ ctl*[1]   (one KT claim).
        """
        E = self.E
        with opened(E, "pq", B) as pv:
            E.mac(prod_rt(pv, self.Dw, self.SIG[0]))
            with opened(E, "dinv", B) as dv, chain_live(E, pv, dv), opened(E, "m", B) as mq:
                # m = p ^ ctl*(p ^ inv(p)): B free copies + B CCX
                rtM = []
                for t_ in range(B):
                    rtM.append(("L", (pv[t_],), 0, (mq[t_],)))
                    rtM.append(norm_term((ctl,), 0, tuple(sorted((pv[t_], dv[t_]))), 0, (mq[t_],)))
                E.mac(rtM)
                yield mq
                E.free(rtM)
            rtPe = []
            bil_terms(
                rtPe,
                KT,
                spec_plain(self.Dw),
                [frozenset((bb_l[0][j], self.SIG[0][j])) for j in range(B)],
                pv,
            )
            rtPe.append(("L", (ctl,), 0, (pv[0],)))
            E.free(rtPe)

    def bank_swaps(self, r, ctl, bb_l):
        """cswap(ctl) of the sigma and B banks over slots 0..KMSV[r]."""
        for k2 in range(KMSV[r] + 1):
            if r == 1 and k2 == 1:
                # provable identity: both branches leave sigma_1 ==
                # B_1 entering this swap (2-branch enumeration)
                continue
            for t_ in range(B):
                self.E.cswap(ctl, self.SIG[k2][t_], bb_l[k2][t_])

    def update(self, r, k0, mq, bb_l):
        """sigma_k ^= m * B_k over slots k0..KMMV[r] (one KT12 each)."""
        for k2 in range(k0, KMMV[r] + 1):
            self.E.xor(prod_rt(self.SIG[k2], mq, bb_l[k2]))

    def push(self, r, ctl=True):
        """Emit the fused-Q push cascade on pqc = dW[SGN] (d<0 forces ctl=0);
        ctl=False spells it with plain swaps (the unconditional r=N-1
        push, 0 Toffoli, no tag).  Extent CAPS[r] = min(DP extent,
        STK_CAP): the CLAMPED Fredkin ladder still acts on pairwise-distinct
        qubits; on a full stack it rotates STK[STK_CAP-1] into Dw (overflow
        drop, the E2 class); on a not-full stack it is the identical
        shift as the uncapped ladder.  Inter-slot swaps carry all STKW
        lanes; the Dw level is value-only.
        """
        E, STK = self.E, self.STK
        capr = CAPS[r]
        if not capr:
            return
        pqc = self.dW[SGN]

        def sw(x, y):
            if ctl:
                E.cswap(pqc, x, y)
            else:
                E.swap(x, y)

        for i2 in range(capr - 2, -1, -1):
            for t_ in range(STKW):
                sw(STK[i2][t_], STK[i2 + 1][t_])
        for t_ in range(B):
            sw(self.Dw[t_], STK[0][t_])
        if not ctl:
            return
        # tag the incoming entry with its round: the vacated slot's tag lanes
        # are |0> below overflow, so free CX from the push flag writes r
        # exactly (overflow shots are the E2 class)
        for bp in bitlist(r, TAGB):
            E.cx(pqc, STK[0][B + bp])

    def erase_dw(self, ctl):
        """Erase Dw.  Post-cascade Dw == ctl * Delta on every branch (d<0:
        pushed, Dw = 0; ctl=0, d>=0: Delta = 0), and ctl=1 left sigma_0 =
        1/Delta: Dw == ctl * inv(sigma_0), one chain on SIG[0] and a B-term
        claim (stack overflow is the E2 class).
        """
        E = self.E
        with opened(E, "wreg", B) as wr, chain_live(E, self.SIG[0], wr):
            E.free([norm_term((ctl,), 0, (wr[t_],), 0, (self.Dw[t_],)) for t_ in range(B)])

    def close_ctl(self, r, ctl):
        """d complement if ctl, then erase ctl (the caller closes its
        window).  Post-complement dW[SGN] = [ctl OR pushed], and a push at
        this round iff the top tag reads r,
        so ctl == dW[SGN] AND NOT [top tag == r].  Rounds the DP proves
        push-free (and r=0) anchor on dW[SGN] alone, for free.  On a stack
        overflow the rotated-out tag corrupts the compare -- overflow shots
        are already the E2 class.
        """
        E, dW = self.E, self.dW
        for i in range(DWB):
            E.cx(ctl, dW[i])
        if CAPS_FULL[r] == 0:
            E.free([norm_term((dW[SGN],), 0, (), 1, (ctl,))])
        else:
            tag = self.STK[0][B:]  # materialized at round_head
            with opened(E, "eqt", TAGB - 1) as eqw:
                flips = bitlist(~r & TAGMASK, TAGB)
                for bp in flips:
                    E.x(tag[bp])
                eqr = E.ladder([((tag[i],), 0) for i in range(TAGB)], eqw)
                E.free([norm_term((dW[SGN],), 0, (eqw[TAGB - 2],), 1, (ctl,))])
                E.frees(eqr)
                for bp in flips:
                    E.x(tag[bp])

    def inc_d(self):
        """d += 1: the constant +1 ripple on dW (DWB-2 CCX, MBU-erased
        journaled carries).
        local: gadgets.arith.add has no constant-operand form (an all-None
        operand is a no-op) and no journal hook.
        """
        with opened(self.E, "inc", DWB - 2) as ic:
            self.E.inc1(self.dW, ic)

    def round0(self, bb_l):
        """r=0 needs no working discrepancy register: D(0)=S_1 lives in
        syn[0].  ctl = [S_1 != 0] off syn[0], and the round quotient is
        m = inv(S_1) on both branches (the chain maps 0 to 0).
        """
        E, SIG, syn0 = self.E, self.SIG, self.syn[0]
        with self.b.phase("bm/r00"):
            bb_l = self.round_head(0, bb_l)
            with opened(E, "ctl", 1) as (ctl,):
                flag_test(E, syn0, (), ctl)
                with opened(E, "m", B) as mq:
                    wins, rts = chain_fwd(E, syn0, mq)
                    # SIG[0]=|1>, SIG[1]=|0>, bb_l[0]=|0>, bb_l[1]=|1> are KNOWN basis
                    # states: the two register cswaps collapse to 4 free CX.
                    assert KMSV[0] == 1
                    E.cx(ctl, SIG[0][0])
                    E.cx(ctl, bb_l[0][0])
                    E.cx(ctl, SIG[1][0])
                    E.cx(ctl, bb_l[1][0])
                    # post-swap bb_l[0] value = ctl, bb_l[1] value = 1^ctl; m nonzero
                    # only when ctl=1, hence m*bb0 = m (free CX) and m*bb1 = 0 (skip).
                    for t_ in range(B):
                        E.cx(mq[t_], SIG[0][t_])
                    # erase m: post-update m == ctl * sigma_0 on both branches; the
                    # chain intermediates are functions of syn[0]
                    E.free([norm_term((ctl,), 0, (SIG[0][t_],), 0, (mq[t_],)) for t_ in range(B)])
                    chain_undo(E, wins, rts[:-1])
                self.close_ctl(0, ctl)
            self.inc_d()
        return bb_l

    def round(self, r, bb_l):
        """Emit a general round 0 < r < N-1."""
        with self.b.phase(f"bm/r{r:02d}"):
            bb_l = self.round_head(r, bb_l)
            with opened(self.E, "ctl", 1) as (ctl,):
                self.discrepancy(r)
                # ctl is set iff Dw != 0 and d >= 0
                flag_test(self.E, self.Dw, (self.dW[SGN],), ctl)
                with self.quotient(ctl, bb_l) as mq:
                    self.bank_swaps(r, ctl, bb_l)
                    self.update(r, 0, mq, bb_l)
                self.push(r)
                self.erase_dw(ctl)
                self.close_ctl(r, ctl)
            if r < N - 2:
                # +1 always -> d = ctl ? -d : d+1 (at r=N-2 the increment is
                # dropped: the counter has no later consumer and the complement
                # alone arms the erase anchor)
                self.inc_d()
        return bb_l

    def round_last(self, bb_l):
        """r=N-1: no ctl.  m = p, computed straight into the m window; the
        update skips slot 0 (with a length change due (w=W mass) zB occupies
        slots [N-1-r*, W] and never wraps to 0; otherwise M=0); the push is
        unconditional (push(ctl=False)), and r=N-1 writes no tag (no later
        round reads it).
        """
        r = N - 1
        with self.b.phase(f"bm/r{r:02d}"):
            bb_l = self.round_head(r, bb_l)
            self.discrepancy(r)
            with opened(self.E, "m", B) as mq:
                rtM = self.E.mac(prod_rt(mq, self.Dw, self.SIG[0]))
                self.update(r, 1, mq, bb_l)
                # m is a function of Dw and the untouched sigma_0
                self.E.free(rtM)
            assert CAPS[r] == STK_CAP
            self.push(r, ctl=False)

    def run(self):
        """Stage 2: the seed, round 0, rounds 1..N-2, round N-1; the journal
        is frozen afterwards (stages 3..9 are mirrored explicitly).
        """
        E, J, A = self.E, self.J, self.A
        J.recording = True
        A.round = -1  # seed segment: banks touched here free last
        A.log = self.pools.setdefault(-1, set())
        with E.using(J=J, A=A):
            with self.b.phase("bm/seed"):
                E.x(self.SIG[0][0])
                E.x(self.BB[0][0])
            bb_l = self.round0(list(self.BB))
            for r in range(1, N - 1):
                bb_l = self.round(r, bb_l)
            self.round_last(bb_l)
        self.final_bb_l = [bb_l[W + 1]] + bb_l[:W + 1]
        J.recording = False
        self.JROUND.append(J.mark())
        A.round = None
        A.log = None
        # every lazy register must have materialized inside BM: a first touch
        # in a later stage would have no mirror round to free it at
        assert all(reg.wires is not None for reg in self.SIG + self.BB + self.STK), (
            "lazy register untouched during BM"
        )

    def reverse(self):
        """Stage 10, the journaled measurement-based reverse pass.  Every BM
        step went through J with its conjugate (macs record
        release-or-recompute per the emit-time pebbling choice), so
        replaying the conjugates in reverse re-materializes each round's
        intermediates at the mirrored points and the replayed Fredkin
        cascades pop the stack at exactly the right moments.  The journal
        replays closures over hard wire ids: each round's window pool is
        re-materialized with reclaim right before its replay and freed
        right after, and every bank slot is freed after the replay of its
        first-touch round (its state there equals the forward state before
        that round: |0>).  The availability lines up by time symmetry -- an
        id a bank slot took at forward round f is released before the
        replays of rounds < f, exactly when their pools want it back -- and
        reclaim fails closed on any gap.
        """
        b, A, JR = self.b, self.A, self.JROUND
        freed = 0
        for k in range(len(JR) - 2, -1, -1):
            rr = k - 1  # round -1 = the seed
            with b.phase("reverse/seed" if rr < 0 else f"reverse/r{rr:02d}"):
                if JR[k + 1] > JR[k]:
                    with A.replay(self.pools.get(rr, ())):
                        self.J.reverse(since=JR[k], until=JR[k + 1])
                for reg in reversed(A.bank_log.get(rr, [])):
                    A.close(reg.name, reg.wires)
                    freed += 1
        assert freed == len(self.SIG) + len(self.BB) + len(self.STK), (
            "lazy registers leaked past the reverse",
            freed,
        )


# =========================================================================
# (Un)monicize (stages 3 and 9) and the in-place omega70 (stages 4 and 8)
# =========================================================================
# ARMORED: zflag = [sigma_0 == 0] XORed into W0[0] and SIG[0][0] around the
# slot loop makes the scaling invertible for ALL data (scale defaults to 1
# when sigma_0 = 0), so truncation deviations can never make phase garbage.


def w0_pass(E, SIG, mid, W0):
    """One w0 window: open the chain scratch, run the SIG[0] inversion
    chain intermediates forward, apply mid to the W0 term list (emit_mac
    to create W0, emit_free to erase it), then erase the intermediates
    and close the chain scratch.  The create and destroy windows differ
    only in mid.
    """
    wins, rts = chain_open(E, SIG[0], W0)
    E.macs(rts[:-1])
    mid(rts[-1])
    E.frees(rts[:-1])
    chain_close(E, wins)


def mon_slot(E, sig_i, scale, eraser, ymon):
    """sig_i <- scale * sig_i: ymon = scale * sig_i, a demolition erase of
    sig_i against eraser x ymon through Em's measurement-erase core
    (per-target hmr + conditioned-CZ repayment), then swap ymon into
    sig_i.  monicize: (scale, eraser) = (W0, SIG[0]); unmonicize: (SIG[0],
    W0).
    """
    E.emit_mac(prod_rt(ymon, scale, sig_i))
    E.emit_free(
        [
            ("P", (eraser[a_],), 0, (ymon[b_],), 0, (sig_i[k2],))
            for k2 in range(B)
            for (a_, b_) in MONB[k2]
        ]
    )
    for t_ in range(B):
        E.b.swap(ymon[t_], sig_i[t_])


def mon_stage(E, SIG, prefix, slot_range, forward):
    """One whole (un)monicize stage: create W0, arm the zflag armor,
    run the W scaling slots, disarm, erase W0.  monicize and
    unmonicize are the same block up to phase prefix, slot order, and
    the (scale, eraser) pair.
    """
    b = E.b
    # W0 and the zflag ladder (eq_test's two halves) outlive the phase that
    # opens them, so they open and close by hand in their first/last phase.
    with b.phase(prefix + "/w0+"):
        W0 = E.A.open("w0", B)
        w0_pass(E, SIG, E.emit_mac, W0)
    with b.phase(prefix + "/zflag+"):
        tr = E.A.open("tree", B - 1)
        rts = E.ladder(eq_rungs(SIG[0], 0), tr)
        top = tr[B - 2]
        b.cx(top, W0[0])
        b.cx(top, SIG[0][0])
    scale, eraser = (W0, SIG[0]) if forward else (SIG[0], W0)
    for i in slot_range:
        with b.phase(f"{prefix}/s{i:02d}"), opened(E, "ymon", B) as ym:
            mon_slot(E, SIG[i], scale, eraser, ym)
    with b.phase(prefix + "/zflag-"):
        b.cx(top, SIG[0][0])
        b.cx(top, W0[0])
        E.frees(rts)
        E.A.close("tree", tr)
    with b.phase(prefix + "/w0-"):
        w0_pass(E, SIG, E.emit_free, W0)
        E.A.close("w0", W0)


def omega_pass(E, macrt, prefix, order):
    """Emit the omega70 block term lists in order, one phase block each."""
    for bi in order:
        with E.b.phase(f"{prefix}/m{bi:02d}"):
            E.emit_mac(macrt[bi])


def build():
    """Build the circuit and assert its pins.  Returns (builder, Toffoli
    count); the occupancy/liveness profilers consume its wire_log.
    """
    # ---- registers, held scratch, allocators --------------------------------
    b = Builder()
    with b.phase("init"):  # profile-stage blocks (metadata; emit nothing)
        syn = list(declare_registers(b, [(f"s{r:02d}", B) for r in range(N)]).values())
        assert all(list(syn[r]) == list(range(B * r, B * r + B)) for r in range(N)), (
            "syn ids not contiguous"
        )  # qubits Br..Br+B-1

        ANC = []

        def An(n):
            qs = b.qubits(n)
            ANC.extend(qs)
            return qs

        Dw = An(B)  # fused-Q working discrepancy register
        dW = An(DWB)  # two's-complement d counter
        OMW = [syn[i] for i in range(W)]  # omega lives IN PLACE in syn[0..W-1]

        # Ownership wire-sets (claim/release bookkeeping; emits no op).  Every
        # long-lived wire joins exactly one named set with its full id list, so
        # the anonymous pool live - sum(owned) is empty from init to cleanup;
        # tools/occupancy_profile.py validates the discipline fail-closed.
        b.claim("syn", [w for reg in syn for w in reg])
        b.claim("dw", Dw)
        b.claim("dcount", dW)

        # Transient scratch round-trips through the allocator (A.open /
        # A.close).  Only state that persists between windows (discrepancies,
        # stack, controls, counters, the Em pivots) stays held; everything that
        # returns to |0> mid-circuit is freed there, so peak_qubits is the
        # measured max co-live width and the liveness profile shows the true
        # windows.  Journal replays (the BM reverse, the Psi route) run
        # closures over hard wire ids, so A logs the window ids a journal
        # touches and re-materializes exactly that set before its replay
        # (A.replay; reclaim fails closed when any id is not free).
        A = Alloc(b)

        sA = An(1)[0]
        b.claim("pivA", [sA])
        sB = An(1)[0]
        b.claim("pivB", [sB])
        sT = An(1)[0]
        b.claim("fanT", [sT])
        E = Em(b, sA, sB, sT, A)
        bm = BM(E, A, syn, Dw, dW)
        SIG = bm.SIG

    # Width ledger: N * B syn + (B + DWB + 3) held (Dw, dW, pivots) + the
    # lazily-materialized wires ((2W+3) B bank + STK_CAP tagged stack slots)
    # + the late BM/reverse rounds' window tiling.  The peak assert below
    # enforces the total.

    # ---- 1. zlin: free Z part sum_i Tr(u_i S_i) -----------------------------
    with b.phase("zlin"):
        for i in range(N):
            for bb in []:  # variable masks: no fixed syndrome-linear split
                b.z(syn[i][bb])

    # ---- 2. bm: BM forward (q-form conditional-swap, single-chain m) --------
    bm.run()
    BORROW = borrow_template(b, E, A, bm, dW) if os.environ.get("DQI_BORROW") else None

    # ---- 3. monicize: SIG[i] <- inv(sigma_0)*SIG[i] for i=1..W --------------
    mon_stage(E, SIG, "monicize", range(1, W + 1), True)

    # ---- 4. omega70 = sigma_m*S mod z^N, IN PLACE in syn[0..N-1] ------------
    # sigma_m is monic: the triangles' diagonals span sigma_1..sigma_{W-1},
    # the off-diagonal tail block's sigma_1..sigma_W (omega_rts).
    # Rows W..N-1 (the TAIL) end at 0 on every correct decode and host the
    # Bezout pair (stage 5).
    MACRT = omega_rts(SIG, syn)
    omega_pass(E, MACRT, "omega", range(len(MACRT)))

    # ---- 5. euclid: the half-degree Bezout pass (Psi route) -----------------
    # sigma_m = E(z)^2 + z O(z)^2 with E_i = sqrt(sigma_2i), O_i =
    # sqrt(sigma_2i+1): SIG[1..35] are square-rooted IN PLACE (a free linear
    # map) and the Euclid bank EB is E = [ONE, SIG2, .., SIG34], O = [SIG1, ..,
    # SIG35] (ONE is a fresh word holding 1 = E_0).  m = floor((35 - L)/2) =
    # (dW + 2) >> 2 (dW = the BM counter, +2 in place), and the barrel pad
    # shifts both halves up by m words so deg E' = 17 on every correct decode.
    # JSET journals the setup (ONE, the in-place square roots, dW += 2) so
    # stage 7 undoes it with one reverse.
    # PSI_IDS logs every scratch id the Psi journals touch; stages R and 7
    # hold exactly that set around their replays.
    with b.phase("euclid/setup"):
        ONE = A.open("one", B)
        EPS = A.open("zprec", 6)  # eps + 5-bit idx
        PRE = A.open("pre", 1)[0]
        PSI_IDS = set()
        JSET, JPAD, JPRE, JF, JSEED = (Journal(), Journal(), Journal(), Journal(), Journal())
        with E.using(J=JSET), A.logging(PSI_IDS):
            E.x(ONE[0])
            for i in range(1, W + 1):
                JSET.step(
                    lambda r=list(SIG[i]): linear_in_place(b, r, SQRT_ROWS),
                    lambda r=list(SIG[i]): linear_in_place(b, r, SQUARE_ROWS),
                )
            # dW += 2: a constant increment on bits 1..7 (ladder carries, MBU-erased)
            with opened(E, "inc", DWB - 3) as tr:
                E.inc1(dW[1:DWB], tr)
        EW = [ONE] + [list(SIG[2 * i]) for i in range(1, NH)]
        OWD = [list(SIG[2 * i + 1]) for i in range(NH)]
        MBITS = dW[2:7]  # m = (dW + 2) >> 2
    with b.phase("euclid/pad"), E.using(J=JPAD):
        pad_barrel(E, MBITS, EW, OWD)
    with b.phase("euclid/pre"), E.using(J=JPRE), A.logging(PSI_IDS):
        psi_pre(E, EW, OWD, PRE)
    with b.phase("euclid/F"), E.using(J=JF), A.logging(PSI_IDS):
        F_FIN, _G_FIN, RECS = psi_forward(E, EW, OWD, EPS[0], EPS[1:6])

    # ---- seed + R: the Bezout pair (a, b) grows in the TAIL + one EXT word --
    # a[(m+1) % 18] <- inv(h) (h = the final F pivot), then R = the journal
    # reverse of F interleaved with the cofactor recurrence: on exit EB holds
    # (E', O') again and (a, b) = (A, B) with A E' + B O' = z^m, deg <= 17.
    with b.phase("euclid/seed"):
        TAILW = [list(syn[i]) for i in range(W, N)]
        AW = TAILW[:NH]
        with E.using(J=JSEED), A.logging(PSI_IDS):
            psi_seed(E, F_FIN[NH - 1], MBITS, AW)
    with b.phase("euclid/R"):
        with A.replay(PSI_IDS, "psipool"):
            EXT = A.open("ext", B)  # opened under the replay hold: never a PSI_IDS id
            BWD = TAILW[NH:] + [EXT]
            JR, AFIN, BFIN = build_R(E, JF, RECS, AW, BWD, PRE, PSI_WIN, (EW, OWD))
            JR.forward()
        A.close("zprec", EPS)

    # ---- 6. sweep: root flag + rank-12 Psi pairing over the M points --------
    with b.phase("sweep"):
        emit_sweep(b, E, A, OMW, EW + OWD, AFIN + BFIN)

    # ---- 7. euclid_inv: R^-1, unseed, F^-1, un-pre, un-pad, un-setup --------
    # The zprec and psipool holds span R..setup, so they open and close by
    # hand (A.replay's two halves) in the first and last phase.
    PSI_POOL = sorted(PSI_IDS)
    with b.phase("euclid_inv/R"):
        A.reopen("zprec", EPS)
        A.reopen("psipool", PSI_POOL)
        JR.reverse()
        A.close("ext", EXT)
    with b.phase("euclid_inv/seed"):
        JSEED.reverse()
    with b.phase("euclid_inv/F"):
        JF.reverse()
    with b.phase("euclid_inv/pre"):
        JPRE.reverse()
    with b.phase("euclid_inv/pad"):
        JPAD.reverse()
    with b.phase("euclid_inv/setup"):
        JSET.reverse()
        A.close("psipool", PSI_POOL)
        A.close("pre", [PRE])
        A.close("zprec", EPS)
        A.close("one", ONE)

    # ---- 8. omega_restore: exact reversed block list ------------------------
    # each block's targets are disjoint from its operands, so re-emitting the
    # same term list un-XORs it; the reversed block order restores S exactly.
    omega_pass(E, MACRT, "omega_restore", reversed(range(len(MACRT))))

    # ---- 9. unmonicize: SIG[i] <- sigma_0*SIG[i] for i=W..1 -----------------
    mon_stage(E, SIG, "unmonicize", range(W, 0, -1), False)

    # ---- 10. reverse: the journaled BM reverse pass -------------------------
    if BORROW is not None:
        BORROW()
    bm.reverse()

    # ---- 11. cleanup ---------------------------------------------------------
    # Release-before-free through the only teardown path: every workspace set
    # returns its wires (full id lists, reverse claim order) ahead of the
    # frees; the N * B syndrome wires stay owned to the end as the declared
    # registers.
    with b.phase("cleanup"):
        b.release("fanT", [sT])
        b.release("pivB", [sB])
        b.release("pivA", [sA])
        b.release("dcount", dW)
        b.release("dw", Dw)
        b.recycle_all(reversed(ANC))

    # =========================================================================
    # Metrics gate, pre-emission: the pins under the measured window
    # schedule and STK_CAP = 1.
    # =========================================================================
    TOF = assert_metrics(b, __file__)

    return b, TOF


if __name__ == "__main__":
    emit_and_report(build()[0])

In [ ]:
%%writefile src/gen_dqi_v6/run_gu_port.py
# SPDX-License-Identifier: Apache-2.0
"""Build the paper-source port with arbitrary classical phase masks.

The published source's fixed mask and loop emitter are replaced by an
in-place Chien scan. The decoder and all constants are generated from
source for the competition field, never translated from binary circuits.
"""
import argparse, hashlib, json, os, pathlib, time, traceback
os.environ['OMP_NUM_THREADS']='1'
import kickmix as km


def borrow_template(b,E,A,bm,dW):
    """Route the final template's two high zero words into a scratch loan.

    The common completed recurrence leaves words d and d+1, modulo 37,
    zero in the cyclic template bank. Routing these into fixed words 35
    and 36 permits a 24-qubit loan. This is an approximate support claim,
    tested by check_borrow.py and the official verifier, not an assertion
    for all inputs. The inverse route restores the remaining template.
    """
    import gu_port as G
    bb=bm.final_bb_l
    def swaps(v):
        m=(v+2)%37
        first=(m+35)%37;second=(m+36)%37
        pairs=[]
        if first!=35:pairs.append((35,first))
        if second==35:second=first
        if second!=36:pairs.append((36,second))
        return pairs
    def route(reverse=False):
        with b.phase('borrow/unroute' if reverse else 'borrow/route'):
            if os.environ.get('DQI_BORROW')=='affine':
                # Two 8-bit values form an affine line: choose a differing
                # pivot, XOR it into the other differing coordinates and
                # test the remaining seven coordinates. Thus each modulo-37
                # selector needs one six-AND ladder, no stored control bank.
                for m in range(37):
                    pairs=swaps(m-2)
                    if not pairs:continue
                    values=[v&255 for v in range(-1,69) if (v+2)%37==m]
                    assert len(values) in (1,2)
                    if len(values)==2:
                        u,v=values;diff=u^v;pivot=(diff&-diff).bit_length()-1
                        changed=[i for i in range(8) if i!=pivot and (diff>>i)&1]
                        for i in changed:E.cx(dW[pivot],dW[i])
                        bits=[dW[i] for i in range(8) if i!=pivot]
                        target=sum((((u>>i)&1)^(((u>>pivot)&1) if i in changed else 0))<<k
                                   for k,i in enumerate(i for i in range(8) if i!=pivot))
                    else:
                        changed=[];bits=dW;target=values[0]
                    with G.eq_test(E,bits,target) as ctl:
                        for x,y in (reversed(pairs) if reverse else pairs):
                            E.swaps(ctl,G.word_pairs([bb[x]],[bb[y]]))
                    for i in reversed(changed):E.cx(dW[pivot],dW[i])
            elif os.environ.get('DQI_BORROW')=='trie':
                mods=[m for m in range(37) if swaps(m-2)]
                ctlwords=A.open('loan_controls',len(mods))
                controls=dict(zip(mods,ctlwords))
                entries={v&255:(v+2)%37 for v in range(-1,69) if (v+2)%37 in controls}
                def visit(values,bit,parent=None):
                    if bit<0:
                        assert len(values)==1
                        E.cx(parent,controls[entries[values[0]]])
                        return
                    one=[v for v in values if (v>>bit)&1]
                    zero=[v for v in values if not (v>>bit)&1]
                    node=A.open('loan_trie',1)[0]
                    parity=0 if one else 1
                    rt=G.norm_term(() if parent is None else (parent,),1 if parent is None else 0,
                                   (dW[bit],),parity,(node,))
                    E.mac([rt])
                    if one:visit(one,bit-1,node)
                    if zero:
                        if one:
                            if parent is None:E.x(node)
                            else:E.cx(parent,node)
                            rt=G.norm_term(() if parent is None else (parent,),1 if parent is None else 0,
                                           (dW[bit],),1,(node,))
                        visit(zero,bit-1,node)
                    E.free([rt]);A.close('loan_trie',[node])
                visit(list(entries),7)
                for m,ctl in controls.items():
                    pairs=swaps(m-2)
                    for x,y in (reversed(pairs) if reverse else pairs):
                        E.swaps(ctl,G.word_pairs([bb[x]],[bb[y]]))
                visit(list(entries),7)
                A.close('loan_controls',ctlwords)
            else:
                for v in range(-1,69):
                    pairs=swaps(v)
                    if not pairs:continue
                    with G.eq_test(E,dW,v&255) as ctl:
                        for x,y in (reversed(pairs) if reverse else pairs):
                            E.swaps(ctl,G.word_pairs([bb[x]],[bb[y]]))
    route()
    loan=list(bb[35])+list(bb[36])
    A.close('borrowed_template',loan)
    def restore():
        A.reopen('borrowed_template',loan)
        route(reverse=True)
    return restore


def dynamic_sweep(b,E,A,omw,ebw,abw):
    import gu_port as G
    from port_runtime import linear_in_place
    raw=b.k;gf=km.GF2Field(12)
    def arr(qs):return km.array([b.physical[q] for q in qs])
    def sum_into(words,target):
        raw.cx(arr([q for word in words for q in word]),
               arr([q for _ in words for q in target]))
    names=[('scan_root',12),('scan_omega',12),('scan_psi',12),('scan_located',12)]
    val,om,ps,loc=[A.open(name,n) for name,n in names]
    all_words=omw+ebw+abw
    logs=list(range(35))+G.LOG_EB+G.LOG_AB
    beta=gf.pow(gf.primitive_element,4094)
    steps=[gf.pow(beta,e) for e in logs]
    arrays=[arr(reg) for reg in all_words]
    for j in range(4095):
        sum_into(omw,om);sum_into(ebw,val);sum_into(abw,ps)
        linear_in_place(b,ps,G.SQUARE_ROWS)
        with G.eq_test(E,val,0) as root:
            raw.init_and(b.physical[root],arr(om),arr(loc))
            raw.gf2_phase_by_product(arr(loc),arr(ps),mask=b.errors[j],field=gf)
            raw.del_and(b.physical[root],arr(om),arr(loc))
        linear_in_place(b,ps,G.SQRT_ROWS)
        sum_into(abw,ps);sum_into(ebw,val);sum_into(omw,om)
        if j+1<4095:
            for reg,factor in zip(arrays,steps):
                if factor!=1:raw.gf2_imul(factor,target=reg,field=gf)
        if j%256==0:
            p={'phase':'sweep','position':j,'positions':4095,'elapsed':time.monotonic()-b.started,
               'physical_qubits':len(b.physical)}
            print(json.dumps(p),flush=True)
            if b.progress:b.progress.write_text(json.dumps(p,indent=2))
    for reg,factor in zip(arrays,steps):
        if factor!=1:raw.gf2_idiv(gf.pow(factor,4094),target=reg,field=gf)
    for (name,_),qs in reversed(list(zip(names,[val,om,ps,loc]))):A.close(name,qs)


def main():
    ap=argparse.ArgumentParser()
    ap.add_argument('--name',required=True)
    ap.add_argument('--shots',type=int,default=2048)
    ap.add_argument('--salts',nargs='+',default=['dqi-gu-source-smoke-20261023'])
    ap.add_argument('--load',action='store_true')
    ap.add_argument('--derive-only',action='store_true')
    ap.add_argument('--borrow',action='store_true')
    ap.add_argument('--loan-routing',choices=['direct','trie','affine'],default='direct')
    ap.add_argument('--build-only',action='store_true')
    ap.add_argument('--expected-sha256')
    ap.add_argument('--lean',action='store_true')
    ap.add_argument('--stack',type=int,default=1)
    ap.add_argument('--wide',action='store_true')
    ap.add_argument('--stream',action='store_true')
    ap.add_argument('--top-first',action='store_true')
    ap.add_argument('--pebble-last',action='store_true')
    ap.add_argument('--norm-pebble',action='store_true')
    ap.add_argument('--wide-late',action='store_true')
    ap.add_argument('--compact-bm-from',type=int,default=70)
    a=ap.parse_args()
    out=pathlib.Path(os.environ['QCC_OUT'])
    host=pathlib.Path(os.environ.get('QCC_CACHE', 'dqi_cache'))/a.name
    host.mkdir(parents=True,exist_ok=True)
    def report(obj,name='summary.json'):
        (out/name).write_text(json.dumps(obj,indent=2))
        print(json.dumps(obj),flush=True)
    from dqi import verify_submission_dqi as V
    t=time.monotonic()
    if a.load:
        circuit=km.Circuit.from_file(str(host/'dqi.kmb'),format='kmb')
    else:
        import gu_formula_derivation as F
        tables,timing=F.run_b12()
        assert F.certify_kt(tables['KT12'],F.FIELD)
        constants=json.loads((pathlib.Path(__file__).parent/'ref/gu_constants.json').read_text())
        constants.update(tables)
        if a.wide or a.wide_late:
            win=constants['bm_windows']
            for r in range(70):
                if a.wide_late and not a.wide and r<64:continue
                win['IMAXV'][r]=min(r,35,(r+1)//2+1)
                win['KMSV'][r]=min(r+1,35,r//2+2)
        constpath=out/'constants_competition.json'
        constpath.write_text(json.dumps(constants,indent=2))
        report({'field_modulus':F.CONTRACT['gf_poly'],'formulas':{k:len(v) for k,v in tables.items()},
                'derivation_seconds':timing},'formula_summary.json')
        if a.derive_only:return
        os.environ['DQI_CONSTANTS']=str(constpath)
        import gu_port as G
        G.emit_sweep=dynamic_sweep
        G.borrow_template=borrow_template
        if a.stack!=1:
            G.STK_CAP=a.stack
            G.CAPS=[min(c,a.stack) for c in G.CAPS_FULL]
        if a.lean:
            import round2_gadgets as R2
            if a.norm_pebble:report(R2.certify_norm_identity(G),'norm_certificate.json')
            R2.install(G,top_first=a.top_first,pebble_last=a.pebble_last,norm_pebble=a.norm_pebble,
                       compact_from=a.compact_bm_from)
            G.emit_sweep=R2.lean_sweep
            if a.stream:G.emit_sweep=R2.stream_sweep
        if a.borrow:os.environ['DQI_BORROW']=a.loan_routing
        b,tof=G.build()
        (out/'phase_stats.json').write_text(json.dumps(b.phase_stats,indent=2))
        (out/'liveness.json').write_text(json.dumps(b.live_profile,indent=2))
        b.k.free(b.measurement)
        circuit=b.k.finish_circuit()
        circuit.to_file(str(host/'dqi.kmb'),format='kmb')
    summary={'name':a.name,'args':vars(a),'path':str(host/'dqi.kmb'),
             'build_seconds':time.monotonic()-t,'qubits':circuit.num_qubits,
             'static_toffoli':circuit.operation_statistics().num_toffoli_like,
             'num_ops':len(circuit),'bytes':(host/'dqi.kmb').stat().st_size,'verification':{}}
    if a.expected_sha256:
        digest=hashlib.sha256()
        with (host/'dqi.kmb').open('rb') as f:
            for chunk in iter(lambda:f.read(8<<20),b''):digest.update(chunk)
        summary['rebuilt_sha256']=digest.hexdigest()
        summary['expected_sha256']=a.expected_sha256
        summary['reproducible']=digest.hexdigest()==a.expected_sha256
        report(summary)
        assert summary['reproducible'], 'Rebuilt circuit differs from verified artifact'
    report(summary)
    if a.build_only:return
    spec=V.ProblemSpecDQI.benchmark_instance(num_shots=a.shots)
    for salt in a.salts:
        report({'stage':'verify','salt':salt,'shots':a.shots},'progress.json')
        t=time.monotonic()
        try:r=V.verify(circuit,spec,salt=salt,max_workers=int(os.environ.get('QCC_JOBS','1')))
        except Exception as e:
            traceback.print_exc();r={'status':'FAIL','error':str(e)}
        r.update(salt=salt,seconds=time.monotonic()-t)
        summary['verification'][salt]=r
        report(summary)
    report({'stage':'complete'},'progress.json')


if __name__=='__main__':main()

In [ ]:
%%writefile src/gen_dqi_v6/round2_gadgets.py
# SPDX-License-Identifier: Apache-2.0
"""Round-two reversible scheduling changes; original algebra: Gu et al.

The quotient swap schedule, early Euclid quotient parking, and early linear
root erasure were developed for this notebook.
"""
from contextlib import contextmanager


def install(G,top_first=False,pebble_last=False,norm_pebble=False,compact_from=70):
    @contextmanager
    def quotient(self, ctl, bb_l):
        E=self.E
        if self.A.round>=compact_from:
            with compact_quotient(G,self,ctl,bb_l) as mq:yield mq
            return
        if pebble_last and self.A.round==68:
            with pebble_quotient(G,self,ctl,bb_l,full=norm_pebble) as mq:yield mq
            return
        with G.opened(E,'pq',G.B) as pv:
            E.mac(G.prod_rt(pv,self.Dw,self.SIG[0]))
            with G.opened(E,'dinv',G.B) as dv, G.chain_live(E,pv,dv):
                # The quotient occupies p's register; swap back before erasing
                # the inversion scaffold, whose source is the original p.
                pairs=G.word_pairs([pv],[dv])
                E.swaps(ctl,pairs)
                yield pv
                E.swaps(ctl,pairs)
            rt=[]
            G.bil_terms(rt,G.KT,G.spec_plain(self.Dw),
                [frozenset((bb_l[0][j],self.SIG[0][j])) for j in range(G.B)],pv)
            rt.append(('L',(ctl,),0,(pv[0],)))
            E.free(rt)
    G.BM.quotient=quotient
    if pebble_last or compact_from<70:
        original_run=G.BM.run
        def run(self):
            original_run(self)
            # An erased scratch window may be reused by a bank later in the
            # SAME round. Replay already owns that bank and restores it to
            # zero before the earlier scratch operations execute. Hold only
            # the other ids; free the bank once at its normal reverse point.
            for rr,regs in self.A.bank_log.items():
                for reg in regs:self.pools[rr].difference_update(reg.wires)
        G.BM.run=run

    # Preserve every journal mark's meaning: q_hi is parked in g17 before
    # q_lo's temporary window opens, and the recorded cofactor operand follows
    # its new location. No polynomial operation is dropped.
    import inspect
    if norm_pebble:
        G.inv_word=lambda E,src:compact_inverse(G,E,src)
    source=inspect.getsource(G.psi_forward)
    source=source.replace('''            with opened(E, "qh", B) as qh, opened(E, "ql", B) as ql:
                E.mac(prod_rt(qh, g[17], fi))''','''            with opened(E, "qh", B) as qh:
                E.mac(prod_rt(qh, g[17], fi))
                E.free(prod_rt(g[17], qh, f[17]))
                E.swaps(None, word_pairs([qh], [g[17]]))
            qh = g[17]
            with opened(E, "ql", B) as ql:''')
    source=source.replace('''                E.free(prod_rt(g[17], qh, f[17]))  # claim g17 == qh f17
''','')
    source=source.replace('''                E.swaps(None, word_pairs([qh, ql], [g[17], g[16]]))''',
                          '''                E.swaps(None, word_pairs([ql], [g[16]]))''')
    if top_first:
        G.top_poly_rt=lambda qh,ql,fw,tgt,high:top_poly_rt(G,qh,ql,fw,tgt,high)
        source=source.replace('''            qh = g[17]
            with opened(E, "ql", B) as ql:
                if j <= 16:
                    with opened(E, "t", B) as t:
                        trt = E.mac(prod_rt(t, qh, f[16]))
                        rtl = []
                        bil_terms(
                            rtl,
                            KT,
                            [frozenset((g[16][i], t[i])) for i in range(B)],
                            spec_plain(fi),
                            ql,
                        )
                        E.mac(rtl)
                        E.free(trt)
                else:
                    E.mac(prod_rt(ql, g[16], fi))''','''            qh = g[17]
            if j <= 16:
                targets = [[g[j - 1 + n]] for n in range(18 - j)]
                fw = [f[k] for k in range(j,17)]
                E.xor(top_poly_rt(qh,None,fw,targets,True))
            with opened(E, "ql", B) as ql:
                E.mac(prod_rt(ql, g[16], fi))''')
        source=source.replace('''                        polyprod_rt(
                            [ql, qh],
                            [f[k] for k in range(j, 17)],
                            [[g[j - 1 + n]] for n in range(18 - j)],
                        ),''','''                        top_poly_rt(qh,ql,fw,targets,False),''')
        assert 'with opened(E, "t", B)' not in source
    ns=dict(vars(G));exec(source,ns)
    G.psi_forward=ns['psi_forward']


@contextmanager
def compact_quotient(G,bm,ctl,bb_l):
    E=bm.E
    with G.opened(E,'pq',12) as pv:
        E.mac(G.prod_rt(pv,bm.Dw,bm.SIG[0]))
        with G.inv_word(E,pv) as dv:
            pairs=G.word_pairs([pv],[dv]);E.swaps(ctl,pairs)
            yield pv
            E.swaps(ctl,pairs)
        rt=[]
        G.bil_terms(rt,G.KT,G.spec_plain(bm.Dw),
            [frozenset((bb_l[0][j],bm.SIG[0][j])) for j in range(12)],pv)
        rt.append(('L',(ctl,),0,(pv[0],)));E.free(rt)


@contextmanager
def pebble_quotient(G,bm,ctl,bb_l,full=False):
    E=bm.E;A=bm.A
    with G.opened(E,'pq',12) as pv:
        E.mac(G.prod_rt(pv,bm.Dw,bm.SIG[0]))
        dv=A.open('dinv',12)
        wins,rts=G.chain_fwd(E,pv,dv)
        nq,m3,s3,yd=wins
        E.free(rts[2]);E.free(rts[1])
        A.close('s3d',s3);A.close('m3d',m3)
        if full:
            E.free(rts[0]);A.close('nqd',nq)
        pairs=G.word_pairs([pv],[dv]);E.swaps(ctl,pairs)
        yield pv
        E.swaps(ctl,pairs)
        # The inverse output needs only p and yd for MBU. Release its twelve
        # wires BEFORE recomputing the two erased three-bit norm windows.
        E.free(rts[4]);A.close('dinv',dv)
        if full:nq=A.open('nqd_re',6)
        nm=A.open('m3d_re',3);ns=A.open('s3d_re',3)
        nr=G.chain_rts(pv,[nq,nm,ns,yd],dv)
        if full:E.mac(nr[0])
        E.mac(nr[1]);E.mac(nr[2]);E.free(nr[3])
        E.free(nr[2]);E.free(nr[1]);E.free(nr[0] if full else rts[0])
        A.close('s3d_re',ns);A.close('m3d_re',nm)
        A.close('yd',yd);A.close('nqd',nq)
        rt=[]
        G.bil_terms(rt,G.KT,G.spec_plain(bm.Dw),
            [frozenset((bb_l[0][j],bm.SIG[0][j])) for j in range(12)],pv)
        rt.append(('L',(ctl,),0,(pv[0],)));E.free(rt)


def norm_rt(G,src,out):
    rt=[]
    G.bil_terms(rt,G.KT6,G.spec_rows(G.A0ROWS,src),G.spec_rows(G.A1ROWS,src),out)
    G.lin_rows_terms(rt,G.LNROWS,src,out)
    return rt


@contextmanager
def compact_inverse(G,E,src):
    """18-wire inverse schedule using Norm(x^-1) = Norm(x)^-1.

    Forward inverse is still 66 Toffolis. Erasing its output costs 39 rather
    than36, but no twelve-bit output overlaps an eighteen-bit scaffold.
    """
    A=E.A
    wins=[A.open(name,n) for name,n in G.CHAIN_WINS]
    nq,m3,s3,yd=wins
    # The dummy output is only used to construct term lists, never emitted.
    rts=G.chain_rts(src,wins,src)
    E.macs(rts[:4])
    E.free(rts[2]);E.free(rts[1]);E.free(rts[0])
    A.close('s3d',s3);A.close('m3d',m3);A.close('nqd',nq)
    out=A.open('inv',12)
    inv=G.chain_rts(src,wins,out)[4];E.mac(inv)
    E.free(norm_rt(G,out,yd));A.close('yd',yd)
    yield out
    y=A.open('yd_un',6);E.mac(norm_rt(G,out,y))
    dummy=[out[:6],out[:3],out[:3],y]
    E.free(G.chain_rts(src,dummy,out)[4]);A.close('inv',out)
    n=A.open('nqd_un',6);m=A.open('m3d_un',3);s=A.open('s3d_un',3)
    rt=G.chain_rts(src,[n,m,s,y],src)
    E.macs(rt[:3]);E.free(rt[3]);E.frees(rt[:3])
    A.close('s3d_un',s);A.close('m3d_un',m);A.close('nqd_un',n);A.close('yd_un',y)


_TOP_V={}

def certify_norm_identity(G):
    """Exhaust all4096 field elements over the actual emitted term lists."""
    src=list(range(12));wins=[list(range(12,18)),list(range(18,21)),list(range(21,24)),list(range(24,30))]
    out=list(range(30,42));normout=list(range(42,48))
    rts=G.chain_rts(src,wins,out);nr=norm_rt(G,out,normout)
    def emit(memory,rt):
        for term in rt:
            if term[0]=='L':v=term[2]^sum(memory[q] for q in term[1])%2
            else:v=(term[2]^sum(memory[q] for q in term[1])%2)&(term[4]^sum(memory[q] for q in term[3])%2)
            for q in term[-1]:memory[q]^=v
    for v in range(4096):
        memory=[(v>>q)&1 for q in range(12)]+[0]*36
        for rt in rts:emit(memory,rt)
        got=sum(memory[q]<<i for i,q in enumerate(out))
        assert got==G.ginv(v),(v,got,G.ginv(v))
        emit(memory,nr)
        assert [memory[q] for q in wins[-1]]==[memory[q] for q in normout],v
    return {'field_elements':4096,'inverse_correct':True,'norm_inverse_identity':True,
            'method':'evaluate exact Gu term lists and norm_rt on every GF4096 element, including zero'}

def top_poly_rt(G,qh,ql,fw,tgt,high):
    """Evaluate a linear-times-polynomial product at finite points + infinity.

    Emit infinity first so its high coefficient changes g16 before q_lo is
    divided from g16. This removes the redundant twelve-bit top-product copy.
    The remaining interpolation columns never write the high coefficient.
    """
    n=len(fw)+1
    if n not in _TOP_V:
        pts=[G.EXP[t] for t in range(n-1)]
        matrix=[[G.gpow(p,k) for k in range(n)] for p in pts]+[[0]*(n-1)+[1]]
        _TOP_V[n]=(pts,G.gf_matinv(matrix))
    pts,iv=_TOP_V[n]
    rt=[]
    for t in ([n-1] if high else range(n-1)):
        if high:
            sa=G.spec_plain(qh);sb=G.spec_plain(fw[-1])
        else:
            sa=[G.xfold([(ql,1<<b),(qh,G.cmrows(pts[t])[b])]) for b in range(G.B)]
            cf=[G.cmrows(G.gpow(pts[t],k)) for k in range(len(fw))]
            sb=[G.xfold([(fw[k],cf[k][b]) for k in range(len(fw))]) for b in range(G.B)]
        def targets(tb,t=t):
            acc={};u=G.tmask(tb)
            for k in range(n):
                v=G.gmul(iv[k][t],u)
                for word in tgt[k]:
                    for b in G.bitlist(v,G.B):acc[word[b]]=acc.get(word[b],0)^1
            return sorted(q for q,odd in acc.items() if odd)
        G.bil_terms(rt,G.KT,sa,sb,tws_of=targets)
    return [term for term in rt if term[-1]]


def lean_sweep(b,E,A,omw,ebw,abw,stream=False):
    import gu_port as G
    import kickmix as km
    from port_runtime import linear_in_place
    raw=b.k;gf=km.GF2Field(12)
    def arr(qs):return km.array([b.physical[q] for q in qs])
    def sum_into(words,target):
        raw.cx(arr([q for word in words for q in word]),arr([q for _ in words for q in target]))
    words=omw+ebw+abw
    logs=list(range(35))+G.LOG_EB+G.LOG_AB
    beta=gf.pow(gf.primitive_element,4094)
    steps=[gf.pow(beta,e) for e in logs]
    arrays=[arr(reg) for reg in words]
    for j in range(4095):
        val=A.open('scan_root',12);sum_into(ebw,val)
        tr=A.open('tree',11);rts=E.ladder(G.eq_rungs(val,0),tr);root=tr[-1]
        linear=[('L',tuple(w[k] for w in ebw),0,(val[k],)) for k in range(12)]
        E.free(linear);A.close('scan_root',val)
        om=A.open('scan_omega',1 if stream else 12);ps=A.open('scan_psi',12);loc=A.open('scan_located',1 if stream else 12)
        sum_into(abw,ps)
        linear_in_place(b,ps,G.SQUARE_ROWS)
        if stream:
            # Bilinearity permits one evaluator bit at a time. The AND
            # ancilla is repaid before reuse: still twelve Toffolis/point.
            for k in range(12):
                raw.cx(arr([w[k] for w in omw]),arr(om*len(omw)))
                raw.init_and(b.physical[root],arr(om),arr(loc))
                for h in range(12):
                    coeff=gf.mul(1<<k,1<<h)
                    for c in range(12):
                        if (coeff>>c)&1:
                            raw.push_condition(b.errors[j][c])
                            raw.cz(b.physical[loc[0]],b.physical[ps[h]])
                            raw.pop_condition()
                raw.del_and(b.physical[root],arr(om),arr(loc))
                raw.cx(arr([w[k] for w in omw]),arr(om*len(omw)))
        else:
            sum_into(omw,om)
            raw.init_and(b.physical[root],arr(om),arr(loc))
            raw.gf2_phase_by_product(arr(loc),arr(ps),mask=b.errors[j],field=gf)
            raw.del_and(b.physical[root],arr(om),arr(loc))
            sum_into(omw,om)
        linear_in_place(b,ps,G.SQRT_ROWS)
        sum_into(abw,ps)
        A.close('scan_located',loc);A.close('scan_psi',ps);A.close('scan_omega',om)
        A.reopen('scan_root',val);sum_into(ebw,val)
        E.frees(rts);A.close('tree',tr)
        sum_into(ebw,val);A.close('scan_root',val)
        if j+1<4095:
            for reg,factor in zip(arrays,steps):
                if factor!=1:raw.gf2_imul(factor,target=reg,field=gf)
        if j%512==0:
            import json,time
            p={'phase':'lean_sweep','position':j,'elapsed':time.monotonic()-b.started}
            print(json.dumps(p),flush=True)
            if b.progress:b.progress.write_text(json.dumps(p))
    for reg,factor in zip(arrays,steps):
        if factor!=1:raw.gf2_idiv(gf.pow(factor,4094),target=reg,field=gf)

def stream_sweep(b,E,A,omw,ebw,abw):
    return lean_sweep(b,E,A,omw,ebw,abw,stream=True)

In [ ]:
%%writefile src/gen_dqi_v6/ref/gu_classical.py
"""uncompute.classical — classical math shared by circuits and their checks.

Nothing here touches a Builder: these are the pure-Python models and
table-builders whose OUTPUTS enter emitted circuits (lookup tables, linear
layers, test vectors, reference values): GF(2^n) scalar arithmetic and
GF(2) linear-map helpers. One exp/log-table field (``GF2n``) serves every
finite-field circuit. ``poly`` is always the full reduction polynomial
including the x^n term (e.g. 0x13 for GF(16)).
"""


# ---- GF(2^n) scalar arithmetic ---------------------------------------------


def _clmul_mod(a, b, n, poly):
    """Return a * b in GF(2^n) mod ``poly`` (LSB-first schoolbook + reduce)."""
    prod = 0
    for i in range(n):
        if (a >> i) & 1:
            prod ^= b << i
    for bit in range(2 * n - 2, n - 1, -1):
        if (prod >> bit) & 1:
            prod ^= poly << (bit - n)
    return prod & ((1 << n) - 1)


class GF2n:
    """GF(2^n) mod ``poly``, by exp/log tables over a primitive element.

    ``gen`` is the smallest primitive element, so ``gen == 2`` (the element
    x) whenever x is primitive. ``exp[k] = gen^k`` for k in [0, 2*order):
    the table repeats once, so ``exp[log[a] + log[b]]`` needs no modulo.
    ``log[a]`` is the exponent k with gen^k = a, for nonzero a. ``log[0]`` is 0, a
    placeholder: callers test for zero first. Hot loops may index ``exp``
    and ``log`` directly. The constructor raises ValueError when ``poly``
    is not irreducible of degree n. Tables cost O(2^n), so n stays small.
    """

    __slots__ = ("exp", "gen", "log", "n", "order")

    def __init__(self, n, poly):
        self.n = n
        self.order = order = (1 << n) - 1  # multiplicative group order
        for gen in range(1, 1 << n) if poly >> n == 1 else ():
            exp, x = [1], gen
            while x != 1 and len(exp) < order:
                exp.append(x)
                x = _clmul_mod(x, gen, n, poly)
            if x == 1 and len(exp) == order:
                break
        else:
            raise ValueError(f"poly {poly:#x} is not irreducible of degree {n}")
        self.gen = gen
        self.exp = exp + exp
        self.log = [0] * (order + 1)
        for k, v in enumerate(exp):
            self.log[v] = k

    def mul(self, a, b):
        """Return a * b."""
        if a == 0 or b == 0:
            return 0
        return self.exp[self.log[a] + self.log[b]]

    def pow(self, a, e):
        """Return a^e for any integer e. 0^0 is 1; 0^e is 0 otherwise."""
        if a == 0:
            return int(e == 0)
        return self.exp[self.log[a] * e % self.order]

    def inv(self, a):
        """Return a^-1. The inverse is totalized: 0 maps to 0."""
        return self.exp[self.order - self.log[a]] if a else 0

    def sqrt(self, a):
        """Return the unique square root, a^(2^(n-1))."""
        return self.exp[(self.log[a] << (self.n - 1)) % self.order] if a else 0

    def trace(self, a, k=None):
        """Return a + a^2 + a^4 + ... + a^(2^(k-1)).

        ``k=None`` means k = n: the absolute trace, 0 or 1 for every a.
        For a in the subfield GF(2^k), k dividing n, the same sum is the
        subfield trace to GF(2).
        """
        t = 0
        for _ in range(self.n if k is None else k):
            t ^= a
            a = self.mul(a, a)
        return t


# ---- GF(2) linear-map helpers ----------------------------------------------


def parity(x):
    """GF(2) parity of an int's bits."""
    return x.bit_count() & 1


def f2_rows_of(fn, n):
    """Return the rows of the GF(2)-linear map ``fn`` on n-bit ints.

    Row i is a packed int: bit j is set when input bit j feeds output
    bit i (LSB-first, the layout ``gadgets.linear_xor`` takes). The call
    checks linearity over all 2^n inputs and raises AssertionError on a
    nonlinear ``fn``, which would otherwise yield a silently wrong
    matrix.
    """
    cols = [fn(1 << j) for j in range(n)]
    for v in range(1 << n):
        image = 0
        for j in range(n):
            if (v >> j) & 1:
                image ^= cols[j]
        assert fn(v) == image, "map is not GF(2)-linear"
    return [sum((((cols[j] >> i) & 1) << j) for j in range(n)) for i in range(n)]


def f2_matinv(rows, n):
    """Inverse of the n x n GF(2) matrix ``rows`` (LSB-first packed ints)
    by partial-pivot Gauss-Jordan; asserts invertibility.
    """
    A = list(rows)
    I = [1 << i for i in range(n)]
    for col in range(n):
        piv = None
        for r in range(col, n):
            if (A[r] >> col) & 1:
                piv = r
                break
        assert piv is not None, "singular matrix"
        A[col], A[piv] = A[piv], A[col]
        I[col], I[piv] = I[piv], I[col]
        for r in range(n):
            if r != col and ((A[r] >> col) & 1):
                A[r] ^= A[col]
                I[r] ^= I[col]
    return I


def f2_transpose(rows, n):
    """Transpose of the n x n GF(2) matrix ``rows`` (LSB-first packed
    ints).
    """
    return [sum(((rows[r] >> c) & 1) << r for r in range(n)) for c in range(n)]

In [ ]:
%%writefile src/gen_dqi_v6/ref/gu_linear.py
"""GF(2) linear-layer synthesis.

Source: conventional Gauss-Jordan row elimination, emitted as the reversed
elimination schedule — the textbook in-place CNOT/SWAP realization of an
invertible F2 matrix, as used across the finite-field literature. ``linear_xor`` is the textbook
out-of-place form: one CNOT per set matrix entry.
"""


def linear_in_place(b, qs, rows):
    """Apply the invertible GF(2) matrix ``rows`` to register ``qs`` in
    place: qs <- rows * qs. ``rows[i]`` is the packed-int row i (bit j =
    column j, little-endian to match register order).

    Synthesis is leftmost-pivot Gauss-Jordan elimination recorded and
    emitted in reverse: 0 Toffoli, only CX and SWAP; no ancillas. Raises
    (via the identity assert) if the matrix is singular.
    """
    n = len(qs)
    a = list(rows)
    assert len(a) == n
    ops = []
    for col in range(n):
        pivot = next(r for r in range(col, n) if (a[r] >> col) & 1)
        if pivot != col:
            a[col], a[pivot] = a[pivot], a[col]
            ops.append(("swap", col, pivot))
        for r in range(n):
            if r != col and (a[r] >> col) & 1:
                a[r] ^= a[col]
                ops.append(("cx", col, r))
    assert a == [1 << i for i in range(n)], "singular matrix"
    for kind, i, j in reversed(ops):
        (b.swap if kind == "swap" else b.cx)(qs[i], qs[j])


def linear_xor(b, src, dst, rows):
    """XOR the GF(2) matrix-vector product onto a second register:
    dst ^= rows * src. ``rows[i]`` is the packed-int row i (bit j =
    column j), for example from :func:`uncompute.classical.f2_rows_of`.

    Emits one CX per set matrix entry in row-major order: for each dst
    bit i, for each src bit j. 0 Toffoli; no ancillas. ``src`` and
    ``dst`` must be disjoint.
    """
    for i, row in enumerate(rows):
        for j in range(len(src)):
            if (row >> j) & 1:
                b.cx(src[j], dst[i])

In [ ]:
%%writefile src/gen_dqi_v6/ref/gu_constants.json
{
 "KT12": [
  [
   [
    0,
    8,
    9
   ],
   [
    0,
    8,
    9
   ],
   [
    0,
    7,
    10,
    11
   ]
  ],
  [
   [
    6,
    7,
    9,
    10
   ],
   [
    6,
    7,
    9,
    10
   ],
   [
    1,
    2,
    3,
    4,
    8,
    9,
    11
   ]
  ],
  [
   [
    0,
    6,
    7,
    8,
    10
   ],
   [
    0,
    6,
    7,
    8,
    10
   ],
   [
    0,
    1,
    2,
    3,
    4,
    7,
    8,
    9,
    10
   ]
  ],
  [
   [
    0,
    1,
    2,
    3,
    4,
    5,
    7,
    8,
    10,
    11
   ],
   [
    0,
    1,
    2,
    3,
    4,
    5,
    7,
    8,
    10,
    11
   ],
   [
    0,
    1,
    4,
    5,
    7,
    9,
    11
   ]
  ],
  [
   [
    6,
    8,
    9
   ],
   [
    6,
    8,
    9
   ],
   [
    1,
    2,
    3,
    4,
    5,
    6,
    7,
    11
   ]
  ],
  [
   [
    0,
    1,
    2,
    3,
    4,
    5,
    6,
    7,
    9,
    10,
    11
   ],
   [
    0,
    1,
    2,
    3,
    4,
    5,
    6,
    7,
    9,
    10,
    11
   ],
   [
    0,
    2,
    3,
    6,
    9
   ]
  ],
  [
   [
    0,
    2,
    3,
    5,
    7,
    8,
    10
   ],
   [
    0,
    2,
    3,
    5,
    7,
    8,
    10
   ],
   [
    0,
    1,
    3,
    5,
    8,
    9,
    10,
    11
   ]
  ],
  [
   [
    1,
    2,
    4,
    5,
    11
   ],
   [
    1,
    2,
    4,
    5,
    11
   ],
   [
    2,
    4,
    6,
    8,
    9,
    10
   ]
  ],
  [
   [
    0,
    1,
    3,
    4,
    7,
    8,
    10,
    11
   ],
   [
    0,
    1,
    3,
    4,
    7,
    8,
    10,
    11
   ],
   [
    0,
    1,
    2,
    3,
    4,
    5,
    6,
    11
   ]
  ],
  [
   [
    0,
    1,
    3,
    4,
    6,
    8,
    11
   ],
   [
    0,
    1,
    3,
    4,
    6,
    8,
    11
   ],
   [
    1,
    4,
    8
   ]
  ],
  [
   [
    1,
    2,
    4,
    5,
    6,
    10
   ],
   [
    1,
    2,
    4,
    5,
    6,
    10
   ],
   [
    3,
    10,
    11
   ]
  ],
  [
   [
    0,
    2,
    3,
    5,
    8,
    10,
    11
   ],
   [
    0,
    2,
    3,
    5,
    8,
    10,
    11
   ],
   [
    1,
    3,
    4,
    8,
    10,
    11
   ]
  ],
  [
   [
    5,
    6,
    7,
    9,
    11
   ],
   [
    5,
    6,
    7,
    9,
    11
   ],
   [
    0,
    3,
    5,
    6,
    7,
    8,
    11
   ]
  ],
  [
   [
    7,
    8,
    11
   ],
   [
    7,
    8,
    11
   ],
   [
    0,
    1,
    2,
    4,
    6,
    7,
    9,
    10
   ]
  ],
  [
   [
    5,
    6,
    8,
    9
   ],
   [
    5,
    6,
    8,
    9
   ],
   [
    1,
    2,
    3,
    4,
    5,
    8,
    9,
    10,
    11
   ]
  ],
  [
   [
    0,
    4,
    6,
    7,
    9,
    11
   ],
   [
    0,
    4,
    6,
    7,
    9,
    11
   ],
   [
    1,
    3,
    6,
    8,
    9,
    11
   ]
  ],
  [
   [
    2,
    3,
    5,
    8,
    9,
    10
   ],
   [
    2,
    3,
    5,
    8,
    9,
    10
   ],
   [
    1,
    2,
    4,
    6,
    10,
    11
   ]
  ],
  [
   [
    0,
    2,
    3,
    4,
    5,
    6,
    7,
    8,
    10,
    11
   ],
   [
    0,
    2,
    3,
    4,
    5,
    6,
    7,
    8,
    10,
    11
   ],
   [
    2,
    3,
    4,
    8,
    9,
    10
   ]
  ],
  [
   [
    1,
    2,
    3,
    4,
    7,
    8,
    10,
    11
   ],
   [
    1,
    2,
    3,
    4,
    7,
    8,
    10,
    11
   ],
   [
    0,
    1,
    2,
    3,
    4,
    5,
    6,
    9,
    10
   ]
  ],
  [
   [
    3,
    7,
    8,
    9,
    10
   ],
   [
    3,
    7,
    8,
    9,
    10
   ],
   [
    0,
    1,
    2,
    6,
    11
   ]
  ],
  [
   [
    1,
    2,
    4,
    9,
    11
   ],
   [
    1,
    2,
    4,
    9,
    11
   ],
   [
    3,
    4,
    5,
    9,
    10,
    11
   ]
  ],
  [
   [
    0,
    1,
    2,
    3,
    6,
    8,
    9,
    10
   ],
   [
    0,
    1,
    2,
    3,
    6,
    8,
    9,
    10
   ],
   [
    1,
    4,
    5,
    6,
    8,
    10
   ]
  ],
  [
   [
    2,
    5,
    7
   ],
   [
    2,
    5,
    7
   ],
   [
    0,
    3,
    5,
    9,
    11
   ]
  ],
  [
   [
    0,
    1,
    3,
    5,
    6,
    7,
    8,
    9,
    10
   ],
   [
    0,
    1,
    3,
    5,
    6,
    7,
    8,
    9,
    10
   ],
   [
    0,
    1,
    3,
    4,
    6,
    8,
    9,
    10,
    11
   ]
  ],
  [
   [
    0,
    2,
    3,
    4,
    5
   ],
   [
    0,
    2,
    3,
    4,
    5
   ],
   [
    0,
    1,
    3,
    5,
    8,
    10,
    11
   ]
  ],
  [
   [
    2,
    3,
    5,
    6,
    8,
    11
   ],
   [
    2,
    3,
    5,
    6,
    8,
    11
   ],
   [
    6,
    9,
    10
   ]
  ],
  [
   [
    0,
    4,
    6,
    8,
    11
   ],
   [
    0,
    4,
    6,
    8,
    11
   ],
   [
    0,
    1,
    3,
    5,
    6,
    8,
    9,
    11
   ]
  ],
  [
   [
    1,
    2,
    4,
    6,
    7,
    8,
    9,
    10,
    11
   ],
   [
    1,
    2,
    4,
    6,
    7,
    8,
    9,
    10,
    11
   ],
   [
    1,
    3,
    5,
    6,
    10,
    11
   ]
  ],
  [
   [
    3,
    6,
    9,
    10,
    11
   ],
   [
    3,
    6,
    9,
    10,
    11
   ],
   [
    1,
    2,
    3,
    4,
    5,
    7,
    8,
    9
   ]
  ],
  [
   [
    1,
    2,
    3,
    4,
    7,
    8
   ],
   [
    1,
    2,
    3,
    4,
    7,
    8
   ],
   [
    2,
    4,
    6,
    7,
    8,
    9,
    10,
    11
   ]
  ],
  [
   [
    0,
    1,
    3,
    5,
    6,
    7,
    8,
    9,
    10,
    11
   ],
   [
    0,
    1,
    3,
    5,
    6,
    7,
    8,
    9,
    10,
    11
   ],
   [
    0,
    2,
    4,
    7,
    9,
    10,
    11
   ]
  ],
  [
   [
    2,
    5,
    8,
    9,
    10
   ],
   [
    2,
    5,
    8,
    9,
    10
   ],
   [
    2,
    4,
    7,
    8,
    11
   ]
  ],
  [
   [
    0,
    1,
    2,
    3,
    6,
    7,
    11
   ],
   [
    0,
    1,
    2,
    3,
    6,
    7,
    11
   ],
   [
    0,
    8,
    9,
    10
   ]
  ],
  [
   [
    0,
    2,
    5,
    8,
    9,
    10
   ],
   [
    0,
    2,
    5,
    8,
    9,
    10
   ],
   [
    1,
    3,
    4,
    8,
    11
   ]
  ],
  [
   [
    3,
    4,
    11
   ],
   [
    3,
    4,
    11
   ],
   [
    1,
    3,
    4,
    5,
    8,
    9
   ]
  ],
  [
   [
    0,
    2,
    3,
    4,
    5,
    8,
    9,
    10,
    11
   ],
   [
    0,
    2,
    3,
    4,
    5,
    8,
    9,
    10,
    11
   ],
   [
    5,
    9,
    11
   ]
  ],
  [
   [
    1,
    4,
    7,
    8,
    9
   ],
   [
    1,
    4,
    7,
    8,
    9
   ],
   [
    1,
    2,
    4,
    5,
    8,
    9,
    11
   ]
  ],
  [
   [
    2,
    3,
    10,
    11
   ],
   [
    2,
    3,
    10,
    11
   ],
   [
    0,
    1,
    5,
    7
   ]
  ],
  [
   [
    1,
    2,
    3,
    4,
    7,
    8,
    9,
    10,
    11
   ],
   [
    1,
    2,
    3,
    4,
    7,
    8,
    9,
    10,
    11
   ],
   [
    0,
    2,
    4,
    7,
    8,
    9,
    11
   ]
  ],
  [
   [
    0,
    1,
    2,
    4,
    5,
    7,
    10
   ],
   [
    0,
    1,
    2,
    4,
    5,
    7,
    10
   ],
   [
    0,
    2,
    4,
    5,
    7,
    8
   ]
  ],
  [
   [
    2,
    4,
    10
   ],
   [
    2,
    4,
    10
   ],
   [
    2,
    3,
    5,
    9
   ]
  ],
  [
   [
    0,
    1,
    5,
    7
   ],
   [
    0,
    1,
    5,
    7
   ],
   [
    0,
    3,
    4,
    7,
    8,
    9
   ]
  ]
 ],
 "KT6": [
  [
   [
    0,
    1,
    2,
    5
   ],
   [
    0,
    1,
    2,
    5
   ],
   [
    3
   ]
  ],
  [
   [
    1,
    2,
    3,
    5
   ],
   [
    1,
    2,
    3,
    5
   ],
   [
    0,
    2,
    3,
    4,
    5
   ]
  ],
  [
   [
    0,
    3
   ],
   [
    0,
    3
   ],
   [
    0,
    2,
    4,
    5
   ]
  ],
  [
   [
    0,
    4,
    5
   ],
   [
    0,
    4,
    5
   ],
   [
    0
   ]
  ],
  [
   [
    3
   ],
   [
    3
   ],
   [
    1,
    2,
    3
   ]
  ],
  [
   [
    0,
    3,
    4,
    5
   ],
   [
    0,
    3,
    4,
    5
   ],
   [
    0,
    1,
    2,
    3
   ]
  ],
  [
   [
    0,
    2,
    3,
    5
   ],
   [
    0,
    2,
    3,
    5
   ],
   [
    2,
    3
   ]
  ],
  [
   [
    2,
    4,
    5
   ],
   [
    2,
    4,
    5
   ],
   [
    3,
    4
   ]
  ],
  [
   [
    0,
    3,
    4
   ],
   [
    0,
    3,
    4
   ],
   [
    2,
    4
   ]
  ],
  [
   [
    0,
    1,
    3,
    4,
    5
   ],
   [
    0,
    1,
    3,
    4,
    5
   ],
   [
    1,
    3
   ]
  ],
  [
   [
    1,
    4
   ],
   [
    1,
    4
   ],
   [
    1,
    3,
    5
   ]
  ],
  [
   [
    0,
    3,
    5
   ],
   [
    0,
    3,
    5
   ],
   [
    5
   ]
  ],
  [
   [
    2,
    3,
    5
   ],
   [
    2,
    3,
    5
   ],
   [
    0,
    4
   ]
  ],
  [
   [
    1,
    3,
    5
   ],
   [
    1,
    3,
    5
   ],
   [
    0,
    1,
    2,
    5
   ]
  ],
  [
   [
    1,
    2
   ],
   [
    1,
    2
   ],
   [
    1,
    2,
    4,
    5
   ]
  ]
 ],
 "KT3": [
  [
   [
    0
   ],
   [
    0
   ],
   [
    0,
    1,
    2
   ]
  ],
  [
   [
    1
   ],
   [
    1
   ],
   [
    0,
    2
   ]
  ],
  [
   [
    2
   ],
   [
    2
   ],
   [
    0
   ]
  ],
  [
   [
    0,
    1
   ],
   [
    0,
    1
   ],
   [
    1
   ]
  ],
  [
   [
    1,
    2
   ],
   [
    1,
    2
   ],
   [
    0,
    1
   ]
  ],
  [
   [
    0,
    2
   ],
   [
    0,
    2
   ],
   [
    2
   ]
  ]
 ],
 "CUBE3Q": [
  [
   [
    1,
    2,
    6
   ],
   [
    1,
    4,
    2
   ],
   [
    2,
    4,
    1
   ]
  ],
  [
   7,
   2,
   4
  ]
 ],
 "PAIR3": [
  [
   1,
   1
  ],
  [
   2,
   6
  ],
  [
   4,
   4
  ]
 ],
 "LIN": {
  "E12": [
   1,
   2677,
   1970,
   228,
   566,
   1180,
   2041,
   919,
   3758,
   2733,
   3851,
   1630
  ],
  "A0ROWS": [
   1217,
   456,
   680,
   912,
   508,
   3032
  ],
  "A1ROWS": [
   1386,
   2892,
   742,
   1696,
   3470,
   1906
  ],
  "LNROWS": [
   1769,
   472,
   320,
   3540,
   3998,
   604
  ],
  "SQ3R": [
   1,
   4,
   6
  ],
  "LMROWS": [
   41,
   52,
   14
  ]
 },
 "bm_windows": {
  "IMAXV": [
   0,
   1,
   2,
   2,
   3,
   3,
   4,
   4,
   5,
   5,
   6,
   6,
   7,
   7,
   8,
   8,
   9,
   9,
   10,
   10,
   11,
   11,
   12,
   12,
   13,
   13,
   14,
   14,
   15,
   15,
   16,
   16,
   17,
   17,
   18,
   18,
   19,
   19,
   20,
   20,
   21,
   21,
   22,
   22,
   23,
   23,
   24,
   24,
   25,
   25,
   26,
   26,
   27,
   27,
   28,
   28,
   29,
   29,
   30,
   30,
   31,
   31,
   32,
   32,
   33,
   33,
   33,
   34,
   34,
   35
  ],
  "KMSV": [
   1,
   2,
   2,
   3,
   3,
   4,
   4,
   5,
   5,
   6,
   6,
   7,
   7,
   8,
   8,
   9,
   9,
   10,
   10,
   11,
   11,
   12,
   12,
   13,
   13,
   14,
   14,
   15,
   15,
   16,
   16,
   17,
   17,
   18,
   18,
   19,
   19,
   20,
   20,
   21,
   21,
   22,
   22,
   23,
   23,
   24,
   24,
   25,
   25,
   26,
   26,
   27,
   27,
   28,
   28,
   29,
   29,
   30,
   30,
   31,
   31,
   32,
   32,
   33,
   33,
   33,
   34,
   34,
   35,
   35
  ],
  "KMMV": [
   0,
   1,
   1,
   2,
   2,
   3,
   3,
   4,
   4,
   5,
   5,
   6,
   6,
   7,
   7,
   8,
   8,
   9,
   9,
   10,
   10,
   11,
   11,
   12,
   12,
   13,
   13,
   14,
   14,
   15,
   15,
   16,
   16,
   17,
   17,
   18,
   18,
   19,
   19,
   20,
   20,
   21,
   21,
   22,
   22,
   23,
   23,
   24,
   24,
   25,
   25,
   26,
   26,
   27,
   27,
   28,
   28,
   29,
   29,
   30,
   30,
   31,
   31,
   32,
   32,
   33,
   33,
   34,
   34,
   35
  ]
 },
 "psi_windows": {
  "mul": {
   "17": [
    0,
    0
   ],
   "16": [
    1,
    1
   ],
   "15": [
    2,
    3
   ],
   "14": [
    3,
    4
   ],
   "13": [
    4,
    5
   ],
   "12": [
    5,
    6
   ],
   "11": [
    6,
    7
   ],
   "10": [
    7,
    8
   ],
   "9": [
    8,
    9
   ],
   "8": [
    9,
    10
   ],
   "7": [
    10,
    11
   ],
   "6": [
    11,
    12
   ],
   "5": [
    12,
    13
   ],
   "4": [
    13,
    14
   ],
   "3": [
    14,
    15
   ],
   "2": [
    15,
    16
   ],
   "1": [
    16,
    17
   ],
   "0": [
    0,
    18
   ]
  },
  "rz": {
   "17": [
    0,
    0
   ],
   "16": [
    0,
    0
   ],
   "15": [
    2,
    2
   ],
   "14": [
    3,
    3
   ],
   "13": [
    4,
    4
   ],
   "12": [
    5,
    5
   ],
   "11": [
    6,
    6
   ],
   "10": [
    7,
    7
   ],
   "9": [
    8,
    8
   ],
   "8": [
    9,
    9
   ],
   "7": [
    10,
    10
   ],
   "6": [
    11,
    11
   ],
   "5": [
    12,
    12
   ],
   "4": [
    13,
    13
   ],
   "3": [
    14,
    14
   ],
   "2": [
    15,
    15
   ],
   "1": [
    16,
    16
   ],
   "0": [
    17,
    17
   ]
  },
  "swap": {
   "17": [
    1,
    1
   ],
   "16": [
    2,
    2
   ],
   "15": [
    3,
    4
   ],
   "14": [
    4,
    5
   ],
   "13": [
    5,
    6
   ],
   "12": [
    6,
    7
   ],
   "11": [
    7,
    8
   ],
   "10": [
    8,
    9
   ],
   "9": [
    9,
    10
   ],
   "8": [
    10,
    11
   ],
   "7": [
    11,
    12
   ],
   "6": [
    12,
    13
   ],
   "5": [
    13,
    14
   ],
   "4": [
    14,
    15
   ],
   "3": [
    15,
    16
   ],
   "2": [
    16,
    17
   ],
   "1": [
    0,
    18
   ],
   "0": [
    0,
    17
   ]
  }
 }
}

In [ ]:
%%writefile src/gen_dqi_v6/ref/gu_contract.json
{
 "_provenance": "Task-defining derived constants shared by grader.py and every circuit. grader.py re-derives all of them from its inline definitions and asserts equality with this file at import. c_stream_hex = counter-mode SHA-256 over c_tag (block i = SHA-256(tag || u32le(i))), truncated to 6143 bytes = the 49140-bit fixup vector c (bit k = bit k%8 of byte k/8; the top 4 bits of the last byte are unused); weight_cum_2p16 = the exact DQI weight law |w_k|^2 of Jordan et al. (2025), arXiv:2408.08292, at this instance (m=4095, l=35, p=2016/4096), quantized to a /2^16 grid by largest-remainder rounding (entry k = P(weight <= k) * 2^16).",
 "gf_poly": 4331,
 "m": 4095,
 "n_synd": 70,
 "b": 12,
 "w_max": 35,
 "c_tag": "kickmix/rs-decode-4095/fixup-c/v1",
 "c_stream_hex": "0ab02275411d99401427df155f38df3d98dc8cd245edf6c545c44fbdfa580586048e10430b242a866358f792f5d4bf309aa3b5df457ec1be714f477bb3674d87bfe6f1e5be30e94aebbc685caf06703c256d6fde56cf1a71a44b26dd3a7f4d39f6ee5a39ed36d49a8e7c160a82ce95c221a9fbc6191b503d5c06976bb53878ace4318e6992c5f4932f6671be9cb705bf20019f44b90345f3b313d959710822c65345a2e2ec8bc270038e5582285d76fe1f53be1cccfe381d130665f33d1269c7aa3dd34878ea8d00369625db16895818731b0694f61f915d05c3f9b62b3008ce551248aa11fb39a8ea189aadb70f573e5f0ef48b8bb73acd01aad27897b0d5b18284284238aca81eed501a557a884a7091224220db6414a2b7c794c187ef6a883e7b375c5b0b0b1497d28aa0f35cfce019afeba54fbb98dc521625e100829b51b15495d8fdf1cfe2175d92668f20a62456ce4ef718b26360f2e83f44de3d452b8a131aac6dca7ef463c96510ca780ff5f76dd3e45894651052e7fae51261b65be4d020647db7f45b233d3bbb4d5941e0b7c303ceb613f3c2ed9b11127a1858f12416d6c957bc5cc58e7f1cb655790578fc64db0720c70190f95ed1c662b81b5e29405e9fa3d8e3ed252853155b5b487dd1c3c8b72771712cb8980b4844211d343db065d60d8e31040e0ebd116bd0a434b38eeede887faad8f4f2a616040b6d605648772342a892f8c571728216671979de87050e3d6a76888b24c3111f257fdb13e5ee92fcc2d851abe85ee3bd80c57404144c2a6fe4049a4a9e8ec74a4642cc8c8eb760b801f14692371add6298127eda4508e4ceb9cfd363d1b55bee8157b116f12c938d5b3f37e147c57f065155e7a4c290fbb4f616acabf0a8ade65294356083304276d56d35790d8bd9b7103f70276ff89d187c0fff73a8fe23cdab99d92f4a176725d43ee8043ce2bb852ec84f834419c4a2ef968aab729a62f09ca081e5041af453a53d7ab399c41cff68e56d30e04bdd4aca3eae09ea16ff96fc3f00fe200aee77e275a3b05f2d6c623b746950fd294fc60bb42826113566cab8358d1a435bdd025cb1905a301f7dd71957c9359f73feb93af6325b84a6628b5097d1ef7d6a8c05949ec66b7a682fcc69ce76c3d736d2e1db88c89c012af8565fd7cfbdcc1f9cb26c73d6af71a4020d6b5edab892539c62e98d81bd7ba114f960e45cc78ca2946d77cec0d232a8bf397e99d9e112c5a576887f718850303d3aba9d958b55b913c9d310a20ff7e946688e5f8f9f01230dc26900594e93581a93a08a2d294d7c9349dc211ef66b729810e3bac545d94aec168270882dbd225bce7af21585fd6d572ccb3ded7903f0211145fb0132d56c07a4834fdbe997c99ed61d9bf391754c75726fcf4896af7d205cabebafee5d8d34a9f7d091ce5557a7c984cbd73076dbc76526668476a9b55c610bb68614be2662ae4e9d4cd823c6fabd7ccf627e672e56dff4555987168ba7b87be4b69cfcd78f6e3a6a2d6faf054b2c5787bbcc92688bdbe59ba6f0925b9d38922df8398022f27dac378b1e19b616eb65215350707341357c5cc4f006b3e12b073a03a1bd8edd06af7e31562effd6ba689130ea72f39106d075d09312625897741a7a82f00a0f5f0b91618ed22da815ad6a3e446e7621b9889c96eedc287785751985ee5622ecd80bfe2324893ba8656c290b3faa2b7c8726515f5e605809a7ae4b6e04fb3fd6ef0165eb01b0b0e6d8dc38b3bb7e7f18a6c983611e8ee601ce55a8897aa244a68d3ddcc1241a28d039874c1c352fbd99e505eb00d5e490d2b033fd4e8472a94a8ef947766e5881e1bba7513e899785b82148119f06676350d4ddb8f35905c53b0e05781d5ea6aa819c5152e4b9d3e4662dfbb5295467af2246af9b3743229f5f4a13edaae3860a99309a7fbd7a5c91d9b373f0cc597e30b9194cb5b3d491c27c6cb227f71939d0d836c588830e39e3e6b8436bcc9af6c76f07513b37cdf63ef37e1c1045a5d333a0c19c3960d02e9cbc2c0b1c196bab8c8ebdb7267dc1daa55e4820a3f351c0aa0e97850d9c8bc887faf4805cff561e93b4c1a25bdb8dd100180360855298b7f7e64bdcfc158a5fef4bdd811832c8e6266accde6f332df161e7ddb2d1627546c966da622d0a6a1a045251dda529b4e10fdd14f9f3fc94ea45ea3b29a56564ec9d30fd8c0f736b5ab7d7a1a45175347d391328170d559dc0da5c6d661a4513bfab803fa1eed3d2c658575ac5de52e6ae74a1addfdcb2d598e96826fd625147491b9bfa2ba270d531600ce2791e32e7fdf4fd7ec1152ee6596894c6e917c6f94e1e79094a82bdac1d5ae4b10c7062256cbc754c2fdc6546080c7242eeca80418cb10fb089f721e9e3552b2f9d0686533d9113d338883f246ac8e5e8028108797952b041ed934c65ec9d541c0d4766682e260e735627b2463d899c1bffe99c9337cf959e2947d01a91cd2d88951029cbc4726991190d1fae04117573905c8619b0afaca7163ef533e9c3bf8c279602488cdc0636c114e28697799b0242cbb310c3f557b0d9e6428ebe3d635d201d5ee183ece5400056943ba890e490a219f4e33bed5809bfea2c74138bfcb3aec67fca982aca9890de419c6054cb0c95f0504c01000fd05e86efab211b4fc7a722712c4f483d72b440065ed3869230623924e3254b2ebb7ffa29f4beea6bba9dc32f3572f78b749fba1540009a0b657aae24fea3d77241ce786ac68618c44c84a19b5f650c1217be19568c28ce5a95316de929f113b876c480456e73eaf94d1c5e1abe81d1f70ade42b87df32f0f7dc56520e7cca6679efe320c0ec1340e9a3f249871f8c7bc36c2e57920e5a2fe96917adf1c9c56809a733e7179246429092cac1a9ca5ffe668fb23a69579de104d3e08e86ce1b0f7bed3abf800608fd02853221cdcafab1ff8dd6eb9aa6577af77df9b3e774a62094f23a6cda60b64a54d1577c1fe0526087b86b3aac84d67db4a937bac13469a330e1fc28974cd2d60c07221de0980b1d46a7102317a381f951c8f0029ab2199b0f5a3fbbb5947cd25c291bff0dd37b635628d2882ff3ed73ad8bdeff3c9444ff97ebe21bb543cc5c3f7f1a0608c5ae28b6a5b591f43895cc088a078ed465f0dd8b289219ee1d1d4df353087b2e96d18c400ce7a8ac5d1e30e3920c5aba753d67e198bd51c57fb90f20ccd518f44a3162213e50f67f712e16d5fab5cbe32f585a817b10cb3e68f53c71c04d9c5a400be6701178f1681a9dc073511118eb668819290697a180380eaae3b68af08fddefdfe233f8cdda37eb90ae6d625b0119d54722ba4ccf62d2b44b272aae1d98ab954130e59e7240cc72dd9566459a2af5dca2ad2663dc4b6b8a48dc68b33ac575ffbd46da82f1bc31223c7e280853a6a2c6667152a3875a6d77f41a54a1f9f6e518f50e6a4e1ae06e81e77fd5461fe87516818a9a67a388e6b8ef41b03d521fb1034a59ce4ada4037844b815f00fe1560aa615052a02c2ec5d288bb441b905aa8fd78c1978632a4288d010b4d2357ec48a2535b827ebab108f2eba026a136043874363916cbd300a02b7624163bd7d2dc4688de111a94d5bce2c0f9b332d3b0f647ca94359d80c7785709942fb553165eb24e09489ba8fcbcb8992b89ab24f715dab250fdb51ccfcf0c2f5aded499a106c45fabd22542baa0c722e78c3e8196136a8ff93c46a1bda8222b6119a90c61fce032b382c52b6f308160d6190c85dabd93b633291969ddcb30b13e3bbf401dcd656944523181086bf5cede9ce66e756c3190df3dcdf268fb9b07dc642d2497625c7f87625780a537318fb5f719d12c384d3baa14094de38c440784b0036ef5e4ec873486ccb35baeab9185f9f5a53227e2e61540c8df2db7428a74aab497e16f4ec9c6ff63083339ad26d2e674b33d7f737f3a391302fa8fa367e8b035d78bc9db74145c5e6d338c8706bf0f70f7f39402d6c29d69760985f4ab54c30b1df29c6d26d3d7ecfd02749718c1decc5ecd7f8b5b007d3a533568217aba921f1bb56bf3c0ec82f1f4ae28acab500338de9a704893b5a33965a295a37f11649a90ce06b37ca0086a7f600e92a3d92a0462726bef2b5cf16818d28ff44cbc6742c4d8c5db383d3e35767c0849797bc2d1d54b3d733128bcf61a2c1f81534abce1170909592c2a4318ab53dcdf6b2c722cf992502ee3d08d460540359df9120a36e0414cbd9e144ad4c62c51f8b5873c3d9c6d6c1ee2e7e58bc40a9eabed857f9c53e5c5174237f5ba8dbe14e6219fb2f85b6364f4c81c295852a54401b57ee9354589283509e880decb0e6aa301a1def95506622fc0d37b6b91ebf5483cf4ea53ec2d452a37b440825beb96070c9d5b9c01485ffa779ba8b06c308b5e50185e335cb4016c4ea86c77878b08fdda99c0bc7376c3845b8c563ea2baff72fe7bf4e5bf2a7522dc4f9527baeeaff5c77dc5b7faa9a731e15d261e264b5ed30c278553b620c43e75addc70145f0c08d5db037d14057e143e5ae55b3fd973f818b008c45b063ee7f0c4f28554af92d6881f1efbc612e6af829525b2dd0f1fe2dc8ab2e4c31deb697699e88ea4fb4d43eea3519bf91730d96a1b554e484d8d95b58be290897ff641b71fd2d786d5ce4062e78b28545b30874b9eb30d9d8b4189919a29ecbc470c907b21ad5964034bbbbcc608d665b028b75627d00c411477fa3fbc78c34727cad31de8db8789d547fbe4a9431c9e03bb61f9e2b993cf127d0bb8402d2da817e9ae35e86134a57d5ce7061f2628b4db18f5d61c4ca8e5955fa126419b45f108dd81969e197473ec06ec736b48b9dd8bb838149042cfd9d9fabfd6b75282936f2ba24ee84b85751a0b1f8892b7a29f0e4c4d35f4d3d6efad602f8794954811d6d59145b2cd0d3d9ee5a2bd88c82664a98aa6a137a6d5046c7c6b23ab544478771b14b8e147e6ecc9a3146b438fffd9c86c708eaf915afb0b05f7b33e666780a4345ee7782624cd3db51ae1dce6e079662ff726a2d54527407eb4b983b9b5592a765a74f5f35cd608b97dc534dec3cbefc1430182a568386ff09755a5034aa2a3b6ea291a0483651f75e8b92bbc3260097ab5121708df9369315e86cb1f1a0a666252263294569115fd5169f034c9ce46897d736d6616cc11d407866bdb0af5c826c534763e810b70b0cae816603fa99ff3148ae17ad2122048cd2384950842d8d4608b32aacc304d5ab4b93f038db83fc45fbc2205d33a780717fc75aaf463f3793952d9a01cc9bb5c7755e4817b2d81d6f0a7a069c2e8d034f26e75ff8f36fa35e82613048e899c022b3b5abe3e2228f41edb25dc7235090644bbcff1287861e6cb545f5903839132f9e397768fc5e3e9160bb188a10c79f511e201e12de36e1b237a47d46bd41293f61e53fa0706ca447d2883ccdea8bc2f8b53eda7d61523bc576ecb381b1139dd286eaaa79fcd22a911554391573776c3a8fbae6ebac203bd993787145b13540582a3e2150de032895e61f3544e1a32ba84bbf95e240d38fae0080528fb1ce9dfdba196e3939b5045c9d6e2ca3a4867e4facc9ef2d620330a235550a489692818542786e256264f3aeb72e565ae46e83f73354573122d7ebf9fe345bdb3d67069e2664c4cdc128bdc43809b1660239411809d4f1cad42a99821bdb111c6c4a6ba41b0145db04198f034803f56c3deee1f3a4334bd9ec37db9bea24c66f342be2512d07a959d1c00a7d36f4c44963df8b22b6b788fca5fccef9871d555af82b8b27a06497a60bf24052cfdbdca7d18731c4cc5c561ede0102fc3679699111c8def6ea7df0701983c3a3722b3fdd6438de336cf7421e2029e40ecbd63c8a2c2479607df55d68b8fcf62941fb3c0a3bc3c21e86fb4e6b4c058b5a5282607640000c2adffb79de8f25d3a25dabebc26b9fe5be6943cd854be634a4e07556a68dc7c999312d4e0f77eded93dc454d44028fe10345e778444bda4c7af6593b650ff63909740c66a34329453ee46eb5014b6ee339ae734eec6c065218f0f22eb5eb4fd2836b7175b9bffe4a85c23dad53e6ab3625ef316e34c31718f3171d38328a143bc28af58184054690e4f16e279a1e5de044c1ac8a1ff5518c13242ddbcbcb97e30c580356bd466f4156a98c66f9a24976765aa396043f3f76e2184e963892e5bb4288c181a48d50aca0da0e99642692948934fdd30d892a3ed225438253abce27472fe1e62c3f2e9bcd01dc34ada72f0ba0ee50210881e25a522ba2b40e5b55695f0e9a0e094957d33e326df57c3fa48ee6931253939b782b015dac6e591fbda9b78d5b83e08d3ac8cc638e0a802762292ce095e51b16ecceee95b5323ea8fc2254a74b5c737ec217301ea14cc44703dec852603056d6dab998e6c34f8f3276216bf10a2e114c56f11d471c49858a95c272dddfcd5c86ae19e4f516b5f7a0a0ab7f7d414de68c0167ea71bbfb1afcec51614a1c0e2c51a4196c6ed44834776d14eda80fc0b446c8c7870990a0c6d1840f3d224d23151bf7fd36bf7e859c6c4c75b1f83bd01171514a438971ffbe5691a9d5b26a121270243eb6183ee60ac4c0b386ad3ce4b5a6d8f2cdc7c61de41d278aa9c473470a54aa27a80d78e9e22e51225f7b47dc3ad97017cc571350a622a9f895f66690773f8770f724f5cd86af9ded84d26972c80b88afd8e02684ac49c41b7d8875daa700f75b56909eb0e4d7c07ce7fbc92d5686aaaab86d53e3b5e5ff626c9b15052f919071c095b044cc3c3ddf194abcc128c07012584c735058b93544f4f736059c37ef3b1c1203c82c33396523b5b3a48b193e6eaff66cf2c88ab8506cfcc90d1e84011d78aa201b63f75ab435eb3326a81edb1531f95dbc19defdce72a66db0530b290fd42d194abd3961af6af54e9ee4ce9ccf5f37f08ab3bb6a9187e4da74675ee1ed963be6dbafb355a106e1656073e433302ca784d59cc10285000cfda8466c27247c1fc4f35bd5daaf2727c6c8e15b10713ac46528be2bba8bb0c4c9bf57109750cef993e2691d12c15d226923027d6f38b735caf4842d2dd8328519591248e5e5e23a533975ffc71276678415c397bdf9b99aee7d853c66fb6ceb3fab015b05e5c51bc8adb2064c629ae9acdead63edde2f1f35f95f40dce4bbe537121356477ef6135da2039e08a1af7db68903ca2bcb14bba9963c5d9df629a0d80dd13162016ecec5ca8fb0794315e2f4293fb3e652e6ca659a13cf45286983da01110f5731577bd14ffb461ace093b5170b405bfed241a957950cc1f5fe3e3914293c86c7e9e786fd464637b4b5980cb99b2fd06e43a535dd665df1c903e9a059ae9e71562cf1ee5fa018b7b8f341a2cd19580d79f915f3fa1ae22ced811a628535f60b823969a3e9b99a926adb1f3696035876d07f149aee05d5b73a714854aa6a1d26ccc926eabd32fbe985e881ba461280c98eee75e87b9af58b447ef6b01da8b6703f832963ecb4ac70fc43098274fbc58e417bdcc17056eb2359cb2d81e439d658a81ba959147d385cd207b10c7d73cd59494d2e4c33f22e6e573d1ea6ec390f7f9c218df196536b5c71fbda6807f8847c6fb5a88ced5e53cac46ac1a9dce71328ce65d41e50d084cc585c431699be3af7587ff6ae1a0aa76da0c70cc0dc4aa7cf3bdf767cc40a23e38e4e2a307214bd3025fbb84876f9d764c08bce44776088ae9b02a502bddb463cfbb8b2eec454e0c1e5159ab5b7a872ec35cce3461ccebc03a666fbcea8d3afce5decc4f61375bee4125eaaf2386655ae2554f7323b1dc22b200da6d6cdb77b64bff6676aa9c45386050d8f6a7025e5b7e765661a06f6eee5e57a330601196a6476a76bca9c2563a4e8738fc183a44b541cc1c42d64856007af7fa2f71cae4504078d187ba875b01f3826e15e98c48946e5b68812268860126a31fc6661f8608d61f5bc44b556f5fd9223aed2f61d55c31c3569716bbc473711812563e566163a93be589a35e469711e1f3b356e5357def24ba087e77b16adf5df3054251731c318a97dd1494ac9b81dbed2050295f336515f4ef346e0ed172f519824a95319f3f6a017ddf495b014e24f37420c7b7125ea9d6766bdc2dafacc2d7aa8363894dc73c5d36008f563a692622c35a55fd100c1e59580fa75589d4fb77f27088b4257b0cb33a400d1be219b69f230b24b3397caaf5ead6d74de2493e5259ed4b1696ea8a9c47c9a1196e8b107282d30ee31527f5b5c7c8e225641b8d78228727e915080279b56c2687cc4b87734c17560911323eeb6dc0c79fa914166968f54099075349fd578dbde760b4642004f9276e5c0fd2a2e4e6f0107c463784c7d42f085232994bd70e924767bc8ccba72c7f4d19ebf8c7839f72911f5354f22b2a305da91ef5fb5c623eb1d55566b4e4d781e87a4df56eea7bd89d0d8c4e3e1ca647baa1c106751cdb66ddb72da2a69e9bb4b366515f3a6db3fa4c9d2bf097e3b5f14caadbf64b2bf119b1b951cd1379299e3af7823a8a4e03c12ef861ad48b13839867a8c3bdb7119cc79b0f0521533f52475dfab5d10c430cb1b57ee6ece34c3b085fbbdb4ae080c7f63a3596cc3088cad73e5fd2fa90d02311540ed63b07c1c3a0be14f88fbf5f5280d9c5ca02c0408f1a2be67837f1af3eb134e57fc8b1b30ac1be1",
 "c_popcount": 24492,
 "c_stream_sha256": "8fcd44e5a419ea29d84bb443d75f10dc9ac85dde2085b7851d92cd198eb3785a",
 "weight_cum_2p16": [
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  1,
  4,
  13,
  41,
  121,
  333,
  843,
  1963,
  4199,
  8230,
  14737,
  24027,
  35550,
  47605,
  57694,
  63723,
  65536
 ]
}

In [ ]:
%%writefile src/gen_dqi_v6/ref/GU_LICENSE

                                 Apache License
                           Version 2.0, January 2004
                        http://www.apache.org/licenses/

   TERMS AND CONDITIONS FOR USE, REPRODUCTION, AND DISTRIBUTION

   1. Definitions.

      "License" shall mean the terms and conditions for use, reproduction,
      and distribution as defined by Sections 1 through 9 of this document.

      "Licensor" shall mean the copyright owner or entity authorized by
      the copyright owner that is granting the License.

      "Legal Entity" shall mean the union of the acting entity and all
      other entities that control, are controlled by, or are under common
      control with that entity. For the purposes of this definition,
      "control" means (i) the power, direct or indirect, to cause the
      direction or management of such entity, whether by contract or
      otherwise, or (ii) ownership of fifty percent (50%) or more of the
      outstanding shares, or (iii) beneficial ownership of such entity.

      "You" (or "Your") shall mean an individual or Legal Entity
      exercising permissions granted by this License.

      "Source" form shall mean the preferred form for making modifications,
      including but not limited to software source code, documentation
      source, and configuration files.

      "Object" form shall mean any form resulting from mechanical
      transformation or translation of a Source form, including but
      not limited to compiled object code, generated documentation,
      and conversions to other media types.

      "Work" shall mean the work of authorship, whether in Source or
      Object form, made available under the License, as indicated by a
      copyright notice that is included in or attached to the work
      (an example is provided in the Appendix below).

      "Derivative Works" shall mean any work, whether in Source or Object
      form, that is based on (or derived from) the Work and for which the
      editorial revisions, annotations, elaborations, or other modifications
      represent, as a whole, an original work of authorship. For the purposes
      of this License, Derivative Works shall not include works that remain
      separable from, or merely link (or bind by name) to the interfaces of,
      the Work and Derivative Works thereof.

      "Contribution" shall mean any work of authorship, including
      the original version of the Work and any modifications or additions
      to that Work or Derivative Works thereof, that is intentionally
      submitted to Licensor for inclusion in the Work by the copyright owner
      or by an individual or Legal Entity authorized to submit on behalf of
      the copyright owner. For the purposes of this definition, "submitted"
      means any form of electronic, verbal, or written communication sent
      to the Licensor or its representatives, including but not limited to
      communication on electronic mailing lists, source code control systems,
      and issue tracking systems that are managed by, or on behalf of, the
      Licensor for the purpose of discussing and improving the Work, but
      excluding communication that is conspicuously marked or otherwise
      designated in writing by the copyright owner as "Not a Contribution."

      "Contributor" shall mean Licensor and any individual or Legal Entity
      on behalf of whom a Contribution has been received by Licensor and
      subsequently incorporated within the Work.

   2. Grant of Copyright License. Subject to the terms and conditions of
      this License, each Contributor hereby grants to You a perpetual,
      worldwide, non-exclusive, no-charge, royalty-free, irrevocable
      copyright license to reproduce, prepare Derivative Works of,
      publicly display, publicly perform, sublicense, and distribute the
      Work and such Derivative Works in Source or Object form.

   3. Grant of Patent License. Subject to the terms and conditions of
      this License, each Contributor hereby grants to You a perpetual,
      worldwide, non-exclusive, no-charge, royalty-free, irrevocable
      (except as stated in this section) patent license to make, have made,
      use, offer to sell, sell, import, and otherwise transfer the Work,
      where such license applies only to those patent claims licensable
      by such Contributor that are necessarily infringed by their
      Contribution(s) alone or by combination of their Contribution(s)
      with the Work to which such Contribution(s) was submitted. If You
      institute patent litigation against any entity (including a
      cross-claim or counterclaim in a lawsuit) alleging that the Work
      or a Contribution incorporated within the Work constitutes direct
      or contributory patent infringement, then any patent licenses
      granted to You under this License for that Work shall terminate
      as of the date such litigation is filed.

   4. Redistribution. You may reproduce and distribute copies of the
      Work or Derivative Works thereof in any medium, with or without
      modifications, and in Source or Object form, provided that You
      meet the following conditions:

      (a) You must give any other recipients of the Work or
          Derivative Works a copy of this License; and

      (b) You must cause any modified files to carry prominent notices
          stating that You changed the files; and

      (c) You must retain, in the Source form of any Derivative Works
          that You distribute, all copyright, patent, trademark, and
          attribution notices from the Source form of the Work,
          excluding those notices that do not pertain to any part of
          the Derivative Works; and

      (d) If the Work includes a "NOTICE" text file as part of its
          distribution, then any Derivative Works that You distribute must
          include a readable copy of the attribution notices contained
          within such NOTICE file, excluding those notices that do not
          pertain to any part of the Derivative Works, in at least one
          of the following places: within a NOTICE text file distributed
          as part of the Derivative Works; within the Source form or
          documentation, if provided along with the Derivative Works; or,
          within a display generated by the Derivative Works, if and
          wherever such third-party notices normally appear. The contents
          of the NOTICE file are for informational purposes only and
          do not modify the License. You may add Your own attribution
          notices within Derivative Works that You distribute, alongside
          or as an addendum to the NOTICE text from the Work, provided
          that such additional attribution notices cannot be construed
          as modifying the License.

      You may add Your own copyright statement to Your modifications and
      may provide additional or different license terms and conditions
      for use, reproduction, or distribution of Your modifications, or
      for any such Derivative Works as a whole, provided Your use,
      reproduction, and distribution of the Work otherwise complies with
      the conditions stated in this License.

   5. Submission of Contributions. Unless You explicitly state otherwise,
      any Contribution intentionally submitted for inclusion in the Work
      by You to the Licensor shall be under the terms and conditions of
      this License, without any additional terms or conditions.
      Notwithstanding the above, nothing herein shall supersede or modify
      the terms of any separate license agreement you may have executed
      with Licensor regarding such Contributions.

   6. Trademarks. This License does not grant permission to use the trade
      names, trademarks, service marks, or product names of the Licensor,
      except as required for reasonable and customary use in describing the
      origin of the Work and reproducing the content of the NOTICE file.

   7. Disclaimer of Warranty. Unless required by applicable law or
      agreed to in writing, Licensor provides the Work (and each
      Contributor provides its Contributions) on an "AS IS" BASIS,
      WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or
      implied, including, without limitation, any warranties or conditions
      of TITLE, NON-INFRINGEMENT, MERCHANTABILITY, or FITNESS FOR A
      PARTICULAR PURPOSE. You are solely responsible for determining the
      appropriateness of using or redistributing the Work and assume any
      risks associated with Your exercise of permissions under this License.

   8. Limitation of Liability. In no event and under no legal theory,
      whether in tort (including negligence), contract, or otherwise,
      unless required by applicable law (such as deliberate and grossly
      negligent acts) or agreed to in writing, shall any Contributor be
      liable to You for damages, including any direct, indirect, special,
      incidental, or consequential damages of any character arising as a
      result of this License or out of the use or inability to use the
      Work (including but not limited to damages for loss of goodwill,
      work stoppage, computer failure or malfunction, or any and all
      other commercial damages or losses), even if such Contributor
      has been advised of the possibility of such damages.

   9. Accepting Warranty or Additional Liability. While redistributing
      the Work or Derivative Works thereof, You may choose to offer,
      and charge a fee for, acceptance of support, warranty, indemnity,
      or other liability obligations and/or rights consistent with this
      License. However, in accepting such obligations, You may act only
      on Your own behalf and on Your sole responsibility, not on behalf
      of any other Contributor, and only if You agree to indemnify,
      defend, and hold each Contributor harmless for any liability
      incurred by, or claims asserted against, such Contributor by reason
      of your accepting any such warranty or additional liability.

   END OF TERMS AND CONDITIONS

   APPENDIX: How to apply the Apache License to your work.

      To apply the Apache License to your work, attach the following
      boilerplate notice, with the fields enclosed by brackets "[]"
      replaced with your own identifying information. (Don't include
      the brackets!)  The text should be enclosed in the appropriate
      comment syntax for the file format. We also recommend that a
      file or class name and description of purpose be included on the
      same "printed page" as the copyright notice for easier
      identification within third-party archives.

   Copyright [yyyy] [name of copyright owner]

   Licensed under the Apache License, Version 2.0 (the "License");
   you may not use this file except in compliance with the License.
   You may obtain a copy of the License at

       http://www.apache.org/licenses/LICENSE-2.0

   Unless required by applicable law or agreed to in writing, software
   distributed under the License is distributed on an "AS IS" BASIS,
   WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
   See the License for the specific language governing permissions and
   limitations under the License.

In [ ]:
%%writefile src/gen_dqi_v6/ref/GU_COMMIT.txt
64b5a13aa249f1360846657392e6c66817b26d0f

In [ ]:
%%writefile src/gen_invsqrt_r3/deferred_lookup.py
"""Deferred lookup demolition and shared one-hot/unary phase repair.

The scheduling idea follows Gu, Kemeny, Yelin and Lukin's Gaussian circuit
(andigu/uncompute, commit 64b5a13aa249f1360846657392e6c66817b26d0f).
This is a new implementation against the kickmix API, Apache-2.0.
"""
import kickmix as km

def demolish(b, register):
 bits=b.alloc_bits(len(register))
 for q,m in zip(register,bits):b.hmr(q,m)
 return list(bits)

def repair_anf(b,address,table,measurements,low_bits=4):
 """Möbius-transform the truth table and share monomials of each half."""
 address=list(address);bits=list(measurements);coef=list(table)
 for i in range(len(address)):
  for mask in range(len(coef)):
   if mask&(1<<i):coef[mask]^=coef[mask^(1<<i)]
 def monomials(register):
  nodes={1<<i:q for i,q in enumerate(register)};undo=[]
  for mask in range(1,1<<len(register)):
   if mask in nodes:continue
   bit=mask&-mask;left=nodes[mask^bit];right=nodes[bit]
   target=b.alloc_qubits(1)[0];b.init_and(left,right,target)
   nodes[mask]=target;undo.append((left,right,target))
  return nodes,undo
 lo,lundo=monomials(address[:low_bits]);hi,hundo=monomials(address[low_bits:])
 one=b.alloc_qubits(1)[0];b.x(one)
 for mask,word in enumerate(coef):
  l=mask&((1<<low_bits)-1);h=mask>>low_bits
  while word:
   pos=(word&-word).bit_length()-1;b.push_condition(bits[pos])
   if l and h:b.cz(lo[l],hi[h])
   else:b.z(lo[l] if l else hi[h] if h else one)
   b.pop_condition();word&=word-1
 b.x(one);b.free(one)
 for left,right,target in reversed(hundo+lundo):
  b.del_and(left,right,target);b.free(target)
 for bit in bits:b.free(bit)

def repair(b,address,table,measurements,low_bits=4,anf=False):
 """Repair (-1)^(table[address] dot measurements), without output wires."""
 if anf:return repair_anf(b,address,table,measurements,low_bits)
 address=list(address);bits=list(measurements)
 low=address[:low_bits];high=address[low_bits:]
 rails=b.alloc_qubits(2);b.cx(low[-1],rails[0]);b.x(rails[0]);b.cx(low[-1],rails[1])
 current=list(rails);levels=[]
 for bit in reversed(low[:-1]):
  following=[];level=[]
  for parent in current:
   pair=b.alloc_qubits(2);lo,hi=pair
   b.init_and(parent,bit,hi);b.cx(parent,lo);b.cx(hi,lo)
   following.extend([lo,hi]);level.append((parent,bit,pair))
  levels.append(level);current=following
 def leaf(prefix,control):
  for j,rail in enumerate(current):
   word=table[(prefix<<low_bits)+j]
   while word:
    pos=(word&-word).bit_length()-1
    b.push_condition(bits[pos]);b.cz(control,rail);b.pop_condition()
    word&=word-1
 def walk(remaining,prefix,parent):
  if not remaining:leaf(prefix,parent);return
  bit=remaining[-1];rest=remaining[:-1]
  if parent is None:
   b.x(bit);walk(rest,prefix*2,bit);b.x(bit);walk(rest,prefix*2+1,bit)
  else:
   tmp=b.alloc_qubits(1)[0]
   b.x(bit);b.init_and(parent,bit,tmp);b.x(bit)
   walk(rest,prefix*2,tmp)
   b.cx(parent,tmp);walk(rest,prefix*2+1,tmp);b.cx(parent,tmp)
   b.x(bit);b.del_and(parent,bit,tmp);b.x(bit);b.free(tmp)
 walk(high,0,None)
 for level in reversed(levels):
  for parent,bit,pair in reversed(level):
   lo,hi=pair;b.cx(hi,lo);b.cx(parent,lo);b.del_and(parent,bit,hi);b.free(pair)
 b.cx(low[-1],rails[1]);b.x(rails[0]);b.cx(low[-1],rails[0]);b.free(rails)
 for bit in bits:b.free(bit)

In [ ]:
%%writefile src/gen_invsqrt_r3/hybrid_add.py
"""Cuccaro MAJ/UMA prefix followed by a measured-carry Gidney suffix.

Trades clean carry workspace for additional Toffolis. New implementation,
Apache-2.0; circuit identities credited to Cuccaro et al. and Gidney.
"""
import signed_digit_lab as A

def add(b,a,t,cin=None,sub=False,borrowed=(),prefix=0):
 a,t=list(a),list(t);k=min(prefix,len(a),len(t)-1)
 if not k:return A.folded_gadd(b,a,t,cin,sub,borrowed=borrowed)
 if sub:
  for q in t:b.x(q)
 initial=None;available=list(borrowed)
 if cin is None:
  if available:cin=available.pop(0)
  else:initial=b.alloc_qubits(1)[0];cin=initial
 carry=cin
 for i in range(k):
  b.cx(a[i],t[i]);b.cx(a[i],carry);b.ccx(t[i],carry,a[i]);carry=a[i]
 A.folded_gadd(b,a[k:],t[k:],carry,borrowed=available)
 for i in reversed(range(k)):
  carry=cin if i==0 else a[i-1]
  b.ccx(t[i],carry,a[i]);b.cx(a[i],carry);b.cx(carry,t[i])
 if initial is not None:b.free(initial)
 if sub:
  for q in t:b.x(q)

In [ ]:
%%writefile src/gen_invsqrt_r3/interpolated_mux_round3.py
"""Synthesize curvature by a tiny table and three short signed additions.

Both base and slope are classical functions of the normalized address.
The small slope is HMR-demolished after each load; its two masks join the
same deferred repair as the two curvature masks. New Apache-2.0 source.
"""
import time_mux_round3 as M
import os,pathlib,json,time,math,concurrent.futures,traceback,argparse
import numpy as np
import sumit_reference as H
import signed_digit_lab as A
from invsqrt import verify_submission_invsqrt as V

def fit(job):
 params,frac,e,g,*other=job;gb=other[0] if other else 3
 c=H.Cfg(*params);G=1<<gb;mid=1+(g+.5)*G/128;scale=2.**((e-1)/2)
 bc=round(.375*scale*mid**-2.5*2**c.f2)
 sc=round(1.25*.375*scale*mid**-3.5*2**c.f2/128*2**frac)
 # 1.25 factor = derivative magnitude 2.5 divided by centered-digit factor2.
 candidates=[];needed=set()
 for base in range(bc-4,bc+5):
  for step in range(max(0,sc-3),sc+4):
   values=tuple(base-sum((2*((low>>i)&1)-1)*int(H.rd(step,frac-i)) for i in range(gb)) for low in range(G))
   if min(values)<0 or max(values)>=1<<c.w2:continue
   candidates.append((base,step,values));needed.update((low,k2) for low,k2 in enumerate(values))
 states={};cache={}
 for low in range(G):
  a=g*G+low;d=np.arange(1<<14,dtype=np.int64);xn=(1<<21)+(a<<14)+d
  d=d[(d%2==0) if e else ((xn<=3133452)|(d%4==0))]
  m=1+(a*(1<<14)+d)*2.**-21;target=scale/np.sqrt(m)*2**c.fy
  mid1=1+(a*(1<<14)+((1<<14)-1)/2)*2.**-21
  k1c=round(.5*scale*mid1**-1.5*2**c.f1);states[low]=(d,target,k1c)
 for low,k2 in sorted(needed):
  d,target,k1c=states[low];best=None
  for k1 in range(k1c-5,k1c+6):
   r,q=H.pipeline(c,k2,k1,d);vals=target+q;center=(float(vals.min())+float(vals.max()))/2+((1<<c.gy)-1)/2
   for k0 in range(math.floor(center)-4,math.ceil(center)+5):
    y=((k0-q)>>c.gy)<<c.gy
    if y.max()>=1<<c.fy:continue
    ratio=float(np.max(abs(y-target)/target))*2**24
    if best is None or ratio<best[0]:best=(ratio,k2,k1,k0)
  cache[low,k2]=best
 best=None
 for base,step,values in candidates:
  rows=[cache[low,k2] for low,k2 in enumerate(values)]
  if any(row is None for row in rows):continue
  score=max(row[0] for row in rows)
  if best is None or score<best['score']:best=dict(score=score,base=base,step=step,rows=[row[1:] for row in rows])
 return (e,g),best

def main():
 p=argparse.ArgumentParser();p.add_argument('--verify',action='store_true');p.add_argument('--fraction-bits',type=int,nargs='+',default=[2,3,4]);args=p.parse_args()
 out=pathlib.Path(os.environ['QCC_OUT']);t0=time.monotonic();params=[9,19,26,2,[5]];cfg=H.Cfg(*params);results=[];best=None
 for frac in args.fraction_bits:
  fits={};start=time.monotonic()
  with concurrent.futures.ProcessPoolExecutor(max_workers=int(os.environ['QCC_JOBS'])) as pool:
   for key,row in pool.map(fit,[(params,frac,e,g) for e in range(2) for g in range(16)],chunksize=1):
    fits[key]=row
    progress=dict(stage='fitting',fraction_bits=frac,completed=len(fits),total=32,elapsed_seconds=time.monotonic()-t0);(out/'progress.json').write_text(json.dumps(progress))
    if len(fits)%8==0:print(json.dumps(progress),flush=True)
  (out/(f'fit-{frac}.json')).write_text(json.dumps([dict(key=k,fit=v) for k,v in fits.items()]))
  tabs={};encoding=[]
  for e in range(2):
   for a in range(128):
    row=fits[e,a//8];tabs[e,a]=tuple(row['rows'][a%8]);encoding.append((row['base'],row['step']))
  for e in range(2):
   for a in range(128):
    base,step=encoding[e*128+a]
    assert tabs[e,a][0]==base-sum((2*((a>>i)&1)-1)*int(H.rd(step,frac-i)) for i in range(3))
  cert=A.exhaustive_certificate(cfg,tabs);interp=dict(fraction_bits=frac,remainder_bits=3,step_width=max(s.bit_length() for _,s in encoding),rows=encoding)
  data=dict(params=params,interpolation=interp,rows=[tabs[e,a] for e in range(2) for a in range(128)])
  (out/(f'tables-{frac}.json')).write_text(json.dumps(data));r=dict(fraction_bits=frac,fit_worst=max(row['score'] for row in fits.values()),certificate=cert,reports=[])
  if cert['pass']:
   try:
    circ,sp=M.build(cfg,tabs,group_bits=None,interpolation=interp);r.update(split=sp,estimate=A.estimated_resources(circ,cert))
    if best is None or r['estimate']['estimated_expected_cost']<best[0]:
     best=(r['estimate']['estimated_expected_cost'],circ,r,data);circ.to_file(str(out/'invsqrt.kmb'),format='kmb');(out/'tables.json').write_text(json.dumps(data))
   except Exception as exc:r.update(error=str(exc),traceback=traceback.format_exc())
  r['seconds']=time.monotonic()-start;results.append(r);(out/'screen.json').write_text(json.dumps(results,indent=2));print(json.dumps(r),flush=True)
 if args.verify and best:
  _,circ,r,data=best
  for suffix in ['A','B']:
   salt='round3-interpolated-curvature-'+str(r['fraction_bits'])+'-'+suffix;start=time.monotonic()
   progress=dict(stage='verifying',salt=salt,completed_salts=len(r['reports']),elapsed_seconds=time.monotonic()-t0);(out/'progress.json').write_text(json.dumps(progress));print(json.dumps(progress),flush=True)
   try:rep=V.verify(circ,V.ProblemSpecInvSqrt.for_n_p(10),salt=salt,max_workers=int(os.environ['QCC_JOBS']))
   except Exception as exc:rep=dict(status='FAIL',error=str(exc),traceback=traceback.format_exc())
   rep.update(salt=salt,seconds=time.monotonic()-start);r['reports'].append(rep);(out/'results.json').write_text(json.dumps(r,indent=2));print(json.dumps(rep),flush=True)
   if rep['status']!='PASS':break
 (out/'screen.json').write_text(json.dumps(results,indent=2))
if __name__=='__main__':main()

In [ ]:
%%writefile src/gen_invsqrt_r3/round2.py
"""Signed-digit oracle schedules, derived from Sumit Chongder and Rudra R4/R5.

Changes made here: configurable precisions and split widths; borrow the
normalized leading one for a clean carry; prune the final phase-shift stage
using the promised s<=10 bound. No verifier logic is changed.
"""
from fractions import Fraction
import numpy as np
import kickmix as km
import sumit_reference as H
import signed_digit_lab as A
import deferred_lookup as D
import hybrid_add as B

def prepare(cfg,tabs):
 d=np.arange(1<<H.DB,dtype=np.int64);Rmax=0;spread=0;minimum={};maxima={}
 for key,(k2,k1,k0) in tabs.items():
  r,_=H.pipeline(cfg,k2,k1,d);minimum[key]=int(r.min());maxima[key]=int(r.max());Rmax=max(Rmax,int(r.max()),k1)
  spread=max(spread,int(r.max()-r.min()))
 # Pick the narrowest unsigned residual which also contains the initial K1.
 for block in range(1,Rmax.bit_length()):
  bases={k:v>>block for k,v in minimum.items()}
  if all(0<=tabs[k][1]-(v<<block)<(1<<(block+1)) and maxima[k]-(v<<block)<(1<<(block+1)) for k,v in bases.items()):break
 return {"block":block,"rw":block+1,"cw":Rmax.bit_length()-block,
         "coarse":bases,"residual":{k:tabs[k][1]-(v<<block) for k,v in bases.items()}}

def shift(b,flags,word,L,inverse,prune=True):
 stages=[(1,1),(2,2),(3,4),(4,8)];occ=[];n=L
 for k,d in stages:
  occ.append(min(n,L+2) if prune and k==4 else n);n=min(n+d,len(word))
 seq=list(enumerate(stages));seq=seq[::-1] if inverse else seq
 for j,(k,d) in seq:
  indices=range(occ[j]) if inverse else reversed(range(occ[j]))
  for i in indices:
   if i+d<len(word):
    zero_boundary = i+d >= L+sum(amount for _,amount in stages[:j])
    if zero_boundary:
     if inverse:b.cx(word[i+d],word[i]);b.del_and(flags[k],word[i],word[i+d])
     else:b.init_and(flags[k],word[i],word[i+d]);b.cx(word[i+d],word[i])
    else:b.cswap(flags[k],word[i],word[i+d])

def build(cfg,tabs,*,borrow=True,prune=True,merge=False,parity_input=False,carry_block=None,phase_cut=0,deferred=False,anf=False):
 lay=H.layout(cfg,tabs,LB=cfg.fy-7,S=cfg.fy-8);split=prepare(cfg,tabs)
 b=km.CircuitBuilder();x=b.create_quantum_register(22,name="x");flags=b.alloc_qubits(4 if parity_input else 5)
 f=([x[21]]+list(flags)) if parity_input else flags
 if parity_input:
  for k,amount in H.NSTAGES[:-1]:
   top=x[22-amount:22];b.x(top);H.and_all_into(b,list(top),f[k]);b.x(top)
   for i in reversed(range(22-amount)):b.cswap(f[k],x[i],x[i+amount])
  b.x(x[21]);b.left_rotate(x[:21],1,control=x[21]);b.cx(x[21],x[0])
 else:H.normalize(b,x,f)
 borrowed=[]
 if borrow and not parity_input:b.x(x[21]);borrowed=[x[21]]
 c2,res,coarse,low,high=[b.alloc_qubits(n) for n in [cfg.w2,split["rw"],split["cw"],lay["LB"],lay["whi"]]]
 checkpoints=[]
 def mark(label):
  z=b # stats collected from final tagged operation counts externally when supported
 def add(a,t,cin=None,sub=False):
  if carry_block:return B.add(b,a,t,cin,sub,borrowed=borrowed,prefix=carry_block)
  return A.folded_gadd(b,a,t,cin,sub,borrowed=borrowed)
 full=km.array(list(low)+list(res)+list(c2)+list(high)+list(coarse))
 rest=km.array(list(low)+list(res)+list(c2)+list(coarse));fulltable=[];resttable=[];hightable=[]
 for e in range(2):
  for i in range(128):
   k=(e,i);common=lay["lo"][k]|(split["residual"][k]<<lay["LB"])|(tabs[k][0]<<(lay["LB"]+split["rw"]))
   resttable.append(common|(split["coarse"][k]<<(lay["LB"]+split["rw"]+cfg.w2)))
   fulltable.append(common|(lay["hi"][k]<<(lay["LB"]+split["rw"]+cfg.w2))|(split["coarse"][k]<<(lay["LB"]+split["rw"]+cfg.w2+lay["whi"])))
   hightable.append(lay["hi"][k])
 address=km.array(list(x[14:21])+[f[0]]);delta=x[:14]
 def phase(reg,lowbit):
  ext=b.alloc_qubits(10-len(borrowed));word=km.array(list(reg)+borrowed+list(ext))
  shift(b,f,word,len(reg),False,prune)
  for i,q in enumerate(word):
   exponent=lowbit+i-cfg.fy-10
   if exponent<0 and not(lowbit==cfg.gy and i<phase_cut):b.z_pow(q,Fraction(2)**(exponent+1))
  shift(b,f,word,len(reg),True,prune);b.free(ext)
 def inner(inverse):
  rows=cfg.rrow_list();rows=rows[::-1] if inverse else rows
  for i in rows:
   sh=cfg.sh2[i]
   if sh>=0:a=c2[sh:];t=res;cin=c2[sh-1] if sh else None
   else:a=c2;t=res[-sh:];cin=None
   b.cx(delta[i],t);add(a,t,cin,sub=inverse);b.cx(delta[i],t)
 def outer(derivative,inverse):
  for i in range(14) if inverse else reversed(range(14)):
   sh=cfg.sh1[i]
   if sh>=0:a=derivative[sh:];t=low;cin=derivative[sh-1] if sh else None
   else:a=derivative;t=low[-sh:];cin=None
   b.cx(delta[i],t);add(a,t,cin,sub=inverse);b.cx(delta[i],t)
 b.init_lookup(table=fulltable,address=address,target=full)
 deferred_bits=[]
 if not merge:
  phase(high,lay["S"])
  if deferred:deferred_bits=D.demolish(b,high)
  else:b.del_lookup(table=hightable,address=address,target=high)
  b.free(high)
 inner(False);add(res[split["block"]:],coarse)
 derivative=res[:split["block"]]+coarse
 outer(derivative,False)
 if merge:
  add(low[lay["S"]:],high)
  phase(km.array(list(low[cfg.gy:lay["S"]])+list(high)),cfg.gy)
  add(low[lay["S"]:],high,sub=True)
 else:phase(low[cfg.gy:],cfg.gy)
 outer(derivative,True);add(res[split["block"]:],coarse,sub=True);inner(True)
 remaining=full if merge else rest
 if deferred:
  deferred_bits+=D.demolish(b,remaining);b.free(remaining)
  table=fulltable if merge else [hi|(rr<<lay['whi']) for hi,rr in zip(hightable,resttable)]
  D.repair(b,address,table,deferred_bits,anf=anf)
 else:
  b.del_lookup(table=fulltable if merge else resttable,address=address,target=remaining);b.free(remaining)
 if borrowed:b.x(x[21])
 if parity_input:
  b.cx(x[21],x[0]);b.right_rotate(x[:21],1,control=x[21]);b.x(x[21])
  for k,amount in reversed(H.NSTAGES[:-1]):
   for i in range(22-amount):b.cswap(f[k],x[i],x[i+amount])
   top=x[22-amount:22];m=b.alloc_bits(1)[0];b.hmr(f[k],m);b.push_condition(m)
   b.x(top);H.phase_all(b,list(top));b.x(top);b.pop_condition();b.free(m)
 else:H.unnormalize(b,x,f)
 b.free(flags)
 return b.finish_circuit(),{k:v for k,v in split.items() if k not in ["coarse","residual"]}

In [ ]:
%%writefile src/gen_invsqrt_r3/tables.json
{"params":[9,19,26,2,[5]],"interpolation":{"fraction_bits":3,"remainder_bits":3,"step_width":4,"rows":[[126,9],[126,9],[126,9],[126,9],[126,9],[126,9],[126,9],[126,9],[108,7],[108,7],[108,7],[108,7],[108,7],[108,7],[108,7],[108,7],[94,6],[94,6],[94,6],[94,6],[94,6],[94,6],[94,6],[94,6],[83,5],[83,5],[83,5],[83,5],[83,5],[83,5],[83,5],[83,5],[73,4],[73,4],[73,4],[73,4],[73,4],[73,4],[73,4],[73,4],[65,3],[65,3],[65,3],[65,3],[65,3],[65,3],[65,3],[65,3],[58,2],[58,2],[58,2],[58,2],[57,2],[57,2],[57,2],[57,2],[52,2],[52,2],[52,2],[52,2],[51,0],[51,0],[50,0],[49,0],[47,2],[47,2],[47,2],[47,2],[46,2],[46,2],[46,2],[46,2],[42,1],[42,1],[42,1],[42,1],[42,1],[42,1],[42,1],[42,1],[39,0],[39,0],[39,0],[39,0],[39,2],[39,2],[39,2],[39,2],[35,2],[35,2],[35,2],[35,2],[35,2],[35,2],[35,2],[35,2],[32,1],[32,1],[32,1],[32,1],[32,1],[32,1],[32,1],[32,1],[30,0],[30,0],[30,0],[30,0],[29,0],[29,0],[29,0],[29,0],[27,0],[27,0],[27,0],[27,0],[26,0],[26,0],[26,0],[26,0],[25,0],[25,0],[25,0],[25,0],[25,0],[24,0],[24,0],[24,0],[178,13],[178,13],[178,13],[178,13],[178,13],[178,13],[178,13],[178,13],[154,11],[154,11],[154,11],[154,11],[154,11],[154,11],[154,11],[154,11],[134,7],[134,7],[134,7],[134,7],[134,7],[134,7],[134,7],[134,7],[117,7],[117,7],[117,7],[117,7],[117,7],[117,7],[117,7],[117,7],[103,6],[103,6],[103,6],[103,6],[103,6],[103,6],[103,6],[103,6],[92,5],[92,5],[92,5],[92,5],[92,5],[92,5],[92,5],[92,5],[82,4],[82,4],[82,4],[82,4],[82,4],[82,4],[82,4],[82,4],[73,4],[73,4],[73,4],[73,4],[73,4],[73,4],[73,4],[73,4],[66,3],[66,3],[66,3],[66,3],[66,3],[66,3],[66,3],[66,3],[60,3],[60,3],[60,3],[60,3],[60,3],[60,3],[60,3],[60,3],[54,2],[54,2],[54,2],[54,2],[54,2],[54,2],[54,2],[54,2],[50,2],[50,2],[50,2],[50,2],[50,2],[50,2],[50,2],[50,2],[45,2],[45,2],[45,2],[45,2],[45,2],[45,2],[45,2],[45,2],[41,2],[41,2],[41,2],[41,2],[41,2],[41,2],[41,2],[41,2],[38,1],[38,1],[38,1],[38,1],[38,1],[38,1],[38,1],[38,1],[35,1],[35,1],[35,1],[35,1],[35,1],[35,1],[35,1],[35,1]],"update_width":8},"threshold":0.99,"rows":[[134,184284,189442912],[132,182154,188710052],[130,180064,187985631],[128,178014,187269491],[124,176003,186561483],[122,174029,185861435],[120,172091,185169211],[118,170190,184484662],[115,168323,183807656],[113,166490,183138044],[111,164690,182475698],[109,162922,181820487],[107,161186,181172283],[105,159481,180530965],[103,157805,179896407],[101,156158,179268497],[100,154540,178647110],[98,152949,178032146],[96,151386,177423488],[94,149849,176821030],[94,148338,176224660],[92,146852,175634293],[90,145391,175049821],[88,143953,174471143],[88,142540,173898159],[86,141149,173330793],[86,139781,172768935],[84,138436,172212514],[82,137110,171661436],[80,135807,171115614],[80,134523,170574958],[78,133260,170039402],[77,132017,169508856],[75,130792,168983250],[75,129587,168462493],[73,128400,167946531],[73,127231,167435271],[71,126080,166928663],[71,124945,166426615],[69,123828,165929081],[68,122727,165435977],[68,121643,164947238],[66,120574,164462814],[66,119521,163982626],[64,118483,163506628],[64,117461,163034742],[62,116452,162566926],[62,115458,162103108],[60,114478,161643246],[60,113512,161187265],[58,112560,160735131],[58,111621,160286773],[57,110695,159842149],[57,109781,159401199],[55,108880,158963889],[55,107991,158530151],[54,107114,158099947],[54,106249,157673222],[52,105396,157249944],[52,104553,156830046],[51,103722,156413500],[51,102903,156000250],[50,102094,155590264],[49,101295,155183492],[49,100507,154779890],[49,99728,154379422],[47,98960,153982054],[47,98201,153587730],[46,97453,153196425],[46,96714,152808091],[44,95984,152422703],[44,95263,152040211],[43,94552,151660585],[43,93848,151283785],[43,93154,150909781],[43,92469,150538535],[41,91791,150170027],[41,91122,149804203],[41,90461,149441037],[41,89807,149080500],[39,89162,148722570],[39,88525,148367196],[39,87895,148014360],[39,87272,147664028],[39,86657,147316170],[39,86048,146970762],[37,85448,146627778],[37,84853,146287178],[37,84267,145948938],[37,83686,145613036],[35,83112,145279450],[35,82544,144948138],[35,81983,144619082],[35,81429,144292257],[33,80881,143967645],[33,80339,143645207],[33,79802,143324927],[33,79272,143006779],[33,78748,142690739],[33,78229,142376787],[31,77716,142064905],[31,77208,141755057],[31,76706,141447225],[31,76210,141141391],[30,75719,140837539],[30,75233,140535635],[30,74752,140235663],[30,74277,139937605],[29,73806,139641443],[29,73341,139347148],[29,72879,139054708],[29,72424,138764100],[27,71973,138475316],[27,71526,138188320],[27,71084,137903100],[27,70647,137619640],[26,70214,137337922],[26,69785,137057926],[26,69361,136779636],[26,68941,136503032],[25,68526,136228104],[25,68114,135954826],[25,67706,135683186],[25,67303,135413170],[25,66903,135144756],[24,66508,134877940],[24,66116,134612692],[24,65728,134349002],[190,260618,267912733],[186,257604,266876317],[184,254649,265851829],[180,251750,264839058],[176,248905,263837776],[172,246114,262847766],[170,243374,261868805],[166,240685,260900717],[164,238044,259943273],[162,235452,258996294],[158,232907,258059602],[156,230407,257132988],[152,227952,256216297],[150,225540,255309331],[146,223170,254411939],[144,220841,253523934],[141,218553,252645166],[139,216303,251775472],[137,214092,250914696],[135,211918,250062687],[133,209781,249219303],[131,207680,248384395],[129,205614,247557821],[127,203582,246739444],[124,201582,245929136],[122,199616,245126758],[120,197681,244332178],[118,195777,243545277],[116,193903,242765929],[114,192059,241994019],[112,190245,241229423],[110,188458,240472031],[109,186700,239721722],[107,184968,238978398],[105,183264,238241948],[103,181585,237512262],[103,179932,236789233],[101,178303,236072775],[99,176699,235362783],[97,175119,234659159],[97,173563,233961797],[95,172029,233270625],[95,170518,232585534],[93,169029,231906454],[91,167561,231233288],[89,166115,230565950],[89,164688,229904346],[87,163283,229248415],[86,161897,228598063],[84,160531,227953219],[84,159184,227313793],[82,157855,226679727],[82,156545,226050927],[80,155254,225427342],[80,153979,224808878],[78,152723,224195488],[77,151482,223587086],[75,150259,222983616],[75,149052,222384996],[73,147861,221791180],[73,146686,221202087],[71,145527,220617673],[71,144382,220037857],[69,143253,219462601],[69,142138,218891821],[69,141038,218325473],[67,139951,217763505],[67,138879,217205848],[65,137820,216652462],[65,136774,216103274],[63,135742,215558250],[63,134723,215017324],[63,133716,214480446],[63,132722,213947572],[61,131740,213418658],[61,130771,212893639],[59,129812,212372483],[59,128865,211855127],[57,127931,211341545],[57,127007,210831671],[56,126095,210325473],[56,125193,209822899],[54,124302,209323919],[54,123422,208828475],[54,122551,208336532],[54,121691,207848048],[52,120841,207362994],[52,120001,206881312],[52,119170,206402970],[52,118350,205927932],[50,117538,205456166],[50,116736,204987620],[50,115943,204522264],[50,115158,204060064],[48,114383,203600993],[48,113616,203144995],[47,112858,202692053],[47,112108,202242123],[45,111366,201795185],[45,110633,201351191],[45,109907,200910111],[45,109190,200471917],[43,108480,200036589],[43,107778,199604073],[43,107083,199174353],[43,106396,198747396],[41,105716,198323182],[41,105043,197901664],[41,104378,197482822],[41,103719,197066628],[39,103068,196653062],[39,102422,196242082],[39,101785,195833668],[39,101153,195427794],[39,100528,195024432],[39,99909,194623556],[37,99298,194225152],[37,98692,193829176],[37,98091,193435613],[37,97497,193044437],[36,96909,192655629],[36,96327,192269157],[36,95751,191884999],[36,95180,191503137],[34,94616,191123553],[34,94057,190746211],[34,93502,190371093],[34,92954,189998181]],"fit_groups":[[3,0,0,0.8148105350137045],[3,0,1,0.8943204643510907],[3,0,2,0.9834761903270947],[3,0,3,0.9677781455074566],[3,0,4,0.8810102937126736],[3,0,5,0.8254088820967135],[2,0,12,0.9165629943487492],[2,0,13,0.8384806352446524],[2,0,14,0.9499016324961326],[0,0,60,null],[0,0,61,null],[0,0,62,null],[0,0,63,null],[2,0,16,0.8548550322180108],[2,0,17,0.953821457350305],[3,0,9,0.9289634566854921],[2,0,20,0.9241306836638363],[2,0,21,0.9741612764427697],[3,0,11,0.9738952253623129],[3,0,12,0.8881409954892974],[2,0,26,0.9116751085323821],[2,0,27,0.9492284958355196],[2,0,28,0.9509008246971019],[2,0,29,0.941565884692494],[2,0,30,0.9752065464741345],[0,0,124,null],[0,0,125,null],[0,0,126,null],[0,0,127,null],[3,1,0,0.7038371877505638],[3,1,1,0.8026629771168182],[3,1,2,0.6948060776302255],[3,1,3,0.611326615579549],[3,1,4,0.6974229041567335],[3,1,5,0.7799141051392934],[3,1,6,0.6650227355176005],[3,1,7,0.657121674848042],[3,1,8,0.6639376266509004],[3,1,9,0.7552662920774285],[3,1,10,0.8034399560684355],[3,1,11,0.812105546040423],[3,1,12,0.7230947601200167],[3,1,13,0.8255134508502201],[3,1,14,0.767192264983811],[3,1,15,0.7254860994118587]],"build_options":{"main_lookup":"anf","affine":false,"group_bits":null,"joint_norm":true}}

In [ ]:
%%writefile src/gen_invsqrt_r3/time_mux_round3.py
"""Curvature time multiplexing with shared classical phase repair.

New Apache-2.0 implementation, building on Sumit Chongder's centered
quadratic, Gu et al.'s deferred repairs, and round-2 arithmetic.
The curvature is constant on groups of adjacent address intervals. It is
looked up only during the inner multiplications, demolished twice, and
both measurement masks are merged into the final shared ANF repair.
"""
import os,sys,pathlib,json,math,time,concurrent.futures,traceback
os.environ['OMP_NUM_THREADS']='1'

sys.path[:0]=[str(pathlib.Path(__file__).parent/'vendor')]
import numpy as np
import kickmix as km
from fractions import Fraction
import sumit_reference as H
import signed_digit_lab as A
import round2 as O
import deferred_lookup as D
from invsqrt import verify_submission_invsqrt as V

def joint_top_normalization_repair(b,x,m8,m16):
 """On positive22-bit inputs, f16=g16 and f8=g8 XOR g16,
 where gk tests whether the highest k input bits are zero.
 Thus the joint phase is m8*g8 XOR (m8 XOR m16)*g16.
 A shared16-bit prefix ladder, or an8-bit phase, costs8.5 mean
 Toffolis instead of10 for the two separate repairs.
 """
 # Classical allocation can recycle a nonzero bit. XOR directly into the
 # fresh HMR result instead of assuming a newly allocated bit is zero.
 parity=m16;b.cx(m8,parity)
 b.push_condition(parity)
 top=list(reversed(x[6:22]));b.x(top);anc=b.alloc_qubits(14)
 b.init_and(top[0],top[1],anc[0])
 for j in range(1,14):b.init_and(anc[j-1],top[j+1],anc[j])
 b.push_condition(m8);b.z(anc[6]);b.pop_condition()
 b.cz(anc[13],top[15])
 for j in reversed(range(1,14)):b.del_and(anc[j-1],top[j+1],anc[j])
 b.del_and(top[0],top[1],anc[0]);b.free(anc);b.x(top);b.pop_condition()
 b.x(parity);b.push_condition(parity);b.push_condition(m8)
 top=x[14:22];b.x(top);H.phase_all(b,list(top));b.x(top)
 b.pop_condition();b.pop_condition()

def anf_lookup(b,address,target,table):
 """Compute an XOR lookup with depth-first ANF monomials, at most
 2**n-n-1 temporary ANDs and n-1 ladder wires. New implementation.
 """
 address=list(address);coef=list(table);n=len(address)
 for i in range(n):
  for mask in range(1<<n):
   if mask>>i&1:coef[mask]^=coef[mask^(1<<i)]
 active={}
 def needed(mask,nextbit):
  children=[(mask|(1<<j),j+1,j) for j in range(nextbit,n)]
  active[mask]=bool(coef[mask])
  for child,nxt,j in children:active[mask]|=needed(child,nxt)
  return active[mask]
 for i in range(n):needed(1<<i,i+1)
 def xor(value,control=None):
  for i,q in enumerate(target):
   if value>>i&1:
    if control is None:b.x(q)
    else:b.cx(control,q)
 xor(coef[0])
 def visit(mask,control,nextbit):
  xor(coef[mask],control)
  for j in range(nextbit,n):
   child=mask|(1<<j)
   if not active[child]:continue
   q=b.alloc_qubits(1)[0];b.init_and(control,address[j],q)
   visit(child,q,j+1);b.del_and(control,address[j],q);b.free(q)
 for i in range(n):
  if active[1<<i]:visit(1<<i,address[i],i+1)

def adaptive_lookup(b,address,target,table,affine=False):
 """Unary decoder pruned wherever every remaining output word is equal.

 A full M-leaf binary tree costs M-2 temporary ANDs; address wires and
 one ladder wire per active depth suffice. Demolition is standard MBU.
 """
 address=list(address)
 def xor_word(value,control):
  for i,q in enumerate(target):
   if value>>i&1:
    if control is None:b.x(q)
    else:b.cx(control,q)
 def walk(bits,rows,parent):
  if len(set(rows))==1:xor_word(rows[0],parent);return
  assert bits
  if affine:
   base=rows[0];columns=[rows[1<<i]^base for i in range(len(bits))]
   test=[base]
   for col in columns:test += [v^col for v in test]
   if rows==test:
    xor_word(base,parent)
    for bit,col in zip(bits,columns):
     if not col:continue
     if parent is None:xor_word(col,bit)
     else:
      q=b.alloc_qubits(1)[0];b.init_and(parent,bit,q);xor_word(col,q);b.del_and(parent,bit,q);b.free(q)
    return
  bit=bits[-1];half=len(rows)//2
  if parent is None:
   b.x(bit);walk(bits[:-1],rows[:half],bit);b.x(bit)
   walk(bits[:-1],rows[half:],bit)
  else:
   tmp=b.alloc_qubits(1)[0]
   b.x(bit);b.init_and(parent,bit,tmp);b.x(bit)
   walk(bits[:-1],rows[:half],tmp)
   b.cx(parent,tmp);walk(bits[:-1],rows[half:],tmp);b.cx(parent,tmp)
   b.x(bit);b.del_and(parent,bit,tmp);b.x(bit);b.free(tmp)
 walk(address,list(table),None)

def fit_group(job):
 params,group_bits,e,g=job;c=H.Cfg(*params)
 d=np.arange(1<<14,dtype=np.int64);G=1<<group_bits
 mid=1+(g+.5)*G/128
 center=round(.375*2.**((e-1)/2)*mid**-2.5*2**c.f2)
 best=None
 for k2 in range(max(0,center-6),min((1<<c.w2)-1,center+6)+1):
  rows=[];scores=[]
  for a in range(g*G,(g+1)*G):
   xn=(1<<21)+(a<<14)+d
   dd=d[(d%2==0) if e else ((xn<=3133452)|(d%4==0))]
   m=1+(a*(1<<14)+dd)*2.**-21;target=2.**((e-1)/2)/np.sqrt(m)*2**c.fy
   mid1=1+(a*(1<<14)+((1<<14)-1)/2)*2.**-21
   k1c=round(.5*2.**((e-1)/2)*mid1**-1.5*2**c.f1)
   segment=None
   for k1 in range(k1c-5,k1c+6):
    r,q=H.pipeline(c,k2,k1,dd);vals=target+q
    center0=(float(vals.min())+float(vals.max()))/2+((1<<c.gy)-1)/2
    for k0 in range(math.floor(center0)-4,math.ceil(center0)+5):
     y=((k0-q)>>c.gy)<<c.gy
     if y.max() >= 1<<c.fy:continue
     ratio=float(np.max(abs(y-target)/target))*2**24
     if segment is None or ratio<segment[0]:segment=(ratio,k2,k1,k0)
   if segment is None:break
   scores.append(segment[0]);rows.append(segment[1:])
  if len(rows)==G and (best is None or max(scores)<best[0]):best=(max(scores),rows)
 return (e,g),best

def build(c,tabs,group_bits=2,mode='mux',interpolation=None,main_lookup='unary',affine=False,joint_norm=False):
 lay=H.layout(c,tabs,LB=c.fy-7,S=c.fy-8);sp=O.prepare(c,tabs)
 b=km.CircuitBuilder();x=b.create_quantum_register(22,name='x');flags=b.alloc_qubits(4);f=[x[21]]+list(flags)
 for k,amount in H.NSTAGES[:-1]:
  top=x[22-amount:22];b.x(top);H.and_all_into(b,list(top),f[k]);b.x(top)
  for i in reversed(range(22-amount)):b.cswap(f[k],x[i],x[i+amount])
 b.x(x[21]);b.left_rotate(x[:21],1,control=x[21]);b.cx(x[21],x[0])
 res,coarse,low,high=[b.alloc_qubits(n) for n in [sp['rw'],sp['cw'],lay['LB'],lay['whi']]]
 full=km.array(list(low)+list(res)+list(high)+list(coarse));rest=km.array(list(low)+list(res)+list(coarse))
 ft=[];rt=[];ht=[];ct=[]
 for e in range(2):
  for a in range(128):
   k=(e,a);common=lay['lo'][k]|(sp['residual'][k]<<lay['LB'])
   rt.append(common|(sp['coarse'][k]<<(lay['LB']+sp['rw'])))
   ft.append(common|(lay['hi'][k]<<(lay['LB']+sp['rw']))|(sp['coarse'][k]<<(lay['LB']+sp['rw']+lay['whi'])))
   ht.append(lay['hi'][k])
  if group_bits is not None:
   for a in range(0,128,1<<group_bits):
    assert len({tabs[e,j][0] for j in range(a,a+(1<<group_bits))})==1
    ct.append(tabs[e,a][0])
  else:ct.extend(tabs[e,a][0] for a in range(128))
 address=km.array(list(x[14:21])+[f[0]]);ca=km.array(list(x[14+(group_bits or 0):21])+[f[0]]);delta=x[:14]
 c2=None;step_masks=[]
 def add(a,t,cin=None,sub=False):A.folded_gadd(b,a,t,cin,sub)
 def phase(reg,lowbit):
  ext=b.alloc_qubits(10);word=km.array(list(reg)+list(ext))
  O.shift(b,f,word,len(reg),False)
  for i,q in enumerate(word):
   ex=lowbit+i-c.fy-10
   if ex<0:b.z_pow(q,Fraction(2)**(ex+1))
  O.shift(b,f,word,len(reg),True);b.free(ext)
 def load():
  nonlocal c2
  c2=b.alloc_qubits(c.w2)
  if interpolation is not None:
   sw=interpolation['step_width'];step=b.alloc_qubits(sw)
   uw=interpolation.get('update_width',c.w2);mask=(1<<uw)-1
   packed=[((base&mask)|(tabs[a//128,a%128][0]&~mask))|(s<<c.w2) for a,(base,s) in enumerate(interpolation['rows'])]
   adaptive_lookup(b,address,km.array(list(c2)+list(step)),packed,affine=affine)
   for i in range(interpolation['remainder_bits']):
    sh=interpolation['fraction_bits']-i
    if sh>=0:a=step[sh:];t=c2[:uw];cin=step[sh-1] if sh else None
    else:a=step;t=c2[-sh:uw];cin=None
    b.cx(x[14+i],t);add(a,t,cin);b.cx(x[14+i],t)
   step_masks.extend(D.demolish(b,step));b.free(step)
  elif group_bits is None:adaptive_lookup(b,ca,c2,ct,affine=affine)
  else:b.init_lookup(table=ct,address=ca,target=c2)
 def erase():
  nonlocal c2
  mm=D.demolish(b,c2);b.free(c2);c2=None;return mm
 def inner(inverse):
  rows=c.rrow_list();rows=rows[::-1] if inverse else rows
  for i in rows:
   sh=c.sh2[i]
   if sh>=0:a=c2[sh:];t=res;cin=c2[sh-1] if sh else None
   else:a=c2;t=res[-sh:];cin=None
   b.cx(delta[i],t);add(a,t,cin,sub=inverse);b.cx(delta[i],t)
 def outer(inverse):
  derivative=res[:sp['block']]+coarse
  for i in range(14) if inverse else reversed(range(14)):
   sh=c.sh1[i]
   if sh>=0:a=derivative[sh:];t=low;cin=derivative[sh-1] if sh else None
   else:a=derivative;t=low[-sh:];cin=None
   b.cx(delta[i],t);add(a,t,cin,sub=inverse);b.cx(delta[i],t)
 if main_lookup=='anf':anf_lookup(b,address,full,ft)
 else:b.init_lookup(table=ft,address=address,target=full)
 phase(high,lay['S']);bits=D.demolish(b,high);b.free(high)
 load();inner(False);cm1=erase();add(res[sp['block']:],coarse);outer(False)
 phase(low[c.gy:],c.gy)
 outer(True);add(res[sp['block']:],coarse,sub=True);load();inner(True);cm2=erase()
 bits+=D.demolish(b,rest);b.free(rest)
 # Separate copies of the curvature word can share its truth table. Repeating
 # its bits is equivalent to XORing the measurement masks, without bit gates.
 bits+=cm1+cm2
 rtwidth=len(rest);hw=lay['whi']
 repairtable=[h|(r<<hw)|(tabs[e,a][0]<<(hw+rtwidth))|(tabs[e,a][0]<<(hw+rtwidth+c.w2)) for e in range(2) for a,(h,r) in enumerate(zip(ht[128*e:128*(e+1)],rt[128*e:128*(e+1)]))]
 if interpolation is not None:
  bits+=step_masks;sw=interpolation['step_width'];off=hw+rtwidth+2*c.w2
  for a,(_,step) in enumerate(interpolation['rows']):
   repairtable[a]|=(step<<off)|(step<<(off+sw))
 D.repair(b,address,repairtable,bits,anf=True)
 b.cx(x[21],x[0]);b.right_rotate(x[:21],1,control=x[21]);b.x(x[21])
 pending=None
 for k,amount in reversed(H.NSTAGES[:-1]):
  for i in range(22-amount):b.cswap(f[k],x[i],x[i+amount])
  top=x[22-amount:22];m=b.alloc_bits(1)[0];b.hmr(f[k],m)
  if joint_norm and k==3:pending=m;continue
  if joint_norm and k==4:
   joint_top_normalization_repair(b,x,pending,m);b.free(pending);b.free(m)
  else:
   b.push_condition(m);b.x(top);H.phase_all(b,list(top));b.x(top);b.pop_condition();b.free(m)
 b.free(flags)
 return b.finish_circuit(),{k:v for k,v in sp.items() if k not in ['coarse','residual']}

def main():
 import argparse
 p=argparse.ArgumentParser();p.add_argument('--params',default='9,19,26,2');p.add_argument('--drop',default='5');p.add_argument('--group-bits',type=int,default=2);p.add_argument('--tables');p.add_argument('--verify',action='store_true');args=p.parse_args()
 params=[*map(int,args.params.split(',')),list(map(int,args.drop.split(','))) if args.drop else []]
 c=H.Cfg(*params);out=pathlib.Path(os.environ['QCC_OUT']);t0=time.monotonic();tabs={};ratios=[]
 if args.tables:
  saved=json.loads(pathlib.Path(args.tables).read_text());tabs={(e,a):tuple(saved['rows'][e*128+a]) for e in range(2) for a in range(128)}
 else:
  G=1<<args.group_bits
  with concurrent.futures.ProcessPoolExecutor(max_workers=int(os.environ['QCC_JOBS'])) as pool:
   for (e,g),row in pool.map(fit_group,[(params,args.group_bits,e,g) for e in range(2) for g in range(128//G)],chunksize=1):
    if row is None:raise ValueError((e,g,'no fit'))
    ratios.append(row[0]);tabs.update({(e,g*G+j):r for j,r in enumerate(row[1])})
 (out/'tables.json').write_text(json.dumps(dict(params=params,group_bits=args.group_bits,rows=[tabs[e,a] for e in range(2) for a in range(128)])))
 cert=A.exhaustive_certificate(c,tabs);result=dict(params=params,group_bits=args.group_bits,certificate=cert,normalized_worst=max(ratios) if ratios else None,reports=[])
 if cert['pass']:
  try:
   circ,sp=build(c,tabs,args.group_bits);circ.to_file(str(out/'invsqrt.kmb'),format='kmb');st=circ.operation_statistics()
   result.update(static=dict(Q=circ.num_qubits,toffoli=int(st.num_toffoli_like),rotations=int(st.num_z_pow(min_clifford_hierarchy_level=4))),split=sp,estimate=A.estimated_resources(circ,cert))
   print(json.dumps(result),flush=True)
   if args.verify:
    for suffix in ['A','B']:
     salt=f'round3-mux-g{args.group_bits}-{args.params}-{suffix}';start=time.monotonic()
     try:r=V.verify(circ,V.ProblemSpecInvSqrt.for_n_p(10),salt=salt,max_workers=int(os.environ['QCC_JOBS']))
     except Exception as exc:r=dict(status='FAIL',error=str(exc),traceback=traceback.format_exc())
     r.update(salt=salt,seconds=time.monotonic()-start);result['reports'].append(r)
     (out/'results.json').write_text(json.dumps(result,indent=2));print(json.dumps(r),flush=True)
     if r['status']!='PASS':break
  except Exception as exc:result.update(error=str(exc),traceback=traceback.format_exc())
 result['seconds']=time.monotonic()-t0;(out/'results.json').write_text(json.dumps(result,indent=2));print(json.dumps(result),flush=True)
if __name__=='__main__':main()

In [ ]:
%%writefile src/gen_invsqrt_r3/vendor/compact_horner.py
"""Compact signed-digit inverse square root, precision (9,19,26,2).
SPDX-License-Identifier: Apache-2.0
Derived from Sumit Chongder's public readable signed-digit quadratic builder:
https://www.kaggle.com/code/sumitchongder/notebook2f196e646b?scriptVersionId=361357149
Changes: parity occupies x[21]; the leading anchor bit is implicit; precision is
recalibrated on the exact reachable domain. Both facts are checked exhaustively.
The centered Horner rows use Gidney additions and measurement uncomputation.
"""
import sys, math
from fractions import Fraction
import numpy as np
import kickmix as km

W = 22; MB = 21; DB = 14; NSEG = 128
F2, F1, FYR, GY, DROP2 = 9, 19, 26, 2, (5,)     # precision configuration (verified)

# ----------------------------------------------------------------- numpy model / tables
def fitc(lo, hi, mid, scale, deg=2, iters=80):
    f = lambda M: scale / np.sqrt(M)
    d = np.linspace(lo - mid, hi - mid, 3001)
    A = np.vander(d, deg + 1, increasing=True)
    w = np.sqrt(mid + d) / scale
    wt = np.ones_like(d)
    for _ in range(iters):                       # Lawson iterations -> weighted minimax
        sw = w * np.sqrt(wt)
        coef = np.linalg.lstsq(A * sw[:, None], f(mid + d) * sw, rcond=None)[0]
        err = np.abs((A @ coef - f(mid + d)) * w)
        wt = wt * err; wt /= wt.sum()
    return coef

class Cfg:
    def __init__(self, f2, f1, fyr, gy, drop2=()):
        self.f2, self.f1, self.fyr, self.gy = f2, f1, fyr, gy
        self.fy = fyr + gy
        self.w2 = f2 - 1
        self.sh2 = [f2 + 22 - i - f1 for i in range(DB)]
        self.sh1 = [f1 + 22 - i - self.fy for i in range(DB)]
        self.drop2 = set(drop2)
    def rrow_list(self):
        return [i for i in reversed(range(DB)) if i not in self.drop2 and self.sh2[i] < self.w2]

def rd(v, s):
    if s < 0: return v << (-s)
    r = v >> s
    if s >= 1: r = r + ((v >> (s - 1)) & 1)
    return r

def pipeline(cfg, K2, K1, d):
    sig = [2 * ((d >> i) & 1) - 1 for i in range(DB)]
    R = np.full_like(d, K1)
    for i in cfg.rrow_list():
        R = R - sig[i] * rd(K2, cfg.sh2[i])
    Q = np.zeros_like(d)
    for i in reversed(range(DB)):
        Q = Q + sig[i] * rd(R, cfg.sh1[i])
    return R, Q

def build_tables(cfg, search=2):
    tabs = {}; worst = 0
    dmid = (2**DB - 1) / 2**(MB + 1)
    d = np.arange(2**DB, dtype=np.int64)
    g = cfg.gy
    for e in (0, 1):
        scale = 2**((e - 1) / 2)
        for a in range(NSEG):
            lo = 1 + a * 2.0**(DB - MB); mid = lo + dmid
            c = fitc(lo, lo + 2.0**(DB - MB), mid, scale)
            C1, C2 = -c[1], c[2]
            M = lo + d * 2.0**-MB
            fM = scale / np.sqrt(M); tgt = fM * 2.0**cfg.fy; w = 1 / fM
            K2c = int(round(C2 * 2**cfg.f2)); K1c = int(round(C1 * 2**cfg.f1))
            best = None
            for K2 in range(K2c - 1, K2c + 2):
                for K1 in range(K1c - search, K1c + search + 1):
                    R, Q = pipeline(cfg, K2, K1, d)
                    def mx(K0):
                        Y = ((K0 - Q) >> g) << g
                        return float(np.max(np.abs(Y - tgt) * w))
                    t = tgt + Q
                    a_, b_ = math.floor(t.min()), math.ceil(t.max())
                    while b_ - a_ > 2:
                        m1 = a_ + (b_ - a_) // 3; m2 = b_ - (b_ - a_) // 3
                        if mx(m1) < mx(m2): b_ = m2
                        else: a_ = m1
                    for K0 in range(a_, b_ + 1):
                        v = mx(K0) / 2.0**cfg.fy
                        if best is None or v < best[0]: best = (v, K2, K1, K0)
            tabs[(e, a)] = best[1:]
            worst = max(worst, best[0])
    return tabs, worst

def layout(cfg, tabs, LB=21, S=20):
    d = np.arange(2**DB, dtype=np.int64)
    assert max(t[0] for t in tabs.values()) < (1 << cfg.w2)
    Rmax = 0; per = {}; Ymax = 0
    for key, (K2, K1, K0) in tabs.items():
        R, Q = pipeline(cfg, K2, K1, d)
        assert R.min() >= 0
        Rmax = max(Rmax, int(R.max()), K1)
        per[key] = (int(Q.min()), int(Q.max()))
        Ymax = max(Ymax, K0 - per[key][0])
    w1 = Rmax.bit_length()
    lo_tab, hi_tab = {}, {}
    for key, (K2, K1, K0) in tabs.items():
        qmin, qmax = per[key]
        base = K0 % (1 << S); j = 0
        while base + j * (1 << S) - qmax < 0: j += 1
        k0lo = base + j * (1 << S)
        assert k0lo - qmin < (1 << LB), 'layout infeasible'
        lo_tab[key] = k0lo; hi_tab[key] = (K0 - k0lo) >> S
        assert hi_tab[key] >= 0
    whi = max(max(hi_tab.values()).bit_length(), 1)
    return dict(w1=w1, LB=LB, S=S, whi=whi, lo=lo_tab, hi=hi_tab)

# ----------------------------------------------------------------- circuit pieces
def and_all_into(b, controls, target):
    k = len(controls)
    if k == 1: b.cx(controls[0], target); return
    if k == 2: b.ccx(controls[0], controls[1], target); return
    anc = b.alloc_qubits(k - 2)
    b.init_and(controls[0], controls[1], anc[0])
    for j in range(1, k - 2): b.init_and(anc[j - 1], controls[j + 1], anc[j])
    b.ccx(anc[k - 3], controls[k - 1], target)
    for j in reversed(range(1, k - 2)): b.del_and(anc[j - 1], controls[j + 1], anc[j])
    b.del_and(controls[0], controls[1], anc[0])
    b.free(anc)

def phase_all(b, controls):
    k = len(controls)
    if k == 1: b.z(controls[0]); return
    if k == 2: b.cz(controls[0], controls[1]); return
    anc = b.alloc_qubits(k - 2)
    b.init_and(controls[0], controls[1], anc[0])
    for j in range(1, k - 2): b.init_and(anc[j - 1], controls[j + 1], anc[j])
    b.cz(anc[k - 3], controls[k - 1])
    for j in reversed(range(1, k - 2)): b.del_and(anc[j - 1], controls[j + 1], anc[j])
    b.del_and(controls[0], controls[1], anc[0])
    b.free(anc)

NSTAGES = [(4, 16), (3, 8), (2, 4), (1, 2), (0, 1)]

def normalize(b, x, f):
    for (k, d) in NSTAGES:
        top = x[W - d:W]
        if d == 1:
            b.cx(x[W - 1], f[0]); b.x(f[0])
        else:
            b.x(top); and_all_into(b, list(top), f[k]); b.x(top)
        for i in reversed(range(0, W - d)): b.cswap(f[k], x[i], x[i + d])

def unnormalize(b, x, f):
    for (k, d) in reversed(NSTAGES):
        for i in range(0, W - d): b.cswap(f[k], x[i], x[i + d])
        top = x[W - d:W]
        if d == 1:
            b.x(f[0]); b.cx(x[W - 1], f[0])
        else:
            m = b.alloc_bits(1)[0]
            b.hmr(f[k], m)
            b.push_condition(m)                     # fixup runs on half the shots on average
            b.x(top); phase_all(b, list(top)); b.x(top)
            b.pop_condition()
            b.free(m)

def gadd(b, a, t, cin=None, sub=False):
    """t += a + cin (or t -= a + cin), Gidney adder: len(t)-1 Toffolis, uncompute by HMR."""
    a = list(a); t = list(t); o = len(a); n = len(t)
    assert o <= n
    if sub: b.x(km.array(t))
    c = b.alloc_qubits(n - 1) if n > 1 else km.array([])
    def C(i): return cin if i == 0 else c[i - 1]
    for i in range(0, n - 1):
        if i == 0 and cin is None:
            if o: b.init_and(a[0], t[0], C(1))
        elif i < o:
            b.cx(C(i), a[i]); b.cx(C(i), t[i]); b.init_and(a[i], t[i], C(i + 1)); b.cx(C(i), C(i + 1))
        else:
            b.init_and(C(i), t[i], C(i + 1))
    if n - 1 < o: b.cx(a[n - 1], t[n - 1])
    if n > 1 or cin is not None: b.cx(C(n - 1), t[n - 1])
    for i in reversed(range(0, n - 1)):
        if i == 0 and cin is None:
            if o: b.del_and(a[0], t[0], C(1)); b.cx(a[0], t[0])
        elif i < o:
            b.cx(C(i), C(i + 1)); b.del_and(a[i], t[i], C(i + 1)); b.cx(C(i), a[i]); b.cx(a[i], t[i])
        else:
            b.del_and(C(i), t[i], C(i + 1)); b.cx(C(i), t[i])
    if n > 1: b.free(c)
    if sub: b.x(km.array(t))

def shift_stages(b, f, reg, L, inverse):
    stages = [(1, 1), (2, 2), (3, 4), (4, 8)]
    occ = []; o = L
    for (k, d) in stages: occ.append(o); o = min(o + d, len(reg))
    seq = list(enumerate(stages))
    if not inverse:
        for idx, (k, d) in seq:
            for i in reversed(range(0, occ[idx])):
                if i + d < len(reg): b.cswap(f[k], reg[i], reg[i + d])
    else:
        for idx, (k, d) in reversed(seq):
            for i in range(0, occ[idx]):
                if i + d < len(reg): b.cswap(f[k], reg[i], reg[i + d])

def build_circuit(cfg, tabs, lay, *, parity_in_input=False, fixed_high=False):
    w2, w1, LB, S, whi = cfg.w2, lay['w1'], lay['LB'], lay['S'], lay['whi']
    gy, fy = cfg.gy, cfg.fy
    b = km.CircuitBuilder()
    x = b.create_quantum_register(W, name='x')
    flags = b.alloc_qubits(4 if parity_in_input else 5)
    f = ([x[W-1]] + list(flags)) if parity_in_input else flags
    if fixed_high:
        assert min(lay['hi'].values()) >= (1 << (whi-1))
        assert max(lay['hi'].values()) < (1 << whi)
        whi -= 1
    c2 = b.alloc_qubits(w2); R = b.alloc_qubits(w1); Lr = b.alloc_qubits(LB); Yh = b.alloc_qubits(whi)
    tgt = km.array(list(Lr) + list(R) + list(c2) + list(Yh))
    table = [0] * 256
    for (e, a), (K2, K1, K0) in tabs.items():
        table[a + 128 * e] = lay['lo'][(e, a)] | (K1 << LB) | (K2 << (LB + w1)) | ((lay['hi'][(e, a)] - ((1 << whi) if fixed_high else 0)) << (LB + w1 + w2))
    addr = km.array(list(x[14:21]) + [f[0]])
    delta = x[0:DB]
    def rrows(inverse):
        seq = cfg.rrow_list()
        if inverse: seq = list(reversed(seq))
        for i in seq:
            sh = cfg.sh2[i]
            if sh >= 0: off = c2[sh:w2]; t = R[0:w1]; cin = c2[sh - 1] if sh >= 1 else None
            else: off = c2[0:w2]; t = R[-sh:w1]; cin = None
            b.cx(delta[i], t); gadd(b, off, t, cin, sub=inverse); b.cx(delta[i], t)
    def yrows(inverse):
        seq = list(reversed(range(DB)))
        if inverse: seq.reverse()
        for i in seq:
            sh = cfg.sh1[i]; assert 0 <= sh < w1
            t = Lr[0:LB]; cin = R[sh - 1] if sh >= 1 else None
            b.cx(delta[i], t); gadd(b, R[sh:w1], t, cin, sub=inverse); b.cx(delta[i], t)
    if parity_in_input:
        for k,amount in NSTAGES[:-1]:
            top=x[W-amount:W];b.x(top);and_all_into(b,list(top),f[k]);b.x(top)
            for i in reversed(range(W-amount)):b.cswap(f[k],x[i],x[i+amount])
        b.x(x[W-1]);b.left_rotate(x[:W-1],1,control=x[W-1]);b.cx(x[W-1],x[0])
    else:normalize(b,x,f)
    b.init_lookup(table=table, address=addr, target=tgt)
    rrows(False); yrows(False)
    ext = b.alloc_qubits(10)
    if LB > S: gadd(b, Lr[S:LB], Yh)
    const=b.alloc_qubits(1) if fixed_high else []
    if fixed_high:b.x(const)
    Cx = km.array(list(Lr[gy:S]) + list(Yh) + list(const) + list(ext))
    shift_stages(b, f, Cx, S - gy + whi + int(fixed_high), False)
    for j in range(len(Cx)):
        e = gy + j - fy - 10                       # bit weight 2^e turns
        if e < 0: b.z_pow(Cx[j], Fraction(2)**(e + 1))
    shift_stages(b, f, Cx, S - gy + whi + int(fixed_high), True)
    if fixed_high:b.x(const);b.free(const)
    if LB > S: gadd(b, Lr[S:LB], Yh, sub=True)
    b.free(ext)
    yrows(True); rrows(True)
    b.del_lookup(table=table, address=addr, target=tgt)
    b.free(tgt)
    if parity_in_input:
        b.cx(x[W-1],x[0]);b.right_rotate(x[:W-1],1,control=x[W-1]);b.x(x[W-1])
        for k,amount in reversed(NSTAGES[:-1]):
            for i in range(W-amount):b.cswap(f[k],x[i],x[i+amount])
            top=x[W-amount:W];m=b.alloc_bits(1)[0];b.hmr(f[k],m);b.push_condition(m)
            b.x(top);phase_all(b,list(top));b.x(top);b.pop_condition();b.free(m)
    else:unnormalize(b,x,f)
    b.free(flags)
    return b.finish_circuit()

In [ ]:
%%writefile src/gen_invsqrt_r3/vendor/domain_tables.py
"""Integer minimax calibration on the exact reachable normalized domain."""
import math,functools
import numpy as np
import compact_horner as H
H.fitc=functools.lru_cache(None)(H.fitc)
MAX_X=3133452

def valid_residuals(e,a):
    d=np.arange(1<<H.DB,dtype=np.int64);xn=(1<<H.MB)+(a<<H.DB)+d
    mask=(d%2==0) if e else ((xn<=MAX_X)|(d%4==0))
    return d[mask]

def calibrate_segment(cfg,e,a,c_radius=1,b_radius=3):
    d=valid_residuals(e,a);lo=1+a*2.**(H.DB-H.MB);mid=lo+((1<<H.DB)-1)/2**(H.MB+1)
    scale=2**((e-1)/2);co=H.fitc(lo,lo+2.**(H.DB-H.MB),mid,scale)
    f=scale/np.sqrt(lo+d*2.**-H.MB);target=f*2**cfg.fy;w=1/f
    K2c=round(co[2]*2**cfg.f2);K1c=round(-co[1]*2**cfg.f1);best=None;quant=1<<cfg.gy
    for K2 in range(max(0,K2c-c_radius),min((1<<cfg.w2)-1,K2c+c_radius)+1):
        for K1 in range(K1c-b_radius,K1c+b_radius+1):
            R,Q=H.pipeline(cfg,K2,K1,d)
            if R.min()<0:continue
            vals=target+Q;center=.5*(vals.min()+vals.max())+.5*(quant-1)
            k0s=np.arange(math.floor(center)-quant-1,math.ceil(center)+quant+2,dtype=np.int64)
            # The stored high anchor and the carry must keep the leading one fixed.
            k0s=k0s[k0s <= (1<<cfg.fy)-1+int(Q.min())]
            if len(k0s)==0:continue
            y=((k0s[:,None]-Q)>>cfg.gy)<<cfg.gy
            scores=np.max(abs(y-target)*w,axis=1)/2**cfg.fy;i=int(scores.argmin())
            v=float(scores[i]);K0=int(k0s[i])
            if best is None or v<best[0]:best=(v,K2,K1,K0)
    if best is None:raise RuntimeError('No legal coefficient triple')
    return best

def build_tables(cfg,refine_threshold=.95):
    tabs={};ratios=[]
    for e in(0,1):
        for a in range(128):
            best=calibrate_segment(cfg,e,a)
            if best[0]*2**24>refine_threshold:
                alt=calibrate_segment(cfg,e,a,c_radius=4,b_radius=7)
                if alt[0]<best[0]:best=alt
            tabs[(e,a)]=best[1:];ratios.append(best[0]*2**24)
    return tabs,dict(worst_ratio=max(ratios),worst_key=[int(np.argmax(ratios))//128,int(np.argmax(ratios))%128],reachable_normalized_inputs=sum(len(valid_residuals(e,a)) for e in(0,1) for a in range(128)))

In [ ]:
%%writefile src/gen_invsqrt_r3/vendor/signed_digit_lab.py
# Attribution: Sumit Chongder, Apache-2.0 as published.
# https://www.kaggle.com/code/sumitchongder/notebook2f196e646b
# Read-only source downloaded from latest version 2. Kirill's v9 cites v1.
# This isolated lab regenerates the circuit from readable source; no binaries or
# remote notebook outputs are inputs. Our added exact integer certificate and
# reporting helpers and memory-schedule adaptations follow the original generator.
"""Builds an optimized kickmix circuit for the invsqrt benchmark (Benchmark I).

|x>|0> -> exp(2 pi i / sqrt(x)) |x>|0>,   x in [1, 3133452], 22-bit register 'x'.

Design (all arithmetic exactly mirrored by the numpy model below):
  1. Normalize x by single-bit binary stages (16, 8, 4, 2, 1) into M = x_norm / 2^21 in [1, 2);
     flags f0..f4 hold the shift count s' in binary.  e = f0 (parity), s = s' >> 1.
     1/sqrt(x) = 2^(s-10) * y,  y = 2^((e-1)/2) / sqrt(M)  in (0.5, 1].
  2. One 256-entry QROM indexed by (7 mantissa bits, e) XORs quadratic coefficients directly into
     the working registers (no separate coefficient copies).
  3. Centered signed-digit Horner: delta_c = sum_i (2 d_i - 1) 2^(i-22) over the 14 low bits.
     R = K1 - C2*delta_c,  y = K0 - R*delta_c.  Each row is an UNcontrolled Gidney addition
     wrapped in CNOT flips (Y - A = ~(~Y + A)), with a rounding carry-in.
  4. The y register is split: rows only touch a 21-bit low register L; the constant high part
     sits in a separate register and the overflow bit is merged in before phasing.
  5. Binary de-normalization (shift by s), one Z_POW per bit, then exact mirror uncompute.
     Lookup uncompute is measurement based (del_lookup); normalization flags are cleared by HMR
     with classically conditioned phase fixups.
Usage: python signed_digit_lab.py OUT.kmb [--baseline] (requires kickmix and NumPy)
"""
import sys, math
from fractions import Fraction
import numpy as np
import kickmix as km

W = 22; MB = 21; DB = 14; NSEG = 128
F2, F1, FYR, GY, DROP2 = 10, 19, 26, 2, (5,)     # precision configuration (verified)

# ----------------------------------------------------------------- numpy model / tables
def fitc(lo, hi, mid, scale, deg=2, iters=80):
    f = lambda M: scale / np.sqrt(M)
    d = np.linspace(lo - mid, hi - mid, 3001)
    A = np.vander(d, deg + 1, increasing=True)
    w = np.sqrt(mid + d) / scale
    wt = np.ones_like(d)
    for _ in range(iters):                       # Lawson iterations -> weighted minimax
        sw = w * np.sqrt(wt)
        coef = np.linalg.lstsq(A * sw[:, None], f(mid + d) * sw, rcond=None)[0]
        err = np.abs((A @ coef - f(mid + d)) * w)
        wt = wt * err; wt /= wt.sum()
    return coef

class Cfg:
    def __init__(self, f2, f1, fyr, gy, drop2=()):
        self.f2, self.f1, self.fyr, self.gy = f2, f1, fyr, gy
        self.fy = fyr + gy
        self.w2 = f2 - 1
        self.sh2 = [f2 + 22 - i - f1 for i in range(DB)]
        self.sh1 = [f1 + 22 - i - self.fy for i in range(DB)]
        self.drop2 = set(drop2)
    def rrow_list(self):
        return [i for i in reversed(range(DB)) if i not in self.drop2 and self.sh2[i] < self.w2]

def rd(v, s):
    if s < 0: return v << (-s)
    r = v >> s
    if s >= 1: r = r + ((v >> (s - 1)) & 1)
    return r

def pipeline(cfg, K2, K1, d):
    sig = [2 * ((d >> i) & 1) - 1 for i in range(DB)]
    R = np.full_like(d, K1)
    for i in cfg.rrow_list():
        R = R - sig[i] * rd(K2, cfg.sh2[i])
    Q = np.zeros_like(d)
    for i in reversed(range(DB)):
        Q = Q + sig[i] * rd(R, cfg.sh1[i])
    return R, Q

def build_tables(cfg, search=2):
    tabs = {}; worst = 0
    dmid = (2**DB - 1) / 2**(MB + 1)
    d = np.arange(2**DB, dtype=np.int64)
    g = cfg.gy
    for e in (0, 1):
        scale = 2**((e - 1) / 2)
        for a in range(NSEG):
            lo = 1 + a * 2.0**(DB - MB); mid = lo + dmid
            c = fitc(lo, lo + 2.0**(DB - MB), mid, scale)
            C1, C2 = -c[1], c[2]
            M = lo + d * 2.0**-MB
            fM = scale / np.sqrt(M); tgt = fM * 2.0**cfg.fy; w = 1 / fM
            K2c = int(round(C2 * 2**cfg.f2)); K1c = int(round(C1 * 2**cfg.f1))
            best = None
            for K2 in range(K2c - 1, K2c + 2):
                for K1 in range(K1c - search, K1c + search + 1):
                    R, Q = pipeline(cfg, K2, K1, d)
                    def mx(K0):
                        Y = ((K0 - Q) >> g) << g
                        return float(np.max(np.abs(Y - tgt) * w))
                    t = tgt + Q
                    a_, b_ = math.floor(t.min()), math.ceil(t.max())
                    while b_ - a_ > 2:
                        m1 = a_ + (b_ - a_) // 3; m2 = b_ - (b_ - a_) // 3
                        if mx(m1) < mx(m2): b_ = m2
                        else: a_ = m1
                    for K0 in range(a_, b_ + 1):
                        v = mx(K0) / 2.0**cfg.fy
                        if best is None or v < best[0]: best = (v, K2, K1, K0)
            tabs[(e, a)] = best[1:]
            worst = max(worst, best[0])
    return tabs, worst

def layout(cfg, tabs, LB=21, S=20):
    d = np.arange(2**DB, dtype=np.int64)
    assert max(t[0] for t in tabs.values()) < (1 << cfg.w2)
    Rmax = 0; per = {}; Ymax = 0
    for key, (K2, K1, K0) in tabs.items():
        R, Q = pipeline(cfg, K2, K1, d)
        assert R.min() >= 0
        Rmax = max(Rmax, int(R.max()), K1)
        per[key] = (int(Q.min()), int(Q.max()))
        Ymax = max(Ymax, K0 - per[key][0])
    w1 = Rmax.bit_length()
    lo_tab, hi_tab = {}, {}
    for key, (K2, K1, K0) in tabs.items():
        qmin, qmax = per[key]
        base = K0 % (1 << S); j = 0
        while base + j * (1 << S) - qmax < 0: j += 1
        k0lo = base + j * (1 << S)
        assert k0lo - qmin < (1 << LB), 'layout infeasible'
        lo_tab[key] = k0lo; hi_tab[key] = (K0 - k0lo) >> S
        assert hi_tab[key] >= 0
    whi = max(max(hi_tab.values()).bit_length(), 1)
    return dict(w1=w1, LB=LB, S=S, whi=whi, lo=lo_tab, hi=hi_tab)

# ----------------------------------------------------------------- circuit pieces
def and_all_into(b, controls, target):
    k = len(controls)
    if k == 1: b.cx(controls[0], target); return
    if k == 2: b.ccx(controls[0], controls[1], target); return
    anc = b.alloc_qubits(k - 2)
    b.init_and(controls[0], controls[1], anc[0])
    for j in range(1, k - 2): b.init_and(anc[j - 1], controls[j + 1], anc[j])
    b.ccx(anc[k - 3], controls[k - 1], target)
    for j in reversed(range(1, k - 2)): b.del_and(anc[j - 1], controls[j + 1], anc[j])
    b.del_and(controls[0], controls[1], anc[0])
    b.free(anc)

def phase_all(b, controls):
    k = len(controls)
    if k == 1: b.z(controls[0]); return
    if k == 2: b.cz(controls[0], controls[1]); return
    anc = b.alloc_qubits(k - 2)
    b.init_and(controls[0], controls[1], anc[0])
    for j in range(1, k - 2): b.init_and(anc[j - 1], controls[j + 1], anc[j])
    b.cz(anc[k - 3], controls[k - 1])
    for j in reversed(range(1, k - 2)): b.del_and(anc[j - 1], controls[j + 1], anc[j])
    b.del_and(controls[0], controls[1], anc[0])
    b.free(anc)

NSTAGES = [(4, 16), (3, 8), (2, 4), (1, 2), (0, 1)]

def normalize(b, x, f):
    for (k, d) in NSTAGES:
        top = x[W - d:W]
        if d == 1:
            b.cx(x[W - 1], f[0]); b.x(f[0])
        else:
            b.x(top); and_all_into(b, list(top), f[k]); b.x(top)
        for i in reversed(range(0, W - d)): b.cswap(f[k], x[i], x[i + d])

def unnormalize(b, x, f):
    for (k, d) in reversed(NSTAGES):
        for i in range(0, W - d): b.cswap(f[k], x[i], x[i + d])
        top = x[W - d:W]
        if d == 1:
            b.x(f[0]); b.cx(x[W - 1], f[0])
        else:
            m = b.alloc_bits(1)[0]
            b.hmr(f[k], m)
            b.push_condition(m)                     # fixup runs on half the shots on average
            b.x(top); phase_all(b, list(top)); b.x(top)
            b.pop_condition()
            b.free(m)

def gadd(b, a, t, cin=None, sub=False):
    """t += a + cin (or t -= a + cin), Gidney adder: len(t)-1 Toffolis, uncompute by HMR."""
    a = list(a); t = list(t); o = len(a); n = len(t)
    assert o <= n
    if sub: b.x(km.array(t))
    c = b.alloc_qubits(n - 1) if n > 1 else km.array([])
    def C(i): return cin if i == 0 else c[i - 1]
    for i in range(0, n - 1):
        if i == 0 and cin is None:
            if o: b.init_and(a[0], t[0], C(1))
        elif i < o:
            b.cx(C(i), a[i]); b.cx(C(i), t[i]); b.init_and(a[i], t[i], C(i + 1)); b.cx(C(i), C(i + 1))
        else:
            b.init_and(C(i), t[i], C(i + 1))
    if n - 1 < o: b.cx(a[n - 1], t[n - 1])
    if n > 1 or cin is not None: b.cx(C(n - 1), t[n - 1])
    for i in reversed(range(0, n - 1)):
        if i == 0 and cin is None:
            if o: b.del_and(a[0], t[0], C(1)); b.cx(a[0], t[0])
        elif i < o:
            b.cx(C(i), C(i + 1)); b.del_and(a[i], t[i], C(i + 1)); b.cx(C(i), a[i]); b.cx(a[i], t[i])
        else:
            b.del_and(C(i), t[i], C(i + 1)); b.cx(C(i), t[i])
    if n > 1: b.free(c)
    if sub: b.x(km.array(t))

def shift_stages(b, f, reg, L, inverse):
    stages = [(1, 1), (2, 2), (3, 4), (4, 8)]
    occ = []; o = L
    for (k, d) in stages: occ.append(o); o = min(o + d, len(reg))
    seq = list(enumerate(stages))
    if not inverse:
        for idx, (k, d) in seq:
            for i in reversed(range(0, occ[idx])):
                if i + d < len(reg): b.cswap(f[k], reg[i], reg[i + d])
    else:
        for idx, (k, d) in reversed(seq):
            for i in range(0, occ[idx]):
                if i + d < len(reg): b.cswap(f[k], reg[i], reg[i + d])

def build_circuit(cfg, tabs, lay, adder=None):
    # The default preserves the public generator exactly. An injected adder
    # supports isolated, explicitly tested memory experiments.
    if adder is None: adder=gadd
    w2, w1, LB, S, whi = cfg.w2, lay['w1'], lay['LB'], lay['S'], lay['whi']
    gy, fy = cfg.gy, cfg.fy
    b = km.CircuitBuilder()
    x = b.create_quantum_register(W, name='x')
    f = b.alloc_qubits(5)
    c2 = b.alloc_qubits(w2); R = b.alloc_qubits(w1); Lr = b.alloc_qubits(LB); Yh = b.alloc_qubits(whi)
    tgt = km.array(list(Lr) + list(R) + list(c2) + list(Yh))
    table = [0] * 256
    for (e, a), (K2, K1, K0) in tabs.items():
        table[a + 128 * e] = lay['lo'][(e, a)] | (K1 << LB) | (K2 << (LB + w1)) | (lay['hi'][(e, a)] << (LB + w1 + w2))
    addr = km.array(list(x[14:21]) + [f[0]])
    delta = x[0:DB]
    def rrows(inverse):
        seq = cfg.rrow_list()
        if inverse: seq = list(reversed(seq))
        for i in seq:
            sh = cfg.sh2[i]
            if sh >= 0: off = c2[sh:w2]; t = R[0:w1]; cin = c2[sh - 1] if sh >= 1 else None
            else: off = c2[0:w2]; t = R[-sh:w1]; cin = None
            b.cx(delta[i], t); adder(b, off, t, cin, sub=inverse); b.cx(delta[i], t)
    def yrows(inverse):
        seq = list(reversed(range(DB)))
        if inverse: seq.reverse()
        for i in seq:
            sh = cfg.sh1[i]; assert 0 <= sh < w1
            t = Lr[0:LB]; cin = R[sh - 1] if sh >= 1 else None
            b.cx(delta[i], t); adder(b, R[sh:w1], t, cin, sub=inverse); b.cx(delta[i], t)
    normalize(b, x, f)
    b.init_lookup(table=table, address=addr, target=tgt)
    rrows(False); yrows(False)
    ext = b.alloc_qubits(10)
    if LB > S: adder(b, Lr[S:LB], Yh)
    Cx = km.array(list(Lr[gy:S]) + list(Yh) + list(ext))
    shift_stages(b, f, Cx, S - gy + whi, False)
    for j in range(len(Cx)):
        e = gy + j - fy - 10                       # bit weight 2^e turns
        if e < 0: b.z_pow(Cx[j], Fraction(2)**(e + 1))
    shift_stages(b, f, Cx, S - gy + whi, True)
    if LB > S: adder(b, Lr[S:LB], Yh, sub=True)
    b.free(ext)
    yrows(True); rrows(True)
    b.del_lookup(table=table, address=addr, target=tgt)
    b.free(tgt)
    unnormalize(b, x, f)
    b.free(f)
    return b.finish_circuit()

def main(out):
    cfg = Cfg(F2, F1, FYR, GY, DROP2)
    tabs, worst = build_tables(cfg)
    print(f'model worst relative error / 2^-24 = {worst * 2**24:.3f}')
    lay = layout(cfg, tabs)
    c = build_circuit(cfg, tabs, lay)
    s = c.operation_statistics()
    print('qubits', c.num_qubits, 'toffoli(static)', s.num_toffoli_like,
          'rotations', s.num_z_pow(min_clifford_hierarchy_level=4))
    c.to_file(out, format='kmb')
    print('wrote', out)


def selected_cfg():
    return Cfg(F2,F1,FYR,GY,DROP2)


def config_record(cfg):
    return dict(f2=cfg.f2,f1=cfg.f1,fyr=cfg.fyr,gy=cfg.gy,drop2=sorted(cfg.drop2))


def load_or_build_tables(path=None):
    """Regenerate open-source coefficients, optionally retaining a local cache."""
    import json
    from pathlib import Path
    cfg=selected_cfg()
    path=Path(path) if path else None
    if path and path.exists():
        saved=json.loads(path.read_text())
        assert saved['config']==config_record(cfg)
        rows=saved['rows']
        assert len(rows)==256 and all(len(row)==3 for row in rows)
        tabs={(i//128,i%128):tuple(map(int,row)) for i,row in enumerate(rows)}
        return cfg,tabs,saved['normalized_worst_relative_error']
    tabs,worst=build_tables(cfg)
    if path:
        path.write_text(json.dumps({'source':'Sumit Chongder open generator, latest v2',
            'config':config_record(cfg),'normalized_worst_relative_error':worst,
            'rows':[tabs[(e,a)] for e in range(2) for a in range(128)]},indent=2))
    return cfg,tabs,worst


def integer_model(xs,cfg,tabs):
    """Exact emitted arithmetic in units 2^-(cfg.fy+10) turns, before mod 1."""
    xs=np.asarray(xs,dtype=np.int64)
    if not ((xs>=1)&(xs<=3133452)).all():
        raise ValueError('The supported competition domain is x=1..3133452')
    # All inputs fit exactly in float64; frexp gives the exact binary exponent.
    shifts=21-(np.frexp(xs)[1].astype(np.int64)-1)
    norm=xs << shifts
    assert ((norm >> 21)==1).all(), 'Normalized top-bit borrowing requires x[21]=1'
    parity=shifts&1
    index=((norm >> 14)&127)+128*parity
    table=np.asarray([tabs[(e,a)] for e in range(2) for a in range(128)],dtype=np.int64)
    K2,K1,K0=(table[index,j] for j in range(3))
    d=norm&((1 << DB)-1)
    signs=[2*((d >> i)&1)-1 for i in range(DB)]
    R=K1.copy()
    for i in cfg.rrow_list():
        R-=signs[i]*rd(K2,cfg.sh2[i])
    Q=np.zeros_like(xs)
    for i in reversed(range(DB)):
        Q+=signs[i]*rd(R,cfg.sh1[i])
    Y=((K0-Q) >> cfg.gy) << cfg.gy
    if (R<0).any() or (Y<0).any():
        raise OverflowError('An unsigned intermediate is negative')
    return Y << (shifts >> 1)


def exhaustive_certificate(cfg,tabs,chunk_size=131072):
    """Every promised basis input, exact integer target and tolerance.

    This is an arithmetic certificate. Ancillas, restored inputs and phase
    feed-forward require fresh circuit simulation and the official verifier.
    """
    output_fraction=cfg.fy+10
    worst_ratio,worst_x,min_margin=0.,0,1 << 64
    for start in range(1,3133453,chunk_size):
        xs=np.arange(start,min(start+chunk_size,3133453),dtype=np.int64)
        values=integer_model(xs,cfg,tabs).astype(np.uint64)
        phase=values << np.uint64(64-output_fraction)
        roots=[math.isqrt((1 << 130)//int(x)) for x in xs]
        target=np.array([((r+1) >> 1)&((1 << 64)-1) for r in roots],dtype=np.uint64)
        tolerance=np.array([min(r >> 25,1 << 63) for r in roots],dtype=np.uint64)
        error=np.abs((phase-target).view(np.int64)).view(np.uint64)
        ratio=error/tolerance
        i=int(np.argmax(ratio))
        if float(ratio[i])>worst_ratio:
            worst_ratio,worst_x=float(ratio[i]),int(xs[i])
        if (error>tolerance).any():
            i=int(np.flatnonzero(error>tolerance)[0])
            return {'pass':False,'first_bad_x':int(xs[i]),'ratio':float(ratio[i])}
        min_margin=min(min_margin,int((tolerance-error).min()))
    return {'pass':True,'tested_inputs':3133452,'worst_ratio':worst_ratio,
            'worst_x':worst_x,'min_margin_units':min_margin,
            'phase_fraction_bits':output_fraction}


def estimated_resources(circuit,certificate):
    """Model-based cost; full unchanged official verification remains required."""
    assert certificate['pass'] and certificate['min_margin_units']>0
    stats=circuit.operation_statistics()
    rotations=int(stats.num_z_pow(min_clifford_hierarchy_level=4))
    epsilon=math.pi*(certificate['min_margin_units']/2**64)/rotations
    per_rotation=0.56*math.log2(1/epsilon)+5.3
    static_toffoli=int(stats.num_toffoli_like)
    # Four X-basis flag measurements independently activate phase_all on half
    # the shots. Its 16,8,4,2-control paths have 14+6+2+0 Toffolis in total.
    normalization_discount=sum(max(0,d-2) for _,d in NSTAGES if d>1)/2
    expected_toffoli=static_toffoli-normalization_discount
    t_like=int(stats.num_t_like)
    extra=(t_like+rotations*per_rotation)/4
    return {'qubits':circuit.num_qubits,'static_toffoli':static_toffoli,
            'expected_mean_toffoli':expected_toffoli,'t_like':t_like,
            'rotations':rotations,'rotation_epsilon':epsilon,
            'estimated_expected_cost':circuit.num_qubits*(expected_toffoli+extra),
            'static_cost_upper_estimate':circuit.num_qubits*(static_toffoli+extra),
            'full_official_verification_required':True}




# ---- Our isolated phase scheduling, folded carry and borrowed workspace ----
def folded_gadd(b,a,t,cin=None,sub=False,borrowed=()):
    a,t=list(a),list(t)
    o,n=len(a),len(t)
    assert o<=n and n>=1
    if sub:
        for q in t:b.x(q)
    if n==1:
        if o:b.cx(a[0],t[0])
        if cin is not None:b.cx(cin,t[0])
        if sub:
            for q in t:b.x(q)
        return
    borrowed=list(borrowed)[:max(0,n-2)]
    count=max(0,n-2-len(borrowed))
    allocated=b.alloc_qubits(count) if count else []
    c=borrowed+list(allocated)
    def C(i):return cin if i==0 else c[i-1]
    for i in range(n-2):
        if i==0 and cin is None:
            if o:b.init_and(a[0],t[0],C(1))
        elif i<o:
            b.cx(C(i),a[i]);b.cx(C(i),t[i])
            b.init_and(a[i],t[i],C(i+1));b.cx(C(i),C(i+1))
        else:
            b.init_and(C(i),t[i],C(i+1))
    i=n-2
    carry=C(i)
    if carry is None:
        if i<o:
            b.ccx(a[i],t[i],t[-1]);b.cx(a[i],t[i])
    elif i<o:
        b.cx(carry,a[i]);b.cx(carry,t[i])
        b.ccx(a[i],t[i],t[-1]);b.cx(carry,t[-1])
        b.cx(carry,a[i]);b.cx(a[i],t[i])
    else:
        b.ccx(carry,t[i],t[-1]);b.cx(carry,t[i])
    if n-1<o:b.cx(a[n-1],t[n-1])
    for i in reversed(range(n-2)):
        if i==0 and cin is None:
            if o:b.del_and(a[0],t[0],C(1));b.cx(a[0],t[0])
        elif i<o:
            b.cx(C(i),C(i+1));b.del_and(a[i],t[i],C(i+1))
            b.cx(C(i),a[i]);b.cx(a[i],t[i])
        else:
            b.del_and(C(i),t[i],C(i+1));b.cx(C(i),t[i])
    if count:b.free(allocated)
    if sub:
        for q in t:b.x(q)


def truth_table_check(max_width=6,borrowed_count=0):
    class Bits:
        def __init__(self):self.v=[]
        def alloc_qubits(self,n):
            ids=list(range(len(self.v),len(self.v)+n));self.v.extend([0]*n);return ids
        def x(self,q):self.v[q]^=1
        def cx(self,a,t):self.v[t]^=self.v[a]
        def ccx(self,a,b,t):self.v[t]^=self.v[a]&self.v[b]
        def init_and(self,a,b,t):
            assert self.v[t]==0
            self.v[t]=self.v[a]&self.v[b]
        def del_and(self,a,b,t):
            assert self.v[t]==(self.v[a]&self.v[b])
            self.v[t]=0
        def free(self,reg):assert not any(self.v[q] for q in reg)
        def put(self,reg,value):
            for i,q in enumerate(reg):self.v[q]=(value >> i)&1
        def read(self,reg):return sum(self.v[q] << i for i,q in enumerate(reg))
    count=0
    for n in range(1,max_width+1):
        for m in range(n+1):
            for a in range(1 << m):
                for value in range(1 << n):
                    for cin_value in (None,0,1):
                        for sub in (False,True):
                            b=Bits();areg=b.alloc_qubits(m);target=b.alloc_qubits(n)
                            cin=b.alloc_qubits(1)[0] if cin_value is not None else None
                            if cin is not None:b.v[cin]=cin_value
                            b.put(areg,a);b.put(target,value)
                            borrowed=b.alloc_qubits(borrowed_count)
                            folded_gadd(b,areg,target,cin,sub,borrowed=borrowed)
                            delta=a+(cin_value or 0)
                            expected=(value-delta if sub else value+delta) % (1 << n)
                            assert b.read(target)==expected,(n,m,a,value,cin_value,sub)
                            assert b.read(areg)==a
                            assert cin is None or b.v[cin]==cin_value
                            assert not any(b.v[q] for q in borrowed)
                            count+=1
    return count


def build_split_circuit(cfg,tabs,lay,adder=None,borrow_top=False):
    if adder is None:adder=gadd
    w2,w1,LB,S,whi=cfg.w2,lay['w1'],lay['LB'],lay['S'],lay['whi']
    b=km.CircuitBuilder()
    x=b.create_quantum_register(W,name='x')
    flags=b.alloc_qubits(5)
    normalize(b,x,flags)
    # Every promised positive input normalizes to x[21]=1. The arithmetic
    # address uses x[14:21] and delta x[:14], so this wire is idle in the core.
    if borrow_top:
        b.x(x[-1])
        raw_adder=adder
        def adder(builder,a,t,cin=None,sub=False):
            return raw_adder(builder,a,t,cin,sub,borrowed=(x[-1],))
    c2=b.alloc_qubits(w2)
    R=b.alloc_qubits(w1)
    Lr=b.alloc_qubits(LB)
    Yh=b.alloc_qubits(whi)
    remaining=km.array(list(Lr)+list(R)+list(c2))
    target=km.array(list(remaining)+list(Yh))
    table=[0]*256
    rest_table=[0]*256
    high_table=[0]*256
    for (e,a),(K2,K1,K0) in tabs.items():
        idx=a+128*e
        rest_table[idx]=lay['lo'][(e,a)] | (K1 << LB) | (K2 << (LB+w1))
        high_table[idx]=lay['hi'][(e,a)]
        table[idx]=rest_table[idx] | (high_table[idx] << (LB+w1+w2))
    address=km.array(list(x[14:21])+[flags[0]])
    delta=x[:DB]
    def rrows(inverse):
        seq=cfg.rrow_list()
        if inverse:seq=list(reversed(seq))
        for i in seq:
            sh=cfg.sh2[i]
            if sh>=0:off=c2[sh:w2];t=R[:w1];cin=c2[sh-1] if sh>=1 else None
            else:off=c2[:w2];t=R[-sh:w1];cin=None
            b.cx(delta[i],t);adder(b,off,t,cin,sub=inverse);b.cx(delta[i],t)
    def yrows(inverse):
        seq=list(reversed(range(DB)))
        if inverse:seq.reverse()
        for i in seq:
            sh=cfg.sh1[i]
            assert 0<=sh<w1
            cin=R[sh-1] if sh>=1 else None
            b.cx(delta[i],Lr);adder(b,R[sh:w1],Lr,cin,sub=inverse);b.cx(delta[i],Lr)
    def phase(reg,base_bit):
        ext=b.alloc_qubits(10)
        word=km.array(list(reg)+list(ext))
        shift_stages(b,flags,word,len(reg),False)
        for j in range(len(word)):
            exponent=base_bit+j-cfg.fy-10
            if exponent<0:b.z_pow(word[j],Fraction(2)**(exponent+1))
        shift_stages(b,flags,word,len(reg),True)
        b.free(ext)
    b.init_lookup(table=table,address=address,target=target)
    phase(Yh,S)
    b.del_lookup(table=high_table,address=address,target=Yh)
    b.free(Yh)
    rrows(False)
    yrows(False)
    phase(Lr[cfg.gy:LB],cfg.gy)
    yrows(True)
    rrows(True)
    b.del_lookup(table=rest_table,address=address,target=remaining)
    b.free(remaining)
    if borrow_top:b.x(x[-1])
    unnormalize(b,x,flags)
    b.free(flags)
    return b.finish_circuit()


def build_selected(cfg,tabs,lay=None):
    """Selected unchanged-Sumit model with split high phase and 93-qubit schedule.

    The normalized input's top bit is known to be one throughout the core;
    it is temporarily zeroed for a carry wire and restored before unnormalizing.
    Every call regenerates a circuit from source and ordinary numeric tables.
    """
    if lay is None:lay=layout(cfg,tabs)
    return build_split_circuit(cfg,tabs,lay,adder=folded_gadd,borrow_top=True)


def selected_descriptor():
    return {
        'variant':'sumit_split_folded_borrow',
        'source_author':'Sumit Chongder',
        'source_url':'https://www.kaggle.com/code/sumitchongder/notebook2f196e646b?scriptVersionId=361357149',
        'source_download':'Latest v2 readable generator; regenerated baseline canonical bytes match cited v1 report',
        'license_as_published':'Apache-2.0',
        'coefficient_changes':'None; original Sumit Cfg(10,19,26,2,(5,)) and generated coefficient table',
        'own_changes':[
            'Allocate coefficient registers after normalization',
            'Phase constant high word and measurement-unlookup it before Horner arithmetic',
            'Phase the low accumulator including its overflow bit separately; remove high/low merge adders',
            'Fold final ripple carry directly into highest target bit, including the rounding carry-in',
            'Borrow normalized x[21] as a clean carry wire; restore its value before unnormalization'],
        'baseline_canonical_sha256':'f19a8207e282859f663abe7f655b06eb3224d32ac99fbf899e1422b3795ca0be',
        'selected_canonical_sha256':'85611d22c524a5e982f2beafd11e32e6d39c86e174d3438734ff5a088679090a',
        'measured_counts':{'qubits':93,'static_toffoli':1582,'rotations':44},
        'expected_mean_toffoli':1571.0,
        'estimated_cost':175203.2709288877,
        'official_full_verification':'Required separately; no arithmetic certificate substitutes for it'}


def comparison_report(cfg,tabs,certificate,lay=None):
    """Measured circuit counts and model-based estimates, without executing verification."""
    if lay is None:lay=layout(cfg,tabs)
    baseline=build_circuit(cfg,tabs,lay)
    selected=build_selected(cfg,tabs,lay)
    return {'baseline':estimated_resources(baseline,certificate),
            'selected':estimated_resources(selected,certificate),
            'selected_descriptor':selected_descriptor()}


def cli():
    import argparse
    import json
    from pathlib import Path
    parser = argparse.ArgumentParser(description='Generate the credited signed-digit circuit and memory-schedule adaptation.')
    parser.add_argument('output', nargs='?', default='invsqrt.kmb')
    parser.add_argument('--baseline', action='store_true')
    args = parser.parse_args()
    cfg, tabs, _ = load_or_build_tables()
    certificate = exhaustive_certificate(cfg, tabs)
    if not certificate['pass']:
        raise ArithmeticError(certificate)
    lay = layout(cfg, tabs)
    circuit = build_circuit(cfg, tabs, lay) if args.baseline else build_selected(cfg, tabs, lay)
    circuit.to_file(args.output, format='kmb')
    report = estimated_resources(circuit, certificate)
    report['arithmetic_certificate'] = certificate
    report['variant'] = 'sumit_baseline' if args.baseline else selected_descriptor()['variant']
    Path(args.output + '.resources.json').write_text(json.dumps(report, indent=2), encoding='utf-8')
    print(json.dumps(report, indent=2))


if __name__ == '__main__':
    cli()

In [ ]:
%%writefile src/gen_invsqrt_r3/vendor/sumit_reference.py
"""Builds an optimized kickmix circuit for the invsqrt benchmark (Benchmark I).

|x>|0> -> exp(2 pi i / sqrt(x)) |x>|0>,   x in [1, 3133452], 22-bit register 'x'.

Design (all arithmetic exactly mirrored by the numpy model below):
  1. Normalize x by single-bit binary stages (16, 8, 4, 2, 1) into M = x_norm / 2^21 in [1, 2);
     flags f0..f4 hold the shift count s' in binary.  e = f0 (parity), s = s' >> 1.
     1/sqrt(x) = 2^(s-10) * y,  y = 2^((e-1)/2) / sqrt(M)  in (0.5, 1].
  2. One 256-entry QROM indexed by (7 mantissa bits, e) XORs quadratic coefficients directly into
     the working registers (no separate coefficient copies).
  3. Centered signed-digit Horner: delta_c = sum_i (2 d_i - 1) 2^(i-22) over the 14 low bits.
     R = K1 - C2*delta_c,  y = K0 - R*delta_c.  Each row is an UNcontrolled Gidney addition
     wrapped in CNOT flips (Y - A = ~(~Y + A)), with a rounding carry-in.
  4. The y register is split: rows only touch a 21-bit low register L; the constant high part
     sits in a separate register and the overflow bit is merged in before phasing.
  5. Binary de-normalization (shift by s), one Z_POW per bit, then exact mirror uncompute.
     Lookup uncompute is measurement based (del_lookup); normalization flags are cleared by HMR
     with classically conditioned phase fixups.
Usage:  python build_invsqrt.py OUT.kmb   (needs kickmix importable and numpy)
"""
import sys, math
from fractions import Fraction
import numpy as np
import kickmix as km

W = 22; MB = 21; DB = 14; NSEG = 128
F2, F1, FYR, GY, DROP2 = 10, 19, 26, 2, (5,)     # precision configuration (verified)

# ----------------------------------------------------------------- numpy model / tables
def fitc(lo, hi, mid, scale, deg=2, iters=80):
    f = lambda M: scale / np.sqrt(M)
    d = np.linspace(lo - mid, hi - mid, 3001)
    A = np.vander(d, deg + 1, increasing=True)
    w = np.sqrt(mid + d) / scale
    wt = np.ones_like(d)
    for _ in range(iters):                       # Lawson iterations -> weighted minimax
        sw = w * np.sqrt(wt)
        coef = np.linalg.lstsq(A * sw[:, None], f(mid + d) * sw, rcond=None)[0]
        err = np.abs((A @ coef - f(mid + d)) * w)
        wt = wt * err; wt /= wt.sum()
    return coef

class Cfg:
    def __init__(self, f2, f1, fyr, gy, drop2=()):
        self.f2, self.f1, self.fyr, self.gy = f2, f1, fyr, gy
        self.fy = fyr + gy
        self.w2 = f2 - 1
        self.sh2 = [f2 + 22 - i - f1 for i in range(DB)]
        self.sh1 = [f1 + 22 - i - self.fy for i in range(DB)]
        self.drop2 = set(drop2)
    def rrow_list(self):
        return [i for i in reversed(range(DB)) if i not in self.drop2 and self.sh2[i] < self.w2]

def rd(v, s):
    if s < 0: return v << (-s)
    r = v >> s
    if s >= 1: r = r + ((v >> (s - 1)) & 1)
    return r

def pipeline(cfg, K2, K1, d):
    sig = [2 * ((d >> i) & 1) - 1 for i in range(DB)]
    R = np.full_like(d, K1)
    for i in cfg.rrow_list():
        R = R - sig[i] * rd(K2, cfg.sh2[i])
    Q = np.zeros_like(d)
    for i in reversed(range(DB)):
        Q = Q + sig[i] * rd(R, cfg.sh1[i])
    return R, Q

def build_tables(cfg, search=2):
    tabs = {}; worst = 0
    dmid = (2**DB - 1) / 2**(MB + 1)
    d = np.arange(2**DB, dtype=np.int64)
    g = cfg.gy
    for e in (0, 1):
        scale = 2**((e - 1) / 2)
        for a in range(NSEG):
            lo = 1 + a * 2.0**(DB - MB); mid = lo + dmid
            c = fitc(lo, lo + 2.0**(DB - MB), mid, scale)
            C1, C2 = -c[1], c[2]
            M = lo + d * 2.0**-MB
            fM = scale / np.sqrt(M); tgt = fM * 2.0**cfg.fy; w = 1 / fM
            K2c = int(round(C2 * 2**cfg.f2)); K1c = int(round(C1 * 2**cfg.f1))
            best = None
            for K2 in range(K2c - 1, K2c + 2):
                for K1 in range(K1c - search, K1c + search + 1):
                    R, Q = pipeline(cfg, K2, K1, d)
                    def mx(K0):
                        Y = ((K0 - Q) >> g) << g
                        return float(np.max(np.abs(Y - tgt) * w))
                    t = tgt + Q
                    a_, b_ = math.floor(t.min()), math.ceil(t.max())
                    while b_ - a_ > 2:
                        m1 = a_ + (b_ - a_) // 3; m2 = b_ - (b_ - a_) // 3
                        if mx(m1) < mx(m2): b_ = m2
                        else: a_ = m1
                    for K0 in range(a_, b_ + 1):
                        v = mx(K0) / 2.0**cfg.fy
                        if best is None or v < best[0]: best = (v, K2, K1, K0)
            tabs[(e, a)] = best[1:]
            worst = max(worst, best[0])
    return tabs, worst

def layout(cfg, tabs, LB=21, S=20):
    d = np.arange(2**DB, dtype=np.int64)
    assert max(t[0] for t in tabs.values()) < (1 << cfg.w2)
    Rmax = 0; per = {}; Ymax = 0
    for key, (K2, K1, K0) in tabs.items():
        R, Q = pipeline(cfg, K2, K1, d)
        assert R.min() >= 0
        Rmax = max(Rmax, int(R.max()), K1)
        per[key] = (int(Q.min()), int(Q.max()))
        Ymax = max(Ymax, K0 - per[key][0])
    w1 = Rmax.bit_length()
    lo_tab, hi_tab = {}, {}
    for key, (K2, K1, K0) in tabs.items():
        qmin, qmax = per[key]
        base = K0 % (1 << S); j = 0
        while base + j * (1 << S) - qmax < 0: j += 1
        k0lo = base + j * (1 << S)
        assert k0lo - qmin < (1 << LB), 'layout infeasible'
        lo_tab[key] = k0lo; hi_tab[key] = (K0 - k0lo) >> S
        assert hi_tab[key] >= 0
    whi = max(max(hi_tab.values()).bit_length(), 1)
    return dict(w1=w1, LB=LB, S=S, whi=whi, lo=lo_tab, hi=hi_tab)

# ----------------------------------------------------------------- circuit pieces
def and_all_into(b, controls, target):
    k = len(controls)
    if k == 1: b.cx(controls[0], target); return
    if k == 2: b.ccx(controls[0], controls[1], target); return
    anc = b.alloc_qubits(k - 2)
    b.init_and(controls[0], controls[1], anc[0])
    for j in range(1, k - 2): b.init_and(anc[j - 1], controls[j + 1], anc[j])
    b.ccx(anc[k - 3], controls[k - 1], target)
    for j in reversed(range(1, k - 2)): b.del_and(anc[j - 1], controls[j + 1], anc[j])
    b.del_and(controls[0], controls[1], anc[0])
    b.free(anc)

def phase_all(b, controls):
    k = len(controls)
    if k == 1: b.z(controls[0]); return
    if k == 2: b.cz(controls[0], controls[1]); return
    anc = b.alloc_qubits(k - 2)
    b.init_and(controls[0], controls[1], anc[0])
    for j in range(1, k - 2): b.init_and(anc[j - 1], controls[j + 1], anc[j])
    b.cz(anc[k - 3], controls[k - 1])
    for j in reversed(range(1, k - 2)): b.del_and(anc[j - 1], controls[j + 1], anc[j])
    b.del_and(controls[0], controls[1], anc[0])
    b.free(anc)

NSTAGES = [(4, 16), (3, 8), (2, 4), (1, 2), (0, 1)]

def normalize(b, x, f):
    for (k, d) in NSTAGES:
        top = x[W - d:W]
        if d == 1:
            b.cx(x[W - 1], f[0]); b.x(f[0])
        else:
            b.x(top); and_all_into(b, list(top), f[k]); b.x(top)
        for i in reversed(range(0, W - d)): b.cswap(f[k], x[i], x[i + d])

def unnormalize(b, x, f):
    for (k, d) in reversed(NSTAGES):
        for i in range(0, W - d): b.cswap(f[k], x[i], x[i + d])
        top = x[W - d:W]
        if d == 1:
            b.x(f[0]); b.cx(x[W - 1], f[0])
        else:
            m = b.alloc_bits(1)[0]
            b.hmr(f[k], m)
            b.push_condition(m)                     # fixup runs on half the shots on average
            b.x(top); phase_all(b, list(top)); b.x(top)
            b.pop_condition()
            b.free(m)

def gadd(b, a, t, cin=None, sub=False):
    """t += a + cin (or t -= a + cin), Gidney adder: len(t)-1 Toffolis, uncompute by HMR."""
    a = list(a); t = list(t); o = len(a); n = len(t)
    assert o <= n
    if sub: b.x(km.array(t))
    c = b.alloc_qubits(n - 1) if n > 1 else km.array([])
    def C(i): return cin if i == 0 else c[i - 1]
    for i in range(0, n - 1):
        if i == 0 and cin is None:
            if o: b.init_and(a[0], t[0], C(1))
        elif i < o:
            b.cx(C(i), a[i]); b.cx(C(i), t[i]); b.init_and(a[i], t[i], C(i + 1)); b.cx(C(i), C(i + 1))
        else:
            b.init_and(C(i), t[i], C(i + 1))
    if n - 1 < o: b.cx(a[n - 1], t[n - 1])
    if n > 1 or cin is not None: b.cx(C(n - 1), t[n - 1])
    for i in reversed(range(0, n - 1)):
        if i == 0 and cin is None:
            if o: b.del_and(a[0], t[0], C(1)); b.cx(a[0], t[0])
        elif i < o:
            b.cx(C(i), C(i + 1)); b.del_and(a[i], t[i], C(i + 1)); b.cx(C(i), a[i]); b.cx(a[i], t[i])
        else:
            b.del_and(C(i), t[i], C(i + 1)); b.cx(C(i), t[i])
    if n > 1: b.free(c)
    if sub: b.x(km.array(t))

def shift_stages(b, f, reg, L, inverse):
    stages = [(1, 1), (2, 2), (3, 4), (4, 8)]
    occ = []; o = L
    for (k, d) in stages: occ.append(o); o = min(o + d, len(reg))
    seq = list(enumerate(stages))
    if not inverse:
        for idx, (k, d) in seq:
            for i in reversed(range(0, occ[idx])):
                if i + d < len(reg): b.cswap(f[k], reg[i], reg[i + d])
    else:
        for idx, (k, d) in reversed(seq):
            for i in range(0, occ[idx]):
                if i + d < len(reg): b.cswap(f[k], reg[i], reg[i + d])

def build_circuit(cfg, tabs, lay):
    w2, w1, LB, S, whi = cfg.w2, lay['w1'], lay['LB'], lay['S'], lay['whi']
    gy, fy = cfg.gy, cfg.fy
    b = km.CircuitBuilder()
    x = b.create_quantum_register(W, name='x')
    f = b.alloc_qubits(5)
    c2 = b.alloc_qubits(w2); R = b.alloc_qubits(w1); Lr = b.alloc_qubits(LB); Yh = b.alloc_qubits(whi)
    tgt = km.array(list(Lr) + list(R) + list(c2) + list(Yh))
    table = [0] * 256
    for (e, a), (K2, K1, K0) in tabs.items():
        table[a + 128 * e] = lay['lo'][(e, a)] | (K1 << LB) | (K2 << (LB + w1)) | (lay['hi'][(e, a)] << (LB + w1 + w2))
    addr = km.array(list(x[14:21]) + [f[0]])
    delta = x[0:DB]
    def rrows(inverse):
        seq = cfg.rrow_list()
        if inverse: seq = list(reversed(seq))
        for i in seq:
            sh = cfg.sh2[i]
            if sh >= 0: off = c2[sh:w2]; t = R[0:w1]; cin = c2[sh - 1] if sh >= 1 else None
            else: off = c2[0:w2]; t = R[-sh:w1]; cin = None
            b.cx(delta[i], t); gadd(b, off, t, cin, sub=inverse); b.cx(delta[i], t)
    def yrows(inverse):
        seq = list(reversed(range(DB)))
        if inverse: seq.reverse()
        for i in seq:
            sh = cfg.sh1[i]; assert 0 <= sh < w1
            t = Lr[0:LB]; cin = R[sh - 1] if sh >= 1 else None
            b.cx(delta[i], t); gadd(b, R[sh:w1], t, cin, sub=inverse); b.cx(delta[i], t)
    normalize(b, x, f)
    b.init_lookup(table=table, address=addr, target=tgt)
    rrows(False); yrows(False)
    ext = b.alloc_qubits(10)
    if LB > S: gadd(b, Lr[S:LB], Yh)
    Cx = km.array(list(Lr[gy:S]) + list(Yh) + list(ext))
    shift_stages(b, f, Cx, S - gy + whi, False)
    for j in range(len(Cx)):
        e = gy + j - fy - 10                       # bit weight 2^e turns
        if e < 0: b.z_pow(Cx[j], Fraction(2)**(e + 1))
    shift_stages(b, f, Cx, S - gy + whi, True)
    if LB > S: gadd(b, Lr[S:LB], Yh, sub=True)
    b.free(ext)
    yrows(True); rrows(True)
    b.del_lookup(table=table, address=addr, target=tgt)
    b.free(tgt)
    unnormalize(b, x, f)
    b.free(f)
    return b.finish_circuit()

def main(out):
    cfg = Cfg(F2, F1, FYR, GY, DROP2)
    tabs, worst = build_tables(cfg)
    print(f'model worst relative error / 2^-24 = {worst * 2**24:.3f}')
    lay = layout(cfg, tabs)
    c = build_circuit(cfg, tabs, lay)
    s = c.operation_statistics()
    print('qubits', c.num_qubits, 'toffoli(static)', s.num_toffoli_like,
          'rotations', s.num_z_pow(min_clifford_hierarchy_level=4))
    c.to_file(out)
    print('wrote', out)

if __name__ == '__main__':
    main(sys.argv[1] if len(sys.argv) > 1 else 'invsqrt.kmb')

In [ ]:
%%writefile src/ports/bank_basis_only.py
# SPDX-License-Identifier: Apache-2.0
"""Isolate GF(2) dependence removal while retaining all original zero columns.

GF(2) output-basis idea: Artem Nazemtsev's attributed public source and Zongao
Bian's published description. Original banked oracle: Sumit Chongder and
Kozykorpesh. New isolation wrapper written here.
"""
from pathlib import Path
import port_pipeline as P

def banked_lookup_basis_only(delta,brot):
    original=P.basis_rows
    def retain_zero_axes(rows):
        basis,pivots=original(rows)
        combined=0
        for value in rows:
            combined|=int(value)
        width=delta.shape[1]*brot
        zeros=[p for p in range(width) if not (combined>>p&1)]
        return basis+[1<<p for p in zeros],pivots+zeros
    try:
        P.basis_rows=retain_zero_axes
        return P.banked_lookup_gf2(delta,brot)
    finally:
        P.basis_rows=original

def build_dfthcsos(out,data_file,brot=10,J=5):
    original=P.P.banked_lookup
    try:
        P.P.banked_lookup=banked_lookup_basis_only
        return P.P.build_dfthcsos(Path(out),Path(data_file),brot=brot,J=J)
    finally:
        P.P.banked_lookup=original

In [ ]:
%%writefile src/ports/evidence_inv/calibration_selected.json
{
  "drop": 0,
  "dropped": [],
  "deltas": [
    28042041,
    48878784,
    5856404,
    -3166547,
    -37185083,
    -21697865,
    -48110676,
    -25431986,
    5127554,
    49699577,
    -453919217,
    -1681535144,
    -3543027284,
    -10890021264,
    -9180331621,
    0,
    0,
    0,
    3703532,
    9460082,
    2304471,
    7417056,
    -4173865,
    8817759,
    -2900329,
    -10538999,
    568751,
    8408801,
    3159226,
    -8460881,
    2206763,
    19861463,
    2388418,
    -1943846,
    -18110848,
    3883407,
    -23518243,
    -69239028,
    -40588013,
    568532883,
    739763839,
    5259299380,
    -1742252662,
    -16441614585,
    -54839950758,
    -68719476736,
    -68719476736
  ],
  "margin": 58924122,
  "estimate": 155798.44602855993,
  "fitted": true,
  "source_kmx_sha256": "4b26d2a6575a5d82f44439a242e0fe5edc7b3acf2215daf20d792f9669299386",
  "all_inputs": 3133452,
  "worst_margin_units": 58924122
}

In [ ]:
%%writefile src/ports/evidence_inv/exact_proof.json
{
  "counts": {
    "ccx_live_literal": 5,
    "ccx_zero_product": 8
  },
  "replacements": [
    {
      "line": 7005,
      "original": "CCX q22 q72 q73",
      "replacement": [
        "CX q22 q73"
      ],
      "reason": "ccx_live_literal",
      "condition_depth": 0
    },
    {
      "line": 7776,
      "original": "CCX q76 q69 q75",
      "replacement": [],
      "reason": "ccx_zero_product",
      "condition_depth": 0
    },
    {
      "line": 7778,
      "original": "CCX q75 q68 q74",
      "replacement": [],
      "reason": "ccx_zero_product",
      "condition_depth": 0
    },
    {
      "line": 7780,
      "original": "CCX q74 q67 q73",
      "replacement": [],
      "reason": "ccx_zero_product",
      "condition_depth": 0
    },
    {
      "line": 7781,
      "original": "CCX q73 q66 q65",
      "replacement": [],
      "reason": "ccx_zero_product",
      "condition_depth": 0
    },
    {
      "line": 7837,
      "original": "CCX q78 q66 q65",
      "replacement": [
        "CX q78 q65"
      ],
      "reason": "ccx_live_literal",
      "condition_depth": 0
    },
    {
      "line": 8760,
      "original": "CCX q69 q43 q44",
      "replacement": [
        "CX q69 q44"
      ],
      "reason": "ccx_live_literal",
      "condition_depth": 0
    },
    {
      "line": 15021,
      "original": "CCX q70 q43 q44",
      "replacement": [
        "CX q70 q44"
      ],
      "reason": "ccx_live_literal",
      "condition_depth": 0
    },
    {
      "line": 15675,
      "original": "CCX q74 q78 q75",
      "replacement": [],
      "reason": "ccx_zero_product",
      "condition_depth": 0
    },
    {
      "line": 15677,
      "original": "CCX q75 q79 q76",
      "replacement": [],
      "reason": "ccx_zero_product",
      "condition_depth": 0
    },
    {
      "line": 15679,
      "original": "CCX q76 q65 q77",
      "replacement": [],
      "reason": "ccx_zero_product",
      "condition_depth": 0
    },
    {
      "line": 15680,
      "original": "CCX q77 q66 q67",
      "replacement": [],
      "reason": "ccx_zero_product",
      "condition_depth": 0
    },
    {
      "line": 15736,
      "original": "CCX q82 q66 q67",
      "replacement": [
        "CX q82 q67"
      ],
      "reason": "ccx_live_literal",
      "condition_depth": 0
    }
  ],
  "max_x": 3133452,
  "domain": "all x=1..3133452",
  "unknown_branch_joins": 0,
  "classical_xor_passthrough": true,
  "source_kmx_sha256": "5097e1d17147dc5999bc275053d9b50958452966a861ffd50f846bae79134b47",
  "result_kmx_sha256": "4b26d2a6575a5d82f44439a242e0fe5edc7b3acf2215daf20d792f9669299386"
}

In [ ]:
%%writefile src/ports/evidence_inv/virtual_tables.json
{
  "params": [
    9,
    19,
    26,
    2,
    [
      5
    ]
  ],
  "interpolation": {
    "fraction_bits": 3,
    "remainder_bits": 3,
    "step_width": 4,
    "rows": [
      [
        126,
        9
      ],
      [
        126,
        9
      ],
      [
        126,
        9
      ],
      [
        126,
        9
      ],
      [
        126,
        9
      ],
      [
        126,
        9
      ],
      [
        126,
        9
      ],
      [
        126,
        9
      ],
      [
        108,
        7
      ],
      [
        108,
        7
      ],
      [
        108,
        7
      ],
      [
        108,
        7
      ],
      [
        108,
        7
      ],
      [
        108,
        7
      ],
      [
        108,
        7
      ],
      [
        108,
        7
      ],
      [
        94,
        6
      ],
      [
        94,
        6
      ],
      [
        94,
        6
      ],
      [
        94,
        6
      ],
      [
        94,
        6
      ],
      [
        94,
        6
      ],
      [
        94,
        6
      ],
      [
        94,
        6
      ],
      [
        83,
        5
      ],
      [
        83,
        5
      ],
      [
        83,
        5
      ],
      [
        83,
        5
      ],
      [
        83,
        5
      ],
      [
        83,
        5
      ],
      [
        83,
        5
      ],
      [
        83,
        5
      ],
      [
        73,
        4
      ],
      [
        73,
        4
      ],
      [
        73,
        4
      ],
      [
        73,
        4
      ],
      [
        73,
        4
      ],
      [
        73,
        4
      ],
      [
        73,
        4
      ],
      [
        73,
        4
      ],
      [
        65,
        3
      ],
      [
        65,
        3
      ],
      [
        65,
        3
      ],
      [
        65,
        3
      ],
      [
        65,
        3
      ],
      [
        65,
        3
      ],
      [
        65,
        3
      ],
      [
        65,
        3
      ],
      [
        58,
        2
      ],
      [
        58,
        2
      ],
      [
        58,
        2
      ],
      [
        58,
        2
      ],
      [
        57,
        2
      ],
      [
        57,
        2
      ],
      [
        57,
        2
      ],
      [
        57,
        2
      ],
      [
        52,
        2
      ],
      [
        52,
        2
      ],
      [
        52,
        2
      ],
      [
        52,
        2
      ],
      [
        51,
        0
      ],
      [
        51,
        0
      ],
      [
        50,
        0
      ],
      [
        49,
        0
      ],
      [
        47,
        2
      ],
      [
        47,
        2
      ],
      [
        47,
        2
      ],
      [
        47,
        2
      ],
      [
        46,
        2
      ],
      [
        46,
        2
      ],
      [
        46,
        2
      ],
      [
        46,
        2
      ],
      [
        42,
        1
      ],
      [
        42,
        1
      ],
      [
        42,
        1
      ],
      [
        42,
        1
      ],
      [
        42,
        1
      ],
      [
        42,
        1
      ],
      [
        42,
        1
      ],
      [
        42,
        1
      ],
      [
        39,
        0
      ],
      [
        39,
        0
      ],
      [
        39,
        0
      ],
      [
        39,
        0
      ],
      [
        39,
        2
      ],
      [
        39,
        2
      ],
      [
        39,
        2
      ],
      [
        39,
        2
      ],
      [
        35,
        2
      ],
      [
        35,
        2
      ],
      [
        35,
        2
      ],
      [
        35,
        2
      ],
      [
        35,
        2
      ],
      [
        35,
        2
      ],
      [
        35,
        2
      ],
      [
        35,
        2
      ],
      [
        32,
        1
      ],
      [
        32,
        1
      ],
      [
        32,
        1
      ],
      [
        32,
        1
      ],
      [
        32,
        1
      ],
      [
        32,
        1
      ],
      [
        32,
        1
      ],
      [
        32,
        1
      ],
      [
        30,
        0
      ],
      [
        30,
        0
      ],
      [
        30,
        0
      ],
      [
        30,
        0
      ],
      [
        29,
        0
      ],
      [
        29,
        0
      ],
      [
        29,
        0
      ],
      [
        29,
        0
      ],
      [
        27,
        0
      ],
      [
        27,
        0
      ],
      [
        27,
        0
      ],
      [
        27,
        0
      ],
      [
        26,
        0
      ],
      [
        26,
        0
      ],
      [
        26,
        0
      ],
      [
        26,
        0
      ],
      [
        25,
        0
      ],
      [
        25,
        0
      ],
      [
        25,
        0
      ],
      [
        25,
        0
      ],
      [
        25,
        0
      ],
      [
        24,
        0
      ],
      [
        24,
        0
      ],
      [
        24,
        0
      ],
      [
        178,
        13
      ],
      [
        178,
        13
      ],
      [
        178,
        13
      ],
      [
        178,
        13
      ],
      [
        178,
        13
      ],
      [
        178,
        13
      ],
      [
        178,
        13
      ],
      [
        178,
        13
      ],
      [
        154,
        11
      ],
      [
        154,
        11
      ],
      [
        154,
        11
      ],
      [
        154,
        11
      ],
      [
        154,
        11
      ],
      [
        154,
        11
      ],
      [
        154,
        11
      ],
      [
        154,
        11
      ],
      [
        134,
        7
      ],
      [
        134,
        7
      ],
      [
        134,
        7
      ],
      [
        134,
        7
      ],
      [
        134,
        7
      ],
      [
        134,
        7
      ],
      [
        134,
        7
      ],
      [
        134,
        7
      ],
      [
        117,
        7
      ],
      [
        117,
        7
      ],
      [
        117,
        7
      ],
      [
        117,
        7
      ],
      [
        117,
        7
      ],
      [
        117,
        7
      ],
      [
        117,
        7
      ],
      [
        117,
        7
      ],
      [
        103,
        6
      ],
      [
        103,
        6
      ],
      [
        103,
        6
      ],
      [
        103,
        6
      ],
      [
        103,
        6
      ],
      [
        103,
        6
      ],
      [
        103,
        6
      ],
      [
        103,
        6
      ],
      [
        92,
        5
      ],
      [
        92,
        5
      ],
      [
        92,
        5
      ],
      [
        92,
        5
      ],
      [
        92,
        5
      ],
      [
        92,
        5
      ],
      [
        92,
        5
      ],
      [
        92,
        5
      ],
      [
        82,
        4
      ],
      [
        82,
        4
      ],
      [
        82,
        4
      ],
      [
        82,
        4
      ],
      [
        82,
        4
      ],
      [
        82,
        4
      ],
      [
        82,
        4
      ],
      [
        82,
        4
      ],
      [
        73,
        4
      ],
      [
        73,
        4
      ],
      [
        73,
        4
      ],
      [
        73,
        4
      ],
      [
        73,
        4
      ],
      [
        73,
        4
      ],
      [
        73,
        4
      ],
      [
        73,
        4
      ],
      [
        66,
        3
      ],
      [
        66,
        3
      ],
      [
        66,
        3
      ],
      [
        66,
        3
      ],
      [
        66,
        3
      ],
      [
        66,
        3
      ],
      [
        66,
        3
      ],
      [
        66,
        3
      ],
      [
        60,
        3
      ],
      [
        60,
        3
      ],
      [
        60,
        3
      ],
      [
        60,
        3
      ],
      [
        60,
        3
      ],
      [
        60,
        3
      ],
      [
        60,
        3
      ],
      [
        60,
        3
      ],
      [
        54,
        2
      ],
      [
        54,
        2
      ],
      [
        54,
        2
      ],
      [
        54,
        2
      ],
      [
        54,
        2
      ],
      [
        54,
        2
      ],
      [
        54,
        2
      ],
      [
        54,
        2
      ],
      [
        50,
        2
      ],
      [
        50,
        2
      ],
      [
        50,
        2
      ],
      [
        50,
        2
      ],
      [
        50,
        2
      ],
      [
        50,
        2
      ],
      [
        50,
        2
      ],
      [
        50,
        2
      ],
      [
        45,
        2
      ],
      [
        45,
        2
      ],
      [
        45,
        2
      ],
      [
        45,
        2
      ],
      [
        45,
        2
      ],
      [
        45,
        2
      ],
      [
        45,
        2
      ],
      [
        45,
        2
      ],
      [
        41,
        2
      ],
      [
        41,
        2
      ],
      [
        41,
        2
      ],
      [
        41,
        2
      ],
      [
        41,
        2
      ],
      [
        41,
        2
      ],
      [
        41,
        2
      ],
      [
        41,
        2
      ],
      [
        38,
        1
      ],
      [
        38,
        1
      ],
      [
        38,
        1
      ],
      [
        38,
        1
      ],
      [
        38,
        1
      ],
      [
        38,
        1
      ],
      [
        38,
        1
      ],
      [
        38,
        1
      ],
      [
        35,
        1
      ],
      [
        35,
        1
      ],
      [
        35,
        1
      ],
      [
        35,
        1
      ],
      [
        35,
        1
      ],
      [
        35,
        1
      ],
      [
        35,
        1
      ],
      [
        35,
        1
      ]
    ],
    "update_width": 8
  },
  "threshold": 0.99,
  "rows": [
    [
      134,
      184284,
      189442912
    ],
    [
      132,
      182154,
      188710052
    ],
    [
      130,
      180063,
      187985630
    ],
    [
      128,
      178014,
      187269492
    ],
    [
      124,
      176003,
      186561484
    ],
    [
      122,
      174029,
      185861436
    ],
    [
      120,
      172092,
      185169212
    ],
    [
      118,
      170190,
      184484662
    ],
    [
      115,
      168323,
      183807656
    ],
    [
      113,
      166490,
      183138044
    ],
    [
      111,
      164690,
      182475698
    ],
    [
      109,
      162922,
      181820488
    ],
    [
      107,
      161186,
      181172284
    ],
    [
      105,
      159481,
      180530966
    ],
    [
      103,
      157805,
      179896408
    ],
    [
      101,
      156158,
      179268498
    ],
    [
      100,
      154540,
      178647110
    ],
    [
      98,
      152949,
      178032146
    ],
    [
      96,
      151386,
      177423488
    ],
    [
      94,
      149849,
      176821030
    ],
    [
      94,
      148338,
      176224660
    ],
    [
      92,
      146852,
      175634294
    ],
    [
      90,
      145391,
      175049822
    ],
    [
      88,
      143953,
      174471144
    ],
    [
      88,
      142540,
      173898160
    ],
    [
      86,
      141149,
      173330794
    ],
    [
      86,
      139781,
      172768936
    ],
    [
      84,
      138436,
      172212514
    ],
    [
      82,
      137110,
      171661436
    ],
    [
      80,
      135807,
      171115614
    ],
    [
      80,
      134523,
      170574958
    ],
    [
      78,
      133260,
      170039402
    ],
    [
      77,
      132017,
      169508856
    ],
    [
      75,
      130792,
      168983250
    ],
    [
      75,
      129587,
      168462494
    ],
    [
      73,
      128400,
      167946532
    ],
    [
      73,
      127231,
      167435272
    ],
    [
      71,
      126080,
      166928664
    ],
    [
      71,
      124945,
      166426616
    ],
    [
      69,
      123828,
      165929082
    ],
    [
      68,
      122728,
      165435976
    ],
    [
      68,
      121643,
      164947238
    ],
    [
      66,
      120574,
      164462814
    ],
    [
      66,
      119521,
      163982626
    ],
    [
      64,
      118483,
      163506628
    ],
    [
      64,
      117461,
      163034742
    ],
    [
      62,
      116452,
      162566926
    ],
    [
      62,
      115458,
      162103108
    ],
    [
      60,
      114478,
      161643246
    ],
    [
      60,
      113512,
      161187266
    ],
    [
      58,
      112560,
      160735132
    ],
    [
      58,
      111621,
      160286774
    ],
    [
      57,
      110695,
      159842150
    ],
    [
      57,
      109781,
      159401200
    ],
    [
      55,
      108880,
      158963890
    ],
    [
      55,
      107991,
      158530152
    ],
    [
      54,
      107114,
      158099948
    ],
    [
      54,
      106249,
      157673222
    ],
    [
      52,
      105396,
      157249944
    ],
    [
      52,
      104553,
      156830046
    ],
    [
      51,
      103722,
      156413500
    ],
    [
      51,
      102903,
      156000250
    ],
    [
      50,
      102094,
      155590264
    ],
    [
      49,
      101295,
      155183492
    ],
    [
      49,
      100507,
      154779890
    ],
    [
      49,
      99728,
      154379422
    ],
    [
      47,
      98960,
      153982054
    ],
    [
      47,
      98201,
      153587730
    ],
    [
      46,
      97453,
      153196426
    ],
    [
      46,
      96714,
      152808092
    ],
    [
      44,
      95984,
      152422704
    ],
    [
      44,
      95263,
      152040212
    ],
    [
      43,
      94552,
      151660586
    ],
    [
      43,
      93848,
      151283786
    ],
    [
      43,
      93154,
      150909782
    ],
    [
      43,
      92469,
      150538536
    ],
    [
      41,
      91791,
      150170028
    ],
    [
      41,
      91122,
      149804204
    ],
    [
      41,
      90461,
      149441038
    ],
    [
      41,
      89807,
      149080500
    ],
    [
      39,
      89162,
      148722570
    ],
    [
      39,
      88525,
      148367196
    ],
    [
      39,
      87895,
      148014360
    ],
    [
      39,
      87272,
      147664028
    ],
    [
      39,
      86657,
      147316170
    ],
    [
      39,
      86048,
      146970762
    ],
    [
      37,
      85448,
      146627778
    ],
    [
      37,
      84853,
      146287178
    ],
    [
      37,
      84267,
      145948938
    ],
    [
      37,
      83686,
      145613036
    ],
    [
      35,
      83112,
      145279450
    ],
    [
      35,
      82544,
      144948138
    ],
    [
      35,
      81983,
      144619082
    ],
    [
      35,
      81429,
      144292258
    ],
    [
      33,
      80881,
      143967646
    ],
    [
      33,
      80339,
      143645208
    ],
    [
      33,
      79802,
      143324928
    ],
    [
      33,
      79272,
      143006780
    ],
    [
      33,
      78748,
      142690740
    ],
    [
      33,
      78229,
      142376788
    ],
    [
      31,
      77716,
      142064906
    ],
    [
      31,
      77208,
      141755058
    ],
    [
      31,
      76706,
      141447226
    ],
    [
      31,
      76210,
      141141392
    ],
    [
      30,
      75719,
      140837540
    ],
    [
      30,
      75233,
      140535636
    ],
    [
      30,
      74752,
      140235664
    ],
    [
      30,
      74277,
      139937606
    ],
    [
      29,
      73806,
      139641444
    ],
    [
      29,
      73341,
      139347148
    ],
    [
      29,
      72879,
      139054708
    ],
    [
      29,
      72424,
      138764100
    ],
    [
      27,
      71973,
      138475316
    ],
    [
      27,
      71526,
      138188320
    ],
    [
      27,
      71084,
      137903100
    ],
    [
      27,
      70647,
      137619640
    ],
    [
      26,
      70214,
      137337922
    ],
    [
      26,
      69785,
      137057926
    ],
    [
      26,
      69361,
      136779636
    ],
    [
      26,
      68941,
      136503032
    ],
    [
      25,
      68526,
      136228104
    ],
    [
      25,
      68114,
      135954826
    ],
    [
      25,
      67706,
      135683186
    ],
    [
      25,
      67303,
      135413170
    ],
    [
      25,
      66903,
      135144756
    ],
    [
      24,
      66508,
      134877940
    ],
    [
      24,
      66116,
      134612692
    ],
    [
      24,
      65728,
      134349002
    ],
    [
      190,
      260618,
      267912734
    ],
    [
      186,
      257604,
      266876318
    ],
    [
      184,
      254649,
      265851828
    ],
    [
      180,
      251750,
      264839058
    ],
    [
      176,
      248905,
      263837776
    ],
    [
      172,
      246114,
      262847766
    ],
    [
      170,
      243374,
      261868806
    ],
    [
      166,
      240685,
      260900718
    ],
    [
      164,
      238045,
      259943274
    ],
    [
      162,
      235452,
      258996294
    ],
    [
      158,
      232907,
      258059602
    ],
    [
      156,
      230407,
      257132988
    ],
    [
      152,
      227952,
      256216298
    ],
    [
      150,
      225540,
      255309332
    ],
    [
      146,
      223170,
      254411940
    ],
    [
      144,
      220841,
      253523934
    ],
    [
      141,
      218553,
      252645166
    ],
    [
      139,
      216303,
      251775472
    ],
    [
      137,
      214092,
      250914696
    ],
    [
      135,
      211918,
      250062688
    ],
    [
      133,
      209781,
      249219304
    ],
    [
      131,
      207680,
      248384396
    ],
    [
      129,
      205614,
      247557822
    ],
    [
      127,
      203582,
      246739444
    ],
    [
      124,
      201582,
      245929136
    ],
    [
      122,
      199616,
      245126758
    ],
    [
      120,
      197681,
      244332178
    ],
    [
      118,
      195777,
      243545278
    ],
    [
      116,
      193903,
      242765930
    ],
    [
      114,
      192059,
      241994020
    ],
    [
      112,
      190245,
      241229424
    ],
    [
      110,
      188459,
      240472030
    ],
    [
      109,
      186700,
      239721722
    ],
    [
      107,
      184968,
      238978398
    ],
    [
      105,
      183264,
      238241948
    ],
    [
      103,
      181585,
      237512262
    ],
    [
      103,
      179931,
      236789232
    ],
    [
      101,
      178303,
      236072776
    ],
    [
      99,
      176699,
      235362784
    ],
    [
      97,
      175119,
      234659160
    ],
    [
      97,
      173563,
      233961798
    ],
    [
      95,
      172030,
      233270626
    ],
    [
      95,
      170518,
      232585534
    ],
    [
      93,
      169029,
      231906454
    ],
    [
      91,
      167561,
      231233288
    ],
    [
      89,
      166115,
      230565950
    ],
    [
      89,
      164688,
      229904346
    ],
    [
      87,
      163283,
      229248416
    ],
    [
      86,
      161897,
      228598064
    ],
    [
      84,
      160531,
      227953220
    ],
    [
      84,
      159184,
      227313794
    ],
    [
      82,
      157855,
      226679728
    ],
    [
      82,
      156545,
      226050928
    ],
    [
      80,
      155254,
      225427342
    ],
    [
      80,
      153979,
      224808878
    ],
    [
      78,
      152723,
      224195488
    ],
    [
      77,
      151482,
      223587086
    ],
    [
      75,
      150259,
      222983616
    ],
    [
      75,
      149052,
      222384996
    ],
    [
      73,
      147861,
      221791180
    ],
    [
      73,
      146686,
      221202088
    ],
    [
      71,
      145527,
      220617674
    ],
    [
      71,
      144382,
      220037858
    ],
    [
      69,
      143253,
      219462602
    ],
    [
      69,
      142138,
      218891822
    ],
    [
      69,
      141038,
      218325474
    ],
    [
      67,
      139951,
      217763506
    ],
    [
      67,
      138879,
      217205848
    ],
    [
      65,
      137820,
      216652462
    ],
    [
      65,
      136774,
      216103274
    ],
    [
      63,
      135742,
      215558250
    ],
    [
      63,
      134723,
      215017324
    ],
    [
      63,
      133716,
      214480446
    ],
    [
      63,
      132722,
      213947572
    ],
    [
      61,
      131740,
      213418658
    ],
    [
      61,
      130771,
      212893640
    ],
    [
      59,
      129812,
      212372484
    ],
    [
      59,
      128865,
      211855128
    ],
    [
      57,
      127931,
      211341546
    ],
    [
      57,
      127007,
      210831672
    ],
    [
      56,
      126095,
      210325474
    ],
    [
      56,
      125193,
      209822900
    ],
    [
      54,
      124302,
      209323920
    ],
    [
      54,
      123422,
      208828476
    ],
    [
      54,
      122551,
      208336532
    ],
    [
      54,
      121691,
      207848048
    ],
    [
      52,
      120841,
      207362994
    ],
    [
      52,
      120001,
      206881312
    ],
    [
      52,
      119170,
      206402970
    ],
    [
      52,
      118350,
      205927932
    ],
    [
      50,
      117538,
      205456166
    ],
    [
      50,
      116736,
      204987620
    ],
    [
      50,
      115943,
      204522264
    ],
    [
      50,
      115158,
      204060064
    ],
    [
      48,
      114383,
      203600994
    ],
    [
      48,
      113616,
      203144996
    ],
    [
      47,
      112858,
      202692054
    ],
    [
      47,
      112108,
      202242124
    ],
    [
      45,
      111366,
      201795186
    ],
    [
      45,
      110633,
      201351192
    ],
    [
      45,
      109907,
      200910112
    ],
    [
      45,
      109190,
      200471918
    ],
    [
      43,
      108480,
      200036590
    ],
    [
      43,
      107778,
      199604074
    ],
    [
      43,
      107083,
      199174354
    ],
    [
      43,
      106396,
      198747396
    ],
    [
      41,
      105716,
      198323182
    ],
    [
      41,
      105043,
      197901664
    ],
    [
      41,
      104378,
      197482822
    ],
    [
      41,
      103719,
      197066628
    ],
    [
      39,
      103068,
      196653062
    ],
    [
      39,
      102422,
      196242082
    ],
    [
      39,
      101785,
      195833668
    ],
    [
      39,
      101153,
      195427794
    ],
    [
      39,
      100528,
      195024432
    ],
    [
      39,
      99909,
      194623556
    ],
    [
      37,
      99298,
      194225152
    ],
    [
      37,
      98692,
      193829176
    ],
    [
      37,
      98091,
      193435614
    ],
    [
      37,
      97497,
      193044438
    ],
    [
      36,
      96909,
      192655630
    ],
    [
      36,
      96327,
      192269158
    ],
    [
      36,
      95751,
      191885000
    ],
    [
      36,
      95180,
      191503138
    ],
    [
      34,
      94616,
      191123554
    ],
    [
      34,
      94057,
      190746212
    ],
    [
      34,
      93502,
      190371094
    ],
    [
      34,
      92954,
      189998182
    ]
  ],
  "fit_groups": [
    [
      3,
      0,
      0,
      0.8148105350137045
    ],
    [
      3,
      0,
      1,
      0.8943204643510907
    ],
    [
      3,
      0,
      2,
      0.9834761903270947
    ],
    [
      3,
      0,
      3,
      0.9677781455074566
    ],
    [
      3,
      0,
      4,
      0.8810102937126736
    ],
    [
      3,
      0,
      5,
      0.8254088820967135
    ],
    [
      2,
      0,
      12,
      0.9165629943487492
    ],
    [
      2,
      0,
      13,
      0.8384806352446524
    ],
    [
      2,
      0,
      14,
      0.9499016324961326
    ],
    [
      0,
      0,
      60,
      null
    ],
    [
      0,
      0,
      61,
      null
    ],
    [
      0,
      0,
      62,
      null
    ],
    [
      0,
      0,
      63,
      null
    ],
    [
      2,
      0,
      16,
      0.8548550322180108
    ],
    [
      2,
      0,
      17,
      0.953821457350305
    ],
    [
      3,
      0,
      9,
      0.9289634566854921
    ],
    [
      2,
      0,
      20,
      0.9241306836638363
    ],
    [
      2,
      0,
      21,
      0.9741612764427697
    ],
    [
      3,
      0,
      11,
      0.9738952253623129
    ],
    [
      3,
      0,
      12,
      0.8881409954892974
    ],
    [
      2,
      0,
      26,
      0.9116751085323821
    ],
    [
      2,
      0,
      27,
      0.9492284958355196
    ],
    [
      2,
      0,
      28,
      0.9509008246971019
    ],
    [
      2,
      0,
      29,
      0.941565884692494
    ],
    [
      2,
      0,
      30,
      0.9752065464741345
    ],
    [
      0,
      0,
      124,
      null
    ],
    [
      0,
      0,
      125,
      null
    ],
    [
      0,
      0,
      126,
      null
    ],
    [
      0,
      0,
      127,
      null
    ],
    [
      3,
      1,
      0,
      0.7038371877505638
    ],
    [
      3,
      1,
      1,
      0.8026629771168182
    ],
    [
      3,
      1,
      2,
      0.6948060776302255
    ],
    [
      3,
      1,
      3,
      0.611326615579549
    ],
    [
      3,
      1,
      4,
      0.6974229041567335
    ],
    [
      3,
      1,
      5,
      0.7799141051392934
    ],
    [
      3,
      1,
      6,
      0.6650227355176005
    ],
    [
      3,
      1,
      7,
      0.657121674848042
    ],
    [
      3,
      1,
      8,
      0.6639376266509004
    ],
    [
      3,
      1,
      9,
      0.7552662920774285
    ],
    [
      3,
      1,
      10,
      0.8034399560684355
    ],
    [
      3,
      1,
      11,
      0.812105546040423
    ],
    [
      3,
      1,
      12,
      0.7230947601200167
    ],
    [
      3,
      1,
      13,
      0.8255134508502201
    ],
    [
      3,
      1,
      14,
      0.767192264983811
    ],
    [
      3,
      1,
      15,
      0.7254860994118587
    ]
  ],
  "build_options": {
    "main_lookup": "anf",
    "affine": false,
    "group_bits": null,
    "joint_norm": true
  }
}

In [ ]:
%%writefile src/ports/fit_even_anchors.py
# SPDX-License-Identifier: Apache-2.0
"""Refit only linear/constant coefficients; retain curvature and R3 grouping."""
import json
import math
import numpy as np

def fit(data, H, log=print, even_linear=False):
    cfg=H.Cfg(*data['params'])
    rows=[]
    details=[]
    d=np.arange(1<<14,dtype=np.int64)
    for n,(k2,k1,k0) in enumerate(data['rows']):
        e,a=divmod(n,128)
        xn=(1<<21)+(a<<14)+d
        dd=d[(d%2==0) if e else ((xn<=3133452)|(d%4==0))]
        target=2.**((e-1)/2)/np.sqrt(1+(a*(1<<14)+dd)*2.**-21)*2.**cfg.fy
        best=None
        for linear in range(k1-4,k1+5):
            if even_linear and linear%2:
                continue
            _,q=H.pipeline(cfg,k2,linear,dd)
            z=target+q
            center=(float(z.min())+float(z.max()))/2+((1<<cfg.gy)-1)/2
            for constant in range(2*(math.floor(center/2)-5),2*(math.ceil(center/2)+5)+1,2):
                y=((constant-q)>>cfg.gy)<<cfg.gy
                score=float(np.max(np.abs(y-target)/target))*2**24
                if best is None or score<best[0]:
                    best=score,k2,linear,constant
        rows.append(list(best[1:]))
        details.append(dict(row=n,ratio=best[0],old=[k2,k1,k0],new=list(best[1:])))
        if n%32==0:
            log(f'even-anchor fit {n+1}/256; worst relative tolerance usage {max(r["ratio"] for r in details):.9f}')
    result=dict(data,rows=rows)
    report=dict(worst_ratio=max(r['ratio'] for r in details),rows=details,
        curvature_unchanged=all(x[0]==y[0] for x,y in zip(rows,data['rows'])),
        full_official_verifier_required=True)
    return result,report

In [ ]:
%%writefile src/ports/invsqrt_ports.py
# SPDX-License-Identifier: Apache-2.0
"""Replay individually measured source-generated invsqrt steps.

All generators use readable coefficients and source. Exact rewrite recipes
record the original line and certified replacement; replay checks both source
and result hashes. No public binary circuit is a generator dependency.
Credits: Sumit Chongder; Rudra Bhaskar; Kozykorpesh; Alex Chilton; Kirill
Tymchishin; Artem Nazemtsev; Gu/Kemeny/Yelin/Lukin; Dmitrii Gluzdov;
lightningv08 (calibration), Mizero Lucky Gall (generic calibration),
Zongao Bian (virtual low-bit parity/on-demand carry).
"""
import hashlib
import json
from pathlib import Path
import sys

ROOT=Path(__file__).resolve().parent

def raw_source(base='r2', *, tables_file=None, virtual_bits=0):
    import kickmix as km
    if base=='r2':
        source=ROOT.parent/'gen_invsqrt_r2'
        sys.path[:0]=[str(source)]
        import sumit_reference as H
        if virtual_bits==2:
            import virtual_two_round2 as M
        elif virtual_bits==1:
            import virtual_round2 as M
        else:
            import round2 as M
        data=json.loads(Path(tables_file or source/'tables.json').read_text())
        tabs={(e,i):tuple(data['rows'][128*e+i]) for e in range(2) for i in range(128)}
        return M.build(H.Cfg(*data['params']),tabs,deferred=True,anf=True,parity_input=True)[0]
    if base!='r3' or virtual_bits not in (0,1):
        raise ValueError('Supported: r2 virtual_bits=0,1,2; r3 virtual_bits=0,1')
    source=ROOT.parent/'gen_invsqrt_r3'
    for name in ('sumit_reference','signed_digit_lab','round2','deferred_lookup','hybrid_add'):
        sys.modules.pop(name,None)
    sys.path[:0]=[str(source),str(source/'vendor')]
    import sumit_reference as H
    if virtual_bits:
        import virtual_time_mux_round3 as M
    else:
        import time_mux_round3 as M
    data=json.loads(Path(tables_file or source/'tables.json').read_text())
    tabs={(e,i):tuple(data['rows'][128*e+i]) for e in range(2) for i in range(128)}
    return M.build(H.Cfg(*data['params']),tabs,interpolation=data['interpolation'],**data['build_options'])[0]


def replay_exact(circuit, certificate_file):
    import kickmix as km
    certificate=json.loads(Path(certificate_file).read_text())
    text=str(circuit)
    if 'source_kmx_sha256' in certificate:
        assert hashlib.sha256(text.encode()).hexdigest()==certificate['source_kmx_sha256']
    patches={r['line']:r for r in certificate['replacements']}
    output=[]
    for i,line in enumerate(text.splitlines(),1):
        if i in patches:
            patch=patches[i]
            assert line==patch['original'], (i,line,patch)
            output.extend(patch['replacement'])
        else:
            output.append(line)
    candidate=km.Circuit('\n'.join(output)+'\n')
    if 'result_kmx_sha256' in certificate:
        assert hashlib.sha256(str(candidate).encode()).hexdigest()==certificate['result_kmx_sha256']
    return candidate


def build_invsqrt(out, *, base='r2', certificate_file=None, calibration_file=None,
                  tables_file=None, virtual_bits=0):
    """Drop-in ``build_invsqrt(out)``; kwargs select a separately measured step.

    Pass its exact_proof.json, optional calibration_selected.json and optional
    virtual_tables.json from the corresponding measured step directory.
    """
    import kickmix as km
    circuit=raw_source(base,tables_file=tables_file,virtual_bits=virtual_bits)
    if certificate_file:
        circuit=replay_exact(circuit,certificate_file)
    if calibration_file:
        from port_pipeline import module
        C=module('phase_replay',ROOT/'public_sources/lightning_v2/embedded/rotation_calibration.py')
        fit=json.loads(Path(calibration_file).read_text())
        text=str(circuit)
        assert hashlib.sha256(text.encode()).hexdigest()==fit['source_kmx_sha256']
        lines,rots=C.parse_kmx(text)
        circuit=km.Circuit(C.rewrite(lines,rots,{j:int(v) for j,v in enumerate(fit['deltas']) if j not in fit['dropped']},set(fit['dropped'])))
    out=Path(out)
    out.mkdir(parents=True,exist_ok=True)
    path=out/'invsqrt.kmb'
    circuit.to_file(str(path),format='kmb')
    return path

In [ ]:
%%writefile src/ports/port_pipeline.py
# SPDX-License-Identifier: Apache-2.0
"""Small source-only ports for the public qcc_pipeline interface.

Artem Nazemtsev: factor gauge, exact-domain Clifford compiler.
Artem Nazemtsev: GF(2) compression of banked difference outputs (his
earlier banked-clean-staging is also cited by Zongao Bian).
Sumit Chongder / Kozykorpesh / Dmitrii Gluzdov: existing banked loader.
The GF(2) implementation here is reconstructed from the published description.
"""
from pathlib import Path
import importlib.util
import numpy as np

ROOT = Path(__file__).resolve().parent


def module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    obj = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(obj)
    return obj


P = module('public_pipeline_ports', ROOT.parent / 'qcc_pipeline.py')


def setup(competition_dir):
    P.setup(competition_dir)


def generate_gauge_words(data_file, destination):
    """Write words/W/H2_I in the public dataset's format; return build evidence."""
    G = module('artem_gauge_port', ROOT / 'public_sources/anazemcev_quantum-circuits-source-exact-compilation/embedded/src/gauge_generator.py')
    data, words, evidence = G.generate(data_file, P.COMP / 'verifiers/dfthcsos/FeMoCo76o113e.fixed_shifts.npz')
    np.savez(destination, words=words, W=data['W'], H2_I=data['H2_I'])
    return evidence


def basis_rows(rows):
    """RREF of output vectors; return basis words and pivot bit positions."""
    basis = {}
    for value in rows:
        v = int(value)
        for pivot in sorted(basis, reverse=True):
            if v >> pivot & 1:
                v ^= basis[pivot]
        if v:
            pivot = v.bit_length() - 1
            for old in list(basis):
                if basis[old] >> pivot & 1:
                    basis[old] ^= v
            basis[pivot] = v
    pivots = sorted(basis)
    return [basis[p] for p in pivots], pivots


def banked_lookup_gf2(delta, brot):
    """Exact XOR oracle, including arbitrary initial target words."""
    import kickmix as km
    A, J = delta.shape
    s, width = (A - 1).bit_length(), J * brot
    table = [sum(int(v) << (j * brot) for j, v in enumerate(row)) for row in delta]
    even = table[::2]
    odd = table[1::2] + ([table[-1]] if A % 2 else [])
    diff = [x ^ y for x, y in zip(even, odd)]
    basis, pivots = basis_rows(diff)
    nonpivot = [v ^ (1 << p) for v, p in zip(basis, pivots)]
    def transform(value):
        return value ^ sum_xor(v for v, p in zip(nonpivot, pivots) if value >> p & 1)
    def coordinates(value):
        return sum(((value >> p) & 1) << i for i, p in enumerate(pivots))
    assert all(transform(v) == sum(((v >> p) & 1) << p for p in pivots) for v in diff)
    coeff = [coordinates(v) for v in diff]
    values = [transform(v) | (c << width) for v, c in zip(even, coeff)]
    b = km.CircuitBuilder()
    sel = b.create_quantum_register(s, 'selection')
    tgt = [q for j in range(J) for q in b.create_quantum_register(brot, f'word_{j}')]
    def target_transform():
        for v, p in zip(nonpivot, pivots):
            for j in range(width):
                if v >> j & 1:
                    b.cx(tgt[p], tgt[j])
    target_transform()
    bank = b.alloc_qubits(len(basis))
    outputs = tgt + list(bank)
    bits = list(sel[1:])[::-1]
    anc = b.alloc_qubits(s - 2)
    H = len(even)
    def node(ctrl, level, lo, hi):
        if lo >= H:
            return
        if hi - lo == 1 or len(set(values[lo:min(H, hi)])) == 1:
            ts = [q for j, q in enumerate(outputs) if values[lo] >> j & 1]
            if ts:
                b.cx(ctrl, km.array(ts))
            return
        mid = (lo + hi) // 2
        if H <= mid:
            node(ctrl, level + 1, lo, mid)
            return
        bit, a = bits[level], anc[level - 1]
        b.x(bit); b.init_and(ctrl, bit, a); b.x(bit)
        node(a, level + 1, lo, mid)
        b.cx(ctrl, a)
        node(a, level + 1, mid, hi)
        b.del_and(ctrl, bit, a)
    half = 1 << (s - 2)
    b.x(bits[0]); node(bits[0], 1, 0, half); b.x(bits[0])
    node(bits[0], 1, half, 2 * half)
    b.free(anc)
    for i, p in enumerate(pivots):
        b.ccx(sel[0], bank[i], tgt[p])
    if len(bank):
        b.del_lookup(table=coeff + [0] * ((1 << (s - 1)) - H), address=sel[1:], target=bank)
    b.free(bank)
    target_transform()
    return b.finish_circuit()


def sum_xor(values):
    result = 0
    for v in values:
        result ^= v
    return result


def build_dfthcsos(out, data_file, brot=10, J=5, gf2=False):
    old = P.banked_lookup
    try:
        if gf2:
            P.banked_lookup = banked_lookup_gf2
        return P.build_dfthcsos(Path(out), Path(data_file), brot=brot, J=J)
    finally:
        P.banked_lookup = old

In [ ]:
%%writefile src/ports/public_sources/anazemcev_quantum-circuits-source-exact-compilation/embedded/src/gauge_generator.py
"""Reproducible exact DFTHC gauge from public numerical input, NumPy only.

SPDX-License-Identifier: Apache-2.0

Input and initial factorization: Dmitrii Gluzdov's public Apache-2.0 dataset
https://www.kaggle.com/datasets/dmitriigluzdov/vqcc-femoco-public-factorization/versions/2
The C30 factor and original sparse-factor work trace to k256.dev and
Kozykorpesh, as credited by the accompanying notebook/source notices.
The log-depth tree and vector decoding follow the organizers' reference.

Own optimization: simultaneous orthogonal Cayley moves reduce the squared
column L1 sum. The augmented [W; H2_I] Gram is preserved; no angle words,
Hamiltonian integrals, symmetry shifts, or acceptance thresholds are changed.
This file imports no research modules or previously saved factorization.
"""
from __future__ import annotations

import argparse
import hashlib
import json
from pathlib import Path
import time

import numpy as np

PUBLIC_SHA256 = "9afd0ca5e90dd1740fe713e4ac1c26736184ea9bc4d4b57a6eeefafd608ee193"
SHIFTS_SHA256 = "7b7facf63fe35a9d9d3d00787ffdb700c863fa845fe4724d7f7c54778151cd87"
FACTORIZATION_KEYS = ("U", "W", "H2_I", "H1_I", "Hsym_lower", "tree_edges")


def log_depth_tree_edges(n=76):
    """Parent-before-child breadth-first order of the official Givens tree."""
    edges, level, depth = [], [0], 0
    while len(edges) < n-1:
        nxt = []
        for label in level:
            nxt.append(label)
            child = label+(1 << depth)
            if child < n:
                edges.append((label, child))
                nxt.append(child)
        level, depth = nxt, depth+1
    return np.asarray(edges, dtype=np.int64)


def decode_words(words, edges, brot=10):
    """Apply the ordered rotations to e0, matching official vector decoding."""
    angles = np.asarray(words, dtype=np.float64)/(1 << (brot-1))
    vectors = np.zeros((len(words), words.shape[1]+1), np.float64)
    vectors[:, 0] = 1.0
    for j, (parent, child) in enumerate(edges):
        parent_value, child_value = vectors[:, parent].copy(), vectors[:, child].copy()
        theta = np.pi*angles[:, j]
        cosine, sine = np.cos(theta), np.sin(theta)
        vectors[:, parent] = cosine*parent_value-sine*child_value
        vectors[:, child] = sine*parent_value+cosine*child_value
    return vectors


def column_l1_square(a, epsilon=0.0):
    absolute = np.sqrt(a*a+epsilon*epsilon) if epsilon else np.abs(a)
    return np.sum(absolute.sum(axis=1)**2, axis=1)


def cayley_stage(a, iterations, epsilon):
    """Orthogonal right rotations, with separate line searches for each rho."""
    r, _, c = a.shape
    eye = np.broadcast_to(np.eye(c), (r, c, c))
    best, best_loss = a.copy(), column_l1_square(a)
    rates = np.full(r, 0.5)
    history = []
    for iteration in range(iterations):
        absolute = np.sqrt(a*a+epsilon*epsilon)
        gradient = 2*absolute.sum(axis=1)[:, None, :]*a/absolute
        raw = a.swapaxes(1, 2) @ gradient
        skew = 0.5*(raw-raw.swapaxes(1, 2))
        initial = column_l1_square(a, epsilon)
        accepted = np.zeros(r, bool)
        next_a, trial_rates = a.copy(), rates.copy()
        for _ in range(12):
            matrix = skew*(0.5*trial_rates)[:, None, None]
            # For skew K, (I+K)^(-1)(I-K) is orthogonal.
            rotation = np.linalg.solve(eye+matrix, eye-matrix)
            trial = a @ rotation
            good = (column_l1_square(trial, epsilon) < initial-1e-13) & ~accepted
            next_a[good] = trial[good]
            accepted |= good
            trial_rates[~accepted] *= 0.5
            if accepted.all():
                break
        a = next_a
        rates = np.minimum(3.0, np.maximum(1e-5, trial_rates*1.2))
        actual = column_l1_square(a)
        improved = actual < best_loss
        best[improved], best_loss[improved] = a[improved], actual[improved]
        if iteration % 40 == 0:
            history.append({"iteration": iteration, "best_l1_square": float(best_loss.sum())})
    return best, history


def generate(public_npz, fixed_shifts_npz, *, copies=30, iterations=100):
    """Return (factorization, unchanged angle words, algebraic build report).

    Dependencies are NumPy, the checksummed public v2 NPZ, and the official
    fixed shifts NPZ. Fresh official verification is a separate caller step.
    """
    started = time.monotonic()
    source, shifts_source = Path(public_npz), Path(fixed_shifts_npz)
    assert hashlib.sha256(source.read_bytes()).hexdigest() == PUBLIC_SHA256
    assert hashlib.sha256(shifts_source.read_bytes()).hexdigest() == SHIFTS_SHA256
    assert copies == 30 and iterations == 100, "reviewed candidate uses C30 and100 iterations/stage"
    with np.load(source, allow_pickle=False) as archive:
        words = archive["words"].astype(np.int64)
        w, h2 = archive["W"].copy(), archive["H2_I"].copy()
    assert words.shape == (3976, 75) and w.shape == (65, 60, 30) and h2.shape == (65, 30)
    assert np.all((words >= 0) & (words < 1024))
    assert np.isfinite(w).all() and np.isfinite(h2).all()
    original_words = words.copy()
    edges = log_depth_tree_edges()
    u = decode_words(words[76:], edges).reshape(65, 60, 76)
    assert np.max(np.abs(np.linalg.norm(u, axis=-1)-1)) < 1e-12
    a = np.concatenate([w, h2[:, None, :]], axis=1)
    original_gram = a @ a.swapaxes(1, 2)
    baseline = float(column_l1_square(a).sum())
    stages = []
    for epsilon in (0.0005, 0.00005, 0.000005):
        a, history = cayley_stage(a, iterations, epsilon)
        stages.append({"epsilon": epsilon, "history": history,
                       "l1_square": float(column_l1_square(a).sum())})
    final = float(column_l1_square(a).sum())
    drift = float(np.max(np.abs(a @ a.swapaxes(1, 2)-original_gram)))
    assert drift < 1e-8 and final < baseline
    # h1 is unchanged. Retain its original encoded eigenvectors: a redundant
    # eigendecomposition could choose different signs and alter angle words.
    assert np.array_equal(words, original_words), "all3976x75 integer angle words must be unchanged"
    with np.load(shifts_source, allow_pickle=False) as archive:
        fixed = {key: archive[key].copy() for key in ("H1_I", "Hsym_lower")}
    data = {"U": u, "W": a[:, :-1, :], "H2_I": a[:, -1, :],
            "tree_edges": edges, **fixed}
    report = {"public_input_sha256": PUBLIC_SHA256, "fixed_shifts_sha256": SHIFTS_SHA256,
              "R_B_C": [65, 60, 30], "brot": 10, "iterations_per_stage": iterations,
              "baseline_l1_square": baseline, "final_l1_square": final,
              "augmented_gram_max_drift": drift,
              "all_3976x75_angle_words_unchanged": True,
              "angle_words_sha256_int64": hashlib.sha256(words.astype("<i8").tobytes()).hexdigest(),
              "stages": stages, "seconds": time.monotonic()-started,
              "numpy_version": np.__version__,
              "correctness_label": "Algebraic invariants checked; full official verifier not invoked by generate()"}
    return data, words, report


def save_factorization(path, data):
    """Save only the public factorization contract and its organizer tree."""
    np.savez(path, **{key: data[key] for key in FACTORIZATION_KEYS})


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("public_npz", type=Path)
    parser.add_argument("fixed_shifts_npz", type=Path)
    parser.add_argument("output_npz", type=Path)
    args = parser.parse_args()
    result, words, report = generate(args.public_npz, args.fixed_shifts_npz)
    args.output_npz.parent.mkdir(parents=True, exist_ok=True)
    save_factorization(args.output_npz, result)
    report["output_npz_sha256"] = hashlib.sha256(args.output_npz.read_bytes()).hexdigest()
    args.output_npz.with_suffix(".build.json").write_text(json.dumps(report, indent=2))
    print(json.dumps(report, indent=2), flush=True)

In [ ]:
%%writefile src/ports/public_sources/lightning_v2/embedded/portable_exact_domain.py
"""Portable exact-domain inverse-square-root Clifford compiler.

Requires the credited signed_digit_invsqrt_v3 module on sys.path, NumPy and
kickmix. The default entry point regenerates from readable source and explicit
coefficient data; it has no filesystem layout, fixture or network dependency.
Full truth-vector equality certifies every selected replacement over every
x=1..3,133,452. Classical branches join exact equal functions or unknown.
"""
from __future__ import annotations
import os
os.environ.update(OPENBLAS_NUM_THREADS='1', OMP_NUM_THREADS='1', MKL_NUM_THREADS='1')
from collections import Counter
import json
import re
import numpy as np
import kickmix as km

MAX_X = 3_133_452

class Known:
    """Immutable truth vector. Samples only reject matches; equality is exact."""
    __slots__ = ('v', 'signature')
    sample = None

    def __init__(self, values):
        self.v = values
        self.signature = values[Known.sample].tobytes()

class DomainCompiler:
    def __init__(self, nqubits, max_x=MAX_X, linear=False, mbu=False, temp_bit=0):
        self.max_x = max_x
        self.nwords = (max_x + 63) // 64
        Known.sample = np.unique(np.r_[np.arange(min(32, self.nwords)),
            np.linspace(0, self.nwords - 1, min(96, self.nwords), dtype=int)])
        self.zero = Known(np.zeros(self.nwords, dtype=np.uint64))
        ones = np.full(self.nwords, np.uint64(2**64 - 1))
        if max_x % 64:
            ones[-1] = np.uint64((1 << (max_x % 64)) - 1)
        self.one = Known(ones)
        xs = np.arange(1, max_x + 1, dtype=np.uint32)
        self.original = []
        for bit in range(22):
            values = np.zeros(self.nwords * 8, dtype=np.uint8)
            packed = np.packbits(((xs >> bit) & 1).astype(np.uint8), bitorder='little')
            values[:len(packed)] = packed
            self.original.append(Known(values.view(np.uint64)))
        self.q = self.original[:] + [self.zero] * (nqubits - 22)
        self.stack = []
        self.replacements = []
        self.counts = Counter()
        self.join_unknown = 0
        self.output = []
        self.linear = linear
        self.linear_false_candidates = 0
        self.mbu = mbu
        self.temp_bit = temp_bit

    @staticmethod
    def equal(a, b):
        if a is b:
            return True
        if a is None or b is None or a.signature != b.signature:
            return False
        return bool(np.array_equal(a.v, b.v))

    def xor(self, a, b):
        if a is None or b is None:
            return None
        if a is self.zero:
            return b
        if b is self.zero:
            return a
        if a is b:
            return self.zero
        return Known(a.v ^ b.v)

    def band(self, a, b):
        if a is self.zero or b is self.zero:
            return self.zero
        if a is None or b is None:
            return None
        if a is self.one:
            return b
        if b is self.one or a is b:
            return a
        return Known(a.v & b.v)

    def literal(self, function, excluded):
        """Return an untouched live literal equal to the full truth vector."""
        if function is None:
            return None
        for q, value in enumerate(self.q):
            if q not in excluded and self.equal(function, value):
                return (q, False)
        complement = self.xor(function, self.one)
        for q, value in enumerate(self.q):
            if q not in excluded and self.equal(complement, value):
                return (q, True)
        return None

    def linear_terms(self, function, excluded):
        """Find an affine GF(2) expression, then prove it with ALL truth bits.

        Gaussian elimination over sampled bits only proposes a combination.
        It cannot certify a rewrite: the combination is always compared with
        the full domain vector before being accepted. Unsampled dependencies
        can cause missed opportunities, never invalid replacements.
        """
        if not self.linear or function is None:
            return None
        variables = [q for q, value in enumerate(self.q)
                     if q not in excluded and value is not None]
        values = [self.q[q] for q in variables] + [self.one]
        extra_words = []
        # A failed full proof supplies another exact input row to the linear
        # system. Refinement removes sample-induced dependencies rather than
        # stopping at the first invalid proposed combination.
        while True:
            def signature(value):
                raw = value.signature
                if extra_words:
                    raw += value.v[extra_words].tobytes()
                return int.from_bytes(raw, 'little')
            basis = {}
            for index, value in enumerate(values):
                vector = signature(value)
                combination = 1 << index
                while vector:
                    pivot = vector.bit_length()
                    if pivot not in basis:
                        basis[pivot] = vector, combination
                        break
                    existing, coefficients = basis[pivot]
                    vector ^= existing
                    combination ^= coefficients
            vector = signature(function)
            combination = 0
            while vector:
                pivot = vector.bit_length()
                if pivot not in basis:
                    return None
                existing, coefficients = basis[pivot]
                vector ^= existing
                combination ^= coefficients
            selected = [i for i in range(len(values)) if combination >> i & 1]
            exact = self.zero
            for i in selected:
                exact = self.xor(exact, values[i])
            if self.equal(exact, function):
                constant = len(variables) in selected
                return [variables[i] for i in selected if i < len(variables)], constant
            self.linear_false_candidates += 1
            witness = int(np.flatnonzero(exact.v != function.v)[0])
            assert witness not in extra_words
            extra_words.append(witness)

    def record(self, line_number, original, emitted, reason):
        self.replacements.append({'line': line_number, 'original': original,
            'replacement': emitted, 'reason': reason, 'condition_depth': len(self.stack)})
        self.counts[reason] += 1

    def execute(self, line, number):
        parts = line.split()
        if not parts:
            return
        op = parts[0]
        qs = [int(s[1:]) for s in parts[1:] if re.fullmatch(r'q\d+', s)]
        emitted = [line]
        # Conditional inline phases have no effect on computational functions.
        if op in {'Z', 'CZ', 'Z_POW', 'NEG', 'INFO_TAG', 'REGISTER',
                  'APPEND_TO_REGISTER', 'BIT_STORE0', 'BIT_INVERT'}:
            pass
        elif op == 'PUSH_CONDITION':
            self.stack.append(self.q[:])
        elif op == 'POP_CONDITION':
            before = self.stack.pop()
            for q, old in enumerate(before):
                if not self.equal(old, self.q[q]):
                    self.q[q] = None
                    self.join_unknown += 1
        elif op in {'HMR', 'R'}:
            assert 'if' not in parts, 'Conditional measurement/reset needs a joined branch.'
            self.q[qs[0]] = self.zero
        elif op == 'X':
            assert 'if' not in parts, 'Inline basis condition is unsupported.'
            self.q[qs[0]] = self.xor(self.q[qs[0]], self.one)
        elif op == 'CX':
            assert 'if' not in parts, 'Inline basis condition is unsupported.'
            a, b = qs
            self.q[b] = self.xor(self.q[b], self.q[a])
        elif op == 'SWAP':
            assert 'if' not in parts, 'Inline basis condition is unsupported.'
            a, b = qs
            self.q[a], self.q[b] = self.q[b], self.q[a]
        elif op == 'CCX':
            assert 'if' not in parts, 'Inline basis condition is unsupported.'
            a, b, t = qs
            product = self.band(self.q[a], self.q[b])
            reason = None
            if self.equal(product, self.zero):
                emitted, reason = [], 'ccx_zero_product'
            elif self.equal(product, self.one):
                emitted, reason = [f'X q{t}'], 'ccx_constant_one'
            else:
                literal = self.literal(product, {t})
                if literal is not None:
                    q, inverted = literal
                    emitted = ([f'X q{t}'] if inverted else []) + [f'CX q{q} q{t}']
                    reason = 'ccx_live_complement' if inverted else 'ccx_live_literal'
                else:
                    terms = self.linear_terms(product, {t})
                    if terms is not None:
                        variables, constant = terms
                        emitted = ([f'X q{t}'] if constant else [])
                        emitted += [f'CX q{q} q{t}' for q in variables]
                        reason = 'ccx_affine_live_span'
            if reason is None and self.mbu:
                result = self.xor(self.q[t], product)
                terms = None
                if self.equal(result, self.zero):
                    terms = [], False
                elif self.equal(result, self.one):
                    terms = [], True
                else:
                    literal = self.literal(result, {t})
                    if literal is not None:
                        q, inverted = literal
                        terms = [q], inverted
                    elif self.linear:
                        terms = self.linear_terms(result, {t})
                if terms is not None:
                    variables, constant = terms
                    prepare = ([f'X q{t}'] if constant else [])
                    prepare += [f'CX q{q} q{t}' for q in variables]
                    # Before: t = a*b XOR L(other wires).  XOR L makes t=a*b.
                    # HMR contributes (-1)**(m*a*b); conditioned CZ cancels it.
                    # The same Clifford expression then restores t=L.
                    emitted = prepare + [f'HMR q{t} b{self.temp_bit}',
                        f'CZ q{a} q{b} if b{self.temp_bit}'] + prepare
                    reason = 'ccx_output_affine_mbu'
            if reason:
                self.record(number, line, emitted, reason)
            self.q[t] = self.xor(self.q[t], product)
        elif op == 'CSWAP':
            assert 'if' not in parts, 'Inline basis condition is unsupported.'
            c, a, b = qs
            delta = self.xor(self.q[a], self.q[b])
            product = self.band(self.q[c], delta)
            reason = None
            if self.equal(product, self.zero):
                emitted, reason = [], 'cswap_zero_delta'
            elif self.equal(product, delta) and delta is not None:
                emitted, reason = [f'SWAP q{a} q{b}'], 'cswap_unconditional'
            else:
                literal = self.literal(product, {a, b})
                if literal is not None:
                    q, inverted = literal
                    emitted = ([f'X q{a}', f'X q{b}'] if inverted else [])
                    emitted += [f'CX q{q} q{a}', f'CX q{q} q{b}']
                    reason = 'cswap_live_complement' if inverted else 'cswap_live_literal'
                else:
                    terms = self.linear_terms(product, {a, b})
                    if terms is not None:
                        variables, constant = terms
                        emitted = ([f'X q{a}', f'X q{b}'] if constant else [])
                        emitted += [gate for q in variables
                                    for gate in (f'CX q{q} q{a}', f'CX q{q} q{b}')]
                        reason = 'cswap_affine_live_span'
            if reason:
                self.record(number, line, emitted, reason)
            self.q[a] = self.xor(self.q[a], product)
            self.q[b] = self.xor(self.q[b], product)
        else:
            raise AssertionError(f'Unsupported operation {op}: {line}')
        self.output.extend(emitted)

    def compile(self, circuit):
        for number, line in enumerate(str(circuit).splitlines(), 1):
            self.execute(line, number)
            if number % 5000 == 0:
                print(json.dumps({'line': number, 'replacements': len(self.replacements),
                    'unknown_qubits': sum(v is None for v in self.q)}), flush=True)
        assert not self.stack
        assert all(self.equal(self.q[q], self.original[q]) for q in range(22)), 'Input function lost.'
        assert all(self.equal(v, self.zero) for v in self.q[22:]), 'Scratch function is not zero.'
        return km.Circuit('\n'.join(self.output) + '\n')

def optimize_circuit(circuit, linear=False, mbu=False):
    """Open in-memory compiler API; no circuit binary or network is required."""
    compiler = DomainCompiler(circuit.num_qubits, linear=linear, mbu=mbu,
                              temp_bit=circuit.num_bits)
    return compiler.compile(circuit), compiler

def build_selected(cfg=None, tables=None, layout=None):
    """Regenerate the credited frozen arithmetic, then prove five reductions.

    This entry point depends only on the readable frozen source and its
    explicit coefficient data. The CLI's existing KMB is an independent
    canonical regression fixture rather than a generator dependency.
    """
    import signed_digit_invsqrt_v3 as core
    if cfg is None or tables is None:
        assert cfg is None and tables is None
        cfg, tables, _ = core.load_or_build_tables()
    source = core.build_selected(cfg, tables, layout)
    circuit, compiler = optimize_circuit(source)
    assert len(compiler.replacements) == 5
    return circuit

In [ ]:
%%writefile src/ports/public_sources/lightning_v2/embedded/rotation_calibration.py
# Source: Mizero v3, Apache-2.0; extends Lightningv08 phase calibration.
"""Exhaustive rotation-angle calibration for an invsqrt kickmix circuit.

The verifier prices every non-Clifford Z_POW by its synthesis T-count at
eps = pi * margin / n_rotations, where margin is the smallest phase slack
(tolerance - |error|, in turns) over all 3,133,452 inputs. Two levers follow:

  * nudge the existing rotation angles to *raise* the worst-case margin, so
    every rotation becomes cheaper to synthesize; and
  * drop the smallest rotations and let the others absorb the change, so
    fewer rotations are paid for at all.

Each Z_POW applies phase theta_j * b_j(x), where b_j(x) is the value of its
qubit at that moment (times its condition bit, if any). We record b_j(x) for
every input by rewriting each rotation as a CX into a fresh record qubit, and
we read the circuit's true total phase from a normal simulation. Then
    phase'(x) = phase(x) + sum_j delta_j b_j(x)       (delta_j = -theta_j if dropped)
is exact, and a linear program over the deltas maximises the common margin.
A cutting-plane loop adds the worst inputs until all inputs are covered.
"""

import math
from decimal import Decimal, localcontext
from fractions import Fraction

import numpy as np
from scipy.optimize import linprog

UNIT = 2 ** 64          # phase units per turn (the verifier's convention)


def parse_kmx(text):
    """Return (lines, rotations); rotation = dict(line, qubit, t, cond, counted)."""
    lines = text.splitlines()
    rots = []
    for i, line in enumerate(lines):
        tok = line.split('#')[0].split()
        if not tok or tok[0] != 'Z_POW':
            continue
        cond = tok[tok.index('if') + 1] if 'if' in tok else None
        t = Fraction(Decimal(tok[2]))
        counted = (t * 4).denominator != 1          # finer than T: a synthesized rotation
        rots.append(dict(line=i, qubit=tok[1], t=t, cond=cond, counted=counted))
    return lines, rots


def instrument(lines, rots, first_record_qubit):
    """Copy of the circuit where rotation j becomes CX qubit -> record_j."""
    out = list(lines)
    for j, r in enumerate(rots):
        rec = f'q{first_record_qubit + j}'
        out[r['line']] = f"CX {r['qubit']} {rec}" + (f" if {r['cond']}" if r['cond'] else '')
    return '\n'.join(out) + '\n'


def theta_units(t):
    """Z_POW exponent (half turns) -> phase units (2^-64 turn), exact if representable."""
    v = t * (UNIT // 2)
    return int(round(v)), v.denominator == 1


def exponent_text(t):
    """Exact decimal text for a dyadic exponent, in kickmix's no-exponent format."""
    with localcontext() as ctx:
        ctx.prec = 400
        d = Decimal(t.numerator) / Decimal(t.denominator)
    s = format(d, 'f')
    if '.' not in s:
        s += '.0'
    return s


def rewrite(lines, rots, deltas_units, dropped):
    out = list(lines)
    remove = set()
    for j, r in enumerate(rots):
        if j in dropped:
            remove.add(r['line'])
            continue
        if deltas_units.get(j, 0):
            t = r['t'] + Fraction(int(deltas_units[j]), UNIT // 2)
            out[r['line']] = f"Z_POW {r['qubit']} {exponent_text(t)}" + (f" if {r['cond']}" if r['cond'] else '')
    return '\n'.join(l for i, l in enumerate(out) if i not in remove) + '\n'


def fit(err_units, tol_units, bits, free, fixed_shift_units, *, bound_units, scale=2.0 ** 28,
        start=4096, add=400, max_iter=60, verbose=True):
    """Maximise m s.t. |err + shift + B[:,free] d| <= tol - m for every input.

    err_units: int64 signed error of the current circuit; tol_units: uint64;
    bits: (n_inputs, n_rot) uint8; fixed_shift_units: float array added to err
    (the effect of dropping rotations). Returns (deltas, margin_units).
    """
    n = len(err_units)
    e = (err_units.astype(np.float64) + fixed_shift_units) / scale
    tol = tol_units.astype(np.float64) / scale
    Bf = bits[:, free].astype(np.float64)
    k = len(free)
    active = set(np.linspace(0, n - 1, start, dtype=np.int64).tolist())
    active.update(np.argsort(tol - np.abs(e))[:start].tolist())
    for it in range(max_iter):
        ii = np.fromiter(sorted(active), dtype=np.int64)
        A = np.vstack([np.c_[Bf[ii], np.ones(len(ii))], np.c_[-Bf[ii], np.ones(len(ii))]])
        b = np.r_[tol[ii] - e[ii], tol[ii] + e[ii]]
        res = linprog(np.r_[np.zeros(k), -1.0], A_ub=A, b_ub=b,
                      bounds=[(-bound_units / scale, bound_units / scale)] * k + [(None, None)], method='highs')
        if not res.success:
            raise RuntimeError(res.message)
        d, m = res.x[:k], res.x[k]
        corrected = e + Bf @ d
        slack = tol - np.abs(corrected)
        worst = float(slack.min())
        if verbose:
            print(f'    LP iter {it}: rows {len(ii):6d}  lp margin {m*scale:.4g}  true margin {worst*scale:.4g} units', flush=True)
        if worst >= m - 1e-9 * max(1.0, abs(m)):
            break
        active.update(np.argpartition(slack, add)[:add].tolist())
    return np.rint(d * scale).astype(np.int64), worst * scale


def exact_margin(err_units, tol_units, bits, deltas_full_units):
    """Integer check of the final, rounded corrections over every input."""
    shift = bits.astype(np.int64) @ deltas_full_units.astype(np.int64)
    corrected = err_units.astype(np.int64) + shift
    slack = tol_units.astype(np.int64) - np.abs(corrected)
    return int(slack.min()), int(np.argmin(slack))


def collect(km, IV, spec, text, rots, *, batch=16384, seed=20261023, log=print):
    """Simulate every input: true phase error (units) and every rotation's bit."""
    c0 = km.Circuit(text)
    Q = c0.num_qubits
    ci = km.Circuit(instrument(text.splitlines(), rots, Q))
    xr, _ = IV.validate_registers(c0, spec)
    rec = km.array([km.q(Q + j) for j in range(len(rots))])
    xs = np.arange(1, spec.max_x + 1, dtype=np.uint64)
    n = len(xs)
    phase = np.empty(n, dtype=np.uint64)
    bits = np.empty((n, len(rots)), dtype=np.uint8)
    s0 = km.Simulator(batch_size=batch, seed=seed, debug_print=False)
    s1 = km.Simulator(batch_size=batch, seed=seed + 1, debug_print=False)
    for start in range(0, n, batch):
        x = np.resize(xs[start:start + batch], batch)
        take = min(batch, n - start)
        for s, c in ((s0, c0), (s1, ci)):
            s.clear_for_shot(); s.write_across_shots(xr, IV.to_bits(x, spec.width)); s.do(c)
        phase[start:start + take] = [IV.half_turns_to_units(s0.read_shot_phase(j)) for j in range(take)]
        bits[start:start + take] = s1.read_across_shots(rec)[:, :take].T
        if start // batch % 40 == 0:
            log(f'  simulated {start + take:,}/{n:,}')
    target, tol = IV.reference_values(spec, xs)
    err = (phase - np.asarray(target, dtype=np.uint64)).view(np.int64)
    return xs, err, np.asarray(tol, dtype=np.uint64), bits, Q


def search(err, tol, bits, rots, report, rotation_t_count, *, max_drop=8, log=print):
    """Try dropping 0..max_drop smallest rotations; return candidates with cost estimates."""
    counted = [j for j, r in enumerate(rots) if r['counted']]
    weight = {j: abs(float(min(rots[j]['t'] % 2, 2 - rots[j]['t'] % 2))) for j in counted}
    order = sorted(counted, key=lambda j: weight[j])
    Qn, tof, tcount = report['num_qubits'], report['toffoli_count'], report['t_count']
    theta = np.array([theta_units(r['t'])[0] for r in rots], dtype=np.float64)

    def estimate(n_rot, margin_units):
        if margin_units <= 0:
            return float('inf')
        eps = math.pi * (margin_units / UNIT) / n_rot
        return Qn * (tof + (tcount + n_rot * rotation_t_count(eps)) / 4)

    m0, _ = exact_margin(err, tol, bits, np.zeros(len(rots), dtype=np.int64))
    base = estimate(len(counted), m0)
    log(f'baseline: {len(counted)} rotations, margin {m0:.6g} units, estimated cost {base:,.3f} '
        f'(verifier reported {report["score"]:,.3f})')
    rows = [dict(drop=0, dropped=[], deltas=np.zeros(len(rots), dtype=np.int64), margin=m0, estimate=base, fitted=False)]
    for k in range(0, max_drop + 1):
        dropped = order[:k]
        free = [j for j in counted if j not in dropped]
        shift = -(bits[:, dropped].astype(np.float64) @ theta[dropped]) if dropped else np.zeros(len(err))
        bound = max([4 * abs(theta[j]) for j in dropped] + [2.0 ** 36])
        log(f'drop {k}: ' + (', '.join(f'{float(rots[j]["t"]):.3e}' for j in dropped) or 'none'))
        try:
            d_free, _ = fit(err, tol, bits, free, shift, bound_units=bound, verbose=False)
        except RuntimeError as exc:
            log(f'    LP failed: {exc}'); continue
        deltas = np.zeros(len(rots), dtype=np.int64)
        deltas[free] = d_free
        for j in dropped:
            deltas[j] = -int(theta_units(rots[j]['t'])[0])
        m, worst_i = exact_margin(err, tol, bits, deltas)
        est = estimate(len(free), m)
        log(f'    exact margin {m:.6g} units, estimated cost {est:,.3f}')
        rows.append(dict(drop=k, dropped=dropped, deltas=deltas, margin=m, estimate=est, fitted=True))
        if m <= 0:
            break
    return rows

In [ ]:
%%writefile src/ports/virtual_low.py
# SPDX-License-Identifier: Apache-2.0
"""Zongao Bian's virtual low-bit idea, reconstructed from notebook prose.

This implementation removes ONE low accumulator bit. For even K0 and rows
sh=0..13, its successive parity values telescope to derivative[sh]. The
carry into the physical suffix is cin XOR d*(a0 XOR cin), in both forward
and inverse row schedules. Compute that carry only while a row needs it.
The two-bit implementation for R2 below also virtualizes the derivative's
low bit; that bit telescopes to the last consecutive rounded curvature bit.
This is a reconstruction, not a copy of the unavailable resource ZIP.
"""

def outer(b, add, derivative, low_suffix, delta, shifts, inverse):
    assert list(reversed(shifts)) == list(range(14))
    for i in range(14) if inverse else reversed(range(14)):
        sh = shifts[i]
        a0 = derivative[sh]
        cin = derivative[sh-1] if sh else None
        a = derivative[sh+1:]
        carry = b.alloc_qubits(1)[0]
        if cin is not None:
            b.cx(cin, a0)
        b.init_and(delta[i], a0, carry)
        if cin is not None:
            b.cx(cin, carry)
            b.cx(cin, a0)
        b.cx(delta[i], low_suffix)
        add(a, low_suffix, carry, sub=inverse)
        b.cx(delta[i], low_suffix)
        if cin is not None:
            b.cx(cin, a0)
            b.cx(cin, carry)
        b.del_and(delta[i], a0, carry)
        if cin is not None:
            b.cx(cin, a0)
        b.free(carry)


def derivative_lsb(cfg):
    shifts=[cfg.sh2[i] for i in cfg.rrow_list()]
    nonnegative=[s for s in shifts if s>=0]
    assert nonnegative == list(range(len(nonnegative))), shifts
    return nonnegative[-1]


def inner(b, add, curvature, res_suffix, delta, cfg, inverse):
    derivative_lsb(cfg)  # certify consecutive rounding terms telescope
    rows=cfg.rrow_list()
    for i in rows[::-1] if inverse else rows:
        sh=cfg.sh2[i]
        if sh<0:
            target=res_suffix[-sh-1:]
            b.cx(delta[i],target)
            add(curvature,target,sub=inverse)
            b.cx(delta[i],target)
            continue
        a0=curvature[sh]
        cin=curvature[sh-1] if sh else None
        carry=b.alloc_qubits(1)[0]
        if cin is not None:
            b.cx(cin,a0)
        b.init_and(delta[i],a0,carry)
        if cin is not None:
            b.cx(cin,carry);b.cx(cin,a0)
        b.cx(delta[i],res_suffix)
        add(curvature[sh+1:],res_suffix,carry,sub=inverse)
        b.cx(delta[i],res_suffix)
        if cin is not None:
            b.cx(cin,a0);b.cx(cin,carry)
        b.del_and(delta[i],a0,carry)
        if cin is not None:
            b.cx(cin,a0)
        b.free(carry)

In [ ]:
%%writefile src/ports/virtual_round2.py
# SPDX-License-Identifier: Apache-2.0
# Modified: one virtual low accumulator bit; see virtual_low.py for credit and identity.
"""Signed-digit oracle schedules, derived from Sumit Chongder and Rudra R4/R5.

Changes made here: configurable precisions and split widths; borrow the
normalized leading one for a clean carry; prune the final phase-shift stage
using the promised s<=10 bound. No verifier logic is changed.
"""
from fractions import Fraction
import numpy as np
import kickmix as km
import sumit_reference as H
import signed_digit_lab as A
import deferred_lookup as D
import hybrid_add as B

def prepare(cfg,tabs):
 d=np.arange(1<<H.DB,dtype=np.int64);Rmax=0;spread=0;minimum={};maxima={}
 for key,(k2,k1,k0) in tabs.items():
  r,_=H.pipeline(cfg,k2,k1,d);minimum[key]=int(r.min());maxima[key]=int(r.max());Rmax=max(Rmax,int(r.max()),k1)
  spread=max(spread,int(r.max()-r.min()))
 # Pick the narrowest unsigned residual which also contains the initial K1.
 for block in range(1,Rmax.bit_length()):
  bases={k:v>>block for k,v in minimum.items()}
  if all(0<=tabs[k][1]-(v<<block)<(1<<(block+1)) and maxima[k]-(v<<block)<(1<<(block+1)) for k,v in bases.items()):break
 return {"block":block,"rw":block+1,"cw":Rmax.bit_length()-block,
         "coarse":bases,"residual":{k:tabs[k][1]-(v<<block) for k,v in bases.items()}}

def shift(b,flags,word,L,inverse,prune=True):
 stages=[(1,1),(2,2),(3,4),(4,8)];occ=[];n=L
 for k,d in stages:
  occ.append(min(n,L+2) if prune and k==4 else n);n=min(n+d,len(word))
 seq=list(enumerate(stages));seq=seq[::-1] if inverse else seq
 for j,(k,d) in seq:
  indices=range(occ[j]) if inverse else reversed(range(occ[j]))
  for i in indices:
   if i+d<len(word):
    zero_boundary = i+d >= L+sum(amount for _,amount in stages[:j])
    if zero_boundary:
     if inverse:b.cx(word[i+d],word[i]);b.del_and(flags[k],word[i],word[i+d])
     else:b.init_and(flags[k],word[i],word[i+d]);b.cx(word[i+d],word[i])
    else:b.cswap(flags[k],word[i],word[i+d])

def build(cfg,tabs,*,borrow=True,prune=True,merge=False,parity_input=False,carry_block=None,phase_cut=0,deferred=False,anf=False):
 lay=H.layout(cfg,tabs,LB=cfg.fy-7,S=cfg.fy-8);split=prepare(cfg,tabs)
 assert not merge and all(v%2==0 for v in lay['lo'].values()), 'virtual low bit requires even anchors and split phases' 
 b=km.CircuitBuilder();x=b.create_quantum_register(22,name="x");flags=b.alloc_qubits(4 if parity_input else 5)
 f=([x[21]]+list(flags)) if parity_input else flags
 if parity_input:
  for k,amount in H.NSTAGES[:-1]:
   top=x[22-amount:22];b.x(top);H.and_all_into(b,list(top),f[k]);b.x(top)
   for i in reversed(range(22-amount)):b.cswap(f[k],x[i],x[i+amount])
  b.x(x[21]);b.left_rotate(x[:21],1,control=x[21]);b.cx(x[21],x[0])
 else:H.normalize(b,x,f)
 borrowed=[]
 if borrow and not parity_input:b.x(x[21]);borrowed=[x[21]]
 c2,res,coarse,low,high=[b.alloc_qubits(n) for n in [cfg.w2,split["rw"],split["cw"],(lay["LB"]-1),lay["whi"]]]
 checkpoints=[]
 def mark(label):
  z=b # stats collected from final tagged operation counts externally when supported
 def add(a,t,cin=None,sub=False):
  if carry_block:return B.add(b,a,t,cin,sub,borrowed=borrowed,prefix=carry_block)
  return A.folded_gadd(b,a,t,cin,sub,borrowed=borrowed)
 full=km.array(list(low)+list(res)+list(c2)+list(high)+list(coarse))
 rest=km.array(list(low)+list(res)+list(c2)+list(coarse));fulltable=[];resttable=[];hightable=[]
 for e in range(2):
  for i in range(128):
   k=(e,i);common=(lay["lo"][k]>>1)|(split["residual"][k]<<(lay["LB"]-1))|(tabs[k][0]<<((lay["LB"]-1)+split["rw"]))
   resttable.append(common|(split["coarse"][k]<<((lay["LB"]-1)+split["rw"]+cfg.w2)))
   fulltable.append(common|(lay["hi"][k]<<((lay["LB"]-1)+split["rw"]+cfg.w2))|(split["coarse"][k]<<((lay["LB"]-1)+split["rw"]+cfg.w2+lay["whi"])))
   hightable.append(lay["hi"][k])
 address=km.array(list(x[14:21])+[f[0]]);delta=x[:14]
 def phase(reg,lowbit):
  ext=b.alloc_qubits(10-len(borrowed));word=km.array(list(reg)+borrowed+list(ext))
  shift(b,f,word,len(reg),False,prune)
  for i,q in enumerate(word):
   exponent=lowbit+i-cfg.fy-10
   if exponent<0 and not(lowbit==cfg.gy and i<phase_cut):b.z_pow(q,Fraction(2)**(exponent+1))
  shift(b,f,word,len(reg),True,prune);b.free(ext)
 def inner(inverse):
  rows=cfg.rrow_list();rows=rows[::-1] if inverse else rows
  for i in rows:
   sh=cfg.sh2[i]
   if sh>=0:a=c2[sh:];t=res;cin=c2[sh-1] if sh else None
   else:a=c2;t=res[-sh:];cin=None
   b.cx(delta[i],t);add(a,t,cin,sub=inverse);b.cx(delta[i],t)
 def outer(derivative,inverse):
  import virtual_low
  virtual_low.outer(b,add,derivative,low,delta,cfg.sh1,inverse)
 b.init_lookup(table=fulltable,address=address,target=full)
 deferred_bits=[]
 if not merge:
  phase(high,lay["S"])
  if deferred:deferred_bits=D.demolish(b,high)
  else:b.del_lookup(table=hightable,address=address,target=high)
  b.free(high)
 inner(False);add(res[split["block"]:],coarse)
 derivative=res[:split["block"]]+coarse
 outer(derivative,False)
 if merge:
  add(low[lay["S"]:],high)
  phase(km.array(list(low[cfg.gy:lay["S"]])+list(high)),cfg.gy)
  add(low[lay["S"]:],high,sub=True)
 else:phase(low[cfg.gy-1:],cfg.gy)
 outer(derivative,True);add(res[split["block"]:],coarse,sub=True);inner(True)
 remaining=full if merge else rest
 if deferred:
  deferred_bits+=D.demolish(b,remaining);b.free(remaining)
  table=fulltable if merge else [hi|(rr<<lay['whi']) for hi,rr in zip(hightable,resttable)]
  D.repair(b,address,table,deferred_bits,anf=anf)
 else:
  b.del_lookup(table=fulltable if merge else resttable,address=address,target=remaining);b.free(remaining)
 if borrowed:b.x(x[21])
 if parity_input:
  b.cx(x[21],x[0]);b.right_rotate(x[:21],1,control=x[21]);b.x(x[21])
  for k,amount in reversed(H.NSTAGES[:-1]):
   for i in range(22-amount):b.cswap(f[k],x[i],x[i+amount])
   top=x[22-amount:22];m=b.alloc_bits(1)[0];b.hmr(f[k],m);b.push_condition(m)
   b.x(top);H.phase_all(b,list(top));b.x(top);b.pop_condition();b.free(m)
 else:H.unnormalize(b,x,f)
 b.free(flags)
 return b.finish_circuit(),{k:v for k,v in split.items() if k not in ["coarse","residual"]}

In [ ]:
%%writefile src/ports/virtual_time_mux_round3.py
# SPDX-License-Identifier: Apache-2.0
# Modified: one virtual low accumulator bit; see virtual_low.py for credit and identity.
"""Curvature time multiplexing with shared classical phase repair.

New Apache-2.0 implementation, building on Sumit Chongder's centered
quadratic, Gu et al.'s deferred repairs, and round-2 arithmetic.
The curvature is constant on groups of adjacent address intervals. It is
looked up only during the inner multiplications, demolished twice, and
both measurement masks are merged into the final shared ANF repair.
"""
import os,sys,pathlib,json,math,time,concurrent.futures,traceback
os.environ['OMP_NUM_THREADS']='1'

sys.path[:0]=[str(pathlib.Path(__file__).parent/'vendor')]
import numpy as np
import kickmix as km
from fractions import Fraction
import sumit_reference as H
import signed_digit_lab as A
import round2 as O
import deferred_lookup as D
from invsqrt import verify_submission_invsqrt as V

def joint_top_normalization_repair(b,x,m8,m16):
 """On positive22-bit inputs, f16=g16 and f8=g8 XOR g16,
 where gk tests whether the highest k input bits are zero.
 Thus the joint phase is m8*g8 XOR (m8 XOR m16)*g16.
 A shared16-bit prefix ladder, or an8-bit phase, costs8.5 mean
 Toffolis instead of10 for the two separate repairs.
 """
 # Classical allocation can recycle a nonzero bit. XOR directly into the
 # fresh HMR result instead of assuming a newly allocated bit is zero.
 parity=m16;b.cx(m8,parity)
 b.push_condition(parity)
 top=list(reversed(x[6:22]));b.x(top);anc=b.alloc_qubits(14)
 b.init_and(top[0],top[1],anc[0])
 for j in range(1,14):b.init_and(anc[j-1],top[j+1],anc[j])
 b.push_condition(m8);b.z(anc[6]);b.pop_condition()
 b.cz(anc[13],top[15])
 for j in reversed(range(1,14)):b.del_and(anc[j-1],top[j+1],anc[j])
 b.del_and(top[0],top[1],anc[0]);b.free(anc);b.x(top);b.pop_condition()
 b.x(parity);b.push_condition(parity);b.push_condition(m8)
 top=x[14:22];b.x(top);H.phase_all(b,list(top));b.x(top)
 b.pop_condition();b.pop_condition()

def anf_lookup(b,address,target,table):
 """Compute an XOR lookup with depth-first ANF monomials, at most
 2**n-n-1 temporary ANDs and n-1 ladder wires. New implementation.
 """
 address=list(address);coef=list(table);n=len(address)
 for i in range(n):
  for mask in range(1<<n):
   if mask>>i&1:coef[mask]^=coef[mask^(1<<i)]
 active={}
 def needed(mask,nextbit):
  children=[(mask|(1<<j),j+1,j) for j in range(nextbit,n)]
  active[mask]=bool(coef[mask])
  for child,nxt,j in children:active[mask]|=needed(child,nxt)
  return active[mask]
 for i in range(n):needed(1<<i,i+1)
 def xor(value,control=None):
  for i,q in enumerate(target):
   if value>>i&1:
    if control is None:b.x(q)
    else:b.cx(control,q)
 xor(coef[0])
 def visit(mask,control,nextbit):
  xor(coef[mask],control)
  for j in range(nextbit,n):
   child=mask|(1<<j)
   if not active[child]:continue
   q=b.alloc_qubits(1)[0];b.init_and(control,address[j],q)
   visit(child,q,j+1);b.del_and(control,address[j],q);b.free(q)
 for i in range(n):
  if active[1<<i]:visit(1<<i,address[i],i+1)

def adaptive_lookup(b,address,target,table,affine=False):
 """Unary decoder pruned wherever every remaining output word is equal.

 A full M-leaf binary tree costs M-2 temporary ANDs; address wires and
 one ladder wire per active depth suffice. Demolition is standard MBU.
 """
 address=list(address)
 def xor_word(value,control):
  for i,q in enumerate(target):
   if value>>i&1:
    if control is None:b.x(q)
    else:b.cx(control,q)
 def walk(bits,rows,parent):
  if len(set(rows))==1:xor_word(rows[0],parent);return
  assert bits
  if affine:
   base=rows[0];columns=[rows[1<<i]^base for i in range(len(bits))]
   test=[base]
   for col in columns:test += [v^col for v in test]
   if rows==test:
    xor_word(base,parent)
    for bit,col in zip(bits,columns):
     if not col:continue
     if parent is None:xor_word(col,bit)
     else:
      q=b.alloc_qubits(1)[0];b.init_and(parent,bit,q);xor_word(col,q);b.del_and(parent,bit,q);b.free(q)
    return
  bit=bits[-1];half=len(rows)//2
  if parent is None:
   b.x(bit);walk(bits[:-1],rows[:half],bit);b.x(bit)
   walk(bits[:-1],rows[half:],bit)
  else:
   tmp=b.alloc_qubits(1)[0]
   b.x(bit);b.init_and(parent,bit,tmp);b.x(bit)
   walk(bits[:-1],rows[:half],tmp)
   b.cx(parent,tmp);walk(bits[:-1],rows[half:],tmp);b.cx(parent,tmp)
   b.x(bit);b.del_and(parent,bit,tmp);b.x(bit);b.free(tmp)
 walk(address,list(table),None)

def fit_group(job):
 params,group_bits,e,g=job;c=H.Cfg(*params)
 d=np.arange(1<<14,dtype=np.int64);G=1<<group_bits
 mid=1+(g+.5)*G/128
 center=round(.375*2.**((e-1)/2)*mid**-2.5*2**c.f2)
 best=None
 for k2 in range(max(0,center-6),min((1<<c.w2)-1,center+6)+1):
  rows=[];scores=[]
  for a in range(g*G,(g+1)*G):
   xn=(1<<21)+(a<<14)+d
   dd=d[(d%2==0) if e else ((xn<=3133452)|(d%4==0))]
   m=1+(a*(1<<14)+dd)*2.**-21;target=2.**((e-1)/2)/np.sqrt(m)*2**c.fy
   mid1=1+(a*(1<<14)+((1<<14)-1)/2)*2.**-21
   k1c=round(.5*2.**((e-1)/2)*mid1**-1.5*2**c.f1)
   segment=None
   for k1 in range(k1c-5,k1c+6):
    r,q=H.pipeline(c,k2,k1,dd);vals=target+q
    center0=(float(vals.min())+float(vals.max()))/2+((1<<c.gy)-1)/2
    for k0 in range(math.floor(center0)-4,math.ceil(center0)+5):
     y=((k0-q)>>c.gy)<<c.gy
     if y.max() >= 1<<c.fy:continue
     ratio=float(np.max(abs(y-target)/target))*2**24
     if segment is None or ratio<segment[0]:segment=(ratio,k2,k1,k0)
   if segment is None:break
   scores.append(segment[0]);rows.append(segment[1:])
  if len(rows)==G and (best is None or max(scores)<best[0]):best=(max(scores),rows)
 return (e,g),best

def build(c,tabs,group_bits=2,mode='mux',interpolation=None,main_lookup='unary',affine=False,joint_norm=False):
 lay=H.layout(c,tabs,LB=c.fy-7,S=c.fy-8);sp=O.prepare(c,tabs)
 assert all(v%2==0 for v in lay['lo'].values()), 'virtual low bit requires even anchors'
 b=km.CircuitBuilder();x=b.create_quantum_register(22,name='x');flags=b.alloc_qubits(4);f=[x[21]]+list(flags)
 for k,amount in H.NSTAGES[:-1]:
  top=x[22-amount:22];b.x(top);H.and_all_into(b,list(top),f[k]);b.x(top)
  for i in reversed(range(22-amount)):b.cswap(f[k],x[i],x[i+amount])
 b.x(x[21]);b.left_rotate(x[:21],1,control=x[21]);b.cx(x[21],x[0])
 res,coarse,low,high=[b.alloc_qubits(n) for n in [sp['rw'],sp['cw'],(lay['LB']-1),lay['whi']]]
 full=km.array(list(low)+list(res)+list(high)+list(coarse));rest=km.array(list(low)+list(res)+list(coarse))
 ft=[];rt=[];ht=[];ct=[]
 for e in range(2):
  for a in range(128):
   k=(e,a);common=(lay['lo'][k]>>1)|(sp['residual'][k]<<(lay['LB']-1))
   rt.append(common|(sp['coarse'][k]<<((lay['LB']-1)+sp['rw'])))
   ft.append(common|(lay['hi'][k]<<((lay['LB']-1)+sp['rw']))|(sp['coarse'][k]<<((lay['LB']-1)+sp['rw']+lay['whi'])))
   ht.append(lay['hi'][k])
  if group_bits is not None:
   for a in range(0,128,1<<group_bits):
    assert len({tabs[e,j][0] for j in range(a,a+(1<<group_bits))})==1
    ct.append(tabs[e,a][0])
  else:ct.extend(tabs[e,a][0] for a in range(128))
 address=km.array(list(x[14:21])+[f[0]]);ca=km.array(list(x[14+(group_bits or 0):21])+[f[0]]);delta=x[:14]
 c2=None;step_masks=[]
 def add(a,t,cin=None,sub=False):A.folded_gadd(b,a,t,cin,sub)
 def phase(reg,lowbit):
  ext=b.alloc_qubits(10);word=km.array(list(reg)+list(ext))
  O.shift(b,f,word,len(reg),False)
  for i,q in enumerate(word):
   ex=lowbit+i-c.fy-10
   if ex<0:b.z_pow(q,Fraction(2)**(ex+1))
  O.shift(b,f,word,len(reg),True);b.free(ext)
 def load():
  nonlocal c2
  c2=b.alloc_qubits(c.w2)
  if interpolation is not None:
   sw=interpolation['step_width'];step=b.alloc_qubits(sw)
   uw=interpolation.get('update_width',c.w2);mask=(1<<uw)-1
   packed=[((base&mask)|(tabs[a//128,a%128][0]&~mask))|(s<<c.w2) for a,(base,s) in enumerate(interpolation['rows'])]
   adaptive_lookup(b,address,km.array(list(c2)+list(step)),packed,affine=affine)
   for i in range(interpolation['remainder_bits']):
    sh=interpolation['fraction_bits']-i
    if sh>=0:a=step[sh:];t=c2[:uw];cin=step[sh-1] if sh else None
    else:a=step;t=c2[-sh:uw];cin=None
    b.cx(x[14+i],t);add(a,t,cin);b.cx(x[14+i],t)
   step_masks.extend(D.demolish(b,step));b.free(step)
  elif group_bits is None:adaptive_lookup(b,ca,c2,ct,affine=affine)
  else:b.init_lookup(table=ct,address=ca,target=c2)
 def erase():
  nonlocal c2
  mm=D.demolish(b,c2);b.free(c2);c2=None;return mm
 def inner(inverse):
  rows=c.rrow_list();rows=rows[::-1] if inverse else rows
  for i in rows:
   sh=c.sh2[i]
   if sh>=0:a=c2[sh:];t=res;cin=c2[sh-1] if sh else None
   else:a=c2;t=res[-sh:];cin=None
   b.cx(delta[i],t);add(a,t,cin,sub=inverse);b.cx(delta[i],t)
 def outer(inverse):
  import virtual_low
  derivative=res[:sp['block']]+coarse
  virtual_low.outer(b,add,derivative,low,delta,c.sh1,inverse)
 if main_lookup=='anf':anf_lookup(b,address,full,ft)
 else:b.init_lookup(table=ft,address=address,target=full)
 phase(high,lay['S']);bits=D.demolish(b,high);b.free(high)
 load();inner(False);cm1=erase();add(res[sp['block']:],coarse);outer(False)
 phase(low[c.gy-1:],c.gy)
 outer(True);add(res[sp['block']:],coarse,sub=True);load();inner(True);cm2=erase()
 bits+=D.demolish(b,rest);b.free(rest)
 # Separate copies of the curvature word can share its truth table. Repeating
 # its bits is equivalent to XORing the measurement masks, without bit gates.
 bits+=cm1+cm2
 rtwidth=len(rest);hw=lay['whi']
 repairtable=[h|(r<<hw)|(tabs[e,a][0]<<(hw+rtwidth))|(tabs[e,a][0]<<(hw+rtwidth+c.w2)) for e in range(2) for a,(h,r) in enumerate(zip(ht[128*e:128*(e+1)],rt[128*e:128*(e+1)]))]
 if interpolation is not None:
  bits+=step_masks;sw=interpolation['step_width'];off=hw+rtwidth+2*c.w2
  for a,(_,step) in enumerate(interpolation['rows']):
   repairtable[a]|=(step<<off)|(step<<(off+sw))
 D.repair(b,address,repairtable,bits,anf=True)
 b.cx(x[21],x[0]);b.right_rotate(x[:21],1,control=x[21]);b.x(x[21])
 pending=None
 for k,amount in reversed(H.NSTAGES[:-1]):
  for i in range(22-amount):b.cswap(f[k],x[i],x[i+amount])
  top=x[22-amount:22];m=b.alloc_bits(1)[0];b.hmr(f[k],m)
  if joint_norm and k==3:pending=m;continue
  if joint_norm and k==4:
   joint_top_normalization_repair(b,x,pending,m);b.free(pending);b.free(m)
  else:
   b.push_condition(m);b.x(top);H.phase_all(b,list(top));b.x(top);b.pop_condition();b.free(m)
 b.free(flags)
 return b.finish_circuit(),{k:v for k,v in sp.items() if k not in ['coarse','residual']}

def main():
 import argparse
 p=argparse.ArgumentParser();p.add_argument('--params',default='9,19,26,2');p.add_argument('--drop',default='5');p.add_argument('--group-bits',type=int,default=2);p.add_argument('--tables');p.add_argument('--verify',action='store_true');args=p.parse_args()
 params=[*map(int,args.params.split(',')),list(map(int,args.drop.split(','))) if args.drop else []]
 c=H.Cfg(*params);out=pathlib.Path(os.environ['QCC_OUT']);t0=time.monotonic();tabs={};ratios=[]
 if args.tables:
  saved=json.loads(pathlib.Path(args.tables).read_text());tabs={(e,a):tuple(saved['rows'][e*128+a]) for e in range(2) for a in range(128)}
 else:
  G=1<<args.group_bits
  with concurrent.futures.ProcessPoolExecutor(max_workers=int(os.environ['QCC_JOBS'])) as pool:
   for (e,g),row in pool.map(fit_group,[(params,args.group_bits,e,g) for e in range(2) for g in range(128//G)],chunksize=1):
    if row is None:raise ValueError((e,g,'no fit'))
    ratios.append(row[0]);tabs.update({(e,g*G+j):r for j,r in enumerate(row[1])})
 (out/'tables.json').write_text(json.dumps(dict(params=params,group_bits=args.group_bits,rows=[tabs[e,a] for e in range(2) for a in range(128)])))
 cert=A.exhaustive_certificate(c,tabs);result=dict(params=params,group_bits=args.group_bits,certificate=cert,normalized_worst=max(ratios) if ratios else None,reports=[])
 if cert['pass']:
  try:
   circ,sp=build(c,tabs,args.group_bits);circ.to_file(str(out/'invsqrt.kmb'),format='kmb');st=circ.operation_statistics()
   result.update(static=dict(Q=circ.num_qubits,toffoli=int(st.num_toffoli_like),rotations=int(st.num_z_pow(min_clifford_hierarchy_level=4))),split=sp,estimate=A.estimated_resources(circ,cert))
   print(json.dumps(result),flush=True)
   if args.verify:
    for suffix in ['A','B']:
     salt=f'round3-mux-g{args.group_bits}-{args.params}-{suffix}';start=time.monotonic()
     try:r=V.verify(circ,V.ProblemSpecInvSqrt.for_n_p(10),salt=salt,max_workers=int(os.environ['QCC_JOBS']))
     except Exception as exc:r=dict(status='FAIL',error=str(exc),traceback=traceback.format_exc())
     r.update(salt=salt,seconds=time.monotonic()-start);result['reports'].append(r)
     (out/'results.json').write_text(json.dumps(result,indent=2));print(json.dumps(r),flush=True)
     if r['status']!='PASS':break
  except Exception as exc:result.update(error=str(exc),traceback=traceback.format_exc())
 result['seconds']=time.monotonic()-t0;(out/'results.json').write_text(json.dumps(result,indent=2));print(json.dumps(result),flush=True)
if __name__=='__main__':main()

In [ ]:
%%writefile src/ports/virtual_two_round2.py
# SPDX-License-Identifier: Apache-2.0
# Modified: virtual low accumulator AND derivative bits; see virtual_low.py.
"""Signed-digit oracle schedules, derived from Sumit Chongder and Rudra R4/R5.

Changes made here: configurable precisions and split widths; borrow the
normalized leading one for a clean carry; prune the final phase-shift stage
using the promised s<=10 bound. No verifier logic is changed.
"""
from fractions import Fraction
import numpy as np
import kickmix as km
import sumit_reference as H
import signed_digit_lab as A
import deferred_lookup as D
import hybrid_add as B

def prepare(cfg,tabs):
 d=np.arange(1<<H.DB,dtype=np.int64);Rmax=0;spread=0;minimum={};maxima={}
 for key,(k2,k1,k0) in tabs.items():
  r,_=H.pipeline(cfg,k2,k1,d);minimum[key]=int(r.min());maxima[key]=int(r.max());Rmax=max(Rmax,int(r.max()),k1)
  spread=max(spread,int(r.max()-r.min()))
 # Pick the narrowest unsigned residual which also contains the initial K1.
 for block in range(1,Rmax.bit_length()):
  bases={k:v>>block for k,v in minimum.items()}
  if all(0<=tabs[k][1]-(v<<block)<(1<<(block+1)) and maxima[k]-(v<<block)<(1<<(block+1)) for k,v in bases.items()):break
 return {"block":block,"rw":block+1,"cw":Rmax.bit_length()-block,
         "coarse":bases,"residual":{k:tabs[k][1]-(v<<block) for k,v in bases.items()}}

def shift(b,flags,word,L,inverse,prune=True):
 stages=[(1,1),(2,2),(3,4),(4,8)];occ=[];n=L
 for k,d in stages:
  occ.append(min(n,L+2) if prune and k==4 else n);n=min(n+d,len(word))
 seq=list(enumerate(stages));seq=seq[::-1] if inverse else seq
 for j,(k,d) in seq:
  indices=range(occ[j]) if inverse else reversed(range(occ[j]))
  for i in indices:
   if i+d<len(word):
    zero_boundary = i+d >= L+sum(amount for _,amount in stages[:j])
    if zero_boundary:
     if inverse:b.cx(word[i+d],word[i]);b.del_and(flags[k],word[i],word[i+d])
     else:b.init_and(flags[k],word[i],word[i+d]);b.cx(word[i+d],word[i])
    else:b.cswap(flags[k],word[i],word[i+d])

def build(cfg,tabs,*,borrow=True,prune=True,merge=False,parity_input=False,carry_block=None,phase_cut=0,deferred=False,anf=False):
 lay=H.layout(cfg,tabs,LB=cfg.fy-7,S=cfg.fy-8);split=prepare(cfg,tabs)
 assert not merge and all(v%2==0 for v in lay['lo'].values()) and all(row[1]%2==0 for row in tabs.values()), 'virtual low bit requires even anchors and split phases' 
 b=km.CircuitBuilder();x=b.create_quantum_register(22,name="x");flags=b.alloc_qubits(4 if parity_input else 5)
 f=([x[21]]+list(flags)) if parity_input else flags
 if parity_input:
  for k,amount in H.NSTAGES[:-1]:
   top=x[22-amount:22];b.x(top);H.and_all_into(b,list(top),f[k]);b.x(top)
   for i in reversed(range(22-amount)):b.cswap(f[k],x[i],x[i+amount])
  b.x(x[21]);b.left_rotate(x[:21],1,control=x[21]);b.cx(x[21],x[0])
 else:H.normalize(b,x,f)
 borrowed=[]
 if borrow and not parity_input:b.x(x[21]);borrowed=[x[21]]
 c2,res,coarse,low,high=[b.alloc_qubits(n) for n in [cfg.w2,(split["rw"]-1),split["cw"],(lay["LB"]-1),lay["whi"]]]
 checkpoints=[]
 def mark(label):
  z=b # stats collected from final tagged operation counts externally when supported
 def add(a,t,cin=None,sub=False):
  if carry_block:return B.add(b,a,t,cin,sub,borrowed=borrowed,prefix=carry_block)
  return A.folded_gadd(b,a,t,cin,sub,borrowed=borrowed)
 full=km.array(list(low)+list(res)+list(c2)+list(high)+list(coarse))
 rest=km.array(list(low)+list(res)+list(c2)+list(coarse));fulltable=[];resttable=[];hightable=[]
 for e in range(2):
  for i in range(128):
   k=(e,i);common=(lay["lo"][k]>>1)|((split["residual"][k]>>1)<<(lay["LB"]-1))|(tabs[k][0]<<((lay["LB"]-1)+(split["rw"]-1)))
   resttable.append(common|(split["coarse"][k]<<((lay["LB"]-1)+(split["rw"]-1)+cfg.w2)))
   fulltable.append(common|(lay["hi"][k]<<((lay["LB"]-1)+(split["rw"]-1)+cfg.w2))|(split["coarse"][k]<<((lay["LB"]-1)+(split["rw"]-1)+cfg.w2+lay["whi"])))
   hightable.append(lay["hi"][k])
 address=km.array(list(x[14:21])+[f[0]]);delta=x[:14]
 def phase(reg,lowbit):
  ext=b.alloc_qubits(10-len(borrowed));word=km.array(list(reg)+borrowed+list(ext))
  shift(b,f,word,len(reg),False,prune)
  for i,q in enumerate(word):
   exponent=lowbit+i-cfg.fy-10
   if exponent<0 and not(lowbit==cfg.gy and i<phase_cut):b.z_pow(q,Fraction(2)**(exponent+1))
  shift(b,f,word,len(reg),True,prune);b.free(ext)
 def inner(inverse):
  import virtual_low
  virtual_low.inner(b,add,c2,res,delta,cfg,inverse)
 def outer(derivative,inverse):
  import virtual_low
  virtual_low.outer(b,add,derivative,low,delta,cfg.sh1,inverse)
 b.init_lookup(table=fulltable,address=address,target=full)
 deferred_bits=[]
 if not merge:
  phase(high,lay["S"])
  if deferred:deferred_bits=D.demolish(b,high)
  else:b.del_lookup(table=hightable,address=address,target=high)
  b.free(high)
 inner(False);add(res[split["block"]-1:],coarse)
 import virtual_low
 derivative=km.array([c2[virtual_low.derivative_lsb(cfg)]]+list(res[:split['block']-1])+list(coarse))
 outer(derivative,False)
 if merge:
  add(low[lay["S"]:],high)
  phase(km.array(list(low[cfg.gy:lay["S"]])+list(high)),cfg.gy)
  add(low[lay["S"]:],high,sub=True)
 else:phase(low[cfg.gy-1:],cfg.gy)
 outer(derivative,True);add(res[split["block"]-1:],coarse,sub=True);inner(True)
 remaining=full if merge else rest
 if deferred:
  deferred_bits+=D.demolish(b,remaining);b.free(remaining)
  table=fulltable if merge else [hi|(rr<<lay['whi']) for hi,rr in zip(hightable,resttable)]
  D.repair(b,address,table,deferred_bits,anf=anf)
 else:
  b.del_lookup(table=fulltable if merge else resttable,address=address,target=remaining);b.free(remaining)
 if borrowed:b.x(x[21])
 if parity_input:
  b.cx(x[21],x[0]);b.right_rotate(x[:21],1,control=x[21]);b.x(x[21])
  for k,amount in reversed(H.NSTAGES[:-1]):
   for i in range(22-amount):b.cswap(f[k],x[i],x[i+amount])
   top=x[22-amount:22];m=b.alloc_bits(1)[0];b.hmr(f[k],m);b.push_condition(m)
   b.x(top);H.phase_all(b,list(top));b.x(top);b.pop_condition();b.free(m)
 else:H.unnormalize(b,x,f)
 b.free(flags)
 return b.finish_circuit(),{k:v for k,v in split.items() if k not in ["coarse","residual"]}

In [ ]:
import time

sys.path.insert(0, str(WORK / "src"))
import qcc_pipeline as P

P.setup(COMP)
sys.path.insert(0, str(WORK / "src" / "ports"))
import invsqrt_ports as IP
import port_pipeline as PP

PP.setup(COMP)
started = time.perf_counter()

print("[1/4] Building inverse-square-root circuit")
EV = WORK / "src" / "ports" / "evidence_inv"
IP.build_invsqrt(
    SUB,
    base="r3",
    virtual_bits=1,
    certificate_file=EV / "exact_proof.json",
    calibration_file=EV / "calibration_selected.json",
    tables_file=EV / "virtual_tables.json",
)

print("[2/4] Building DQI decoder")
P.build_dqi(SUB, WORK / "build")

print("[3/4] Building FeMoCo factorization and angle lookups")
GAUGED = WORK / "build" / "femoco_gauge_words.npz"
GAUGED.parent.mkdir(parents=True, exist_ok=True)
PP.generate_gauge_words(CHEM_DATA, GAUGED)
PP.build_dfthcsos(SUB, GAUGED, brot=10, J=5, gf2=True)

print("[4/4] Running all official full-shot verifiers")
RESULTS = P.verify_all(SUB, "notebook-verification", workers=WORKERS)
failed = {name: result for name, result in RESULTS.items() if result["status"] != "PASS"}
if failed:
    raise AssertionError(f"Official verification failed: {failed}")

(WORK / "verification.json").write_text(json.dumps(RESULTS, indent=1, default=str))
print(f"All verifiers passed in {(time.perf_counter() - started) / 60:.1f} minutes")


## 1. Inverse square root

The circuit applies $e^{2\pi i/\sqrt{x}}$ to a 22-bit input $x$. Binary shift stages of 16, 8, 4, 2 and 1 positions normalize $x$ to a mantissa $M \in [1, 2)$. Their flags record the shift; $e$ is its parity and $s$ half of it, so that

$$\frac{1}{\sqrt{x}} = 2^{\,s-10}\, y, \qquad y = \frac{2^{(e-1)/2}}{\sqrt{M}} \in (0.5, 1].$$

A 256-row lookup addressed by $e$ and seven mantissa bits loads integer coefficients $(K_0, K_1, C_2)$ of a quadratic fit. The 14 low mantissa bits form a centered offset whose digits are all $\pm 1$:

$$\delta = \sum_{i=0}^{13} (2d_i - 1)\, 2^{\,i-22}, \qquad R = K_1 - C_2\,\delta, \qquad y = K_0 - R\,\delta .$$

Because every digit is $\pm 1$, each Horner row is a plain addition or subtraction (an uncontrolled adder wrapped in CNOT gates) instead of a controlled multiplication. After the shift by $s$, every bit of the result is phased with one Z-power rotation, and all intermediate values are uncomputed.

Memory is kept low in five ways:
- the derivative register is split into a short residual and a few coarse bits;
- the constant high part of the result is phased early, and its register is erased later together with the lookup's phase repair;
- the last carry is folded into its target;
- the leading input bit, always one after normalization, serves as a clean carry wire;
- phase-register wires known to be zero are erased with temporary logical-AND gates.

The coefficients use short precisions and are fitted exactly on the reachable inputs. The curvature $C_2$ is shared by groups of adjacent lookup rows and stored as a short base-and-rate code; it is loaded only for the inner multiplications, erased by measurement, and both erasure masks are merged into the final phase repair. The lowest accumulator bit is kept virtual: with even coefficient anchors its value follows from the parity of adjacent rounded rows, and the carry is formed only when an addition needs it. Gates whose outcome is fixed on every legal input are replaced by exact Clifford rewrites, and the rotation angles are recalibrated over all 3,133,452 inputs to widen the synthesis margin. The circuit has 84 qubits and 44 non-Clifford rotations.

## 2. DQI decoder

The decoder receives 70 syndrome symbols over GF($2^{12}$) and a classical record $c$. It applies $(-1)^{\langle c, e\rangle}$ for the unique error $e$ of weight at most 35, then returns all workspace to zero. The error locator $\sigma$ solves the key equation

$$S(z)\,\sigma(z) \equiv \Omega(z) \pmod{z^{70}}.$$

It is found by a reversible Berlekamp–Massey recursion that needs the locator only up to a nonzero factor. The two branches of the update are

$$\sigma \leftarrow \sigma - \alpha\tau \quad\text{(retain)}, \qquad (\sigma, \tau) \leftarrow (\tau - \alpha^{-1}\sigma,\ \sigma) \quad\text{(replace)}.$$

The replace branch equals $-\alpha^{-1}(\sigma - \alpha\tau)$, so both branches share one multiplication path and no polynomial history is stored.

- The discrepancy is recovered from the constant coefficients of the locator.
- A history stack of one entry suffices except on about one input in ten thousand. Those inputs fail, and the verifier prices the failures through $R(\varepsilon_+)$.
- After the recursion, the syndrome register is multiplied in place by $\sigma$. The upper half of $S\sigma \bmod z^{70}$ is zero and serves as workspace.
- A sweep over all 4,095 positions tests $\sigma(\alpha^{-j}) = 0$ and applies the sign of the recovered symbol under the record bits $c_j$.

Field multiplication uses tower-field formulas. They are derived here for the competition modulus $\alpha^{12}+\alpha^6+\alpha^4+\alpha+1$; the published circuit uses a different field polynomial and a fixed record, so the sweep is rewritten to read $c_j$ at every position. Three steps lower the peak qubit count:
- two template words that are idle after the recursion are lent to the later stages;
- the evaluator bits of the sweep stream through one logical-AND bit;
- the final norm values are recomputed from $N(x^{-1}) = N(x)^{-1}$ instead of being stored.

The decoder has 1,789 qubits.

## 3. FeMoCo angle lookups

The lookup circuits stream the 75 Givens-rotation angles of every vector of a factorized FeMoCo Hamiltonian into word registers, $J$ angles per circuit. The score multiplies three quantities: the effective normalization $\lambda_\text{eff} = \sqrt{E_\text{gap}(2\Lambda - E_\text{gap})}$, the Toffoli count of the lookups plus the organizers' state-preparation costs, and the matching qubit count.

The factorization has shape $(R, B, C) = (65, 60, 30)$ and $\lambda_\text{eff} = 47.56$. It is a public refinement of the organizers' baseline; its 10-bit angle words and weights are read from the attached dataset. Before use, the weight columns are rotated by an orthogonal matrix that keeps the fitted tensor unchanged and lowers the normalization slightly. The angles are encoded on the organizers' log-depth Givens tree in batches of $J = 5$, which gives 15 circuits for the 3,976 addresses.

Each circuit is a banked lookup. Even addresses load their words directly. A second bank loads the even–odd difference, which the low address bit copies into the words. The bank is then erased by measurement-based unlookup. Only a linearly independent set of difference columns is stored in the bank, after a reversible change of output basis that is applied to both tables. Each circuit therefore enumerates half of the addresses and stays within the 128-qubit limit.

## 4. Results

All values come from the official verifiers at full shot counts in this run. The decoder's failure rate and the mean Toffoli count depend slightly on the verification seed.

In [ ]:
r = RESULTS
rows = [
    ('Inverse square root', r['invsqrt']['num_qubits'], f"{r['invsqrt']['toffoli_count']:,.0f}",
     f"C = {r['invsqrt']['toffoli_equivalent']:,.0f} ({r['invsqrt']['rotation_count']:.0f} rotations)", 'invsqrt'),
    ('DQI decoder', r['dqi']['num_qubits'], f"{r['dqi']['toffoli_count']:,.0f}",
     f"R = {r['dqi']['repetition_factor']:.3f} ({r['dqi']['value_errors'] + r['dqi']['phase_errors']} failed of {r['dqi']['verified_shots']:,})", 'dqi'),
    ('FeMoCo lookups', r['dfthcsos']['num_qubits'], f"{r['dfthcsos']['toffoli_count']:,.0f}",
     f"λ_eff = {r['dfthcsos']['lambda_eff']:.2f}", 'dfthcsos'),
]
table = pd.DataFrame([{'Task': name, 'Qubits': q, 'Toffoli': t, 'Other': other,
                       'Cost': f"{r[k]['score']:.4g}", 'Reference cost': f'{REFERENCE[k]:.4g}',
                       'Reference / cost': f"{REFERENCE[k] / r[k]['score']:.2f}"} for name, q, t, other, k in rows])
score = math.prod(REFERENCE[k] / r[k]['score'] for k in REFERENCE) ** (1 / 3)
display(table.style.hide(axis='index'))
print(f'Score (geometric mean of reference / cost): {score:.3f}')

## Sources

- Organizers: task definitions, verifiers, kickmix and the log-depth Givens tree ([competition](https://www.kaggle.com/competitions/google-verifiable-quantum-circuit-compilation), [arXiv:2610.23832](https://arxiv.org/abs/2610.23832)).
- Decoder: A. Gu, A. H. Kemeny, S. F. Yelin, M. D. Lukin, [arXiv:2610.22367](https://arxiv.org/abs/2610.22367), with their source [andigu/uncompute](https://github.com/andigu/uncompute), commit `64b5a13`, Apache 2.0. Added here: field-formula regeneration, the record-reading sweep, the template loan and the leaner schedule.
- Inverse square root: signed-digit construction by [Sumit Chongder](https://www.kaggle.com/code/sumitchongder/notebook2f196e646b); derivative split and early high-word phase by [Rudra](https://www.kaggle.com/code/rudrabhaskar/highest-public-score-direct-polynomial-phasing); folded carry, borrowed top bit and the signed-digit lab by [Artem1410](https://www.kaggle.com/code/anazemcev/quantum-circuits-signed-digit-memory); compact precisions and reachable-domain fitting by [kozykappa](https://www.kaggle.com/code/kozykappa/quantum-ai-compact-arithmetic-banked-qrom). Virtual low bits and on-demand carry by [zongaobian](https://www.kaggle.com/code/zongaobian/quantum-circuits-signed-digits-bm-psi-qrom) (reconstructed from the published description); exact-domain Clifford rewrites by [Artem1410](https://www.kaggle.com/code/anazemcev/quantum-circuits-source-exact-compilation); phase calibration by [lightningv08](https://www.kaggle.com/code/lightningv08/quantum-circuits-memory-algebra) and [Mizero](https://www.kaggle.com/code/mizeroluckygall/quantum-circuits-rotation-calibration-39-char). Added here: the deferred high-word erasure with joint phase repair, erasure of the zero phase wires, and the grouped curvature lookup.
- FeMoCo: factorization refined by k256.dev, [kozykappa](https://www.kaggle.com/code/kozykappa/quantum-circuits-sparse-sos-factors-and-circuit) and [Artem1410](https://www.kaggle.com/code/anazemcev/quantum-circuits-signed-digit-memory), with angle words and weights in the attached dataset; banked lookup by Sumit Chongder and [kozykappa](https://www.kaggle.com/code/kozykappa/quantum-ai-compact-arithmetic-banked-qrom); factor gauge and GF(2) compression of the bank by [Artem1410](https://www.kaggle.com/code/anazemcev/quantum-circuits-source-exact-compilation), also used by zongaobian.
- C. Gidney, [Halving the cost of quantum addition](https://quantum-journal.org/papers/q-2018-06-18-74/) (2018); G. H. Low et al., [PRX 15, 041016](https://journals.aps.org/prx/abstract/10.1103/pb2g-j9cw) (2025).

Code released under the Apache 2.0 licence.

In [ ]:
import hashlib
import zipfile

ZIP_PATH = P.write_zip(SUB, WORK / "submission.zip")
expected_members = {
    "invsqrt.kmb",
    "dqi.kmb",
    "dfthcsos/submission.npz",
    *(f"dfthcsos/angle_k_{index}.kmb" for index in range(15)),
}

with zipfile.ZipFile(ZIP_PATH) as archive:
    infos = archive.infolist()
    actual_members = {info.filename for info in infos}
    if len(infos) != len(actual_members):
        raise AssertionError("submission.zip contains duplicate member names")
    if actual_members != expected_members:
        raise AssertionError(
            "Unexpected submission manifest. "
            f"Missing: {sorted(expected_members - actual_members)}; "
            f"extra: {sorted(actual_members - expected_members)}"
        )
    empty_members = [info.filename for info in infos if info.file_size == 0]
    if empty_members:
        raise AssertionError(f"Empty submission members: {empty_members}")
    bad_member = archive.testzip()
    if bad_member is not None:
        raise AssertionError(f"CRC failure in {bad_member}")

zip_digest = hashlib.sha256(ZIP_PATH.read_bytes()).hexdigest()
print(f"Submission ready: {ZIP_PATH}")
print(f"Files           : {len(expected_members)}")
print(f"Archive size    : {ZIP_PATH.stat().st_size / 2**20:.2f} MiB")
print(f"SHA-256         : {zip_digest}")
ZIP_PATH
